In [1]:
import os
import glob
import pandas as pd

ROOT_DEPRESSED = r"D:\Downloads\Depressed"
LLAMA_SAVES_DIR = os.path.join(ROOT_DEPRESSED, "LlamaFactory", "saves")

patterns = [
    "**/generated_predictions.jsonl",
    "**/predictions.jsonl",
    "**/predict*.jsonl",
    "**/predict*.json",
    "**/*test*.jsonl",
    "**/*test*.json",
    "**/*prediction*.jsonl",
    "**/*prediction*.json",
]

found = []
for pat in patterns:
    files = glob.glob(os.path.join(LLAMA_SAVES_DIR, pat), recursive=True)
    for f in files:
        found.append(os.path.normpath(f))

found = sorted(set(found))

print(f"共找到 {len(found)} 个候选预测文件：")
for i, f in enumerate(found):
    print(f"[{i}] {f}")

pd.DataFrame({"candidate_file": found}).to_csv(
    os.path.join(ROOT_DEPRESSED, "llamafactory_prediction_candidates.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n已保存候选文件表到：")
print(os.path.join(ROOT_DEPRESSED, "llamafactory_prediction_candidates.csv"))

共找到 180 个候选预测文件：
[0] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-02-45-11\generated_predictions.jsonl
[1] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-02-45-11\predict_results.json
[2] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-08-43-30\generated_predictions.jsonl
[3] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-08-43-30\predict_results.json
[4] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-11-36-32\generated_predictions.jsonl
[5] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-11-36-32\predict_results.json
[6] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\eval_2026-03-07-06-02-36\generated_predictions.jsonl
[7] D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\eval_2026-03-07-06-02-36\predict_results.json
[8] D

In [ ]:
import os
import re
import json
import glob
import ast
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)


ROOT_DEPRESSED = r"D:\Downloads\Depressed"


DEPTWEET_DIR = os.path.join(ROOT_DEPRESSED, "Deptweet")
CLOSE_JSON_DIR = os.path.join(ROOT_DEPRESSED, "Close_LLM_Deptweet")
OPEN_JSON_DIR = os.path.join(ROOT_DEPRESSED, "Open_LLM_Deptweet")
LLAMA_SAVES_DIR = os.path.join(ROOT_DEPRESSED, "LlamaFactory", "saves")

DEPRESSION_EMO_TEST = os.path.join(ROOT_DEPRESSED, "DepressionEmo-main", "Dataset", "test.json")

EXPORT_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4")
PRED_EXPORT_DIR = os.path.join(EXPORT_DIR, "predictions")
LOG_DIR = os.path.join(EXPORT_DIR, "logs")

os.makedirs(EXPORT_DIR, exist_ok=True)
os.makedirs(PRED_EXPORT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


EMOTION_CODES = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
EMOTION_NAMES = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
CODE_TO_NAME = dict(zip(EMOTION_CODES, EMOTION_NAMES))

CHAR4 = {'甲': 0, '乙': 1, '丙': 2, '丁': 3}
CHAR2 = {'甲': 0, '乙': 1}

label_definition = {
    "binary": {
        "0": "non-depressed",
        "1": "depressed"
    },
    "fourclass": {
        "0": "non-depressed",
        "1": "mild",
        "2": "moderate",
        "3": "severe"
    },
    "eightlabel_order": EMOTION_NAMES,
    "class255_rule": "8-dimensional binary symptom vector mapped to decimal, excluding all-zero pattern"
}


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def vec_to_str(v):
    return "".join(str(int(x)) for x in v)

def str_to_vec(s):
    s = str(s).strip()
    if len(s) == 8 and set(s) <= set("01"):
        return [int(x) for x in s]
    return None

def vec_to_255(v):
    return int(vec_to_str(v), 2)

def dec255_to_vec(x):
    x = int(x)
    if x < 0 or x > 255:
        return None
    return [int(c) for c in format(x, "08b")]

def detect_id_col(df):
    for c in ["id", "Id", "ID", "sample_id"]:
        if c in df.columns:
            return c
    return None

def detect_text_col(df):
    for c in ["text", "tweet", "Text", "sentence", "content", "post"]:
        if c in df.columns:
            return c
    return None

def detect_label_col(df):
    for c in ["target", "label", "labels", "class", "y"]:
        if c in df.columns:
            return c
    return None

def find_files_recursive(root_dir, patterns):
    out = []
    for pat in patterns:
        out.extend(glob.glob(os.path.join(root_dir, pat), recursive=True))
    return sorted(set(out))

def infer_model_name_from_file(fp):
    base = os.path.basename(fp)
    base = base.replace("_Depression_BIN.json", "")
    base = base.replace("_Depression_EIGHT.json", "")
    base = base.replace("_Depression.json", "")
    base = base.replace("generated_predictions.jsonl", "")
    base = base.replace("predict_results.json", "")
    base = base.replace(".jsonl", "")
    base = base.replace(".json", "")
    return base or os.path.basename(os.path.dirname(fp))

def infer_model_name_from_jsonl_path(fp):
    norm = os.path.normpath(fp)
    parts = norm.split(os.sep)
    if "saves" in parts:
        i = parts.index("saves")
        if i + 1 < len(parts):
            return parts[i + 1]
    return os.path.basename(os.path.dirname(fp))

def infer_run_name_from_path(fp):
    return os.path.basename(os.path.dirname(fp))

def infer_api_run_name(fp):
    return os.path.splitext(os.path.basename(fp))[0]

def infer_setting_from_path(fp):
    path_low = fp.lower()

    if "native_eval" in path_low:
        return "native_base_eval"

    if "stage" in path_low or "to_" in path_low or "2to" in path_low or "4to" in path_low or "8to" in path_low or "255" in path_low:
        return "stage"

    if "mtl" in path_low or "multi" in path_low:
        return "mtl"

    if "eval_" in path_low:
        return "ft_eval"

    if "lora" in path_low:
        return "direct_ft"

    if "close_llm" in path_low or "open_llm" in path_low:
        return "api_direct"

    return "unknown"

def infer_task_hint_from_path(fp):
    path_low = fp.lower()

    if "class255" in path_low or "255class" in path_low or "255_cls" in path_low:
        return "class255"

    if "8label" in path_low or "eightlabel" in path_low or "eight_label" in path_low:
        return "eightlabel"

    if "fourclass" in path_low or "4class" in path_low:
        return "fourclass"

    if "binary" in path_low or "bin" in path_low:
        return "binary"

    # 常见阶段任务提示
    if "4to255" in path_low or "4_to_255" in path_low or "4-255" in path_low:
        return "class255"
    if "8to255" in path_low or "8_to_255" in path_low or "8-255" in path_low:
        return "class255"
    if "255to4" in path_low or "255_to_4" in path_low:
        return "fourclass"
    if "255to8" in path_low or "255_to_8" in path_low:
        return "eightlabel"

    return None

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x)).strip()

def to_binary_label(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    if s == "甲":
        return 0
    if s in {"乙", "丙", "丁"}:
        return 1

    s_low = s.lower()
    if "non-depressed" in s_low:
        return 0
    if "depressed" in s_low or "mild" in s_low or "moderate" in s_low or "severe" in s_low:
        return 1

    try:
        v = int(float(s))
        return 0 if v == 0 else 1
    except:
        return None

def parse_list_like(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []

    if isinstance(x, list):
        return x

    s = str(x).strip()
    if s == "":
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            return json.loads(s.replace("'", '"'))
        except Exception:
            try:
                return ast.literal_eval(s)
            except Exception:
                pass

    if "," in s or "，" in s or "、" in s:
        parts = re.split(r"[,，、]\s*", s)
        return [p.strip() for p in parts if p.strip()]

    return [s]

def collect_json_keys(obj, prefix=""):
    keys = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            keys.add(str(k))
            keys |= collect_json_keys(v, prefix + "." + str(k))
    elif isinstance(obj, list):
        for item in obj:
            keys |= collect_json_keys(item, prefix)
    return keys

def read_sidecar_predict_results_keys(jsonl_fp):
    sidecar = os.path.join(os.path.dirname(jsonl_fp), "predict_results.json")
    if not os.path.isfile(sidecar):
        return set()
    try:
        obj = safe_read_json(sidecar)
        return collect_json_keys(obj)
    except Exception:
        return set()


def parse_fourclass_predict(s):
    s = "" if s is None else str(s)

    m = re.findall(r'(?<!\d)([0-3])(?!\d)', s)
    if m:
        return int(m[0]), True

    for ch, val in CHAR4.items():
        if ch in s:
            return val, True

    s_low = s.lower()
    if "non-depressed" in s_low:
        return 0, True
    if "mild" in s_low:
        return 1, True
    if "moderate" in s_low:
        return 2, True
    if "severe" in s_low:
        return 3, True

    return None, False

def parse_binary_predict(s):
    s = "" if s is None else str(s)

    m = re.findall(r'(?<!\d)([0-1])(?!\d)', s)
    if m:
        return int(m[0]), True

    for ch, val in CHAR2.items():
        if ch in s:
            return val, True

    s_low = s.lower()
    if "non-depressed" in s_low:
        return 0, True
    if "depressed" in s_low:
        return 1, True

    return None, False

def parse_eight_predict_vec(s):
    s = "" if s is None else str(s)

    vec = str_to_vec(s)
    if vec is not None:
        return vec, True

    codes = {c for c in EMOTION_CODES if c in s}
    if len(codes) > 0:
        vec = [1 if c in codes else 0 for c in EMOTION_CODES]
        return vec, True

    s_low = s.lower()
    vec = [1 if name.lower() in s_low else 0 for name in EMOTION_NAMES]
    if sum(vec) > 0:
        return vec, True

    return [0] * 8, False

def parse_255_predict(s):
    s = "" if s is None else str(s).strip()

    # 直接整数
    m = re.findall(r'(?<!\d)([1-9]\d{0,2})(?!\d)', s)
    for t in m:
        x = int(t)
        if 1 <= x <= 255:
            return x, dec255_to_vec(x), True

    # 8位二进制
    vec = str_to_vec(s)
    if vec is not None:
        x = vec_to_255(vec)
        if x != 0:
            return x, vec, True

    # 中文代码 / 英文标签集合
    vec, ok = parse_eight_predict_vec(s)
    if ok:
        x = vec_to_255(vec)
        if x != 0:
            return x, vec, True

    return None, None, False


def compute_binary_metrics(y_true, y_pred):
    y_true2, y_pred2 = [], []

    for yt, yp in zip(y_true, y_pred):
        yt_bin = to_binary_label(yt)
        yp_bin = to_binary_label(yp)
        if yt_bin is None or yp_bin is None:
            continue
        y_true2.append(yt_bin)
        y_pred2.append(yp_bin)

    if len(y_true2) == 0:
        return {
            "accuracy": None,
            "precision": None,
            "recall": None,
            "f1": None,
        }

    return {
        "accuracy": accuracy_score(y_true2, y_pred2),
        "precision": precision_score(y_true2, y_pred2, pos_label=1, zero_division=0),
        "recall": recall_score(y_true2, y_pred2, pos_label=1, zero_division=0),
        "f1": f1_score(y_true2, y_pred2, pos_label=1, zero_division=0),
    }

def compute_multiclass_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

def compute_multilabel_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return {
        "exact_match": accuracy_score(y_true, y_pred),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    }

def compute_class255_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


dept_test_4 = pd.read_csv(os.path.join(DEPTWEET_DIR, "Deptweet_test.csv"))
dept_test_bin = pd.read_csv(os.path.join(DEPTWEET_DIR, "Deptweet_test_binary.csv"))
emo_test = pd.read_json(DEPRESSION_EMO_TEST, lines=True)

if "id" in emo_test.columns:
    emo_test["id"] = emo_test["id"].astype(str)

save_json(label_definition, os.path.join(EXPORT_DIR, "label_definition.json"))

summary_rows = []
report_rows = []


def get_true_vec_from_emo_row(item):
    """
    从 emo_test 的一行里尽最大可能恢复 8维真值向量
    返回长度为8的 0/1 list
    """

    # -------------------------------------------------
    # 1) 优先处理 label_id
    # -------------------------------------------------
    if "label_id" in emo_test.columns:
        val = item["label_id"]

        # 1a) 单个整数 class255 编码，例如 149
        try:
            if not isinstance(val, list):
                s = str(val).strip()
                if re.fullmatch(r"\d+", s):
                    x = int(s)
                    if 1 <= x <= 255:
                        vec = dec255_to_vec(x)
                        if vec is not None:
                            return vec
        except Exception:
            pass

        # 1b) 已经是 8维 one-hot list
        if isinstance(val, list) and len(val) == 8:
            try:
                return [int(x) for x in val]
            except Exception:
                pass

        parsed = parse_list_like(val)

        # 1c) 8维 one-hot
        if len(parsed) == 8 and all(str(x).strip() in {"0", "1", "0.0", "1.0"} for x in parsed):
            try:
                return [int(float(x)) for x in parsed]
            except Exception:
                pass

        # 1d) index list，例如 [0, 3, 5]
        try:
            idxs = [int(float(x)) for x in parsed]
            if len(idxs) > 0 and all(0 <= z <= 7 for z in idxs):
                vec = [0] * 8
                for z in idxs:
                    vec[z] = 1
                return vec
        except Exception:
            pass


    if "emotions" in emo_test.columns:
        parsed = parse_list_like(item["emotions"])

        # 2a) 英文标签名集合
        parsed_set = set(str(x).strip() for x in parsed)
        vec = [1 if name in parsed_set else 0 for name in EMOTION_NAMES]
        if sum(vec) > 0:
            return vec

        # 2b) 中文代码集合 甲乙丙...
        vec = [1 if code in parsed_set else 0 for code in EMOTION_CODES]
        if sum(vec) > 0:
            return vec

    # -------------------------------------------------
    # 3) label / target 列
    # -------------------------------------------------
    for col in ["label", "target"]:
        if col in emo_test.columns:
            parsed = parse_list_like(item[col])

            # 3a) 8维 0/1
            if len(parsed) == 8 and all(str(x).strip() in {"0", "1", "0.0", "1.0"} for x in parsed):
                try:
                    return [int(float(x)) for x in parsed]
                except Exception:
                    pass

            # 3b) index list
            try:
                idxs = [int(float(x)) for x in parsed]
                if len(idxs) > 0 and all(0 <= z <= 7 for z in idxs):
                    vec = [0] * 8
                    for z in idxs:
                        vec[z] = 1
                    return vec
            except Exception:
                pass

            # 3c) 英文标签名集合
            parsed_set = set(str(x).strip() for x in parsed)
            vec = [1 if name in parsed_set else 0 for name in EMOTION_NAMES]
            if sum(vec) > 0:
                return vec

            # 3d) 中文代码集合
            vec = [1 if code in parsed_set else 0 for code in EMOTION_CODES]
            if sum(vec) > 0:
                return vec

  
    row_text = " ".join([str(v) for v in item.values])


    if any(code in row_text for code in EMOTION_CODES):
        vec = [1 if code in row_text else 0 for code in EMOTION_CODES]
        if sum(vec) > 0:
            return vec

 
    row_low = row_text.lower()
    vec = [1 if name.lower() in row_low else 0 for name in EMOTION_NAMES]
    if sum(vec) > 0:
        return vec

  
    return [0] * 8

print("emo_test columns:", emo_test.columns.tolist())
print("\n前3条样本恢复的 true_vec:")
for i in range(min(3, len(emo_test))):
    item = emo_test.iloc[i]
    tv = get_true_vec_from_emo_row(item)
    print(i, item["id"] if "id" in emo_test.columns else i, tv, vec_to_255(tv))
print("\n查看前3条样本的原始 label_id / emotions：")
for i in range(min(3, len(emo_test))):
    item = emo_test.iloc[i]
    print(f"\n--- sample {i} ---")
    print("id:", item["id"] if "id" in emo_test.columns else i)
    print("label_id:", item["label_id"] if "label_id" in emo_test.columns else None)
    print("emotions:", item["emotions"] if "emotions" in emo_test.columns else None)
    tv = get_true_vec_from_emo_row(item)
    print("true_vec:", tv)
    print("true_255:", vec_to_255(tv))

def export_api_json_dir(json_dir):
    if not os.path.isdir(json_dir):
        return

    json_files = sorted(glob.glob(os.path.join(json_dir, "*.json")))
    print(f"\n扫描 API 结果目录: {json_dir} | 找到 {len(json_files)} 个 json")

    for fp in json_files:
        model_name = infer_model_name_from_file(fp)
        setting = infer_setting_from_path(fp)
        run_name = infer_api_run_name(fp)
        base = os.path.basename(fp)

        try:
            data = safe_read_json(fp)
        except Exception as e:
            report_rows.append({
                "file": fp, "run_name": run_name, "task": "unknown", "status": "read_failed", "error": str(e)
            })
            continue

        # ---------- 四分类 ----------
        if base.endswith("_Depression.json"):
            rows = []
            y_true, y_pred = [], []
            parse_ok_count = 0

            for item in data:
                pred_raw = item.get("predict", "")
                pred_label, ok = parse_fourclass_predict(pred_raw)
                parse_ok_count += int(ok)

                yt = int(item["label"])
                rows.append({
                    "sample_id": str(item.get("Id", item.get("id", ""))),
                    "text": item.get("text", ""),
                    "true_label": yt,
                    "pred_label": pred_label if pred_label is not None else "",
                    "raw_predict": pred_raw,
                    "parse_ok": ok,
                    "model_name": model_name,
                    "setting": setting,
                    "run_name": run_name,
                    "source_file": fp
                })
                if pred_label is not None:
                    y_true.append(yt)
                    y_pred.append(pred_label)

            out_csv = os.path.join(PRED_EXPORT_DIR, f"fourclass__{model_name}__{setting}__{run_name}.csv")
            pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

            if len(y_true) > 0:
                met = compute_multiclass_metrics(y_true, y_pred)
                summary_rows.append({
                    "task": "fourclass",
                    "model_name": model_name,
                    "setting": setting,
                    "run_name": run_name,
                    **met,
                    "notes": f"api_json parsed={len(y_true)}/{len(data)}"
                })

            report_rows.append({
                "file": fp,
                "run_name": run_name,
                "task": "fourclass",
                "status": "ok",
                "n_total": len(data),
                "n_parse_ok": parse_ok_count,
                "out_csv": out_csv
            })

        # ---------- 二分类 ----------
        elif base.endswith("_Depression_BIN.json"):
            rows = []
            y_true, y_pred = [], []
            parse_ok_count = 0

            for item in data:
                pred_raw = item.get("predict", "")
                pred_label, ok = parse_binary_predict(pred_raw)
                parse_ok_count += int(ok)

                yt = to_binary_label(item["label"])

                rows.append({
                    "sample_id": str(item.get("Id", item.get("id", ""))),
                    "text": item.get("text", ""),
                    "true_label": yt if yt is not None else "",
                    "pred_label": pred_label if pred_label is not None else "",
                    "raw_predict": pred_raw,
                    "parse_ok": ok,
                    "model_name": model_name,
                    "setting": setting,
                    "run_name": run_name,
                    "source_file": fp
                })

                if pred_label is not None and yt is not None:
                    y_true.append(yt)
                    y_pred.append(pred_label)

            out_csv = os.path.join(PRED_EXPORT_DIR, f"binary__{model_name}__{setting}__{run_name}.csv")
            pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

            if len(y_true) > 0:
                met = compute_binary_metrics(y_true, y_pred)
                summary_rows.append({
                    "task": "binary",
                    "model_name": model_name,
                    "setting": setting,
                    "run_name": run_name,
                    **met,
                    "notes": f"api_json parsed={len(y_true)}/{len(data)}"
                })

            report_rows.append({
                "file": fp,
                "run_name": run_name,
                "task": "binary",
                "status": "ok",
                "n_total": len(data),
                "n_parse_ok": parse_ok_count,
                "out_csv": out_csv
            })

        # ---------- 8维 ----------
        elif base.endswith("_Depression_EIGHT.json"):
            rows = []
            y_true, y_pred = [], []
            parse_ok_count = 0

            for item in data:
                pred_raw = item.get("predict", "")
                pred_vec, ok = parse_eight_predict_vec(pred_raw)
                parse_ok_count += int(ok)

                true_emotions = set(item["target"])
                true_vec = [1 if name in true_emotions else 0 for name in EMOTION_NAMES]

                row = {
                    "sample_id": str(item.get("Id", item.get("id", ""))),
                    "text": item.get("text", ""),
                    "true_vec": "b" + vec_to_str(true_vec),
                    "pred_vec": "b" + vec_to_str(pred_vec),
                    "true_255": vec_to_255(true_vec),
                    "pred_255": vec_to_255(pred_vec) if sum(pred_vec) > 0 else "",
                    "raw_predict": pred_raw,
                    "parse_ok": ok,
                    "model_name": model_name,
                    "setting": setting,
                    "run_name": run_name,
                    "source_file": fp
                }
                for i, name in enumerate(EMOTION_NAMES):
                    row[f"true_{name}"] = true_vec[i]
                    row[f"pred_{name}"] = pred_vec[i]
                rows.append(row)

                y_true.append(true_vec)
                y_pred.append(pred_vec)

            out_csv = os.path.join(PRED_EXPORT_DIR, f"eightlabel__{model_name}__{setting}__{run_name}.csv")
            pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

            met = compute_multilabel_metrics(y_true, y_pred)
            summary_rows.append({
                "task": "eightlabel",
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                **met,
                "notes": f"api_json parsed={parse_ok_count}/{len(data)}"
            })

            report_rows.append({
                "file": fp,
                "run_name": run_name,
                "task": "eightlabel",
                "status": "ok",
                "n_total": len(data),
                "n_parse_ok": parse_ok_count,
                "out_csv": out_csv
            })

for json_dir in [CLOSE_JSON_DIR, OPEN_JSON_DIR]:
    export_api_json_dir(json_dir)


def infer_jsonl_task(records, fp=None):
    n = len(records)
    preds = [str(r.get("predict", "")) for r in records[:50]]

    # 1) 路径硬提示优先
    task_hint = infer_task_hint_from_path(fp) if fp is not None else None
    if task_hint is not None:
        return task_hint

    # 2) 读同目录 predict_results.json 侧文件辅助判断
    sidecar_keys = read_sidecar_predict_results_keys(fp) if fp is not None else set()
    sidecar_keys_low = set(k.lower() for k in sidecar_keys)

    # 多标签典型指标
    if any(k in sidecar_keys_low for k in ["exact_match", "hamming_loss", "micro_f1"]):
        return "eightlabel"

    # 255类典型：样本数=emo_test，且有 accuracy / macro_f1，但没有多标签指标
    if n == len(emo_test):
        if ("accuracy" in sidecar_keys_low or "macro_f1" in sidecar_keys_low or "weighted_f1" in sidecar_keys_low) and not any(
            k in sidecar_keys_low for k in ["exact_match", "hamming_loss", "micro_f1"]
        ):
            # 再结合输出内容，如果像 255 就认 255
            hit255 = 0
            for p in preds:
                x, vec, ok = parse_255_predict(p)
                if ok and x is not None and x != 0:
                    hit255 += 1
            if hit255 >= max(3, len(preds) // 6):
                return "class255"

    # 3) 再靠输出内容判断 —— 先 255，后 8维
    if n == len(emo_test):
        hit255 = 0
        for p in preds:
            x, vec, ok = parse_255_predict(p)
            if ok and x is not None and x != 0:
                hit255 += 1
        if hit255 >= max(3, len(preds) // 5):
            return "class255"

        eight_like = 0
        for p in preds:
            if str_to_vec(str(p).strip()) is not None:
                eight_like += 1
                continue
            if any(code in p for code in EMOTION_CODES):
                eight_like += 1
                continue
        if eight_like >= max(3, len(preds) // 5):
            return "eightlabel"

    # 4) 再识别四/二分类
    parsed4 = [parse_fourclass_predict(p)[0] for p in preds]
    parsed2 = [parse_binary_predict(p)[0] for p in preds]

    n4 = sum(x is not None for x in parsed4)
    n2 = sum(x is not None for x in parsed2)

    uniq4 = sorted(set([x for x in parsed4 if x is not None]))
    uniq2 = sorted(set([x for x in parsed2 if x is not None]))

    if n == len(dept_test_4):
        # 若 sidecar 更像 multiclass
        if ("macro_f1" in sidecar_keys_low or "weighted_f1" in sidecar_keys_low) and len(uniq4) >= 3:
            return "fourclass"
        if len(uniq4) >= 3 and n4 >= max(3, len(preds) // 5):
            return "fourclass"
        if len(uniq2) <= 2 and n2 >= max(3, len(preds) // 5):
            return "binary"

    return "unknown"


def export_jsonl_prediction_file(fp):
    model_name = infer_model_name_from_jsonl_path(fp)
    setting = infer_setting_from_path(fp)
    run_name = infer_run_name_from_path(fp)
    task_hint = infer_task_hint_from_path(fp)

    try:
        records = safe_read_jsonl(fp)
    except Exception as e:
        report_rows.append({
            "file": fp,
            "run_name": run_name,
            "task_hint": task_hint,
            "task": "unknown",
            "status": "jsonl_read_failed",
            "error": str(e)
        })
        return

    task = infer_jsonl_task(records, fp=fp)

    # ---------- 四分类 ----------
    if task == "fourclass":
        rows = []
        y_true, y_pred = [], []

        label_col = detect_label_col(dept_test_4)
        id_col = detect_id_col(dept_test_4)
        text_col = detect_text_col(dept_test_4)

        n = min(len(dept_test_4), len(records))
        for i in range(n):
            pred_raw = records[i].get("predict", "")
            pred_label, ok = parse_fourclass_predict(pred_raw)
            yt = int(dept_test_4.iloc[i][label_col])

            rows.append({
                "sample_id": str(dept_test_4.iloc[i][id_col]),
                "text": dept_test_4.iloc[i][text_col],
                "true_label": yt,
                "pred_label": pred_label if pred_label is not None else "",
                "raw_predict": pred_raw,
                "parse_ok": ok,
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                "source_jsonl": fp
            })

            if pred_label is not None:
                y_true.append(yt)
                y_pred.append(pred_label)

        out_csv = os.path.join(PRED_EXPORT_DIR, f"fourclass__{model_name}__{setting}__{run_name}.csv")
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

        if len(y_true) > 0:
            met = compute_multiclass_metrics(y_true, y_pred)
            summary_rows.append({
                "task": "fourclass",
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                **met,
                "notes": f"jsonl parsed={len(y_true)}/{n}"
            })

        report_rows.append({
            "file": fp,
            "run_name": run_name,
            "task_hint": task_hint,
            "task": "fourclass",
            "status": "ok",
            "n_total": len(records),
            "n_used": n,
            "out_csv": out_csv
        })

    # ---------- 二分类 ----------
    elif task == "binary":
        rows = []
        y_true, y_pred = [], []

        label_col = detect_label_col(dept_test_bin)
        id_col = detect_id_col(dept_test_bin)
        text_col = detect_text_col(dept_test_bin)

        n = min(len(dept_test_bin), len(records))
        for i in range(n):
            pred_raw = records[i].get("predict", "")
            pred_label, ok = parse_binary_predict(pred_raw)
            yt = to_binary_label(dept_test_bin.iloc[i][label_col])

            rows.append({
                "sample_id": str(dept_test_bin.iloc[i][id_col]),
                "text": dept_test_bin.iloc[i][text_col],
                "true_label": yt if yt is not None else "",
                "pred_label": pred_label if pred_label is not None else "",
                "raw_predict": pred_raw,
                "parse_ok": ok,
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                "source_jsonl": fp
            })

            if pred_label is not None and yt is not None:
                y_true.append(yt)
                y_pred.append(pred_label)

        out_csv = os.path.join(PRED_EXPORT_DIR, f"binary__{model_name}__{setting}__{run_name}.csv")
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

        if len(y_true) > 0:
            met = compute_binary_metrics(y_true, y_pred)
            summary_rows.append({
                "task": "binary",
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                **met,
                "notes": f"jsonl parsed={len(y_true)}/{n}"
            })

        report_rows.append({
            "file": fp,
            "run_name": run_name,
            "task_hint": task_hint,
            "task": "binary",
            "status": "ok",
            "n_total": len(records),
            "n_used": n,
            "out_csv": out_csv
        })

    # ---------- 8维 ----------
    elif task == "eightlabel":
        rows = []
        y_true, y_pred = [], []

        n = min(len(emo_test), len(records))
        for i in range(n):
            pred_raw = records[i].get("predict", "")
            pred_vec, ok = parse_eight_predict_vec(pred_raw)

            item = emo_test.iloc[i]
            true_vec = get_true_vec_from_emo_row(item)

            row = {
                "sample_id": str(item["id"]),
                "text": item["text"] if "text" in emo_test.columns else (str(item["title"]) + " ### " + str(item["post"])),
                "true_vec": "b" + vec_to_str(true_vec),
                "pred_vec": "b" + vec_to_str(pred_vec),
                "true_255": vec_to_255(true_vec),
                "pred_255": vec_to_255(pred_vec) if sum(pred_vec) > 0 else "",
                "raw_predict": pred_raw,
                "parse_ok": ok,
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                "source_jsonl": fp
            }
            for j, name in enumerate(EMOTION_NAMES):
                row[f"true_{name}"] = true_vec[j]
                row[f"pred_{name}"] = pred_vec[j]
            rows.append(row)

            y_true.append(true_vec)
            y_pred.append(pred_vec)

        out_csv = os.path.join(PRED_EXPORT_DIR, f"eightlabel__{model_name}__{setting}__{run_name}.csv")
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

        met = compute_multilabel_metrics(y_true, y_pred)
        summary_rows.append({
            "task": "eightlabel",
            "model_name": model_name,
            "setting": setting,
            "run_name": run_name,
            "task_hint": task_hint,
            **met,
            "notes": f"jsonl parsed={n}/{n}"
        })

        report_rows.append({
            "file": fp,
            "run_name": run_name,
            "task_hint": task_hint,
            "task": "eightlabel",
            "status": "ok",
            "n_total": len(records),
            "n_used": n,
            "out_csv": out_csv
        })

    # ---------- 255类 ----------
    elif task == "class255":
        rows = []
        y_true, y_pred = [], []

        n = min(len(emo_test), len(records))
        for i in range(n):
            pred_raw = records[i].get("predict", "")
            pred_255, pred_vec, ok = parse_255_predict(pred_raw)

            item = emo_test.iloc[i]
            true_vec = get_true_vec_from_emo_row(item)
            true_255 = vec_to_255(true_vec)

            rows.append({
                "sample_id": str(item["id"]),
                "text": item["text"] if "text" in emo_test.columns else (str(item["title"]) + " ### " + str(item["post"])),
                "true_255": true_255,
                "pred_255": pred_255 if pred_255 is not None else "",
                "true_vec": "b" + vec_to_str(true_vec),
                "pred_vec": "b" + vec_to_str(pred_vec)  if pred_vec is not None else "",
                "raw_predict": pred_raw,
                "parse_ok": ok,
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                "source_jsonl": fp
            })

            if pred_255 is not None:
                y_true.append(true_255)
                y_pred.append(pred_255)

        out_csv = os.path.join(PRED_EXPORT_DIR, f"class255__{model_name}__{setting}__{run_name}.csv")
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

        if len(y_true) > 0:
            met = compute_class255_metrics(y_true, y_pred)
            summary_rows.append({
                "task": "class255",
                "model_name": model_name,
                "setting": setting,
                "run_name": run_name,
                "task_hint": task_hint,
                **met,
                "notes": f"class255 parsed={len(y_true)}/{n}"
            })

        report_rows.append({
            "file": fp,
            "run_name": run_name,
            "task_hint": task_hint,
            "task": "class255",
            "status": "ok",
            "n_total": len(records),
            "n_used": n,
            "out_csv": out_csv
        })

    else:
        report_rows.append({
            "file": fp,
            "run_name": run_name,
            "task_hint": task_hint,
            "task": "unknown",
            "status": "skipped_unknown_task"
        })

# 只扫描 generated_predictions.jsonl
all_jsonls = sorted(
    glob.glob(
        os.path.join(LLAMA_SAVES_DIR, "**", "generated_predictions.jsonl"),
        recursive=True
    )
)

print(f"\n在 LlamaFactory/saves 下共找到 {len(all_jsonls)} 个 generated_predictions.jsonl")
for fp in all_jsonls[:20]:
    print(fp)

for fp in all_jsonls:
    export_jsonl_prediction_file(fp)


meta_rows = []

def append_split_meta(df, split_name, dataset_name):
    id_col = detect_id_col(df)
    if id_col is None:
        return
    for _, row in df.iterrows():
        meta_rows.append({
            "sample_id": str(row[id_col]),
            "split": split_name,
            "source_dataset": dataset_name
        })

split_files = [
    ("train", os.path.join(DEPTWEET_DIR, "Deptweet_train.csv")),
    ("valid", os.path.join(DEPTWEET_DIR, "Deptweet_valid.csv")),
    ("test", os.path.join(DEPTWEET_DIR, "Deptweet_test.csv")),
    ("train_binary", os.path.join(DEPTWEET_DIR, "Deptweet_train_binary.csv")),
    ("valid_binary", os.path.join(DEPTWEET_DIR, "Deptweet_valid_binary.csv")),
    ("test_binary", os.path.join(DEPTWEET_DIR, "Deptweet_test_binary.csv")),
]

for split_name, fp in split_files:
    if os.path.isfile(fp):
        append_split_meta(pd.read_csv(fp), split_name, "Deptweet")

meta_df = pd.DataFrame(meta_rows).drop_duplicates() if len(meta_rows) > 0 else pd.DataFrame(columns=["sample_id", "split", "source_dataset"])

candidate_conf_files = find_files_recursive(ROOT_DEPRESSED, [
    "**/*deptweet*dataset*.xlsx",
    "**/*deptweet*dataset*.csv",
    "**/*confidence*.xlsx",
    "**/*confidence*.csv",
])

confidence_map = {}
for fp in candidate_conf_files:
    try:
        if fp.lower().endswith(".xlsx"):
            df = pd.read_excel(fp)
        else:
            df = pd.read_csv(fp)

        id_col = detect_id_col(df)
        conf_col = None
        for c in df.columns:
            if str(c).lower() == "confidence_score":
                conf_col = c
                break

        if id_col is not None and conf_col is not None:
            for _, row in df.iterrows():
                confidence_map[str(row[id_col])] = row[conf_col]
            print(f"已读取 confidence_score: {fp}")
            break
    except Exception:
        pass

if len(confidence_map) > 0:
    meta_df["confidence_score"] = meta_df["sample_id"].map(confidence_map)

candidate_disagree_files = find_files_recursive(ROOT_DEPRESSED, [
    "**/*discarded*.xlsx",
    "**/*discarded*.csv",
    "**/*disagreement*.xlsx",
    "**/*disagreement*.csv",
])

disagree_ids = set()
for fp in candidate_disagree_files:
    try:
        if fp.lower().endswith(".xlsx"):
            df = pd.read_excel(fp)
        else:
            df = pd.read_csv(fp)

        id_col = detect_id_col(df)
        if id_col is not None:
            disagree_ids.update(df[id_col].astype(str).tolist())
            print(f"已读取 disagreement/discarded: {fp}")
            break
    except Exception:
        pass

if len(disagree_ids) > 0:
    meta_df["is_disagreement_sample"] = meta_df["sample_id"].isin(disagree_ids).astype(int)

meta_df.to_csv(os.path.join(EXPORT_DIR, "sample_meta.csv"), index=False, encoding="utf-8-sig")


summary_df = pd.DataFrame(summary_rows)
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["task", "model_name", "setting", "run_name"]).reset_index(drop=True)
summary_df.to_csv(os.path.join(EXPORT_DIR, "result_summary.csv"), index=False, encoding="utf-8-sig")

report_df = pd.DataFrame(report_rows)
report_df.to_csv(os.path.join(LOG_DIR, "export_report.csv"), index=False, encoding="utf-8-sig")


candidate_rows = []
for fp in all_jsonls:
    candidate_rows.append({
        "generated_predictions_jsonl": fp,
        "model_name": infer_model_name_from_jsonl_path(fp),
        "run_name": infer_run_name_from_path(fp),
        "setting": infer_setting_from_path(fp),
        "task_hint": infer_task_hint_from_path(fp),
        "sidecar_keys": ";".join(sorted(read_sidecar_predict_results_keys(fp)))
    })

pd.DataFrame(candidate_rows).to_csv(
    os.path.join(EXPORT_DIR, "llamafactory_generated_predictions_list.csv"),
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 100)
print("导出完成")
print("EXPORT_DIR =", EXPORT_DIR)
print("标准化预测目录 =", PRED_EXPORT_DIR)
print("结果汇总 =", os.path.join(EXPORT_DIR, "result_summary.csv"))
print("样本元信息 =", os.path.join(EXPORT_DIR, "sample_meta.csv"))
print("日志 =", os.path.join(LOG_DIR, "export_report.csv"))
print("LlamaFactory 预测文件列表 =", os.path.join(EXPORT_DIR, "llamafactory_generated_predictions_list.csv"))
print("=" * 100)

emo_test columns: ['id', 'title', 'post', 'text', 'upvotes', 'date', 'emotions', 'label_id']

前3条样本恢复的 true_vec:
0 w83pst [0, 0, 0, 1, 0, 1, 0, 0] 20
1 f2234m [0, 0, 0, 1, 0, 1, 0, 0] 20
2 17d3zn2 [1, 0, 1, 1, 1, 1, 0, 1] 189

查看前3条样本的原始 label_id / emotions：

--- sample 0 ---
id: w83pst
label_id: 10100
emotions: ['hopelessness', 'sadness']
true_vec: [0, 0, 0, 1, 0, 1, 0, 0]
true_255: 20

--- sample 1 ---
id: f2234m
label_id: 10100
emotions: ['hopelessness', 'sadness']
true_vec: [0, 0, 0, 1, 0, 1, 0, 0]
true_255: 20

--- sample 2 ---
id: 17d3zn2
label_id: 10111101
emotions: ['worthlessness', 'hopelessness', 'emptiness', 'sadness', 'loneliness', 'anger']
true_vec: [1, 0, 1, 1, 1, 1, 0, 1]
true_255: 189

扫描 API 结果目录: D:\Downloads\Depressed\Close_LLM_Deptweet | 找到 36 个 json

在 LlamaFactory/saves 下共找到 90 个 generated_predictions.jsonl
D:\Downloads\Depressed\LlamaFactory\saves\Qwen3.5-0.8B-Base\lora\Native_eval_26-03-18-02-45-11\generated_predictions.jsonl
D:\Downloads\Depressed\LlamaFactory\

##analysis

In [ ]:
# -*- coding: utf-8 -*-
"""
Nature-grade staged coupling analysis for class255 target task
FINAL ROBUST VERSION

Inputs
------
1) final_stage_mapping/final_run_task_mapping.csv
2) export_bundle_v4/result_summary.csv
3) export_bundle_v4/predictions/*.csv

Outputs
-------
Tables:
- class255_stage_run_inventory.csv
- class255_stage_vs_direct_summary.csv
- class255_stage_type_aggregate.csv
- class255_gain_by_symptom_count.csv
- class255_symptom_delta_table.csv
- class255_confusion_reduction_table.csv
- class255_core_shift_table.csv
- class255_reference_centrality.csv

Figures:
- FigS1_class255_stage_performance_matrix
- FigS2_class255_gain_by_stage_type
- FigS3_class255_gain_by_symptom_count
- FigS4_class255_symptom_delta_F1
- FigS5_class255_core_shift_direct_vs_staged
- FigS6_class255_confusion_pair_reduction
- FigS7_class255_omission_commission_delta
- FigS8_class255_stage_size_scaling
"""

import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

warnings.filterwarnings("ignore")


ROOT_DEPRESSED = r"D:\Downloads\Depressed"

FINAL_MAP_CSV = os.path.join(ROOT_DEPRESSED, "final_stage_mapping", "final_run_task_mapping.csv")
SUMMARY_CSV = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "result_summary.csv")
PRED_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "predictions")

OUT_DIR = os.path.join(ROOT_DEPRESSED, "nature_stage_coupling_class255")
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
TXT_DIR = os.path.join(OUT_DIR, "texts")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

SHORT_NAMES = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#222222",
    "white": "#FFFFFF",
    "linegray": "#CFCBC5",
}


def set_pub_style():
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "font.size": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.linewidth": 0.9,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.unicode_minus": False,
    })

def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)
    ax.tick_params(axis="both", which="major", length=3.0, width=0.9, colors=PALETTE["black"])
    for lab in ax.get_xticklabels():
        lab.set_fontweight("bold")
    for lab in ax.get_yticklabels():
        lab.set_fontweight("bold")

def save_fig(fig, stem):
    png = os.path.join(FIG_DIR, f"{stem}.png")
    eps = os.path.join(FIG_DIR, f"{stem}.eps")
    fig.savefig(png, dpi=800, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(eps, format="eps", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    print("[Saved]", png)
    print("[Saved]", eps)

def finite_text(ax, x, y, s, **kwargs):
    if np.isfinite(x) and np.isfinite(y):
        ax.text(x, y, s, **kwargs)

set_pub_style()


def parse_model_size(model_name):
    m = re.search(r"Qwen3\.5-(0\.8|2|4|9)B-Base", str(model_name))
    return float(m.group(1)) if m else np.nan

def bvec_to_list(s):
    s = str(s).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) == 8 and set(s) <= set("01"):
        return [int(c) for c in s]
    return None

def dec255_to_vec(x):
    try:
        if pd.isna(x):
            return None
        x = int(float(x))
        if x < 0 or x > 255:
            return None
        return [int(c) for c in format(x, "08b")]
    except Exception:
        return None

def vec_to_dec255(vec):
    try:
        if vec is None or len(vec) != 8:
            return np.nan
        return int("".join(str(int(v)) for v in vec), 2)
    except Exception:
        return np.nan

def robust_true_vec(true_vec_raw, true_255_raw):
    vec = bvec_to_list(true_vec_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "true_vec"

    vec = dec255_to_vec(true_255_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "true_255"

    return [0, 0, 0, 0, 0, 0, 0, 0], 0, "fallback_zero"

def robust_pred_vec(pred_vec_raw, pred_255_raw):
    vec = bvec_to_list(pred_vec_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "pred_vec"

    vec = dec255_to_vec(pred_255_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "pred_255"

    return [0, 0, 0, 0, 0, 0, 0, 0], 0, "fallback_zero"

def vec_core_score(vec, core):
    if vec is None:
        return np.nan
    idx = [i for i, v in enumerate(vec) if v == 1]
    if len(idx) == 0:
        return np.nan
    return float(np.mean(core[idx]))

def normalize01(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.array([])
    lo, hi = np.min(x), np.max(x)
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)

def correlation_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return np.corrcoef(x[mask], y[mask])[0, 1]

def parse_prediction_filename(fp):
    """
    Example:
    class255__Qwen3.5-9B-Base__ft_eval__eval_2026-03-12-09-33-16_2.csv
    """
    base = os.path.basename(fp)
    stem = os.path.splitext(base)[0]
    parts = stem.split("__")
    if len(parts) < 4:
        return None
    task = parts[0]
    model_name = parts[1]
    phase = parts[2]
    run_name = "__".join(parts[3:])
    return {
        "pred_file": fp,
        "task": task,
        "model_name": model_name,
        "phase": phase,
        "run_name": run_name
    }

def coalesce_columns(df, target_name, candidates):
    existing = [c for c in candidates if c in df.columns]
    if len(existing) == 0:
        df[target_name] = np.nan
        return df

    s = None
    for c in existing:
        if s is None:
            s = df[c]
        else:
            s = s.combine_first(df[c])
    df[target_name] = s
    return df

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df


if not os.path.isfile(FINAL_MAP_CSV):
    raise FileNotFoundError(f"Missing: {FINAL_MAP_CSV}")
if not os.path.isfile(SUMMARY_CSV):
    raise FileNotFoundError(f"Missing: {SUMMARY_CSV}")
if not os.path.isdir(PRED_DIR):
    raise FileNotFoundError(f"Missing directory: {PRED_DIR}")

map_df = pd.read_csv(FINAL_MAP_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)

pred_rows = []
for fp in sorted(glob.glob(os.path.join(PRED_DIR, "*.csv"))):
    x = parse_prediction_filename(fp)
    if x is not None:
        pred_rows.append(x)
pred_idx = pd.DataFrame(pred_rows)

map_df["model_size_b"] = map_df["model_name"].apply(parse_model_size)

# ---------- normalize summary ----------
sum_keep = summary_df.copy()
if "task" in sum_keep.columns:
    sum_keep = sum_keep.rename(columns={"task": "task_summary"})
if "setting" in sum_keep.columns:
    sum_keep = sum_keep.rename(columns={"setting": "phase"})

# only class255 summary rows help here
if "task_summary" in sum_keep.columns:
    sum_keep = sum_keep[sum_keep["task_summary"] == "class255"].copy()

# ---------- normalize pred index ----------
if "task" in pred_idx.columns:
    pred_idx = pred_idx.rename(columns={"task": "task_pred"})
if "task_pred" in pred_idx.columns:
    pred_idx = pred_idx[pred_idx["task_pred"] == "class255"].copy()

# ---------- robust merge ----------
df = map_df.merge(
    sum_keep,
    on=["model_name", "run_name", "phase"],
    how="left",
    suffixes=("", "_sum")
)

df = df.merge(
    pred_idx,
    on=["model_name", "run_name", "phase"],
    how="left",
    suffixes=("", "_pred")
)

# ---------- coalesce duplicated columns ----------
df = coalesce_columns(df, "task_summary_final", ["task_summary", "task_summary_sum"])
df = coalesce_columns(df, "task_pred_final", ["task_pred", "task_pred_pred"])

df = coalesce_columns(df, "accuracy_final", ["accuracy", "accuracy_sum", "accuracy_x", "accuracy_y"])
df = coalesce_columns(df, "macro_f1_final", ["macro_f1", "macro_f1_sum", "macro_f1_x", "macro_f1_y"])
df = coalesce_columns(df, "micro_f1_final", ["micro_f1", "micro_f1_sum", "micro_f1_x", "micro_f1_y"])
df = coalesce_columns(df, "exact_match_final", ["exact_match", "exact_match_sum", "exact_match_x", "exact_match_y"])
df = coalesce_columns(df, "hamming_loss_final", ["hamming_loss", "hamming_loss_sum", "hamming_loss_x", "hamming_loss_y"])
df = coalesce_columns(df, "pred_file_final", ["pred_file", "pred_file_pred", "file", "file_pred"])

# ---------- final task ----------
if "final_task" not in df.columns:
    if "task_pred_final" in df.columns:
        df["final_task"] = df["task_pred_final"]
    elif "task_summary_final" in df.columns:
        df["final_task"] = df["task_summary_final"]
    else:
        df["final_task"] = "unknown"


target_df = df[df["final_task"] == "class255"].copy()

def classify_stage_type(row):
    phase = str(row.get("phase", ""))
    final_task = str(row.get("final_task", ""))
    final_stage_name = str(row.get("final_stage_name", ""))
    final_stage_role = str(row.get("final_stage_role", ""))

    if phase == "native_base_eval" and final_task == "class255":
        return "native_class255"

    if phase == "ft_eval" and final_task == "class255":
        if final_stage_name == "from2to255" and final_stage_role == "target_task_eval":
            return "from2to255"
        if final_stage_name == "from4to255" and final_stage_role == "target_task_eval":
            return "from4to255"
        if final_stage_name.strip() == "":
            return "direct_class255"

    return "other"

target_df["stage_type"] = target_df.apply(classify_stage_type, axis=1)
target_df = target_df[target_df["stage_type"].isin([
    "native_class255", "direct_class255", "from2to255", "from4to255"
])].copy()

# pred file sanity
target_df = target_df[target_df["pred_file_final"].notna()].copy()
target_df = target_df[target_df["pred_file_final"].apply(lambda x: os.path.isfile(str(x)))].copy()

if len(target_df) == 0:
    raise ValueError("No valid class255 target runs found after merge. Please inspect final_run_task_mapping.csv and predictions/.")

print("\n[DEBUG] target_df columns:")
print(sorted(target_df.columns.tolist()))

print("\n[DEBUG] target_df preview:")
debug_cols = [c for c in [
    "model_name", "run_name", "phase", "final_task",
    "stage_type", "final_stage_name", "final_stage_role",
    "accuracy_final", "macro_f1_final", "micro_f1_final",
    "exact_match_final", "hamming_loss_final", "pred_file_final"
] if c in target_df.columns]
print(target_df[debug_cols].head(30).to_string(index=False))

# inventory
inventory_cols = [
    "model_name", "model_size_b", "run_name", "phase", "stage_type",
    "final_stage_name", "final_stage_role",
    "accuracy_final", "macro_f1_final", "micro_f1_final",
    "exact_match_final", "hamming_loss_final",
    "pred_file_final"
]

for c in inventory_cols:
    if c not in target_df.columns:
        target_df[c] = np.nan

inventory = target_df[inventory_cols].copy()
inventory = inventory.rename(columns={
    "accuracy_final": "accuracy",
    "macro_f1_final": "macro_f1",
    "micro_f1_final": "micro_f1",
    "exact_match_final": "exact_match",
    "hamming_loss_final": "hamming_loss",
    "pred_file_final": "pred_file",
})
inventory = inventory.sort_values(["model_size_b", "stage_type", "run_name"]).reset_index(drop=True)
inventory.to_csv(os.path.join(TAB_DIR, "class255_stage_run_inventory.csv"), index=False, encoding="utf-8-sig")


best_class255_row = target_df.sort_values("macro_f1_final", ascending=False).iloc[0]
best_pred = pd.read_csv(best_class255_row["pred_file_final"], dtype={"sample_id": str})

best_pred["true_255"] = pd.to_numeric(best_pred["true_255"], errors="coerce")
best_pred["pred_255"] = pd.to_numeric(best_pred["pred_255"], errors="coerce")

tmp_true = best_pred.apply(
    lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)),
    axis=1
)
best_pred["true_vec_list"] = [x[0] for x in tmp_true]
best_pred["true_vec_valid"] = [x[1] for x in tmp_true]
best_pred["true_vec_source"] = [x[2] for x in tmp_true]

tmp_pred = best_pred.apply(
    lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)),
    axis=1
)
best_pred["pred_vec_list"] = [x[0] for x in tmp_pred]
best_pred["pred_vec_valid"] = [x[1] for x in tmp_pred]
best_pred["pred_vec_source"] = [x[2] for x in tmp_pred]

X_true_best = np.vstack(best_pred["true_vec_list"].tolist()).astype(int)
cooc_true = (X_true_best.T @ X_true_best) / X_true_best.shape[0]
cooc_off = cooc_true.copy()
np.fill_diagonal(cooc_off, 0.0)
strength = cooc_off.sum(axis=1)
core = normalize01(strength)

centrality_df = pd.DataFrame({
    "symptom": EMOTION_NAMES,
    "symptom_short": [SHORT_NAMES[x] for x in EMOTION_NAMES],
    "strength_centrality": strength,
    "strength_centrality_01": core,
    "prevalence_true": X_true_best.mean(axis=0)
})
centrality_df.to_csv(os.path.join(TAB_DIR, "class255_reference_centrality.csv"), index=False, encoding="utf-8-sig")


def compute_run_metrics(pred_file):
    d = pd.read_csv(pred_file, dtype={"sample_id": str})

    # ---------- true side ----------
    d["true_255"] = pd.to_numeric(d["true_255"], errors="coerce")
    tmp_true = d.apply(
        lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)),
        axis=1
    )
    d["true_vec_list"] = [x[0] for x in tmp_true]
    d["true_vec_valid"] = [x[1] for x in tmp_true]
    d["true_vec_source"] = [x[2] for x in tmp_true]

    # ---------- pred side ----------
    d["pred_255"] = pd.to_numeric(d["pred_255"], errors="coerce")
    tmp_pred = d.apply(
        lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)),
        axis=1
    )
    d["pred_vec_list"] = [x[0] for x in tmp_pred]
    d["pred_vec_valid"] = [x[1] for x in tmp_pred]
    d["pred_vec_source"] = [x[2] for x in tmp_pred]

    # filled decimal ids
    d["true_255_filled"] = d.apply(
        lambda r: r["true_255"] if pd.notna(r["true_255"]) else vec_to_dec255(r["true_vec_list"]),
        axis=1
    )
    d["pred_255_filled"] = d.apply(
        lambda r: r["pred_255"] if pd.notna(r["pred_255"]) else vec_to_dec255(r["pred_vec_list"]),
        axis=1
    )

    # correctness at decimal level
    d["correct"] = (d["true_255_filled"] == d["pred_255_filled"])

    # dense matrices
    X_true = np.vstack(d["true_vec_list"].tolist()).astype(int)
    X_pred = np.vstack(d["pred_vec_list"].tolist()).astype(int)

    d["k_true"] = X_true.sum(axis=1)
    d["k_pred"] = X_pred.sum(axis=1)

    d["true_core"] = [vec_core_score(v, core) for v in d["true_vec_list"]]
    d["pred_core"] = [vec_core_score(v, core) for v in d["pred_vec_list"]]
    d["core_shift"] = d["pred_core"] - d["true_core"]

    # by symptom count
    by_k = d.groupby("k_true").agg(
        n=("sample_id", "size"),
        exact_acc=("correct", "mean")
    ).reset_index()

    # symptom metrics
    symptom_rows = []
    for i, name in enumerate(EMOTION_NAMES):
        yt = X_true[:, i]
        yp = X_pred[:, i]
        symptom_rows.append({
            "symptom": name,
            "support": int(yt.sum()),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "f1": f1_score(yt, yp, zero_division=0),
            "true_prev": yt.mean(),
            "pred_prev": yp.mean(),
            "prev_bias": yp.mean() - yt.mean(),
            "omission_rate": ((yt == 1) & (yp == 0)).sum() / max(1, (yt == 1).sum()),
            "commission_rate": ((yt == 0) & (yp == 1)).sum() / max(1, (yt == 0).sum()),
        })
    symptom_df = pd.DataFrame(symptom_rows)

    # confusion pairs
    wrong = d[d["correct"] == False].copy()
    conf_pairs = (
        wrong.groupby(["true_255_filled", "pred_255_filled"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    summary = {
        "n": len(d),
        "accuracy_recomputed": d["correct"].mean(),
        "mean_core_shift": d["core_shift"].mean(),
        "median_core_shift": d["core_shift"].median(),
        "mean_k_shift": (d["k_pred"] - d["k_true"]).mean(),
        "pred_vec_valid_rate": d["pred_vec_valid"].mean(),
        "pred_vec_from_text_rate": (d["pred_vec_source"] == "pred_vec").mean(),
        "pred_vec_from_decimal_rate": (d["pred_vec_source"] == "pred_255").mean(),
        "pred_vec_fallback_zero_rate": (d["pred_vec_source"] == "fallback_zero").mean(),
    }

    return d, by_k, symptom_df, conf_pairs, summary


run_cache = {}
for _, row in target_df.iterrows():
    pred_file = row["pred_file_final"]
    run_cache[pred_file] = compute_run_metrics(pred_file)


baseline_map = {}
for size, g in target_df[target_df["stage_type"] == "direct_class255"].groupby("model_size_b"):
    g = g.sort_values("macro_f1_final", ascending=False).head(1)
    baseline_map[size] = g.iloc[0]["pred_file_final"]


run_metric_rows = []
symptom_delta_rows = []
gain_by_k_rows = []
confusion_pair_rows = []
core_shift_rows = []

for _, row in target_df.iterrows():
    pred_file = row["pred_file_final"]
    size = row["model_size_b"]
    stage_type = row["stage_type"]

    d, by_k, symptom_df, conf_pairs, run_summary = run_cache[pred_file]

    base_file = baseline_map.get(size, "")
    base_exists = isinstance(base_file, str) and (base_file in run_cache) and (stage_type != "direct_class255")

    row_out = {
        "model_name": row["model_name"],
        "model_size_b": size,
        "run_name": row["run_name"],
        "stage_type": stage_type,
        "accuracy": row.get("accuracy_final", np.nan),
        "macro_f1": row.get("macro_f1_final", np.nan),
        "micro_f1": row.get("micro_f1_final", np.nan),
        "exact_match": row.get("exact_match_final", np.nan),
        "hamming_loss": row.get("hamming_loss_final", np.nan),
        "mean_core_shift": run_summary["mean_core_shift"],
        "median_core_shift": run_summary["median_core_shift"],
        "mean_k_shift": run_summary["mean_k_shift"],
        "pred_vec_valid_rate": run_summary["pred_vec_valid_rate"],
        "pred_vec_from_text_rate": run_summary["pred_vec_from_text_rate"],
        "pred_vec_from_decimal_rate": run_summary["pred_vec_from_decimal_rate"],
        "pred_vec_fallback_zero_rate": run_summary["pred_vec_fallback_zero_rate"],
    }

    core_shift_rows.append({
        "model_name": row["model_name"],
        "model_size_b": size,
        "run_name": row["run_name"],
        "stage_type": stage_type,
        "mean_core_shift": run_summary["mean_core_shift"],
        "median_core_shift": run_summary["median_core_shift"],
        "mean_k_shift": run_summary["mean_k_shift"],
        "pred_vec_valid_rate": run_summary["pred_vec_valid_rate"],
        "pred_vec_from_text_rate": run_summary["pred_vec_from_text_rate"],
        "pred_vec_from_decimal_rate": run_summary["pred_vec_from_decimal_rate"],
        "pred_vec_fallback_zero_rate": run_summary["pred_vec_fallback_zero_rate"],
    })

    if base_exists:
        base_d, base_by_k, base_symptom_df, base_conf_pairs, base_summary = run_cache[base_file]
        base_row = target_df[target_df["pred_file_final"] == base_file].iloc[0]

        row_out["delta_macro_f1_vs_direct"] = row.get("macro_f1_final", np.nan) - base_row.get("macro_f1_final", np.nan)
        row_out["delta_accuracy_vs_direct"] = row.get("accuracy_final", np.nan) - base_row.get("accuracy_final", np.nan)
        row_out["delta_mean_core_shift_vs_direct"] = run_summary["mean_core_shift"] - base_summary["mean_core_shift"]
        row_out["delta_mean_k_shift_vs_direct"] = run_summary["mean_k_shift"] - base_summary["mean_k_shift"]
    else:
        row_out["delta_macro_f1_vs_direct"] = np.nan
        row_out["delta_accuracy_vs_direct"] = np.nan
        row_out["delta_mean_core_shift_vs_direct"] = np.nan
        row_out["delta_mean_k_shift_vs_direct"] = np.nan

    run_metric_rows.append(row_out)

    # symptom deltas
    if base_exists:
        tmp = symptom_df.merge(
            base_symptom_df[["symptom", "f1", "precision", "recall", "prev_bias", "omission_rate", "commission_rate"]],
            on="symptom", how="left", suffixes=("", "_base")
        )
        for _, rr in tmp.iterrows():
            symptom_delta_rows.append({
                "model_name": row["model_name"],
                "model_size_b": size,
                "run_name": row["run_name"],
                "stage_type": stage_type,
                "symptom": rr["symptom"],
                "delta_f1_vs_direct": rr["f1"] - rr["f1_base"],
                "delta_precision_vs_direct": rr["precision"] - rr["precision_base"],
                "delta_recall_vs_direct": rr["recall"] - rr["recall_base"],
                "delta_prev_bias_vs_direct": rr["prev_bias"] - rr["prev_bias_base"],
                "delta_omission_vs_direct": rr["omission_rate"] - rr["omission_rate_base"],
                "delta_commission_vs_direct": rr["commission_rate"] - rr["commission_rate_base"],
            })

        # gain by symptom count
        tmpk = by_k.merge(
            base_by_k[["k_true", "exact_acc"]],
            on="k_true", how="outer", suffixes=("", "_base")
        ).sort_values("k_true")
        for _, rr in tmpk.iterrows():
            gain_by_k_rows.append({
                "model_name": row["model_name"],
                "model_size_b": size,
                "run_name": row["run_name"],
                "stage_type": stage_type,
                "k_true": rr["k_true"],
                "stage_exact_acc": rr["exact_acc"],
                "direct_exact_acc": rr["exact_acc_base"],
                "delta_exact_acc_vs_direct": rr["exact_acc"] - rr["exact_acc_base"]
            })

        # confusion reduction
        base_cp = base_conf_pairs.copy()
        stage_cp = conf_pairs.copy()
        cp = base_cp.merge(
            stage_cp,
            on=["true_255_filled", "pred_255_filled"],
            how="outer",
            suffixes=("_base", "_stage")
        ).fillna(0)
        cp["delta_count"] = cp["count_stage"] - cp["count_base"]
        cp["reduction"] = cp["count_base"] - cp["count_stage"]
        cp = cp.sort_values("reduction", ascending=False).reset_index(drop=True)

        for _, rr in cp.head(50).iterrows():
            confusion_pair_rows.append({
                "model_name": row["model_name"],
                "model_size_b": size,
                "run_name": row["run_name"],
                "stage_type": stage_type,
                "true_255": rr["true_255_filled"],
                "pred_255": rr["pred_255_filled"],
                "base_count": rr["count_base"],
                "stage_count": rr["count_stage"],
                "reduction": rr["reduction"]
            })


run_metric_df = pd.DataFrame(run_metric_rows)
symptom_delta_df = pd.DataFrame(symptom_delta_rows)
gain_by_k_df = pd.DataFrame(gain_by_k_rows)
confusion_delta_df = pd.DataFrame(confusion_pair_rows)
core_shift_df = pd.DataFrame(core_shift_rows)

run_metric_df = ensure_cols(run_metric_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "accuracy", "macro_f1", "micro_f1", "exact_match", "hamming_loss",
    "mean_core_shift", "median_core_shift", "mean_k_shift",
    "pred_vec_valid_rate", "pred_vec_from_text_rate",
    "pred_vec_from_decimal_rate", "pred_vec_fallback_zero_rate",
    "delta_macro_f1_vs_direct", "delta_accuracy_vs_direct",
    "delta_mean_core_shift_vs_direct", "delta_mean_k_shift_vs_direct"
])

symptom_delta_df = ensure_cols(symptom_delta_df, [
    "model_name", "model_size_b", "run_name", "stage_type", "symptom",
    "delta_f1_vs_direct", "delta_precision_vs_direct", "delta_recall_vs_direct",
    "delta_prev_bias_vs_direct", "delta_omission_vs_direct",
    "delta_commission_vs_direct"
])

gain_by_k_df = ensure_cols(gain_by_k_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "k_true", "stage_exact_acc", "direct_exact_acc", "delta_exact_acc_vs_direct"
])

confusion_delta_df = ensure_cols(confusion_delta_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "true_255", "pred_255", "base_count", "stage_count", "reduction"
])

core_shift_df = ensure_cols(core_shift_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "mean_core_shift", "median_core_shift", "mean_k_shift",
    "pred_vec_valid_rate", "pred_vec_from_text_rate",
    "pred_vec_from_decimal_rate", "pred_vec_fallback_zero_rate"
])

run_metric_df.to_csv(os.path.join(TAB_DIR, "class255_stage_vs_direct_summary.csv"), index=False, encoding="utf-8-sig")
symptom_delta_df.to_csv(os.path.join(TAB_DIR, "class255_symptom_delta_table.csv"), index=False, encoding="utf-8-sig")
gain_by_k_df.to_csv(os.path.join(TAB_DIR, "class255_gain_by_symptom_count.csv"), index=False, encoding="utf-8-sig")
confusion_delta_df.to_csv(os.path.join(TAB_DIR, "class255_confusion_reduction_table.csv"), index=False, encoding="utf-8-sig")
core_shift_df.to_csv(os.path.join(TAB_DIR, "class255_core_shift_table.csv"), index=False, encoding="utf-8-sig")


agg_stage = run_metric_df.groupby("stage_type").agg(
    n_runs=("run_name", "size"),
    mean_macro_f1=("macro_f1", "mean"),
    mean_accuracy=("accuracy", "mean"),
    mean_delta_macro_f1_vs_direct=("delta_macro_f1_vs_direct", "mean"),
    mean_delta_accuracy_vs_direct=("delta_accuracy_vs_direct", "mean"),
    mean_core_shift=("mean_core_shift", "mean"),
    mean_delta_core_shift_vs_direct=("delta_mean_core_shift_vs_direct", "mean"),
    mean_pred_vec_valid_rate=("pred_vec_valid_rate", "mean"),
    mean_pred_vec_fallback_zero_rate=("pred_vec_fallback_zero_rate", "mean"),
).reset_index()

agg_stage.to_csv(os.path.join(TAB_DIR, "class255_stage_type_aggregate.csv"), index=False, encoding="utf-8-sig")


# FigS1: performance matrix
fig, ax = plt.subplots(figsize=(4.8, 3.6))
plot_df = run_metric_df.copy()
order = ["native_class255", "direct_class255", "from2to255", "from4to255"]
plot_df["stage_type"] = pd.Categorical(plot_df["stage_type"], categories=order, ordered=True)
plot_df = plot_df.sort_values(["stage_type", "model_size_b"])

x_labels = []
mat = []
for st in order:
    dd = plot_df[plot_df["stage_type"] == st].sort_values("model_size_b")
    x_labels.append(st.replace("_", "\n"))
    if len(dd) > 0:
        mat.append([dd["macro_f1"].mean(), dd["accuracy"].mean()])
    else:
        mat.append([np.nan, np.nan])
mat = np.array(mat).T

valid_vals = mat[np.isfinite(mat)]
vmin = np.nanmin(valid_vals) if len(valid_vals) > 0 else 0
vmax = np.nanmax(valid_vals) if len(valid_vals) > 0 else 1

cmap = LinearSegmentedColormap.from_list(
    "stageperf",
    [PALETTE["white"], "#D9EEF8", PALETTE["blue"]]
)
ax.imshow(mat, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
ax.set_xticks(np.arange(len(order)), labels=x_labels)
ax.set_yticks([0, 1], labels=["Macro-F1", "Accuracy"])

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if np.isfinite(mat[i, j]):
            finite_text(ax, j, i, f"{mat[i,j]:.3f}", ha="center", va="center",
                        fontsize=6.5, fontweight="bold")

style_ax(ax)
fig.tight_layout()
save_fig(fig, "FigS1_class255_stage_performance_matrix")

# FigS2: gain by stage type
fig, ax = plt.subplots(figsize=(4.8, 3.4))
plot_df = run_metric_df[run_metric_df["stage_type"].isin(["from2to255", "from4to255"])].copy()
agg = plot_df.groupby("stage_type").agg(
    mean_gain=("delta_macro_f1_vs_direct", "mean"),
    n=("run_name", "size")
).reset_index()

if len(agg) > 0:
    x = np.arange(len(agg))
    colors = [PALETTE["orange"], PALETTE["pink"]][:len(agg)]
    ax.bar(x, agg["mean_gain"], color=colors, edgecolor="none", width=0.62)
    for i, row in agg.iterrows():
        finite_text(ax, i, row["mean_gain"] + 0.01, f"{row['mean_gain']:.3f}\n(n={int(row['n'])})",
                    ha="center", va="bottom", fontsize=6.4, fontweight="bold")
    ax.axhline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xticks(x, labels=agg["stage_type"].tolist())
    ax.set_ylabel("Δ Macro-F1 vs direct class255", fontweight="bold")
    style_ax(ax)
    fig.tight_layout()
    save_fig(fig, "FigS2_class255_gain_by_stage_type")
else:
    plt.close(fig)

# FigS3: gain by symptom count
fig, ax = plt.subplots(figsize=(5.0, 3.5))
drawn = False
if len(gain_by_k_df) > 0 and "stage_type" in gain_by_k_df.columns:
    for st, color in [("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
        dd = gain_by_k_df[gain_by_k_df["stage_type"] == st].copy()
        if len(dd) == 0:
            continue
        aggk = dd.groupby("k_true").agg(mean_gain=("delta_exact_acc_vs_direct", "mean")).reset_index()
        if len(aggk) == 0:
            continue
        ax.plot(aggk["k_true"], aggk["mean_gain"], marker="o", markersize=4.5, linewidth=1.5, color=color)
        last = aggk.iloc[-1]
        finite_text(ax, last["k_true"] + 0.1, last["mean_gain"], st, fontsize=6.8, fontweight="bold", color=color, va="center")
        drawn = True

if drawn:
    ax.axhline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xlabel("No. of symptoms in true combination", fontweight="bold")
    ax.set_ylabel("Δ exact accuracy vs direct", fontweight="bold")
    style_ax(ax)
    fig.tight_layout()
    save_fig(fig, "FigS3_class255_gain_by_symptom_count")
else:
    plt.close(fig)

# FigS4: symptom delta F1
fig, ax = plt.subplots(figsize=(5.4, 3.6))
if len(symptom_delta_df) > 0 and "stage_type" in symptom_delta_df.columns:
    agg_sym = symptom_delta_df.groupby(["stage_type", "symptom"]).agg(mean_delta_f1=("delta_f1_vs_direct", "mean")).reset_index()
    sym_order = EMOTION_NAMES
    y = np.arange(len(sym_order))

    has_points = False
    for st, color, offset in [("from2to255", PALETTE["orange"], -0.12), ("from4to255", PALETTE["pink"], 0.12)]:
        dd = agg_sym[agg_sym["stage_type"] == st].copy()
        dd["symptom"] = pd.Categorical(dd["symptom"], categories=sym_order, ordered=True)
        dd = dd.sort_values("symptom")
        if len(dd) == 0:
            continue
        ax.scatter(dd["mean_delta_f1"], y + offset, s=55, color=color, zorder=3)
        for xv, yv in zip(dd["mean_delta_f1"], y + offset):
            ax.plot([0, xv], [yv, yv], color=PALETTE["linegray"], linewidth=0.8, zorder=1)
        has_points = True

    if has_points:
        ax.axvline(0, color=PALETTE["black"], linewidth=1.0)
        ax.set_yticks(y, labels=[SHORT_NAMES[s] for s in sym_order])
        ax.set_xlabel("Δ symptom F1 vs direct", fontweight="bold")
        ax.text(
            0.98, 0.95,
            "orange=from2to255\npink=from4to255",
            transform=ax.transAxes,
            ha="right", va="top",
            fontsize=6.6, fontweight="bold"
        )
        style_ax(ax)
        fig.tight_layout()
        save_fig(fig, "FigS4_class255_symptom_delta_F1")
    else:
        plt.close(fig)
else:
    plt.close(fig)

# FigS5: core shift direct vs staged
fig, ax = plt.subplots(figsize=(4.8, 3.5))
box_data = []
box_labels = []
box_colors = []
for st, c in [("direct_class255", PALETTE["gray"]), ("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
    vals = run_metric_df[run_metric_df["stage_type"] == st]["mean_core_shift"].dropna().to_list()
    if len(vals) > 0:
        box_data.append(vals)
        box_labels.append(st.replace("_", "\n"))
        box_colors.append(c)

if len(box_data) > 0:
    bp = ax.boxplot(
        box_data, widths=0.55, patch_artist=True, showfliers=False,
        medianprops=dict(color=PALETTE["black"], linewidth=1.0),
        whiskerprops=dict(color=PALETTE["black"], linewidth=0.8),
        capprops=dict(color=PALETTE["black"], linewidth=0.8),
        boxprops=dict(color=PALETTE["black"], linewidth=0.8)
    )
    for patch, c in zip(bp["boxes"], box_colors):
        patch.set_facecolor(c)
    ax.axhline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xticks(np.arange(1, len(box_labels) + 1), labels=box_labels)
    ax.set_ylabel("Mean core-shift", fontweight="bold")
    style_ax(ax)
    fig.tight_layout()
    save_fig(fig, "FigS5_class255_core_shift_direct_vs_staged")
else:
    plt.close(fig)

# FigS6: confusion pair reduction
fig, ax = plt.subplots(figsize=(5.0, 3.6))
if len(confusion_delta_df) > 0 and "stage_type" in confusion_delta_df.columns:
    agg_conf = confusion_delta_df.groupby(["stage_type", "true_255", "pred_255"]).agg(mean_reduction=("reduction", "mean")).reset_index()
    top = agg_conf.sort_values("mean_reduction", ascending=False).head(10).copy()
    if len(top) > 0:
        top["pair"] = top["true_255"].astype(int).astype(str) + "→" + top["pred_255"].astype(int).astype(str)
        top = top.sort_values("mean_reduction", ascending=True).reset_index(drop=True)
        y = np.arange(len(top))
        colors = [PALETTE["orange"] if st == "from2to255" else PALETTE["pink"] for st in top["stage_type"]]
        ax.barh(y, top["mean_reduction"], color=colors, edgecolor="none", height=0.66)
        ax.set_yticks(y, labels=top["pair"].tolist())
        ax.set_xlabel("Mean reduction in confusion count", fontweight="bold")
        style_ax(ax)
        fig.tight_layout()
        save_fig(fig, "FigS6_class255_confusion_pair_reduction")
    else:
        plt.close(fig)
else:
    plt.close(fig)

# FigS7: omission/commission delta
fig, ax = plt.subplots(figsize=(5.2, 3.6))
if len(symptom_delta_df) > 0 and "stage_type" in symptom_delta_df.columns:
    agg_oc = symptom_delta_df.groupby(["stage_type", "symptom"]).agg(
        delta_omission=("delta_omission_vs_direct", "mean"),
        delta_commission=("delta_commission_vs_direct", "mean")
    ).reset_index()

    show_stage = ["from2to255", "from4to255"]
    mats = []
    stage_headers = []
    for st in show_stage:
        dd = agg_oc[agg_oc["stage_type"] == st].copy()
        dd["symptom"] = pd.Categorical(dd["symptom"], categories=EMOTION_NAMES, ordered=True)
        dd = dd.sort_values("symptom")
        if len(dd) == 0:
            continue
        mat = np.vstack([dd["delta_omission"].to_numpy(), dd["delta_commission"].to_numpy()])
        mats.append(mat)
        stage_headers.append(st)

    if len(mats) > 0:
        big = np.hstack(mats)
        valid_vals = big[np.isfinite(big)]
        vmax = np.nanmax(np.abs(valid_vals)) if len(valid_vals) > 0 else 1.0
        cmap = LinearSegmentedColormap.from_list(
            "deltas",
            ["#F7F7F7", "#FDE1CC", "#F39F4E"]
        )
        ax.imshow(big, cmap=cmap, aspect="auto", vmin=-vmax, vmax=vmax)

        xticklabels = []
        for _ in stage_headers:
            xticklabels += [SHORT_NAMES[s] for s in EMOTION_NAMES]

        ax.set_xticks(np.arange(big.shape[1]), labels=xticklabels)
        for lab in ax.get_xticklabels():
            lab.set_rotation(35)
            lab.set_ha("right")
        ax.set_yticks([0, 1], labels=["Δ omission", "Δ commission"])

        if len(stage_headers) == 2:
            finite_text(ax, 3.5, -0.7, stage_headers[0], ha="center", va="bottom", fontsize=6.8, fontweight="bold", color=PALETTE["orange"])
            finite_text(ax, 11.5, -0.7, stage_headers[1], ha="center", va="bottom", fontsize=6.8, fontweight="bold", color=PALETTE["pink"])
        elif len(stage_headers) == 1:
            finite_text(ax, 3.5, -0.7, stage_headers[0], ha="center", va="bottom", fontsize=6.8, fontweight="bold", color=PALETTE["orange"])

        style_ax(ax)
        fig.tight_layout()
        save_fig(fig, "FigS7_class255_omission_commission_delta")
    else:
        plt.close(fig)
else:
    plt.close(fig)

# FigS8: stage size scaling
fig, ax = plt.subplots(figsize=(4.8, 3.4))
drawn = False
if len(run_metric_df) > 0 and "stage_type" in run_metric_df.columns:
    for st, c in [("direct_class255", PALETTE["gray"]), ("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
        dd = run_metric_df[run_metric_df["stage_type"] == st].copy()
        if len(dd) == 0:
            continue
        agg_sz = dd.groupby("model_size_b").agg(mean_macro_f1=("macro_f1", "mean")).reset_index()
        if len(agg_sz) == 0:
            continue
        ax.plot(agg_sz["model_size_b"], agg_sz["mean_macro_f1"], marker="o", markersize=5, linewidth=1.5, color=c)
        last = agg_sz.iloc[-1]
        finite_text(ax, last["model_size_b"] + 0.1, last["mean_macro_f1"], st.replace("_", " "),
                    fontsize=6.8, fontweight="bold", color=c, va="center")
        drawn = True

if drawn:
    ax.set_xlabel("Model size (B)", fontweight="bold")
    ax.set_ylabel("Macro-F1", fontweight="bold")
    ax.set_xticks(sorted(run_metric_df["model_size_b"].dropna().unique()))
    style_ax(ax)
    fig.tight_layout()
    save_fig(fig, "FigS8_class255_stage_size_scaling")
else:
    plt.close(fig)

lines = []
lines.append("Class255 staged coupling summary")
lines.append("")
lines.append("Inventory:")
lines.append(inventory.to_string(index=False))
lines.append("")

lines.append("Stage aggregate:")
lines.append(agg_stage.to_string(index=False))
lines.append("")

if len(symptom_delta_df) > 0:
    lines.append("Mean symptom deltas by stage type:")
    tmp = symptom_delta_df.groupby(["stage_type", "symptom"]).agg(
        delta_f1=("delta_f1_vs_direct", "mean"),
        delta_omission=("delta_omission_vs_direct", "mean"),
        delta_commission=("delta_commission_vs_direct", "mean"),
    ).reset_index()
    lines.append(tmp.to_string(index=False))
    lines.append("")

if len(gain_by_k_df) > 0:
    lines.append("Gain by symptom count:")
    tmp = gain_by_k_df.groupby(["stage_type", "k_true"]).agg(
        delta_exact_acc=("delta_exact_acc_vs_direct", "mean")
    ).reset_index()
    lines.append(tmp.to_string(index=False))
    lines.append("")

with open(os.path.join(TXT_DIR, "class255_stage_summary.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("\nDone.")
print("Output root:", OUT_DIR)
print("Figures:", FIG_DIR)
print("Tables:", TAB_DIR)
print("Texts:", TXT_DIR)


[DEBUG] target_df columns:
['accuracy', 'accuracy_final', 'accuracy_sum', 'context_files', 'context_files_n', 'count_bucket', 'exact_match', 'exact_match_final', 'exact_match_sum', 'explicit_stage_from_context', 'explicit_task_from_context', 'f1', 'f1_sum', 'final_stage_name', 'final_stage_role', 'final_task', 'final_task_source', 'full_n_records', 'hamming_loss', 'hamming_loss_final', 'hamming_loss_sum', 'jsonl_path', 'macro_f1', 'macro_f1_final', 'macro_f1_sum', 'micro_f1', 'micro_f1_final', 'micro_f1_sum', 'model_name', 'model_size_b', 'notes', 'notes_sum', 'paired_group_key', 'phase', 'precision', 'pred_file', 'pred_file_final', 'pred_file_pred', 'recall', 'run_name', 'stage_type', 'suffix_idx', 'task_from_predfile', 'task_from_summary', 'task_hint', 'task_pred', 'task_pred_final', 'task_score_binary', 'task_score_class255', 'task_score_eightlabel', 'task_score_fourclass', 'task_summary', 'task_summary_final', 'time_stem', 'train_dir', 'weighted_f1']

[DEBUG] target_df preview:
  

In [ ]:
# -*- coding: utf-8 -*-
"""
Goal
----
Only analyze the *missing deeper layers* beyond:
1) performance layer
2) basic structural layer

This script targets the missing NHB-level layers:
A. construct closure:
   symptom centrality -> omission / commission / prevalence bias
B. human uncertainty anchoring:
   confidence / disagreement -> model error
C. cross-source & hierarchical transfer generalization:
   native / direct / from2to255 / from4to255
D. cue-dependence stress tests:
   explicit-cue subsets, not requiring re-inference
E. risk-sensitive utility:
   severe false negatives, suicide-intent misses, top-risk capture
F. run-level stability:
   forest-style summary of core structural indicators

Inputs
------
1) final_stage_mapping/final_run_task_mapping.csv
2) export_bundle_v4/result_summary.csv
3) export_bundle_v4/sample_meta.csv
4) export_bundle_v4/predictions/*.csv

Outputs
-------
figures/: PNG + EPS, dpi=800
tables/: CSV summaries
texts/: auto-written result notes
"""

import os
import re
import glob
import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings("ignore")

ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"

FINAL_MAP_CSV = os.path.join(ROOT_DEPRESSED, "final_stage_mapping", "final_run_task_mapping.csv")
SUMMARY_CSV = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "result_summary.csv")
META_CSV = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "sample_meta.csv")
PRED_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "predictions")

OUT_DIR = os.path.join(ROOT_DEPRESSED, "nature_hb_missing_layers")
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
TXT_DIR = os.path.join(OUT_DIR, "texts")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)


EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

SHORT_NAMES = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#222222",
    "white": "#FFFFFF",
    "linegray": "#CFCBC5",
    "red": "#D95F5F",
    "purple": "#B39DDB",
}

# lexical stress-test cue sets
LEXICON = {
    "suicide_explicit": [
        "suicide", "suicidal", "kill myself", "end my life", "die", "dying", "don't want to live",
        "want to die", "want to disappear", "self harm", "self-harm"
    ],
    "hopelessness_explicit": [
        "hopeless", "no future", "nothing will change", "what's the point", "pointless", "can't go on"
    ],
    "sadness_explicit": [
        "sad", "cry", "crying", "empty", "miserable", "depressed", "down", "numb"
    ],
    "worthlessness_explicit": [
        "worthless", "useless", "burden", "hate myself", "failure", "not good enough"
    ],
    "cognitive_explicit": [
        "can't focus", "forget", "forgetting", "memory", "brain fog", "can't think", "concentrate"
    ],
    "first_person_dense": [
        " i ", " i'm ", " ive ", " i've ", " me ", " my ", " myself "
    ],
    "negation_markers": [
        "not", "never", "no ", "nothing", "nobody", "can't", "cannot", "won't", "don't", "doesn't"
    ]
}


def set_pub_style():
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "font.size": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.linewidth": 0.9,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.unicode_minus": False,
    })


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)
    ax.tick_params(axis="both", which="major", length=3.0, width=0.9, colors=PALETTE["black"])
    for lab in ax.get_xticklabels():
        lab.set_fontweight("bold")
    for lab in ax.get_yticklabels():
        lab.set_fontweight("bold")


def save_fig(fig, stem):
    png = os.path.join(FIG_DIR, f"{stem}.png")
    eps = os.path.join(FIG_DIR, f"{stem}.eps")
    fig.savefig(png, dpi=800, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(eps, format="eps", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    print("[Saved]", png)
    print("[Saved]", eps)


def finite_text(ax, x, y, s, **kwargs):
    if np.isfinite(x) and np.isfinite(y):
        ax.text(x, y, s, **kwargs)


set_pub_style()


def parse_model_size(model_name):
    m = re.search(r"Qwen3\.5-(0\.8|2|4|9)B-Base", str(model_name))
    return float(m.group(1)) if m else np.nan


def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df


def bvec_to_list(s):
    s = str(s).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) == 8 and set(s) <= set("01"):
        return [int(c) for c in s]
    return None


def dec255_to_vec(x):
    try:
        if pd.isna(x):
            return None
        x = int(float(x))
        if x < 0 or x > 255:
            return None
        return [int(c) for c in format(x, "08b")]
    except Exception:
        return None


def vec_to_dec255(vec):
    try:
        if vec is None or len(vec) != 8:
            return np.nan
        return int("".join(str(int(v)) for v in vec), 2)
    except Exception:
        return np.nan


def robust_true_vec(true_vec_raw, true_255_raw):
    vec = bvec_to_list(true_vec_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "true_vec"
    vec = dec255_to_vec(true_255_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "true_255"
    return [0, 0, 0, 0, 0, 0, 0, 0], 0, "fallback_zero"


def robust_pred_vec(pred_vec_raw, pred_255_raw):
    vec = bvec_to_list(pred_vec_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "pred_vec"
    vec = dec255_to_vec(pred_255_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "pred_255"
    return [0, 0, 0, 0, 0, 0, 0, 0], 0, "fallback_zero"


def normalize01(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.array([])
    lo, hi = np.min(x), np.max(x)
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def correlation_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return np.corrcoef(x[mask], y[mask])[0, 1]


def parse_prediction_filename(fp):
    base = os.path.basename(fp)
    stem = os.path.splitext(base)[0]
    parts = stem.split("__")
    if len(parts) < 4:
        return None
    task = parts[0]
    model_name = parts[1]
    phase = parts[2]
    run_name = "__".join(parts[3:])
    return {
        "pred_file": fp,
        "task": task,
        "model_name": model_name,
        "phase": phase,
        "run_name": run_name
    }


def normalize_text(x):
    return re.sub(r"\s+", " ", str(x)).strip().lower()


def contains_any_phrase(text, phrases):
    t = f" {normalize_text(text)} "
    return int(any(p.lower() in t for p in phrases))


def first_person_density(text):
    t = f" {normalize_text(text)} "
    hits = sum(t.count(p) for p in LEXICON["first_person_dense"])
    n_words = max(1, len(t.split()))
    return hits / n_words


def negation_density(text):
    t = f" {normalize_text(text)} "
    hits = sum(t.count(p) for p in LEXICON["negation_markers"])
    n_words = max(1, len(t.split()))
    return hits / n_words


def vec_core_score(vec, core):
    if vec is None:
        return np.nan
    idx = [i for i, v in enumerate(vec) if v == 1]
    if len(idx) == 0:
        return np.nan
    return float(np.mean(core[idx]))


def qcut_safe(s, q):
    try:
        return pd.qcut(s, q=q, duplicates="drop")
    except Exception:
        # fallback to rank-based equal bins
        return pd.cut(s.rank(method="average"), bins=q, duplicates="drop")


def odds_ratio_from_table(a, b, c, d):
    # table:
    # group1 event=a non-event=b
    # group0 event=c non-event=d
    # add 0.5 continuity correction
    a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    return (a * d) / (b * c)


def top_log_odds_terms(text_pos, text_neg, topn=30, min_df=3):
    texts = list(text_pos) + list(text_neg)
    y = np.array([1] * len(text_pos) + [0] * len(text_neg))
    if len(text_pos) < 10 or len(text_neg) < 10:
        return pd.DataFrame(columns=["term", "log_odds"])

    vec = CountVectorizer(
        stop_words="english",
        min_df=min_df,
        max_features=6000,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b"
    )
    X = vec.fit_transform(texts)
    vocab = np.array(vec.get_feature_names_out())

    X_pos = X[y == 1].sum(axis=0).A1.astype(float)
    X_neg = X[y == 0].sum(axis=0).A1.astype(float)

    alpha = 1.0
    p_pos = (X_pos + alpha) / (X_pos.sum() + alpha * len(X_pos))
    p_neg = (X_neg + alpha) / (X_neg.sum() + alpha * len(X_neg))

    log_odds = np.log(p_pos) - np.log(p_neg)
    out = pd.DataFrame({"term": vocab, "log_odds": log_odds})
    out = out.sort_values("log_odds", ascending=False).reset_index(drop=True)
    return out.head(topn)

if not os.path.isfile(FINAL_MAP_CSV):
    raise FileNotFoundError(f"Missing: {FINAL_MAP_CSV}")
if not os.path.isfile(SUMMARY_CSV):
    raise FileNotFoundError(f"Missing: {SUMMARY_CSV}")
if not os.path.isfile(META_CSV):
    raise FileNotFoundError(f"Missing: {META_CSV}")
if not os.path.isdir(PRED_DIR):
    raise FileNotFoundError(f"Missing: {PRED_DIR}")

map_df = pd.read_csv(FINAL_MAP_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)
meta_df = pd.read_csv(META_CSV, dtype={"sample_id": str})

pred_rows = []
for fp in sorted(glob.glob(os.path.join(PRED_DIR, "*.csv"))):
    x = parse_prediction_filename(fp)
    if x is not None:
        pred_rows.append(x)
pred_idx = pd.DataFrame(pred_rows)

map_df["model_size_b"] = map_df["model_name"].apply(parse_model_size)

# normalize summary
sum_keep = summary_df.copy()
if "task" in sum_keep.columns:
    sum_keep = sum_keep.rename(columns={"task": "task_summary"})
if "setting" in sum_keep.columns:
    sum_keep = sum_keep.rename(columns={"setting": "phase"})

# normalize pred index
if "task" in pred_idx.columns:
    pred_idx = pred_idx.rename(columns={"task": "task_pred"})

# merge map + summary + prediction index
merged = map_df.merge(
    sum_keep,
    on=["model_name", "run_name", "phase"],
    how="left",
    suffixes=("", "_sum")
)
merged = merged.merge(
    pred_idx,
    on=["model_name", "run_name", "phase"],
    how="left",
    suffixes=("", "_pred")
)

# coalesce some columns
def coalesce_columns(df, target_name, candidates):
    existing = [c for c in candidates if c in df.columns]
    if len(existing) == 0:
        df[target_name] = np.nan
        return df
    s = None
    for c in existing:
        if s is None:
            s = df[c]
        else:
            s = s.combine_first(df[c])
    df[target_name] = s
    return df

merged = coalesce_columns(merged, "accuracy_final", ["accuracy", "accuracy_sum", "accuracy_x", "accuracy_y"])
merged = coalesce_columns(merged, "macro_f1_final", ["macro_f1", "macro_f1_sum", "macro_f1_x", "macro_f1_y"])
merged = coalesce_columns(merged, "micro_f1_final", ["micro_f1", "micro_f1_sum", "micro_f1_x", "micro_f1_y"])
merged = coalesce_columns(merged, "exact_match_final", ["exact_match", "exact_match_sum", "exact_match_x", "exact_match_y"])
merged = coalesce_columns(merged, "hamming_loss_final", ["hamming_loss", "hamming_loss_sum", "hamming_loss_x", "hamming_loss_y"])
merged = coalesce_columns(merged, "pred_file_final", ["pred_file", "pred_file_pred", "file", "file_pred"])

# keep valid pred files
merged = merged[merged["pred_file_final"].notna()].copy()
merged = merged[merged["pred_file_final"].apply(lambda x: os.path.isfile(str(x)))].copy()

# helpful subsets
four_runs = merged[merged["final_task"] == "fourclass"].copy()
class255_runs = merged[merged["final_task"] == "class255"].copy()
binary_runs = merged[merged["final_task"] == "binary"].copy()

# best fourclass
if len(four_runs) == 0:
    raise ValueError("No fourclass runs found.")
best_four = four_runs.sort_values("macro_f1_final", ascending=False).iloc[0]

# best class255
if len(class255_runs) == 0:
    raise ValueError("No class255 runs found.")
best_255 = class255_runs.sort_values("macro_f1_final", ascending=False).iloc[0]

# native / direct / from2 / from4 class255 inventory
def classify_class255_stage_type(row):
    phase = str(row.get("phase", ""))
    final_stage_name = str(row.get("final_stage_name", ""))
    final_stage_role = str(row.get("final_stage_role", ""))

    if phase == "native_base_eval":
        return "native_class255"
    if phase == "ft_eval":
        if final_stage_name == "from2to255" and final_stage_role == "target_task_eval":
            return "from2to255"
        if final_stage_name == "from4to255" and final_stage_role == "target_task_eval":
            return "from4to255"
        if final_stage_name.strip() == "":
            return "direct_class255"
    return "other"

class255_runs["stage_type"] = class255_runs.apply(classify_class255_stage_type, axis=1)
class255_runs = class255_runs[class255_runs["stage_type"].isin([
    "native_class255", "direct_class255", "from2to255", "from4to255"
])].copy()

class255_runs["model_size_b"] = class255_runs["model_name"].apply(parse_model_size)

# save inventory
class255_inventory = class255_runs[[
    "model_name", "model_size_b", "run_name", "phase", "stage_type",
    "final_stage_name", "final_stage_role",
    "accuracy_final", "macro_f1_final", "micro_f1_final",
    "exact_match_final", "hamming_loss_final", "pred_file_final"
]].sort_values(["model_size_b", "stage_type", "run_name"]).reset_index(drop=True)
class255_inventory.to_csv(os.path.join(TAB_DIR, "inventory_class255_stage_runs.csv"), index=False, encoding="utf-8-sig")

four_df = pd.read_csv(best_four["pred_file_final"], dtype={"sample_id": str})
class255_df = pd.read_csv(best_255["pred_file_final"], dtype={"sample_id": str})

# fourclass merge meta
if "sample_id" in four_df.columns and "sample_id" in meta_df.columns:
    four_df = four_df.merge(meta_df, on="sample_id", how="left", suffixes=("", "_meta"))

ensure_cols(four_df, ["true_label", "pred_label", "text"])
four_df["true_label"] = pd.to_numeric(four_df["true_label"], errors="coerce")
four_df["pred_label"] = pd.to_numeric(four_df["pred_label"], errors="coerce")

four_df["abs_err"] = (four_df["true_label"] - four_df["pred_label"]).abs()
four_df["is_error"] = (four_df["abs_err"] > 0).astype(int)
four_df["is_adjacent_error"] = (four_df["abs_err"] == 1).astype(int)
four_df["is_far_error"] = (four_df["abs_err"] >= 2).astype(int)

four_df["error_type"] = np.where(
    four_df["abs_err"] == 0, "Correct",
    np.where(four_df["abs_err"] == 1, "Adjacent", "Far")
)

# class255 robust decode
ensure_cols(class255_df, ["true_vec", "pred_vec", "true_255", "pred_255", "text"])
class255_df["true_255"] = pd.to_numeric(class255_df["true_255"], errors="coerce")
class255_df["pred_255"] = pd.to_numeric(class255_df["pred_255"], errors="coerce")

tmp_true = class255_df.apply(
    lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)),
    axis=1
)
class255_df["true_vec_list"] = [x[0] for x in tmp_true]
class255_df["true_vec_valid"] = [x[1] for x in tmp_true]
class255_df["true_vec_source"] = [x[2] for x in tmp_true]

tmp_pred = class255_df.apply(
    lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)),
    axis=1
)
class255_df["pred_vec_list"] = [x[0] for x in tmp_pred]
class255_df["pred_vec_valid"] = [x[1] for x in tmp_pred]
class255_df["pred_vec_source"] = [x[2] for x in tmp_pred]

class255_df["true_255_filled"] = class255_df.apply(
    lambda r: r["true_255"] if pd.notna(r["true_255"]) else vec_to_dec255(r["true_vec_list"]),
    axis=1
)
class255_df["pred_255_filled"] = class255_df.apply(
    lambda r: r["pred_255"] if pd.notna(r["pred_255"]) else vec_to_dec255(r["pred_vec_list"]),
    axis=1
)

X_true = np.vstack(class255_df["true_vec_list"].tolist()).astype(int)
X_pred = np.vstack(class255_df["pred_vec_list"].tolist()).astype(int)

class255_df["correct"] = (class255_df["true_255_filled"] == class255_df["pred_255_filled"])
class255_df["k_true"] = X_true.sum(axis=1)
class255_df["k_pred"] = X_pred.sum(axis=1)


# A1. true co-occurrence centrality
cooc_true = (X_true.T @ X_true) / X_true.shape[0]
cooc_off = cooc_true.copy()
np.fill_diagonal(cooc_off, 0.0)
strength = cooc_off.sum(axis=1)
centrality = normalize01(strength)

# A2. symptom metrics
symptom_rows = []
for i, name in enumerate(EMOTION_NAMES):
    yt = X_true[:, i]
    yp = X_pred[:, i]
    symptom_rows.append({
        "symptom": name,
        "symptom_short": SHORT_NAMES[name],
        "support": int(yt.sum()),
        "true_prev": yt.mean(),
        "pred_prev": yp.mean(),
        "prev_bias": yp.mean() - yt.mean(),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "omission_rate": ((yt == 1) & (yp == 0)).sum() / max(1, (yt == 1).sum()),
        "commission_rate": ((yt == 0) & (yp == 1)).sum() / max(1, (yt == 0).sum()),
        "centrality": centrality[i]
    })
symptom_df = pd.DataFrame(symptom_rows)
symptom_df.to_csv(os.path.join(TAB_DIR, "A_construct_closure_symptom_metrics.csv"), index=False, encoding="utf-8-sig")

# A3. boundary lexical archetypes
lex_1up = top_log_odds_terms(
    four_df[(four_df["true_label"] == 1) & (four_df["pred_label"] == 2)]["text"].astype(str).tolist(),
    four_df[(four_df["true_label"] == 1) & (four_df["pred_label"] == 1)]["text"].astype(str).tolist(),
    topn=30
)
lex_2down = top_log_odds_terms(
    four_df[(four_df["true_label"] == 2) & (four_df["pred_label"] == 1)]["text"].astype(str).tolist(),
    four_df[(four_df["true_label"] == 2) & (four_df["pred_label"] == 2)]["text"].astype(str).tolist(),
    topn=30
)
lex_1up.to_csv(os.path.join(TAB_DIR, "A_boundary_lexicon_true1_to_pred2.csv"), index=False, encoding="utf-8-sig")
lex_2down.to_csv(os.path.join(TAB_DIR, "A_boundary_lexicon_true2_to_pred1.csv"), index=False, encoding="utf-8-sig")

# A4. figure: centrality -> omission / commission / prev_bias
fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.2))
for ax, ycol, ylabel, color in [
    (axes[0], "omission_rate", "Omission rate", PALETTE["orange"]),
    (axes[1], "commission_rate", "Commission rate", PALETTE["pink"]),
    (axes[2], "prev_bias", "Predicted prevalence − true prevalence", PALETTE["blue"]),
]:
    x = symptom_df["centrality"].to_numpy()
    y = symptom_df[ycol].to_numpy()
    ax.scatter(x, y, s=70, color=color, edgecolors="white", linewidths=0.7)
    for _, r in symptom_df.iterrows():
        finite_text(ax, r["centrality"] + 0.015, r[ycol], r["symptom_short"], fontsize=6.0, fontweight="bold")
    r = correlation_pearson(x, y)
    ax.text(0.98, 0.96, f"r={r:.2f}", transform=ax.transAxes,
            ha="right", va="top", fontsize=6.8, fontweight="bold")
    if ycol == "prev_bias":
        ax.axhline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xlabel("Symptom centrality", fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    style_ax(ax)
fig.tight_layout()
save_fig(fig, "FigA_construct_closure_centrality_bias_triptych")


# B1. confidence decile error anchor
b_conf = pd.DataFrame()
if "confidence_score" in four_df.columns and four_df["confidence_score"].notna().sum() > 30:
    tmp = four_df.dropna(subset=["confidence_score"]).copy()
    tmp["conf_decile"] = qcut_safe(tmp["confidence_score"], q=10)
    b_conf = tmp.groupby("conf_decile").agg(
        n=("sample_id", "size"),
        conf_mean=("confidence_score", "mean"),
        error_rate=("is_error", "mean"),
        adjacent_rate=("is_adjacent_error", "mean"),
        far_rate=("is_far_error", "mean"),
    ).reset_index(drop=True)
    b_conf.to_csv(os.path.join(TAB_DIR, "B_confidence_decile_error_rates.csv"), index=False, encoding="utf-8-sig")

# B2. disagreement enrichment
b_dis = pd.DataFrame()
if "is_disagreement_sample" in four_df.columns:
    tmp = four_df.copy()
    tmp["group"] = np.where(tmp["is_disagreement_sample"] == 1, "disagreement", "non_disagreement")
    b_dis = tmp.groupby("group").agg(
        n=("sample_id", "size"),
        error_rate=("is_error", "mean"),
        adjacent_rate=("is_adjacent_error", "mean"),
        far_rate=("is_far_error", "mean")
    ).reset_index()
    b_dis.to_csv(os.path.join(TAB_DIR, "B_disagreement_enrichment.csv"), index=False, encoding="utf-8-sig")

# B3. error-type odds ratio in disagreement
b_or_rows = []
if "is_disagreement_sample" in four_df.columns:
    for col, label in [("is_error", "any_error"), ("is_adjacent_error", "adjacent_error"), ("is_far_error", "far_error")]:
        g1 = four_df[four_df["is_disagreement_sample"] == 1]
        g0 = four_df[four_df["is_disagreement_sample"] == 0]
        a = int(g1[col].sum())
        b = int(len(g1) - g1[col].sum())
        c = int(g0[col].sum())
        d = int(len(g0) - g0[col].sum())
        b_or_rows.append({
            "error_type": label,
            "odds_ratio_disagreement_vs_non": odds_ratio_from_table(a, b, c, d)
        })
b_or_df = pd.DataFrame(b_or_rows)
b_or_df.to_csv(os.path.join(TAB_DIR, "B_disagreement_odds_ratios.csv"), index=False, encoding="utf-8-sig")

# B4. figure
if len(b_conf) > 0 or len(b_or_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.1))
    if len(b_conf) > 0:
        x = np.arange(len(b_conf))
        axes[0].plot(x, b_conf["error_rate"], marker="o", linewidth=1.5, color=PALETTE["orange"])
        axes[0].plot(x, b_conf["adjacent_rate"], marker="o", linewidth=1.3, color=PALETTE["yellow"])
        axes[0].plot(x, b_conf["far_rate"], marker="o", linewidth=1.3, color=PALETTE["pink"])
        axes[0].set_xticks(x, labels=[str(i+1) for i in range(len(x))])
        axes[0].set_xlabel("Confidence decile (low → high)", fontweight="bold")
        axes[0].set_ylabel("Fraction", fontweight="bold")
        style_ax(axes[0])
    else:
        axes[0].axis("off")

    if len(b_or_df) > 0:
        x = np.arange(len(b_or_df))
        axes[1].bar(x, b_or_df["odds_ratio_disagreement_vs_non"], color=PALETTE["blue"], edgecolor="none", width=0.62)
        for i, r in b_or_df.iterrows():
            finite_text(axes[1], i, r["odds_ratio_disagreement_vs_non"] + 0.08,
                        f"{r['odds_ratio_disagreement_vs_non']:.2f}",
                        ha="center", va="bottom", fontsize=6.4, fontweight="bold")
        axes[1].axhline(1.0, color=PALETTE["linegray"], linewidth=1.0)
        axes[1].set_xticks(x, labels=["Any\nerror", "Adjacent", "Far"])
        axes[1].set_ylabel("Odds ratio\n(disagreement / non)", fontweight="bold")
        style_ax(axes[1])
    else:
        axes[1].axis("off")
    fig.tight_layout()
    save_fig(fig, "FigB_human_uncertainty_anchor")


# C1. class255 stage inventory summary
# C1. class255 stage inventory summary
c_stage = class255_runs.copy()
c_stage["model_size_b"] = c_stage["model_name"].apply(parse_model_size)
c_stage = ensure_cols(c_stage, ["macro_f1_final", "accuracy_final", "stage_type"])

# choose direct baseline by model size
baseline_rows = []
direct_stage = c_stage[c_stage["stage_type"] == "direct_class255"].copy()

if len(direct_stage) > 0:
    for size, g in direct_stage.groupby("model_size_b"):
        g = g.sort_values("macro_f1_final", ascending=False).head(1)
        if len(g) > 0:
            baseline_rows.append(g.iloc[0].to_dict())

baseline_df = pd.DataFrame(baseline_rows)

# ensure baseline_df has expected columns even when empty
baseline_df = ensure_cols(baseline_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "macro_f1_final", "accuracy_final"
])

c_rows = []
for _, r in c_stage.iterrows():
    size = r["model_size_b"]
    stage_type = r["stage_type"]

    out = {
        "model_name": r["model_name"],
        "model_size_b": size,
        "run_name": r["run_name"],
        "stage_type": stage_type,
        "macro_f1": r.get("macro_f1_final", np.nan),
        "accuracy": r.get("accuracy_final", np.nan),
    }

    # baseline may legitimately be missing for this size
    if len(baseline_df) > 0 and "model_size_b" in baseline_df.columns:
        base = baseline_df[baseline_df["model_size_b"] == size]
    else:
        base = pd.DataFrame()

    if len(base) > 0 and stage_type != "direct_class255":
        base = base.iloc[0]
        out["delta_macro_f1_vs_direct"] = r.get("macro_f1_final", np.nan) - base.get("macro_f1_final", np.nan)
        out["delta_accuracy_vs_direct"] = r.get("accuracy_final", np.nan) - base.get("accuracy_final", np.nan)
        out["baseline_available"] = 1
    else:
        out["delta_macro_f1_vs_direct"] = np.nan
        out["delta_accuracy_vs_direct"] = np.nan
        out["baseline_available"] = 0

    c_rows.append(out)

c_stage_df = pd.DataFrame(c_rows)
c_stage_df = ensure_cols(c_stage_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "macro_f1", "accuracy",
    "delta_macro_f1_vs_direct", "delta_accuracy_vs_direct",
    "baseline_available"
])

c_stage_df.to_csv(
    os.path.join(TAB_DIR, "C_stage_generalization_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("\n[DEBUG] class255 staged summary:")
print(c_stage_df.to_string(index=False))
print("\n[DEBUG] direct baselines found:")
print(baseline_df[["model_name", "model_size_b", "run_name", "macro_f1_final", "accuracy_final"]].to_string(index=False))


# C3. figure
fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.2))
# panel 1: mean delta by stage type
# panel 1: mean delta by stage type
tmp = c_stage_df[c_stage_df["stage_type"].isin(["from2to255", "from4to255"])].copy()
tmp = tmp[tmp["baseline_available"] == 1].copy()

if len(tmp) > 0:
    agg = tmp.groupby("stage_type").agg(
        mean_delta_macro_f1=("delta_macro_f1_vs_direct", "mean"),
        mean_delta_acc=("delta_accuracy_vs_direct", "mean"),
        n=("run_name", "size")
    ).reset_index()
    x = np.arange(len(agg))
    axes[0].bar(x - 0.14, agg["mean_delta_macro_f1"], width=0.28, color=PALETTE["orange"], edgecolor="none")
    axes[0].bar(x + 0.14, agg["mean_delta_acc"], width=0.28, color=PALETTE["blue"], edgecolor="none")
    axes[0].axhline(0, color=PALETTE["linegray"], linewidth=1.0)
    axes[0].set_xticks(x, labels=agg["stage_type"].tolist())
    axes[0].set_ylabel("Δ vs direct class255", fontweight="bold")
    style_ax(axes[0])
else:
    axes[0].axis("off")

# panel 2: scaling by model size
drawn = False
for st, color in [("direct_class255", PALETTE["gray"]), ("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
    dd = c_stage_df[c_stage_df["stage_type"] == st].copy()
    if len(dd) == 0:
        continue
    agg = dd.groupby("model_size_b").agg(mean_macro_f1=("macro_f1", "mean")).reset_index()
    if len(agg) == 0:
        continue
    axes[1].plot(agg["model_size_b"], agg["mean_macro_f1"], marker="o", linewidth=1.5, color=color)
    drawn = True
if drawn:
    axes[1].set_xlabel("Model size (B)", fontweight="bold")
    axes[1].set_ylabel("Macro-F1", fontweight="bold")
    style_ax(axes[1])
else:
    axes[1].axis("off")

fig.tight_layout()
save_fig(fig, "FigC_cross_source_hierarchical_generalization")


# D1. fourclass lexical cue dependence
four_stress = four_df.copy()
four_stress["text"] = four_stress["text"].astype(str)

for cue_name, phrases in LEXICON.items():
    if cue_name in ["first_person_dense", "negation_markers"]:
        continue
    four_stress[f"cue__{cue_name}"] = four_stress["text"].apply(lambda x: contains_any_phrase(x, phrases))

four_stress["first_person_density"] = four_stress["text"].apply(first_person_density)
four_stress["negation_density"] = four_stress["text"].apply(negation_density)
four_stress["high_first_person"] = (four_stress["first_person_density"] >= four_stress["first_person_density"].median()).astype(int)
four_stress["high_negation"] = (four_stress["negation_density"] >= four_stress["negation_density"].median()).astype(int)

d_rows = []
cue_cols = [c for c in four_stress.columns if c.startswith("cue__")] + ["high_first_person", "high_negation"]
for c in cue_cols:
    for group, g in [("with_cue", four_stress[four_stress[c] == 1]), ("without_cue", four_stress[four_stress[c] == 0])]:
        if len(g) == 0:
            continue
        d_rows.append({
            "cue": c.replace("cue__", ""),
            "group": group,
            "n": len(g),
            "error_rate": g["is_error"].mean(),
            "adjacent_rate": g["is_adjacent_error"].mean(),
            "far_rate": g["is_far_error"].mean(),
            "mean_true_label": g["true_label"].mean()
        })
d_stress_df = pd.DataFrame(d_rows)
d_stress_df.to_csv(os.path.join(TAB_DIR, "D_fourclass_cue_dependence.csv"), index=False, encoding="utf-8-sig")

# D2. class255 cue dependence on top-risk symptoms
class255_stress = class255_df.copy()
class255_stress["text"] = class255_stress["text"].astype(str)

for cue_name, phrases in LEXICON.items():
    if cue_name in ["first_person_dense", "negation_markers"]:
        continue
    class255_stress[f"cue__{cue_name}"] = class255_stress["text"].apply(lambda x: contains_any_phrase(x, phrases))

class255_stress["first_person_density"] = class255_stress["text"].apply(first_person_density)
class255_stress["negation_density"] = class255_stress["text"].apply(negation_density)
class255_stress["high_first_person"] = (class255_stress["first_person_density"] >= class255_stress["first_person_density"].median()).astype(int)
class255_stress["high_negation"] = (class255_stress["negation_density"] >= class255_stress["negation_density"].median()).astype(int)

# suicide intent label index
idx_suicide = EMOTION_NAMES.index("suicide intent")
class255_stress["true_suicide"] = X_true[:, idx_suicide]
class255_stress["pred_suicide"] = X_pred[:, idx_suicide]
class255_stress["suicide_fn"] = ((class255_stress["true_suicide"] == 1) & (class255_stress["pred_suicide"] == 0)).astype(int)

d2_rows = []
cue_cols2 = [c for c in class255_stress.columns if c.startswith("cue__")] + ["high_first_person", "high_negation"]
for c in cue_cols2:
    for group, g in [("with_cue", class255_stress[class255_stress[c] == 1]), ("without_cue", class255_stress[class255_stress[c] == 0])]:
        if len(g) == 0:
            continue
        d2_rows.append({
            "cue": c.replace("cue__", ""),
            "group": group,
            "n": len(g),
            "exact_acc": g["correct"].mean(),
            "mean_k_true": g["k_true"].mean(),
            "suicide_fn_rate": g[g["true_suicide"] == 1]["suicide_fn"].mean() if (g["true_suicide"] == 1).sum() > 0 else np.nan
        })
d2_stress_df = pd.DataFrame(d2_rows)
d2_stress_df.to_csv(os.path.join(TAB_DIR, "D_class255_cue_dependence.csv"), index=False, encoding="utf-8-sig")

# D3. figure: selected cues only
selected_cues = ["suicide_explicit", "hopelessness_explicit", "worthlessness_explicit", "high_first_person", "high_negation"]
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.4))

tmp = d_stress_df[d_stress_df["cue"].isin(selected_cues)].copy()
if len(tmp) > 0:
    pivot = tmp.pivot(index="cue", columns="group", values="error_rate").reset_index()
    pivot["delta"] = pivot["with_cue"] - pivot["without_cue"]
    pivot = pivot.sort_values("delta", ascending=True).reset_index(drop=True)
    y = np.arange(len(pivot))
    axes[0].barh(y, pivot["delta"], color=PALETTE["orange"], edgecolor="none", height=0.66)
    axes[0].set_yticks(y, labels=pivot["cue"].tolist())
    axes[0].set_xlabel("Δ fourclass error rate\n(with cue − without)", fontweight="bold")
    style_ax(axes[0])
else:
    axes[0].axis("off")

tmp = d2_stress_df[d2_stress_df["cue"].isin(selected_cues)].copy()
if len(tmp) > 0:
    pivot = tmp.pivot(index="cue", columns="group", values="suicide_fn_rate").reset_index()
    if "with_cue" in pivot.columns and "without_cue" in pivot.columns:
        pivot["delta"] = pivot["with_cue"] - pivot["without_cue"]
        pivot = pivot.sort_values("delta", ascending=True).reset_index(drop=True)
        y = np.arange(len(pivot))
        axes[1].barh(y, pivot["delta"], color=PALETTE["red"], edgecolor="none", height=0.66)
        axes[1].set_yticks(y, labels=pivot["cue"].tolist())
        axes[1].set_xlabel("Δ suicide false-negative rate\n(with cue − without)", fontweight="bold")
        style_ax(axes[1])
    else:
        axes[1].axis("off")
else:
    axes[1].axis("off")

fig.tight_layout()
save_fig(fig, "FigD_cue_dependence_stress_tests")


# E1. fourclass thresholds
e_thresh_rows = []
for t in [1, 2, 3]:
    yt = (four_df["true_label"] >= t).astype(int)
    yp = (four_df["pred_label"] >= t).astype(int)
    e_thresh_rows.append({
        "threshold": f">={t}",
        "positive_definition": f"severity >= {t}",
        "support_positive": int(yt.sum()),
        "accuracy": accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
    })
e_thresh_df = pd.DataFrame(e_thresh_rows)
e_thresh_df.to_csv(os.path.join(TAB_DIR, "E_fourclass_threshold_utility.csv"), index=False, encoding="utf-8-sig")

# E2. severe false negative routing
severe = four_df[four_df["true_label"] == 3].copy()
e_severe_route = severe.groupby("pred_label").size().reset_index(name="count")
e_severe_route["fraction"] = e_severe_route["count"] / max(1, e_severe_route["count"].sum())
e_severe_route.to_csv(os.path.join(TAB_DIR, "E_severe_false_negative_routing.csv"), index=False, encoding="utf-8-sig")

# E3. suicide-intent false negative
e_suicide = pd.DataFrame({
    "true_suicide": X_true[:, idx_suicide],
    "pred_suicide": X_pred[:, idx_suicide]
})
e_suicide["false_negative"] = ((e_suicide["true_suicide"] == 1) & (e_suicide["pred_suicide"] == 0)).astype(int)
e_suicide_summary = pd.DataFrame([{
    "support_positive": int(e_suicide["true_suicide"].sum()),
    "precision": precision_score(e_suicide["true_suicide"], e_suicide["pred_suicide"], zero_division=0),
    "recall": recall_score(e_suicide["true_suicide"], e_suicide["pred_suicide"], zero_division=0),
    "f1": f1_score(e_suicide["true_suicide"], e_suicide["pred_suicide"], zero_division=0),
    "false_negative_rate": e_suicide[e_suicide["true_suicide"] == 1]["false_negative"].mean() if e_suicide["true_suicide"].sum() > 0 else np.nan
}])
e_suicide_summary.to_csv(os.path.join(TAB_DIR, "E_suicide_intent_risk_summary.csv"), index=False, encoding="utf-8-sig")

# E4. top-risk capture by predicted symptom burden
class255_df["risk_score_pred"] = class255_df["k_pred"] + X_pred[:, idx_suicide] * 2.0
class255_df["risk_score_true"] = class255_df["k_true"] + X_true[:, idx_suicide] * 2.0

top_capture_rows = []
for frac in [0.05, 0.10, 0.20]:
    n_top = max(1, int(len(class255_df) * frac))
    top_idx = class255_df.sort_values("risk_score_pred", ascending=False).head(n_top).index

    capture_severe_proxy = X_true[:, idx_suicide][top_idx].mean()
    capture_high_burden = (class255_df.loc[top_idx, "k_true"] >= 4).mean()

    top_capture_rows.append({
        "top_fraction": frac,
        "n_top": n_top,
        "suicide_positive_rate_in_top": capture_severe_proxy,
        "high_burden_rate_in_top": capture_high_burden
    })
e_capture_df = pd.DataFrame(top_capture_rows)
e_capture_df.to_csv(os.path.join(TAB_DIR, "E_top_risk_capture.csv"), index=False, encoding="utf-8-sig")

# E5. figure
fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.1))

# threshold utility
x = np.arange(len(e_thresh_df))
axes[0].plot(x, e_thresh_df["precision"], marker="o", linewidth=1.4, color=PALETTE["orange"])
axes[0].plot(x, e_thresh_df["recall"], marker="o", linewidth=1.4, color=PALETTE["blue"])
axes[0].plot(x, e_thresh_df["f1"], marker="o", linewidth=1.4, color=PALETTE["green"])
axes[0].set_xticks(x, labels=e_thresh_df["threshold"].tolist())
axes[0].set_xlabel("Positive definition", fontweight="bold")
axes[0].set_ylabel("Score", fontweight="bold")
style_ax(axes[0])

# severe routing
if len(e_severe_route) > 0:
    x = np.arange(len(e_severe_route))
    axes[1].bar(x, e_severe_route["fraction"], color=PALETTE["pink"], edgecolor="none", width=0.62)
    axes[1].set_xticks(x, labels=e_severe_route["pred_label"].astype(int).astype(str).tolist())
    axes[1].set_xlabel("Predicted label when true=3", fontweight="bold")
    axes[1].set_ylabel("Fraction", fontweight="bold")
    style_ax(axes[1])
else:
    axes[1].axis("off")

# top-risk capture
x = np.arange(len(e_capture_df))
axes[2].bar(x - 0.14, e_capture_df["suicide_positive_rate_in_top"], width=0.28, color=PALETTE["red"], edgecolor="none")
axes[2].bar(x + 0.14, e_capture_df["high_burden_rate_in_top"], width=0.28, color=PALETTE["purple"], edgecolor="none")
axes[2].set_xticks(x, labels=[f"{int(v*100)}%" for v in e_capture_df["top_fraction"]])
axes[2].set_xlabel("Top predicted-risk slice", fontweight="bold")
axes[2].set_ylabel("Rate in slice", fontweight="bold")
style_ax(axes[2])

fig.tight_layout()
save_fig(fig, "FigE_risk_sensitive_utility")


# F1. compute key structural indicators for all fourclass runs
def compute_fourclass_key_metrics(pred_file):
    d = pd.read_csv(pred_file, dtype={"sample_id": str})
    ensure_cols(d, ["true_label", "pred_label", "text", "sample_id"])
    d["true_label"] = pd.to_numeric(d["true_label"], errors="coerce")
    d["pred_label"] = pd.to_numeric(d["pred_label"], errors="coerce")
    d["abs_err"] = (d["true_label"] - d["pred_label"]).abs()
    d["adjacent_error_frac"] = (d["abs_err"] == 1).mean()
    d["far_error_frac"] = (d["abs_err"] >= 2).mean()

    adj = d[d["abs_err"] == 1].copy()
    if len(adj) > 0:
        adj["boundary"] = adj.apply(lambda r: f"{int(min(r['true_label'], r['pred_label']))}↔{int(max(r['true_label'], r['pred_label']))}", axis=1)
        share_12 = (adj["boundary"] == "1↔2").mean()
    else:
        share_12 = np.nan

    return {
        "adjacent_error_frac": d["adjacent_error_frac"].iloc[0] if len(d) > 0 else np.nan,
        "far_error_frac": d["far_error_frac"].iloc[0] if len(d) > 0 else np.nan,
        "share_12_within_adjacent": share_12,
    }

four_stab_rows = []
for _, r in four_runs.iterrows():
    pred_file = r["pred_file_final"]
    if not os.path.isfile(str(pred_file)):
        continue
    met = compute_fourclass_key_metrics(pred_file)
    four_stab_rows.append({
        "model_name": r["model_name"],
        "model_size_b": parse_model_size(r["model_name"]),
        "run_name": r["run_name"],
        "phase": r["phase"],
        "macro_f1": r.get("macro_f1_final", np.nan),
        **met
    })
four_stab_df = pd.DataFrame(four_stab_rows)
four_stab_df.to_csv(os.path.join(TAB_DIR, "F_fourclass_run_stability.csv"), index=False, encoding="utf-8-sig")

# F2. compute key structural indicators for all class255 runs
def compute_class255_key_metrics(pred_file):
    d = pd.read_csv(pred_file, dtype={"sample_id": str})
    ensure_cols(d, ["true_vec", "pred_vec", "true_255", "pred_255", "text"])
    d["true_255"] = pd.to_numeric(d["true_255"], errors="coerce")
    d["pred_255"] = pd.to_numeric(d["pred_255"], errors="coerce")

    tmp_true = d.apply(lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)), axis=1)
    d["true_vec_list"] = [x[0] for x in tmp_true]

    tmp_pred = d.apply(lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)), axis=1)
    d["pred_vec_list"] = [x[0] for x in tmp_pred]
    d["pred_vec_source"] = [x[2] for x in tmp_pred]

    X_t = np.vstack(d["true_vec_list"].tolist()).astype(int)
    X_p = np.vstack(d["pred_vec_list"].tolist()).astype(int)

    d["k_true"] = X_t.sum(axis=1)
    d["k_pred"] = X_p.sum(axis=1)

    # local centrality from true labels of this run
    cooc = (X_t.T @ X_t) / X_t.shape[0]
    off = cooc.copy()
    np.fill_diagonal(off, 0.0)
    cent = normalize01(off.sum(axis=1))

    d["true_core"] = [vec_core_score(v, cent) for v in d["true_vec_list"]]
    d["pred_core"] = [vec_core_score(v, cent) for v in d["pred_vec_list"]]
    d["core_shift"] = d["pred_core"] - d["true_core"]

    # support-weighted omission/commission for selected symptoms
    idx_sad = EMOTION_NAMES.index("sadness")
    idx_brain = EMOTION_NAMES.index("brain dysfunction (forget)")

    yt_sad, yp_sad = X_t[:, idx_sad], X_p[:, idx_sad]
    yt_br, yp_br = X_t[:, idx_brain], X_p[:, idx_brain]

    return {
        "mean_core_shift": d["core_shift"].mean(),
        "brain_omission": ((yt_br == 1) & (yp_br == 0)).sum() / max(1, (yt_br == 1).sum()),
        "sadness_commission": ((yt_sad == 0) & (yp_sad == 1)).sum() / max(1, (yt_sad == 0).sum()),
        "symptom_count_u_shape_proxy": d.groupby("k_true").apply(lambda x: (x["k_pred"] == x["k_true"]).mean()).var(),
        "pred_vec_fallback_zero_rate": (pd.Series(d["pred_vec_source"]) == "fallback_zero").mean(),
    }

class255_stab_rows = []
for _, r in class255_runs.iterrows():
    pred_file = r["pred_file_final"]
    if not os.path.isfile(str(pred_file)):
        continue
    met = compute_class255_key_metrics(pred_file)
    class255_stab_rows.append({
        "model_name": r["model_name"],
        "model_size_b": parse_model_size(r["model_name"]),
        "run_name": r["run_name"],
        "phase": r["phase"],
        "stage_type": r["stage_type"],
        "macro_f1": r.get("macro_f1_final", np.nan),
        **met
    })
class255_stab_df = pd.DataFrame(class255_stab_rows)
class255_stab_df.to_csv(os.path.join(TAB_DIR, "F_class255_run_stability.csv"), index=False, encoding="utf-8-sig")

# F3. figure: forest-like summary
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6))

# fourclass structure stability
if len(four_stab_df) > 0:
    agg = four_stab_df.groupby("model_size_b").agg(
        mean_adj=("adjacent_error_frac", "mean"),
        min_adj=("adjacent_error_frac", "min"),
        max_adj=("adjacent_error_frac", "max"),
        mean_share12=("share_12_within_adjacent", "mean"),
        min_share12=("share_12_within_adjacent", "min"),
        max_share12=("share_12_within_adjacent", "max"),
    ).reset_index()

    for _, r in agg.iterrows():
        axes[0].plot([r["min_adj"], r["max_adj"]], [r["model_size_b"], r["model_size_b"]], color=PALETTE["orange"], linewidth=2.0)
        axes[0].scatter(r["mean_adj"], r["model_size_b"], s=42, color=PALETTE["orange"], zorder=3)

        axes[0].plot([r["min_share12"], r["max_share12"]], [r["model_size_b"] + 0.12, r["model_size_b"] + 0.12], color=PALETTE["blue"], linewidth=2.0)
        axes[0].scatter(r["mean_share12"], r["model_size_b"] + 0.12, s=42, color=PALETTE["blue"], zorder=3)

    axes[0].set_xlabel("Range / mean", fontweight="bold")
    axes[0].set_ylabel("Model size (B)", fontweight="bold")
    style_ax(axes[0])
else:
    axes[0].axis("off")

# class255 structure stability
if len(class255_stab_df) > 0:
    tmp = class255_stab_df[class255_stab_df["stage_type"].isin(["native_class255", "direct_class255", "from2to255", "from4to255"])].copy()
    ymap = {k: i for i, k in enumerate(["native_class255", "direct_class255", "from2to255", "from4to255"])}
    for st, color in [("direct_class255", PALETTE["gray"]), ("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
        dd = tmp[tmp["stage_type"] == st]
        if len(dd) == 0:
            continue
        axes[1].scatter(dd["mean_core_shift"], dd["model_size_b"], s=52, color=color, label=st)
    axes[1].axvline(0, color=PALETTE["linegray"], linewidth=1.0)
    axes[1].set_xlabel("Mean core-shift", fontweight="bold")
    axes[1].set_ylabel("Model size (B)", fontweight="bold")
    style_ax(axes[1])
else:
    axes[1].axis("off")

fig.tight_layout()
save_fig(fig, "FigF_run_level_stability")


lines = []
lines.append("NHB missing-layer analysis summary")
lines.append("")

lines.append("A. Construct closure")
lines.append(f"Centrality vs omission r = {correlation_pearson(symptom_df['centrality'], symptom_df['omission_rate']):.3f}")
lines.append(f"Centrality vs commission r = {correlation_pearson(symptom_df['centrality'], symptom_df['commission_rate']):.3f}")
lines.append(f"Centrality vs prevalence-bias r = {correlation_pearson(symptom_df['centrality'], symptom_df['prev_bias']):.3f}")
lines.append("")

if len(b_dis) > 0:
    lines.append("B. Human uncertainty anchoring")
    lines.append(b_dis.to_string(index=False))
    lines.append("")
if len(b_or_df) > 0:
    lines.append("Disagreement odds ratios")
    lines.append(b_or_df.to_string(index=False))
    lines.append("")

if len(c_stage_df) > 0:
    lines.append("C. Cross-source / hierarchical generalization")
    lines.append(c_stage_df.sort_values(['stage_type', 'model_size_b', 'run_name']).to_string(index=False))
    lines.append("")

if len(d_stress_df) > 0:
    lines.append("D. Cue-dependence stress test (fourclass)")
    lines.append(d_stress_df.head(50).to_string(index=False))
    lines.append("")
if len(d2_stress_df) > 0:
    lines.append("D. Cue-dependence stress test (class255)")
    lines.append(d2_stress_df.head(50).to_string(index=False))
    lines.append("")

lines.append("E. Risk-sensitive utility")
lines.append(e_thresh_df.to_string(index=False))
lines.append("")
lines.append(e_suicide_summary.to_string(index=False))
lines.append("")
lines.append(e_capture_df.to_string(index=False))
lines.append("")

if len(four_stab_df) > 0:
    lines.append("F. Fourclass run stability")
    lines.append(four_stab_df.head(50).to_string(index=False))
    lines.append("")
if len(class255_stab_df) > 0:
    lines.append("F. Class255 run stability")
    lines.append(class255_stab_df.head(50).to_string(index=False))
    lines.append("")

with open(os.path.join(TXT_DIR, "NHB_missing_layers_summary.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))


table_index = pd.DataFrame({
    "table_name": [
        "A_construct_closure_symptom_metrics.csv",
        "A_boundary_lexicon_true1_to_pred2.csv",
        "A_boundary_lexicon_true2_to_pred1.csv",
        "B_confidence_decile_error_rates.csv",
        "B_disagreement_enrichment.csv",
        "B_disagreement_odds_ratios.csv",
        "C_stage_generalization_summary.csv",
        "C_source_target_pair_roles.csv",
        "D_fourclass_cue_dependence.csv",
        "D_class255_cue_dependence.csv",
        "E_fourclass_threshold_utility.csv",
        "E_severe_false_negative_routing.csv",
        "E_suicide_intent_risk_summary.csv",
        "E_top_risk_capture.csv",
        "F_fourclass_run_stability.csv",
        "F_class255_run_stability.csv",
    ]
})
table_index.to_csv(os.path.join(TAB_DIR, "TABLE_INDEX.csv"), index=False, encoding="utf-8-sig")

print("\nDone.")
print("Output root:", OUT_DIR)
print("Figures:", FIG_DIR)
print("Tables:", TAB_DIR)
print("Texts:", TXT_DIR)

[Saved] D:\Downloads\Depressed\Depressed\nature_hb_missing_layers\figures\FigA_construct_closure_centrality_bias_triptych.png
[Saved] D:\Downloads\Depressed\Depressed\nature_hb_missing_layers\figures\FigA_construct_closure_centrality_bias_triptych.eps
[Saved] D:\Downloads\Depressed\Depressed\nature_hb_missing_layers\figures\FigB_human_uncertainty_anchor.png
[Saved] D:\Downloads\Depressed\Depressed\nature_hb_missing_layers\figures\FigB_human_uncertainty_anchor.eps

[DEBUG] class255 staged summary:
       model_name  model_size_b                      run_name      stage_type  macro_f1  accuracy  delta_macro_f1_vs_direct  delta_accuracy_vs_direct  baseline_available
Qwen3.5-0.8B-Base           0.8    eval_2026-03-11-03-48-30_2      from2to255  0.053869  0.157837                       NaN                       NaN                   0
Qwen3.5-0.8B-Base           0.8    eval_2026-03-12-09-33-16_2      from2to255  0.047823  0.161148                       NaN                       NaN         

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")



ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"

FINAL_MAP_CSV = os.path.join(ROOT_DEPRESSED, "final_stage_mapping", "final_run_task_mapping.csv")
SUMMARY_CSV = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "result_summary.csv")
META_CSV = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "sample_meta.csv")
PRED_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "predictions")

OUT_DIR = os.path.join(ROOT_DEPRESSED, "final_all_in_one_outputs_v2")
FIG_DIR = os.path.join(OUT_DIR, "figures")
FIGCSV_DIR = os.path.join(OUT_DIR, "figure_csv")
TAB_DIR = os.path.join(OUT_DIR, "tables")
TXT_DIR = os.path.join(OUT_DIR, "texts")

for d in [OUT_DIR, FIG_DIR, FIGCSV_DIR, TAB_DIR, TXT_DIR]:
    os.makedirs(d, exist_ok=True)


EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

SHORT_NAMES = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#222222",
    "white": "#FFFFFF",
    "linegray": "#CFCBC5",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "deepblue": "#6EA9C4",
    "deeporange": "#D88537",
}

LEXICON = {
    "suicide_explicit": [
        "suicide", "suicidal", "kill myself", "end my life", "die", "dying",
        "don't want to live", "want to die", "want to disappear", "self harm", "self-harm"
    ],
    "hopelessness_explicit": [
        "hopeless", "no future", "nothing will change", "what's the point", "pointless", "can't go on"
    ],
    "sadness_explicit": [
        "sad", "cry", "crying", "empty", "miserable", "depressed", "down", "numb"
    ],
    "worthlessness_explicit": [
        "worthless", "useless", "burden", "hate myself", "failure", "not good enough"
    ],
    "cognitive_explicit": [
        "can't focus", "forget", "forgetting", "memory", "brain fog", "can't think", "concentrate"
    ],
    "first_person_dense": [
        " i ", " i'm ", " ive ", " i've ", " me ", " my ", " myself "
    ],
    "negation_markers": [
        "not", "never", "no ", "nothing", "nobody", "can't", "cannot", "won't", "don't", "doesn't"
    ]
}


def set_pub_style():
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "font.size": 9,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8,
        "axes.linewidth": 0.9,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.unicode_minus": False,
    })


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.9)
    ax.spines["bottom"].set_linewidth(0.9)
    ax.tick_params(axis="both", which="major", length=3.0, width=0.9, colors=PALETTE["black"])
    for lab in ax.get_xticklabels():
        lab.set_fontweight("bold")
    for lab in ax.get_yticklabels():
        lab.set_fontweight("bold")


def save_fig(fig, stem):
    png = os.path.join(FIG_DIR, f"{stem}.png")
    eps = os.path.join(FIG_DIR, f"{stem}.eps")
    fig.savefig(png, dpi=800, bbox_inches="tight", pad_inches=0.03)
    fig.savefig(eps, format="eps", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    print("[Saved]", png)
    print("[Saved]", eps)


def finite_text(ax, x, y, s, **kwargs):
    if np.isfinite(x) and np.isfinite(y):
        ax.text(x, y, s, **kwargs)


def add_clean_legend(ax, handles, loc="best", ncol=1, anchor=None):
    handles = [h for h in handles if h is not None]
    if len(handles) == 0:
        return
    kwargs = dict(frameon=False, loc=loc, ncol=ncol, handlelength=1.8, columnspacing=1.0, borderaxespad=0.3)
    if anchor is not None:
        kwargs["bbox_to_anchor"] = anchor
    leg = ax.legend(handles=handles, **kwargs)
    if leg is not None:
        for t in leg.get_texts():
            t.set_fontweight("bold")


def add_series_legend(ax, labels_colors, loc="best", anchor=None, ncol=1):
    handles = [Line2D([0], [0], color=color, lw=1.8, marker='o', markersize=5, label=label) for label, color in labels_colors]
    add_clean_legend(ax, handles, loc=loc, anchor=anchor, ncol=ncol)


def add_patch_legend(ax, labels_colors, loc="best", anchor=None, ncol=1):
    handles = [Patch(facecolor=color, edgecolor='none', label=label) for label, color in labels_colors]
    add_clean_legend(ax, handles, loc=loc, anchor=anchor, ncol=ncol)


def add_range_mean_legend(ax, color, loc="best", anchor=None):
    handles = [
        Line2D([0], [0], color=color, lw=2.0, label='Range across runs'),
        Line2D([0], [0], color=color, marker='o', linestyle='None', markersize=5, label='Mean')
    ]
    add_clean_legend(ax, handles, loc=loc, anchor=anchor)


set_pub_style()


def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df


def coalesce_columns(df, target_name, candidates):
    existing = [c for c in candidates if c in df.columns]
    if len(existing) == 0:
        df[target_name] = np.nan
        return df
    s = None
    for c in existing:
        if s is None:
            s = df[c]
        else:
            s = s.combine_first(df[c])
    df[target_name] = s
    return df


def parse_model_size(model_name):
    m = re.search(r"Qwen3\.5-(0\.8|2|4|9)B-Base", str(model_name))
    return float(m.group(1)) if m else np.nan


def normalize_text(x):
    return re.sub(r"\s+", " ", str(x)).strip().lower()


def qcut_safe(s, q):
    try:
        return pd.qcut(s, q=q, duplicates="drop")
    except Exception:
        return pd.cut(s.rank(method="average"), bins=q, duplicates="drop")


def correlation_pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    return np.corrcoef(x[mask], y[mask])[0, 1]


def normalize01(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.array([])
    lo, hi = np.min(x), np.max(x)
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def odds_ratio_from_table(a, b, c, d):
    a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    return (a * d) / (b * c)


def contains_any_phrase(text, phrases):
    t = f" {normalize_text(text)} "
    return int(any(p.lower() in t for p in phrases))


def first_person_density(text):
    t = f" {normalize_text(text)} "
    hits = sum(t.count(p) for p in LEXICON["first_person_dense"])
    n_words = max(1, len(t.split()))
    return hits / n_words


def negation_density(text):
    t = f" {normalize_text(text)} "
    hits = sum(t.count(p) for p in LEXICON["negation_markers"])
    n_words = max(1, len(t.split()))
    return hits / n_words


def parse_prediction_filename(fp):
    base = os.path.basename(fp)
    stem = os.path.splitext(base)[0]
    parts = stem.split("__")
    if len(parts) < 4:
        return None
    task = parts[0]
    model_name = parts[1]
    phase = parts[2]
    run_name = "__".join(parts[3:])
    return {
        "pred_file": fp,
        "task_pred": task,
        "model_name": model_name,
        "phase": phase,
        "run_name": run_name
    }


def bvec_to_list(s):
    s = str(s).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) == 8 and set(s) <= set("01"):
        return [int(c) for c in s]
    return None


def dec255_to_vec(x):
    try:
        if pd.isna(x):
            return None
        x = int(float(x))
        if x < 0 or x > 255:
            return None
        return [int(c) for c in format(x, "08b")]
    except Exception:
        return None


def vec_to_dec255(vec):
    try:
        if vec is None or len(vec) != 8:
            return np.nan
        return int("".join(str(int(v)) for v in vec), 2)
    except Exception:
        return np.nan


def robust_true_vec(true_vec_raw, true_255_raw):
    vec = bvec_to_list(true_vec_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "true_vec"
    vec = dec255_to_vec(true_255_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "true_255"
    return [0, 0, 0, 0, 0, 0, 0, 0], 0, "fallback_zero"


def robust_pred_vec(pred_vec_raw, pred_255_raw):
    vec = bvec_to_list(pred_vec_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "pred_vec"
    vec = dec255_to_vec(pred_255_raw)
    if vec is not None and len(vec) == 8:
        return vec, 1, "pred_255"
    return [0, 0, 0, 0, 0, 0, 0, 0], 0, "fallback_zero"


def vec_core_score(vec, core):
    if vec is None:
        return np.nan
    idx = [i for i, v in enumerate(vec) if v == 1]
    if len(idx) == 0:
        return np.nan
    return float(np.mean(core[idx]))


def top_log_odds_terms(text_pos, text_neg, topn=30):
    from sklearn.feature_extraction.text import CountVectorizer

    texts = list(text_pos) + list(text_neg)
    y = np.array([1] * len(text_pos) + [0] * len(text_neg))

    if len(text_pos) < 8 or len(text_neg) < 8:
        return pd.DataFrame(columns=["term", "log_odds"])

    vec = CountVectorizer(
        stop_words="english",
        min_df=2,
        max_features=4000,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b"
    )
    X = vec.fit_transform(texts)
    vocab = np.array(vec.get_feature_names_out())

    X_pos = X[y == 1].sum(axis=0).A1.astype(float)
    X_neg = X[y == 0].sum(axis=0).A1.astype(float)

    alpha = 1.0
    p_pos = (X_pos + alpha) / (X_pos.sum() + alpha * len(X_pos))
    p_neg = (X_neg + alpha) / (X_neg.sum() + alpha * len(X_neg))

    log_odds = np.log(p_pos) - np.log(p_neg)
    out = pd.DataFrame({"term": vocab, "log_odds": log_odds})
    return out.sort_values("log_odds", ascending=False).head(topn).reset_index(drop=True)


def save_plot_csv(df, stem):
    path = os.path.join(FIGCSV_DIR, f"{stem}.csv")
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("[Saved CSV]", path)


for p in [FINAL_MAP_CSV, SUMMARY_CSV, META_CSV]:
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Missing required file: {p}")
if not os.path.isdir(PRED_DIR):
    raise FileNotFoundError(f"Missing required directory: {PRED_DIR}")

map_df = pd.read_csv(FINAL_MAP_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)
meta_df = pd.read_csv(META_CSV, dtype={"sample_id": str})

pred_rows = []
for fp in sorted(glob.glob(os.path.join(PRED_DIR, "*.csv"))):
    x = parse_prediction_filename(fp)
    if x is not None:
        pred_rows.append(x)
pred_idx = pd.DataFrame(pred_rows)

if len(pred_idx) == 0:
    raise ValueError("No standardized prediction CSVs found in predictions/")

# normalize summary
sum_keep = summary_df.copy()
if "task" in sum_keep.columns:
    sum_keep = sum_keep.rename(columns={"task": "task_summary"})
if "setting" in sum_keep.columns:
    sum_keep = sum_keep.rename(columns={"setting": "phase"})

# normalize mapping
map_keep = map_df.copy()
map_keep["model_size_b"] = map_keep["model_name"].apply(parse_model_size)

# IMPORTANT: prediction inventory is the primary table
run_base = pred_idx.copy()

# merge summary
merged = run_base.merge(
    sum_keep,
    on=["model_name", "run_name", "phase"],
    how="left",
    suffixes=("", "_sum")
)

# merge final mapping as enrichment
merged = merged.merge(
    map_keep,
    on=["model_name", "run_name", "phase"],
    how="left",
    suffixes=("", "_map")
)

# coalesce metrics
merged = coalesce_columns(merged, "accuracy_final", ["accuracy", "accuracy_sum", "accuracy_x", "accuracy_y"])
merged = coalesce_columns(merged, "macro_f1_final", ["macro_f1", "macro_f1_sum", "macro_f1_x", "macro_f1_y"])
merged = coalesce_columns(merged, "micro_f1_final", ["micro_f1", "micro_f1_sum", "micro_f1_x", "micro_f1_y"])
merged = coalesce_columns(merged, "exact_match_final", ["exact_match", "exact_match_sum", "exact_match_x", "exact_match_y"])
merged = coalesce_columns(merged, "hamming_loss_final", ["hamming_loss", "hamming_loss_sum", "hamming_loss_x", "hamming_loss_y"])

# row-wise final task coalesce
merged = coalesce_columns(merged, "task_final_coalesced", ["final_task", "task_summary", "task_pred"])
merged["final_task"] = merged["task_final_coalesced"].fillna("unknown")

merged["model_size_b"] = merged["model_name"].apply(parse_model_size)
merged = merged[merged["pred_file"].notna()].copy()
merged = merged[merged["pred_file"].apply(lambda x: os.path.isfile(str(x)))].copy()

# subsets
binary_runs = merged[merged["final_task"] == "binary"].copy()
four_runs = merged[merged["final_task"] == "fourclass"].copy()
eight_runs = merged[merged["final_task"] == "eightlabel"].copy()
class255_runs = merged[merged["final_task"] == "class255"].copy()

print(f"[Info] binary runs found: {len(binary_runs)}")
print(f"[Info] fourclass runs found: {len(four_runs)}")
print(f"[Info] eightlabel runs found: {len(eight_runs)}")
print(f"[Info] class255 runs found: {len(class255_runs)}")

if len(four_runs) == 0:
    raise ValueError("No fourclass runs found.")
if len(class255_runs) == 0:
    raise ValueError("No class255 runs found.")

# best runs
best_four = four_runs.sort_values("macro_f1_final", ascending=False).iloc[0]
best_binary = binary_runs.sort_values("f1", ascending=False).iloc[0] if ("f1" in binary_runs.columns and len(binary_runs) > 0) else None
best_eight = eight_runs.sort_values("macro_f1_final", ascending=False).iloc[0] if len(eight_runs) > 0 else None
best_255 = class255_runs.sort_values("macro_f1_final", ascending=False).iloc[0]

# staged class255
def classify_class255_stage_type(row):
    phase = str(row.get("phase", ""))
    final_stage_name = str(row.get("final_stage_name", ""))
    final_stage_role = str(row.get("final_stage_role", ""))

    if phase == "native_base_eval":
        return "native_class255"
    if phase == "ft_eval":
        if final_stage_name == "from2to255" and final_stage_role == "target_task_eval":
            return "from2to255"
        if final_stage_name == "from4to255" and final_stage_role == "target_task_eval":
            return "from4to255"
        if (pd.isna(final_stage_name)) or (str(final_stage_name).strip() == ""):
            return "direct_class255"
    return "other"

class255_runs["stage_type"] = class255_runs.apply(classify_class255_stage_type, axis=1)
class255_runs = class255_runs[class255_runs["stage_type"].isin([
    "native_class255", "direct_class255", "from2to255", "from4to255"
])].copy()

# direct baseline by size if available
baseline_rows = []
direct_stage = class255_runs[class255_runs["stage_type"] == "direct_class255"].copy()
if len(direct_stage) > 0:
    for size, g in direct_stage.groupby("model_size_b"):
        g = g.sort_values("macro_f1_final", ascending=False).head(1)
        if len(g) > 0:
            baseline_rows.append(g.iloc[0].to_dict())
baseline_df = pd.DataFrame(baseline_rows)
baseline_df = ensure_cols(baseline_df, ["model_name", "model_size_b", "run_name", "macro_f1_final", "accuracy_final"])

# inventory save
class255_inventory = class255_runs[[
    "model_name", "model_size_b", "run_name", "phase", "stage_type",
    "final_stage_name", "final_stage_role",
    "accuracy_final", "macro_f1_final", "micro_f1_final",
    "exact_match_final", "hamming_loss_final", "pred_file"
]].sort_values(["model_size_b", "stage_type", "run_name"]).reset_index(drop=True)
class255_inventory.to_csv(os.path.join(TAB_DIR, "inventory_class255_stage_runs.csv"), index=False, encoding="utf-8-sig")

# save full run inventory too
merged[[
    "task_pred", "task_summary", "final_task",
    "model_name", "model_size_b", "run_name", "phase", "pred_file",
    "accuracy_final", "macro_f1_final", "micro_f1_final", "exact_match_final", "hamming_loss_final"
]].to_csv(os.path.join(TAB_DIR, "inventory_all_runs.csv"), index=False, encoding="utf-8-sig")


four_df = pd.read_csv(best_four["pred_file"], dtype={"sample_id": str})
ensure_cols(four_df, ["sample_id", "text", "true_label", "pred_label"])
four_df = four_df.merge(meta_df, on="sample_id", how="left", suffixes=("", "_meta"))
four_df["true_label"] = pd.to_numeric(four_df["true_label"], errors="coerce")
four_df["pred_label"] = pd.to_numeric(four_df["pred_label"], errors="coerce")
four_df["abs_err"] = (four_df["true_label"] - four_df["pred_label"]).abs()
four_df["is_error"] = (four_df["abs_err"] > 0).astype(int)
four_df["is_adjacent_error"] = (four_df["abs_err"] == 1).astype(int)
four_df["is_far_error"] = (four_df["abs_err"] >= 2).astype(int)
four_df["error_type"] = np.where(
    four_df["abs_err"] == 0, "Correct",
    np.where(four_df["abs_err"] == 1, "Adjacent", "Far")
)

class255_df = pd.read_csv(best_255["pred_file"], dtype={"sample_id": str})
ensure_cols(class255_df, ["sample_id", "text", "true_vec", "pred_vec", "true_255", "pred_255"])
class255_df["true_255"] = pd.to_numeric(class255_df["true_255"], errors="coerce")
class255_df["pred_255"] = pd.to_numeric(class255_df["pred_255"], errors="coerce")

tmp_true = class255_df.apply(lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)), axis=1)
class255_df["true_vec_list"] = [x[0] for x in tmp_true]
class255_df["true_vec_valid"] = [x[1] for x in tmp_true]
class255_df["true_vec_source"] = [x[2] for x in tmp_true]

tmp_pred = class255_df.apply(lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)), axis=1)
class255_df["pred_vec_list"] = [x[0] for x in tmp_pred]
class255_df["pred_vec_valid"] = [x[1] for x in tmp_pred]
class255_df["pred_vec_source"] = [x[2] for x in tmp_pred]

class255_df["true_255_filled"] = class255_df.apply(
    lambda r: r["true_255"] if pd.notna(r["true_255"]) else vec_to_dec255(r["true_vec_list"]), axis=1
)
class255_df["pred_255_filled"] = class255_df.apply(
    lambda r: r["pred_255"] if pd.notna(r["pred_255"]) else vec_to_dec255(r["pred_vec_list"]), axis=1
)

X_true = np.vstack(class255_df["true_vec_list"].tolist()).astype(int)
X_pred = np.vstack(class255_df["pred_vec_list"].tolist()).astype(int)
class255_df["correct"] = (class255_df["true_255_filled"] == class255_df["pred_255_filled"])
class255_df["k_true"] = X_true.sum(axis=1)
class255_df["k_pred"] = X_pred.sum(axis=1)

# best eightlabel if available
eight_df = None
Xe_true, Xe_pred = None, None
if best_eight is not None:
    eight_df = pd.read_csv(best_eight["pred_file"], dtype={"sample_id": str})
    ensure_cols(eight_df, ["sample_id", "text", "true_vec", "pred_vec", "true_255", "pred_255"])

    tmp_true = eight_df.apply(lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)), axis=1)
    eight_df["true_vec_list"] = [x[0] for x in tmp_true]
    eight_df["true_vec_valid"] = [x[1] for x in tmp_true]

    tmp_pred = eight_df.apply(lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)), axis=1)
    eight_df["pred_vec_list"] = [x[0] for x in tmp_pred]
    eight_df["pred_vec_valid"] = [x[1] for x in tmp_pred]

    Xe_true = np.vstack(eight_df["true_vec_list"].tolist()).astype(int)
    Xe_pred = np.vstack(eight_df["pred_vec_list"].tolist()).astype(int)


cm = confusion_matrix(four_df["true_label"], four_df["pred_label"], labels=[0, 1, 2, 3])
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

cm_df = pd.DataFrame(cm_norm, index=[0, 1, 2, 3], columns=[0, 1, 2, 3]).reset_index().rename(columns={"index": "true_label"})
save_plot_csv(cm_df, "Fig1A_fourclass_confusion_normalized")

fig, ax = plt.subplots(figsize=(3.6, 3.2))
cmap = LinearSegmentedColormap.from_list("cm", [PALETTE["white"], "#D9EEF8", PALETTE["blue"]])
im = ax.imshow(cm_norm, cmap=cmap, aspect="auto", vmin=0, vmax=np.max(cm_norm))
ax.set_xticks(np.arange(4), labels=["0", "1", "2", "3"])
ax.set_yticks(np.arange(4), labels=["0", "1", "2", "3"])
ax.set_xlabel("Predicted label", fontweight="bold")
ax.set_ylabel("True label", fontweight="bold")
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        finite_text(ax, j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center", fontsize=6.5, fontweight="bold")
style_ax(ax)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=7, width=0.8, length=2.5)
fig.tight_layout()
save_fig(fig, "Fig1A_fourclass_confusion_normalized")


adjacent_share = four_df["is_adjacent_error"].mean()
far_share = four_df["is_far_error"].mean()
correct_share = (four_df["abs_err"] == 0).mean()

adj_only = four_df[four_df["is_adjacent_error"] == 1].copy()
if len(adj_only) > 0:
    adj_only["boundary"] = adj_only.apply(
        lambda r: f"{int(min(r['true_label'], r['pred_label']))}↔{int(max(r['true_label'], r['pred_label']))}",
        axis=1
    )
    bshare = adj_only["boundary"].value_counts(normalize=True).rename_axis("boundary").reset_index(name="fraction")
else:
    bshare = pd.DataFrame(columns=["boundary", "fraction"])

error_struct_df = pd.DataFrame({
    "category": ["Correct", "Adjacent", "Far"],
    "fraction": [correct_share, adjacent_share, far_share]
})
save_plot_csv(error_struct_df, "Fig1B_fourclass_error_structure")
save_plot_csv(bshare, "Fig1B2_fourclass_adjacent_boundary_breakdown")

fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.8))
axes[0].bar(
    np.arange(3),
    error_struct_df["fraction"],
    color=[PALETTE["gray"], PALETTE["orange"], PALETTE["pink"]],
    edgecolor="none",
    width=0.62
)
axes[0].set_xticks(np.arange(3), labels=error_struct_df["category"].tolist())
axes[0].set_ylabel("Fraction", fontweight="bold")
style_ax(axes[0])
add_patch_legend(axes[0], [("Correct", PALETTE["gray"]), ("Adjacent", PALETTE["orange"]), ("Far", PALETTE["pink"])], loc="upper right")

if len(bshare) > 0:
    axes[1].bar(np.arange(len(bshare)), bshare["fraction"], color=PALETTE["blue"], edgecolor="none", width=0.62)
    axes[1].set_xticks(np.arange(len(bshare)), labels=bshare["boundary"].tolist())
    axes[1].set_ylabel("Fraction within adjacent errors", fontweight="bold")
    style_ax(axes[1])
else:
    axes[1].axis("off")

fig.tight_layout()
save_fig(fig, "Fig1B_fourclass_error_structure_and_boundary")


conf_df = pd.DataFrame()
if "confidence_score" in four_df.columns and four_df["confidence_score"].notna().sum() > 30:
    tmp = four_df.dropna(subset=["confidence_score"]).copy()
    tmp["conf_decile"] = qcut_safe(tmp["confidence_score"], q=10)
    conf_df = tmp.groupby("conf_decile").agg(
        n=("sample_id", "size"),
        conf_mean=("confidence_score", "mean"),
        error_rate=("is_error", "mean"),
        adjacent_rate=("is_adjacent_error", "mean"),
        far_rate=("is_far_error", "mean")
    ).reset_index(drop=True)
save_plot_csv(conf_df, "Fig1C_fourclass_confidence_anchor")

dis_df = pd.DataFrame()
or_df = pd.DataFrame()
if "is_disagreement_sample" in four_df.columns:
    tmp = four_df.copy()
    tmp["group"] = np.where(tmp["is_disagreement_sample"] == 1, "disagreement", "non_disagreement")
    dis_df = tmp.groupby("group").agg(
        n=("sample_id", "size"),
        error_rate=("is_error", "mean"),
        adjacent_rate=("is_adjacent_error", "mean"),
        far_rate=("is_far_error", "mean")
    ).reset_index()

    rows = []
    for col, label in [("is_error", "any_error"), ("is_adjacent_error", "adjacent_error"), ("is_far_error", "far_error")]:
        g1 = tmp[tmp["is_disagreement_sample"] == 1]
        g0 = tmp[tmp["is_disagreement_sample"] == 0]
        a = int(g1[col].sum())
        b = int(len(g1) - g1[col].sum())
        c = int(g0[col].sum())
        d = int(len(g0) - g0[col].sum())
        rows.append({
            "error_type": label,
            "odds_ratio_disagreement_vs_non": odds_ratio_from_table(a, b, c, d)
        })
    or_df = pd.DataFrame(rows)

save_plot_csv(dis_df, "Fig1C2_disagreement_enrichment")
save_plot_csv(or_df, "Fig1C3_disagreement_odds_ratio")

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0))
if len(conf_df) > 0:
    x = np.arange(len(conf_df))
    axes[0].plot(x, conf_df["error_rate"], marker="o", linewidth=1.5, color=PALETTE["orange"])
    axes[0].plot(x, conf_df["adjacent_rate"], marker="o", linewidth=1.3, color=PALETTE["yellow"])
    axes[0].plot(x, conf_df["far_rate"], marker="o", linewidth=1.3, color=PALETTE["pink"])
    axes[0].set_xticks(x, labels=[str(i + 1) for i in range(len(x))])
    axes[0].set_xlabel("Confidence decile (low → high)", fontweight="bold")
    axes[0].set_ylabel("Fraction", fontweight="bold")
    style_ax(axes[0])
    add_series_legend(axes[0], [("Any error", PALETTE["orange"]), ("Adjacent", PALETTE["yellow"]), ("Far", PALETTE["pink"])], loc="upper right")
else:
    axes[0].axis("off")

if len(or_df) > 0:
    x = np.arange(len(or_df))
    axes[1].bar(x, or_df["odds_ratio_disagreement_vs_non"], color=PALETTE["blue"], edgecolor="none", width=0.62)
    axes[1].axhline(1.0, color=PALETTE["linegray"], linewidth=1.0)
    axes[1].set_xticks(x, labels=["Any\nerror", "Adjacent", "Far"])
    axes[1].set_ylabel("Odds ratio\n(disagreement / non)", fontweight="bold")
    style_ax(axes[1])
else:
    axes[1].axis("off")
fig.tight_layout()
save_fig(fig, "Fig1C_human_uncertainty_anchor")


th_rows = []
for t in [1, 2, 3]:
    yt = (four_df["true_label"] >= t).astype(int)
    yp = (four_df["pred_label"] >= t).astype(int)
    th_rows.append({
        "threshold": f">={t}",
        "support_positive": int(yt.sum()),
        "accuracy": accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0)
    })
th_df = pd.DataFrame(th_rows)
save_plot_csv(th_df, "Fig1D_fourclass_threshold_utility")

severe = four_df[four_df["true_label"] == 3].copy()
sev_route = severe.groupby("pred_label").size().reset_index(name="count")
if len(sev_route) > 0:
    sev_route["fraction"] = sev_route["count"] / sev_route["count"].sum()
save_plot_csv(sev_route, "Fig1D2_severe_routing")

fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.9))
x = np.arange(len(th_df))
axes[0].plot(x, th_df["precision"], marker="o", linewidth=1.4, color=PALETTE["orange"])
axes[0].plot(x, th_df["recall"], marker="o", linewidth=1.4, color=PALETTE["blue"])
axes[0].plot(x, th_df["f1"], marker="o", linewidth=1.4, color=PALETTE["green"])
axes[0].set_xticks(x, labels=th_df["threshold"].tolist())
axes[0].set_xlabel("Positive definition", fontweight="bold")
axes[0].set_ylabel("Score", fontweight="bold")
style_ax(axes[0])
add_series_legend(axes[0], [("Precision", PALETTE["orange"]), ("Recall", PALETTE["blue"]), ("F1", PALETTE["green"])], loc="lower left")

if len(sev_route) > 0:
    axes[1].bar(np.arange(len(sev_route)), sev_route["fraction"], color=PALETTE["pink"], edgecolor="none", width=0.62)
    axes[1].set_xticks(np.arange(len(sev_route)), labels=sev_route["pred_label"].astype(int).astype(str).tolist())
    axes[1].set_xlabel("Predicted label when true=3", fontweight="bold")
    axes[1].set_ylabel("Fraction", fontweight="bold")
    style_ax(axes[1])
else:
    axes[1].axis("off")
fig.tight_layout()
save_fig(fig, "Fig1D_fourclass_risk_sensitive_utility")

# Prefer eightlabel for symptom-space validation if available; otherwise fallback to class255
if eight_df is not None:
    Xs_true, Xs_pred = Xe_true, Xe_pred
    symptom_source = "eightlabel"
else:
    Xs_true, Xs_pred = X_true, X_pred
    symptom_source = "class255"

cooc_true = (Xs_true.T @ Xs_true) / Xs_true.shape[0]
cooc_off = cooc_true.copy()
np.fill_diagonal(cooc_off, 0.0)
centrality = normalize01(cooc_off.sum(axis=1))

sym_rows = []
for i, name in enumerate(EMOTION_NAMES):
    yt = Xs_true[:, i]
    yp = Xs_pred[:, i]
    sym_rows.append({
        "source_task": symptom_source,
        "symptom": name,
        "symptom_short": SHORT_NAMES[name],
        "support": int(yt.sum()),
        "true_prev": yt.mean(),
        "pred_prev": yp.mean(),
        "prev_bias": yp.mean() - yt.mean(),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "omission_rate": ((yt == 1) & (yp == 0)).sum() / max(1, (yt == 1).sum()),
        "commission_rate": ((yt == 0) & (yp == 1)).sum() / max(1, (yt == 0).sum()),
        "centrality": centrality[i]
    })
symptom_df = pd.DataFrame(sym_rows)
save_plot_csv(symptom_df, "Fig2_symptom_structure_metrics")

# 2A omission vs centrality
fig, ax = plt.subplots(figsize=(3.4, 3.0))
x = symptom_df["centrality"].to_numpy()
y = symptom_df["omission_rate"].to_numpy()
ax.scatter(x, y, s=72, color=PALETTE["orange"], edgecolors="white", linewidths=0.7)
for _, r in symptom_df.iterrows():
    finite_text(ax, r["centrality"] + 0.015, r["omission_rate"], r["symptom_short"], fontsize=6.0, fontweight="bold")
ax.text(0.98, 0.96, f"r={correlation_pearson(x, y):.2f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=6.8, fontweight="bold")
ax.set_xlabel("Symptom centrality", fontweight="bold")
ax.set_ylabel("Omission rate", fontweight="bold")
style_ax(ax)
fig.tight_layout()
save_fig(fig, "Fig2A_centrality_vs_omission")

# 2B commission vs centrality
fig, ax = plt.subplots(figsize=(3.4, 3.0))
x = symptom_df["centrality"].to_numpy()
y = symptom_df["commission_rate"].to_numpy()
ax.scatter(x, y, s=72, color=PALETTE["pink"], edgecolors="white", linewidths=0.7)
for _, r in symptom_df.iterrows():
    finite_text(ax, r["centrality"] + 0.015, r["commission_rate"], r["symptom_short"], fontsize=6.0, fontweight="bold")
ax.text(0.98, 0.96, f"r={correlation_pearson(x, y):.2f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=6.8, fontweight="bold")
ax.set_xlabel("Symptom centrality", fontweight="bold")
ax.set_ylabel("Commission rate", fontweight="bold")
style_ax(ax)
fig.tight_layout()
save_fig(fig, "Fig2B_centrality_vs_commission")

# 2C prevalence bias vs centrality
fig, ax = plt.subplots(figsize=(3.4, 3.0))
x = symptom_df["centrality"].to_numpy()
y = symptom_df["prev_bias"].to_numpy()
ax.scatter(x, y, s=72, color=PALETTE["blue"], edgecolors="white", linewidths=0.7)
for _, r in symptom_df.iterrows():
    finite_text(ax, r["centrality"] + 0.015, r["prev_bias"], r["symptom_short"], fontsize=6.0, fontweight="bold")
ax.axhline(0, color=PALETTE["linegray"], linewidth=1.0)
ax.text(0.98, 0.96, f"r={correlation_pearson(x, y):.2f}", transform=ax.transAxes,
        ha="right", va="top", fontsize=6.8, fontweight="bold")
ax.set_xlabel("Symptom centrality", fontweight="bold")
ax.set_ylabel("Predicted prevalence − true prevalence", fontweight="bold")
style_ax(ax)
fig.tight_layout()
save_fig(fig, "Fig2C_centrality_vs_prevalence_bias")

# 2D per-symptom F1
symptom_plot_df = symptom_df.sort_values("f1", ascending=True).reset_index(drop=True)
save_plot_csv(symptom_plot_df, "Fig2D_per_symptom_f1")
fig, ax = plt.subplots(figsize=(4.6, 3.3))
ax.barh(np.arange(len(symptom_plot_df)), symptom_plot_df["f1"], color=PALETTE["green"], edgecolor="none", height=0.66)
ax.set_yticks(np.arange(len(symptom_plot_df)), labels=symptom_plot_df["symptom_short"].tolist())
ax.set_xlabel("F1", fontweight="bold")
style_ax(ax)
fig.tight_layout()
save_fig(fig, "Fig2D_per_symptom_f1")

# Optional eightlabel-specific co-occurrence heatmap
if eight_df is not None:
    cooc_df = pd.DataFrame(cooc_true, index=[SHORT_NAMES[x] for x in EMOTION_NAMES], columns=[SHORT_NAMES[x] for x in EMOTION_NAMES])
    cooc_save = cooc_df.reset_index().rename(columns={"index": "symptom"})
    save_plot_csv(cooc_save, "Fig2E_eightlabel_true_cooccurrence")
    fig, ax = plt.subplots(figsize=(4.6, 3.9))
    cmap = LinearSegmentedColormap.from_list("cooc", [PALETTE["white"], "#FDE1CC", PALETTE["orange"]])
    im = ax.imshow(cooc_true, cmap=cmap, aspect="auto", vmin=0, vmax=np.max(cooc_true))
    ax.set_xticks(np.arange(len(EMOTION_NAMES)), labels=[SHORT_NAMES[x] for x in EMOTION_NAMES], rotation=35, ha="right")
    ax.set_yticks(np.arange(len(EMOTION_NAMES)), labels=[SHORT_NAMES[x] for x in EMOTION_NAMES])
    style_ax(ax)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=7, width=0.8, length=2.5)
    fig.tight_layout()
    save_fig(fig, "Fig2E_eightlabel_true_cooccurrence")



kacc_df = class255_df.groupby("k_true").agg(
    n=("sample_id", "size"),
    exact_acc=("correct", "mean")
).reset_index()
save_plot_csv(kacc_df, "Fig3A_class255_accuracy_by_symptom_count")

fig, ax = plt.subplots(figsize=(4.0, 3.0))
ax.plot(kacc_df["k_true"], kacc_df["exact_acc"], marker="o", linewidth=1.5, color=PALETTE["orange"])
ax.set_xlabel("No. of symptoms in true combination", fontweight="bold")
ax.set_ylabel("Exact accuracy", fontweight="bold")
style_ax(ax)
fig.tight_layout()
save_fig(fig, "Fig3A_class255_accuracy_by_symptom_count")

c_rows = []
for _, r in class255_runs.iterrows():
    size = r["model_size_b"]
    st = r["stage_type"]
    out = {
        "model_name": r["model_name"],
        "model_size_b": size,
        "run_name": r["run_name"],
        "stage_type": st,
        "macro_f1": r.get("macro_f1_final", np.nan),
        "accuracy": r.get("accuracy_final", np.nan)
    }

    if len(baseline_df) > 0 and "model_size_b" in baseline_df.columns:
        base = baseline_df[baseline_df["model_size_b"] == size]
    else:
        base = pd.DataFrame()

    if len(base) > 0 and st != "direct_class255":
        base = base.iloc[0]
        out["delta_macro_f1_vs_direct"] = r.get("macro_f1_final", np.nan) - base.get("macro_f1_final", np.nan)
        out["delta_accuracy_vs_direct"] = r.get("accuracy_final", np.nan) - base.get("accuracy_final", np.nan)
        out["baseline_available"] = 1
    else:
        out["delta_macro_f1_vs_direct"] = np.nan
        out["delta_accuracy_vs_direct"] = np.nan
        out["baseline_available"] = 0

    c_rows.append(out)

c_stage_df = pd.DataFrame(c_rows)
c_stage_df = ensure_cols(c_stage_df, [
    "model_name", "model_size_b", "run_name", "stage_type",
    "macro_f1", "accuracy", "delta_macro_f1_vs_direct",
    "delta_accuracy_vs_direct", "baseline_available"
])
save_plot_csv(c_stage_df, "Fig3_stage_transfer_summary")

# 3B performance matrix
order = ["native_class255", "direct_class255", "from2to255", "from4to255"]
mat = []
for st in order:
    dd = c_stage_df[c_stage_df["stage_type"] == st].copy()
    if len(dd) > 0:
        mat.append([dd["macro_f1"].mean(), dd["accuracy"].mean()])
    else:
        mat.append([np.nan, np.nan])
mat = np.array(mat).T
mat_df = pd.DataFrame(mat, index=["Macro-F1", "Accuracy"], columns=order).reset_index().rename(columns={"index": "metric"})
save_plot_csv(mat_df, "Fig3B_stage_performance_matrix")

fig, ax = plt.subplots(figsize=(4.6, 3.3))
valid = mat[np.isfinite(mat)]
vmin = np.nanmin(valid) if len(valid) > 0 else 0
vmax = np.nanmax(valid) if len(valid) > 0 else 1
cmap = LinearSegmentedColormap.from_list("stageperf", [PALETTE["white"], "#D9EEF8", PALETTE["blue"]])
im = ax.imshow(mat, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
ax.set_xticks(np.arange(len(order)), labels=[s.replace("_", "\n") for s in order])
ax.set_yticks([0, 1], labels=["Macro-F1", "Accuracy"])
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if np.isfinite(mat[i, j]):
            finite_text(ax, j, i, f"{mat[i,j]:.3f}", ha="center", va="center", fontsize=6.4, fontweight="bold")
style_ax(ax)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=7, width=0.8, length=2.5)
fig.tight_layout()
save_fig(fig, "Fig3B_stage_performance_matrix")

# 3C gain vs direct
tmp = c_stage_df[c_stage_df["stage_type"].isin(["from2to255", "from4to255"])].copy()
tmp = tmp[tmp["baseline_available"] == 1].copy()
gain_df = pd.DataFrame()
if len(tmp) > 0:
    gain_df = tmp.groupby("stage_type").agg(
        mean_delta_macro_f1=("delta_macro_f1_vs_direct", "mean"),
        mean_delta_acc=("delta_accuracy_vs_direct", "mean"),
        n=("run_name", "size")
    ).reset_index()
save_plot_csv(gain_df, "Fig3C_stage_gain_vs_direct")

fig, ax = plt.subplots(figsize=(4.4, 3.1))
if len(gain_df) > 0:
    x = np.arange(len(gain_df))
    ax.bar(x - 0.14, gain_df["mean_delta_macro_f1"], width=0.28, color=PALETTE["orange"], edgecolor="none")
    ax.bar(x + 0.14, gain_df["mean_delta_acc"], width=0.28, color=PALETTE["blue"], edgecolor="none")
    ax.axhline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xticks(x, labels=gain_df["stage_type"].tolist())
    ax.set_ylabel("Δ vs direct class255", fontweight="bold")
    style_ax(ax)
    add_patch_legend(ax, [("Macro-F1 gain", PALETTE["orange"]), ("Accuracy gain", PALETTE["blue"])], loc="upper right")
    fig.tight_layout()
    save_fig(fig, "Fig3C_stage_gain_vs_direct")
else:
    plt.close(fig)

# 3D scaling
scale_df = c_stage_df.copy()
save_plot_csv(scale_df, "Fig3D_stage_scaling")

fig, ax = plt.subplots(figsize=(4.4, 3.1))
drawn = False
for st, color in [("direct_class255", PALETTE["gray"]), ("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
    dd = scale_df[scale_df["stage_type"] == st].copy()
    if len(dd) == 0:
        continue
    agg = dd.groupby("model_size_b").agg(mean_macro_f1=("macro_f1", "mean")).reset_index()
    if len(agg) == 0:
        continue
    ax.plot(agg["model_size_b"], agg["mean_macro_f1"], marker="o", linewidth=1.5, color=color, label=st.replace("_", " "))
    drawn = True
if drawn:
    ax.set_xlabel("Model size (B)", fontweight="bold")
    ax.set_ylabel("Macro-F1", fontweight="bold")
    style_ax(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    save_fig(fig, "Fig3D_stage_scaling")
else:
    plt.close(fig)

four_stress = four_df.copy()
four_stress["text"] = four_stress["text"].astype(str)

for cue_name, phrases in LEXICON.items():
    if cue_name in ["first_person_dense", "negation_markers"]:
        continue
    four_stress[f"cue__{cue_name}"] = four_stress["text"].apply(lambda x: contains_any_phrase(x, phrases))

four_stress["first_person_density"] = four_stress["text"].apply(first_person_density)
four_stress["negation_density"] = four_stress["text"].apply(negation_density)
four_stress["high_first_person"] = (four_stress["first_person_density"] >= four_stress["first_person_density"].median()).astype(int)
four_stress["high_negation"] = (four_stress["negation_density"] >= four_stress["negation_density"].median()).astype(int)

d_rows = []
cue_cols = [c for c in four_stress.columns if c.startswith("cue__")] + ["high_first_person", "high_negation"]
for c in cue_cols:
    for group, g in [("with_cue", four_stress[four_stress[c] == 1]), ("without_cue", four_stress[four_stress[c] == 0])]:
        if len(g) == 0:
            continue
        d_rows.append({
            "cue": c.replace("cue__", ""),
            "group": group,
            "n": len(g),
            "error_rate": g["is_error"].mean(),
            "adjacent_rate": g["is_adjacent_error"].mean(),
            "far_rate": g["is_far_error"].mean(),
            "mean_true_label": g["true_label"].mean()
        })
four_stress_df = pd.DataFrame(d_rows)
save_plot_csv(four_stress_df, "Fig4A_fourclass_cue_dependence")

class255_stress = class255_df.copy()
class255_stress["text"] = class255_stress["text"].astype(str)
for cue_name, phrases in LEXICON.items():
    if cue_name in ["first_person_dense", "negation_markers"]:
        continue
    class255_stress[f"cue__{cue_name}"] = class255_stress["text"].apply(lambda x: contains_any_phrase(x, phrases))
class255_stress["first_person_density"] = class255_stress["text"].apply(first_person_density)
class255_stress["negation_density"] = class255_stress["text"].apply(negation_density)
class255_stress["high_first_person"] = (class255_stress["first_person_density"] >= class255_stress["first_person_density"].median()).astype(int)
class255_stress["high_negation"] = (class255_stress["negation_density"] >= class255_stress["negation_density"].median()).astype(int)

idx_suicide = EMOTION_NAMES.index("suicide intent")
class255_stress["true_suicide"] = X_true[:, idx_suicide]
class255_stress["pred_suicide"] = X_pred[:, idx_suicide]
class255_stress["suicide_fn"] = ((class255_stress["true_suicide"] == 1) & (class255_stress["pred_suicide"] == 0)).astype(int)

d2_rows = []
cue_cols2 = [c for c in class255_stress.columns if c.startswith("cue__")] + ["high_first_person", "high_negation"]
for c in cue_cols2:
    for group, g in [("with_cue", class255_stress[class255_stress[c] == 1]), ("without_cue", class255_stress[class255_stress[c] == 0])]:
        if len(g) == 0:
            continue
        d2_rows.append({
            "cue": c.replace("cue__", ""),
            "group": group,
            "n": len(g),
            "exact_acc": g["correct"].mean(),
            "mean_k_true": g["k_true"].mean(),
            "suicide_fn_rate": g[g["true_suicide"] == 1]["suicide_fn"].mean() if (g["true_suicide"] == 1).sum() > 0 else np.nan
        })
class255_stress_df = pd.DataFrame(d2_rows)
save_plot_csv(class255_stress_df, "Fig4B_class255_cue_dependence")

selected_cues = ["suicide_explicit", "hopelessness_explicit", "worthlessness_explicit", "high_first_person", "high_negation"]

fig, ax = plt.subplots(figsize=(4.4, 3.3))
tmp = four_stress_df[four_stress_df["cue"].isin(selected_cues)].copy()
if len(tmp) > 0:
    pivot = tmp.pivot(index="cue", columns="group", values="error_rate").reset_index()
    if "with_cue" in pivot.columns and "without_cue" in pivot.columns:
        pivot["delta"] = pivot["with_cue"] - pivot["without_cue"]
        pivot = pivot.sort_values("delta", ascending=True).reset_index(drop=True)
        ax.barh(np.arange(len(pivot)), pivot["delta"], color=PALETTE["orange"], edgecolor="none", height=0.66)
        ax.set_yticks(np.arange(len(pivot)), labels=pivot["cue"].tolist())
        ax.set_xlabel("Δ fourclass error rate\n(with cue − without)", fontweight="bold")
        style_ax(ax)
        fig.tight_layout()
        save_fig(fig, "Fig4A_fourclass_cue_dependence")
    else:
        plt.close(fig)
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(4.4, 3.3))
tmp = class255_stress_df[class255_stress_df["cue"].isin(selected_cues)].copy()
if len(tmp) > 0:
    pivot = tmp.pivot(index="cue", columns="group", values="suicide_fn_rate").reset_index()
    if "with_cue" in pivot.columns and "without_cue" in pivot.columns:
        pivot["delta"] = pivot["with_cue"] - pivot["without_cue"]
        pivot = pivot.sort_values("delta", ascending=True).reset_index(drop=True)
        ax.barh(np.arange(len(pivot)), pivot["delta"], color=PALETTE["red"], edgecolor="none", height=0.66)
        ax.set_yticks(np.arange(len(pivot)), labels=pivot["cue"].tolist())
        ax.set_xlabel("Δ suicide false-negative rate\n(with cue − without)", fontweight="bold")
        style_ax(ax)
        fig.tight_layout()
        save_fig(fig, "Fig4B_class255_cue_dependence")
    else:
        plt.close(fig)
else:
    plt.close(fig)

def compute_fourclass_key_metrics(pred_file):
    d = pd.read_csv(pred_file, dtype={"sample_id": str})
    ensure_cols(d, ["true_label", "pred_label"])
    d["true_label"] = pd.to_numeric(d["true_label"], errors="coerce")
    d["pred_label"] = pd.to_numeric(d["pred_label"], errors="coerce")
    d["abs_err"] = (d["true_label"] - d["pred_label"]).abs()
    adj_frac = (d["abs_err"] == 1).mean()
    far_frac = (d["abs_err"] >= 2).mean()

    adj = d[d["abs_err"] == 1].copy()
    if len(adj) > 0:
        adj["boundary"] = adj.apply(
            lambda r: f"{int(min(r['true_label'], r['pred_label']))}↔{int(max(r['true_label'], r['pred_label']))}",
            axis=1
        )
        share_12 = (adj["boundary"] == "1↔2").mean()
    else:
        share_12 = np.nan

    return {
        "adjacent_error_frac": adj_frac,
        "far_error_frac": far_frac,
        "share_12_within_adjacent": share_12,
    }


def compute_multilabel_key_metrics(pred_file):
    d = pd.read_csv(pred_file, dtype={"sample_id": str})
    ensure_cols(d, ["true_vec", "pred_vec", "true_255", "pred_255"])

    tmp_true = d.apply(lambda r: robust_true_vec(r.get("true_vec", ""), r.get("true_255", np.nan)), axis=1)
    tmp_pred = d.apply(lambda r: robust_pred_vec(r.get("pred_vec", ""), r.get("pred_255", np.nan)), axis=1)

    X_t = np.vstack([x[0] for x in tmp_true]).astype(int)
    X_p = np.vstack([x[0] for x in tmp_pred]).astype(int)

    cooc = (X_t.T @ X_t) / X_t.shape[0]
    off = cooc.copy()
    np.fill_diagonal(off, 0.0)
    cent = normalize01(off.sum(axis=1))

    true_core = [vec_core_score(v, cent) for v in [x[0] for x in tmp_true]]
    pred_core = [vec_core_score(v, cent) for v in [x[0] for x in tmp_pred]]
    core_shift = np.array(pred_core) - np.array(true_core)

    idx_sad = EMOTION_NAMES.index("sadness")
    idx_brain = EMOTION_NAMES.index("brain dysfunction (forget)")
    yt_sad, yp_sad = X_t[:, idx_sad], X_p[:, idx_sad]
    yt_br, yp_br = X_t[:, idx_brain], X_p[:, idx_brain]

    pred_sources = pd.Series([x[2] for x in tmp_pred])

    return {
        "mean_core_shift": np.nanmean(core_shift),
        "brain_omission": ((yt_br == 1) & (yp_br == 0)).sum() / max(1, (yt_br == 1).sum()),
        "sadness_commission": ((yt_sad == 0) & (yp_sad == 1)).sum() / max(1, (yt_sad == 0).sum()),
        "pred_vec_fallback_zero_rate": (pred_sources == "fallback_zero").mean(),
    }


# fourclass stability
four_stab_rows = []
for _, r in four_runs.iterrows():
    pf = r["pred_file"]
    if not os.path.isfile(str(pf)):
        continue
    met = compute_fourclass_key_metrics(pf)
    four_stab_rows.append({
        "model_name": r["model_name"],
        "model_size_b": parse_model_size(r["model_name"]),
        "run_name": r["run_name"],
        "phase": r["phase"],
        "macro_f1": r.get("macro_f1_final", np.nan),
        **met
    })
four_stab_df = pd.DataFrame(four_stab_rows)
save_plot_csv(four_stab_df, "Fig4C_fourclass_run_stability")
four_stab_df.to_csv(os.path.join(TAB_DIR, "fourclass_run_stability.csv"), index=False, encoding="utf-8-sig")

# class255 stability
class255_stab_rows = []
for _, r in class255_runs.iterrows():
    pf = r["pred_file"]
    if not os.path.isfile(str(pf)):
        continue
    met = compute_multilabel_key_metrics(pf)
    class255_stab_rows.append({
        "model_name": r["model_name"],
        "model_size_b": parse_model_size(r["model_name"]),
        "run_name": r["run_name"],
        "phase": r["phase"],
        "stage_type": r["stage_type"],
        "macro_f1": r.get("macro_f1_final", np.nan),
        **met
    })
class255_stab_df = pd.DataFrame(class255_stab_rows)
save_plot_csv(class255_stab_df, "Fig4D_class255_run_stability")
class255_stab_df.to_csv(os.path.join(TAB_DIR, "class255_run_stability.csv"), index=False, encoding="utf-8-sig")

# eightlabel stability
eight_stab_df = pd.DataFrame()
if len(eight_runs) > 0:
    eight_stab_rows = []
    for _, r in eight_runs.iterrows():
        pf = r["pred_file"]
        if not os.path.isfile(str(pf)):
            continue
        met = compute_multilabel_key_metrics(pf)
        eight_stab_rows.append({
            "model_name": r["model_name"],
            "model_size_b": parse_model_size(r["model_name"]),
            "run_name": r["run_name"],
            "phase": r["phase"],
            "macro_f1": r.get("macro_f1_final", np.nan),
            **met
        })
    eight_stab_df = pd.DataFrame(eight_stab_rows)
    save_plot_csv(eight_stab_df, "Fig4E_eightlabel_run_stability")
    eight_stab_df.to_csv(os.path.join(TAB_DIR, "eightlabel_run_stability.csv"), index=False, encoding="utf-8-sig")

# 4C plot
fig, ax = plt.subplots(figsize=(4.6, 3.3))
if len(four_stab_df) > 0:
    agg = four_stab_df.groupby("model_size_b").agg(
        mean_adj=("adjacent_error_frac", "mean"),
        min_adj=("adjacent_error_frac", "min"),
        max_adj=("adjacent_error_frac", "max"),
    ).reset_index()
    for _, r in agg.iterrows():
        ax.plot([r["min_adj"], r["max_adj"]], [r["model_size_b"], r["model_size_b"]], color=PALETTE["orange"], linewidth=2.0)
        ax.scatter(r["mean_adj"], r["model_size_b"], s=42, color=PALETTE["orange"], zorder=3)
    ax.set_xlabel("Adjacent-error fraction", fontweight="bold")
    ax.set_ylabel("Model size (B)", fontweight="bold")
    style_ax(ax)
    add_range_mean_legend(ax, PALETTE["orange"], loc="lower right")
    fig.tight_layout()
    save_fig(fig, "Fig4C_fourclass_run_stability")
else:
    plt.close(fig)

# 4D plot
fig, ax = plt.subplots(figsize=(4.6, 3.3))
if len(class255_stab_df) > 0:
    for st, color in [("direct_class255", PALETTE["gray"]), ("from2to255", PALETTE["orange"]), ("from4to255", PALETTE["pink"])]:
        dd = class255_stab_df[class255_stab_df["stage_type"] == st].copy()
        if len(dd) == 0:
            continue
        ax.scatter(dd["mean_core_shift"], dd["model_size_b"], s=52, color=color, label=st)
    ax.axvline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xlabel("Mean core-shift", fontweight="bold")
    ax.set_ylabel("Model size (B)", fontweight="bold")
    style_ax(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    save_fig(fig, "Fig4D_class255_run_stability")
else:
    plt.close(fig)

# 4E plot for eightlabel
if len(eight_stab_df) > 0:
    fig, ax = plt.subplots(figsize=(4.6, 3.3))
    ax.scatter(eight_stab_df["mean_core_shift"], eight_stab_df["macro_f1"], s=54, color=PALETTE["blue"])
    ax.axvline(0, color=PALETTE["linegray"], linewidth=1.0)
    ax.set_xlabel("Mean core-shift", fontweight="bold")
    ax.set_ylabel("Macro-F1", fontweight="bold")
    style_ax(ax)
    add_patch_legend(ax, [("Eightlabel runs", PALETTE["blue"])], loc="lower right")
    fig.tight_layout()
    save_fig(fig, "Fig4E_eightlabel_run_stability")

report = classification_report(
    four_df["true_label"],
    four_df["pred_label"],
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})
report_df.to_csv(os.path.join(TAB_DIR, "best_fourclass_classification_report.csv"), index=False, encoding="utf-8-sig")

wrong = class255_df[class255_df["correct"] == False].copy()
conf_pairs_df = (
    wrong.groupby(["true_255_filled", "pred_255_filled"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
conf_pairs_df.to_csv(os.path.join(TAB_DIR, "best_class255_confusion_pairs.csv"), index=False, encoding="utf-8-sig")

# eightlabel per-symptom report
if eight_df is not None:
    e_rows = []
    for i, name in enumerate(EMOTION_NAMES):
        yt = Xe_true[:, i]
        yp = Xe_pred[:, i]
        e_rows.append({
            "symptom": name,
            "support": int(yt.sum()),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "f1": f1_score(yt, yp, zero_division=0),
        })
    eight_sym_report = pd.DataFrame(e_rows)
    eight_sym_report.to_csv(os.path.join(TAB_DIR, "best_eightlabel_per_symptom_report.csv"), index=False, encoding="utf-8-sig")

idx_suicide = EMOTION_NAMES.index("suicide intent")
suicide_df = pd.DataFrame({
    "true_suicide": X_true[:, idx_suicide],
    "pred_suicide": X_pred[:, idx_suicide]
})
suicide_df["false_negative"] = ((suicide_df["true_suicide"] == 1) & (suicide_df["pred_suicide"] == 0)).astype(int)
suicide_summary = pd.DataFrame([{
    "support_positive": int(suicide_df["true_suicide"].sum()),
    "precision": precision_score(suicide_df["true_suicide"], suicide_df["pred_suicide"], zero_division=0),
    "recall": recall_score(suicide_df["true_suicide"], suicide_df["pred_suicide"], zero_division=0),
    "f1": f1_score(suicide_df["true_suicide"], suicide_df["pred_suicide"], zero_division=0),
    "false_negative_rate": suicide_df[suicide_df["true_suicide"] == 1]["false_negative"].mean() if suicide_df["true_suicide"].sum() > 0 else np.nan
}])
suicide_summary.to_csv(os.path.join(TAB_DIR, "suicide_intent_risk_summary.csv"), index=False, encoding="utf-8-sig")

class255_df["risk_score_pred"] = class255_df["k_pred"] + X_pred[:, idx_suicide] * 2.0
capture_rows = []
for frac in [0.05, 0.10, 0.20]:
    n_top = max(1, int(len(class255_df) * frac))
    top_idx = class255_df.sort_values("risk_score_pred", ascending=False).head(n_top).index
    capture_rows.append({
        "top_fraction": frac,
        "n_top": n_top,
        "suicide_positive_rate_in_top": X_true[:, idx_suicide][top_idx].mean(),
        "high_burden_rate_in_top": (class255_df.loc[top_idx, "k_true"] >= 4).mean()
    })
capture_df = pd.DataFrame(capture_rows)
capture_df.to_csv(os.path.join(TAB_DIR, "top_risk_capture.csv"), index=False, encoding="utf-8-sig")

lex_1up = top_log_odds_terms(
    four_df[(four_df["true_label"] == 1) & (four_df["pred_label"] == 2)]["text"].astype(str).tolist(),
    four_df[(four_df["true_label"] == 1) & (four_df["pred_label"] == 1)]["text"].astype(str).tolist(),
    topn=30
)
lex_2down = top_log_odds_terms(
    four_df[(four_df["true_label"] == 2) & (four_df["pred_label"] == 1)]["text"].astype(str).tolist(),
    four_df[(four_df["true_label"] == 2) & (four_df["pred_label"] == 2)]["text"].astype(str).tolist(),
    topn=30
)
lex_1up.to_csv(os.path.join(TAB_DIR, "boundary_lexicon_true1_to_pred2.csv"), index=False, encoding="utf-8-sig")
lex_2down.to_csv(os.path.join(TAB_DIR, "boundary_lexicon_true2_to_pred1.csv"), index=False, encoding="utf-8-sig")

# save best-run registry
best_rows = []
if best_binary is not None:
    best_rows.append({
        "task": "binary",
        "model_name": best_binary["model_name"],
        "run_name": best_binary["run_name"],
        "phase": best_binary["phase"],
        "pred_file": best_binary["pred_file"],
    })
best_rows.append({
    "task": "fourclass",
    "model_name": best_four["model_name"],
    "run_name": best_four["run_name"],
    "phase": best_four["phase"],
    "pred_file": best_four["pred_file"],
})
if best_eight is not None:
    best_rows.append({
        "task": "eightlabel",
        "model_name": best_eight["model_name"],
        "run_name": best_eight["run_name"],
        "phase": best_eight["phase"],
        "pred_file": best_eight["pred_file"],
    })
best_rows.append({
    "task": "class255",
    "model_name": best_255["model_name"],
    "run_name": best_255["run_name"],
    "phase": best_255["phase"],
    "pred_file": best_255["pred_file"],
})
pd.DataFrame(best_rows).to_csv(os.path.join(TAB_DIR, "best_run_registry.csv"), index=False, encoding="utf-8-sig")

# summary note
lines = []
lines.append("Final all-in-one analysis summary v2")
lines.append("")
lines.append("Best runs")
if best_binary is not None:
    lines.append(f"Best binary: {best_binary['model_name']} | {best_binary['run_name']}")
lines.append(f"Best fourclass: {best_four['model_name']} | {best_four['run_name']} | macro_f1={best_four.get('macro_f1_final', np.nan):.4f}")
if best_eight is not None:
    lines.append(f"Best eightlabel: {best_eight['model_name']} | {best_eight['run_name']} | macro_f1={best_eight.get('macro_f1_final', np.nan):.4f}")
else:
    lines.append("Best eightlabel: not available")
lines.append(f"Best class255: {best_255['model_name']} | {best_255['run_name']} | macro_f1={best_255.get('macro_f1_final', np.nan):.4f}")
lines.append("")
lines.append(f"binary runs found: {len(binary_runs)}")
lines.append(f"fourclass runs found: {len(four_runs)}")
lines.append(f"eightlabel runs found: {len(eight_runs)}")
lines.append(f"class255 runs found: {len(class255_runs)}")
lines.append("")
lines.append("Generated figures:")
for fp in sorted(glob.glob(os.path.join(FIG_DIR, "*.png"))):
    lines.append(os.path.basename(fp))
lines.append("")
lines.append("Generated figure CSVs:")
for fp in sorted(glob.glob(os.path.join(FIGCSV_DIR, "*.csv"))):
    lines.append(os.path.basename(fp))

with open(os.path.join(TXT_DIR, "analysis_summary.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("\nDone.")
print("Figures:", FIG_DIR)
print("Figure CSVs:", FIGCSV_DIR)
print("Tables:", TAB_DIR)
print("Texts:", TXT_DIR)

[Info] binary runs found: 43
[Info] fourclass runs found: 21
[Info] eightlabel runs found: 12
[Info] class255 runs found: 32
[Saved CSV] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figure_csv\Fig1A_fourclass_confusion_normalized.csv
[Saved] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figures\Fig1A_fourclass_confusion_normalized.png
[Saved] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figures\Fig1A_fourclass_confusion_normalized.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figure_csv\Fig1B_fourclass_error_structure.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figure_csv\Fig1B2_fourclass_adjacent_boundary_breakdown.csv
[Saved] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figures\Fig1B_fourclass_error_structure_and_boundary.png
[Saved] D:\Downloads\Depressed\Depressed\final_all_in_one_outputs_v2\figures\Fig1B_fourclass_error_structure_and_boundary.eps
[Saved CS

In [ ]:
# -*- coding: utf-8 -*-
"""
All-in-one depression analysis
 Covers both existing core figures and the missing deeper analyses:
   - human uncertainty anchoring
   - severity boundary closure
   - symptom core-periphery closure
   - cross-task / cross-source consistency
   - risk-sensitive utility
   - run-level stability
"""

import os
import re
import glob
import json
import math
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    hamming_loss
)

warnings.filterwarnings("ignore")

ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"

OUTPUT_DIR = os.path.join(ROOT_DEPRESSED, "final_pub_outputs")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
CSV_DIR = os.path.join(OUTPUT_DIR, "figure_csv")
TAB_DIR = os.path.join(OUTPUT_DIR, "tables")
TXT_DIR = os.path.join(OUTPUT_DIR, "texts")

for d in [OUTPUT_DIR, FIG_DIR, CSV_DIR, TAB_DIR, TXT_DIR]:
    os.makedirs(d, exist_ok=True)


plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.unicode_minus": False,
    "font.size": 18,
    "axes.labelsize": 24,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
    "axes.linewidth": 2.2,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 8,
    "ytick.major.size": 8,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#202020",
    "lightgray": "#CFCBC3",
    "beige": "#EAD9C6",
    "softpink": "#E9C7D8",
    "softorange": "#F2C792",
    "softblue": "#CBE1EC",
    "softgreen": "#CBE9C6",
}

CMAP_BLUE = LinearSegmentedColormap.from_list(
    "nature_blue", ["#F5F5F5", "#DCE8EE", "#B9D5E2", "#98CFE6"]
)
CMAP_PINK = LinearSegmentedColormap.from_list(
    "nature_pink", ["#F8F6F7", "#F0DDE7", "#E7C3D7", "#EEB7D3"]
)
CMAP_ORANGE = LinearSegmentedColormap.from_list(
    "nature_orange", ["#F9F8F4", "#F1E2CF", "#F0C48E", "#F39F4E"]
)

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

EMOTION_SHORT = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

EMOTION_TO_SHORT = {k: EMOTION_SHORT[k] for k in EMOTION_NAMES}


def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def save_text(text, name):
    fp = os.path.join(TXT_DIR, name)
    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[Saved Text] {fp}")

def save_df(df, name):
    fp = os.path.join(CSV_DIR, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved Table] {fp}")

def save_fig(fig, name):
    png_fp = os.path.join(FIG_DIR, f"{name}.png")
    eps_fp = os.path.join(FIG_DIR, f"{name}.eps")
    fig.savefig(png_fp, dpi=DPI, facecolor="white")
    fig.savefig(eps_fp, dpi=DPI, facecolor="white", format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")

def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.2)
    ax.spines["bottom"].set_linewidth(2.2)
    ax.tick_params(axis="both", width=2.0, length=8, pad=10)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")

def safe_col(df, col, default=np.nan):
    if col not in df.columns:
        df[col] = default
    return df

def parse_model_size_b(model_name):
    if pd.isna(model_name):
        return np.nan
    s = str(model_name)
    m = re.search(r"(\d+(?:\.\d+)?)B", s, re.I)
    if m:
        return float(m.group(1))
    return np.nan

def find_first_file(patterns):
    out = []
    for pat in patterns:
        out.extend(glob.glob(os.path.join(ROOT_DEPRESSED, pat), recursive=True))
    out = sorted(set(out))
    return out[0] if len(out) > 0 else None

def find_all_files(patterns):
    out = []
    for pat in patterns:
        out.extend(glob.glob(os.path.join(ROOT_DEPRESSED, pat), recursive=True))
    return sorted(set(out))

def pick_latest_path(paths):
    if len(paths) == 0:
        return None
    paths2 = [(p, os.path.getmtime(p)) for p in paths if os.path.exists(p)]
    if len(paths2) == 0:
        return None
    paths2.sort(key=lambda x: x[1], reverse=True)
    return paths2[0][0]

def infer_run_name_from_row(row):
    for c in ["run_name", "eval_dir_name"]:
        if c in row and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c]).strip()
    for c in ["source_jsonl", "source_file", "pred_file", "pred_file_final"]:
        if c in row and pd.notna(row[c]):
            s = str(row[c])
            base = os.path.basename(os.path.dirname(s))
            if base:
                return base
    return "unknown_run"

def extract_task_from_filename(fp):
    base = os.path.basename(fp).lower()
    if base.startswith("binary__"):
        return "binary"
    if base.startswith("fourclass__"):
        return "fourclass"
    if base.startswith("eightlabel__"):
        return "eightlabel"
    if base.startswith("class255__"):
        return "class255"
    return "unknown"

def extract_model_from_filename(fp):
    base = os.path.basename(fp)
    parts = base.split("__")
    if len(parts) >= 2:
        return parts[1]
    return "unknown_model"

def parse_vec8(x):
    """
    Robust parsing:
    - b00010100
    - 00010100
    - [0,1,0,...]
    - '0 0 0 1 0 1 0 0'
    """
    if isinstance(x, list) and len(x) == 8:
        try:
            return [int(v) for v in x]
        except:
            return None

    if pd.isna(x):
        return None

    s = str(x).strip()

    if s.startswith("b") and len(s) == 9 and set(s[1:]) <= {"0", "1"}:
        return [int(c) for c in s[1:]]

    if len(s) == 8 and set(s) <= {"0", "1"}:
        return [int(c) for c in s]

    nums = re.findall(r"[01]", s)
    if len(nums) == 8:
        return [int(c) for c in nums]

    return None

def vec_to_int(v):
    if v is None:
        return np.nan
    return int("".join(str(int(z)) for z in v), 2)

def symptom_count_from_vec(v):
    if v is None:
        return np.nan
    return int(np.sum(v))

def odds_ratio_2x2(a, b, c, d, eps=1e-9):
    # [[a,b],[c,d]]
    return ((a + eps) * (d + eps)) / ((b + eps) * (c + eps))

def mean_no_nan(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return float(x.mean())

def median_no_nan(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return float(x.median())

def corr_or_nan(x, y, method="pearson"):
    xx = pd.Series(x)
    yy = pd.Series(y)
    mask = xx.notna() & yy.notna()
    xx = xx[mask]
    yy = yy[mask]
    if len(xx) < 3:
        return np.nan
    return float(xx.corr(yy, method=method))

def bootstrap_metric_ci(y_true, y_pred, metric_fn, n_boot=300, random_state=42):
    rng = np.random.RandomState(random_state)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)
    vals = []
    if n == 0:
        return np.nan, np.nan, np.nan
    for _ in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    vals = np.array(vals)
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


pred_dirs = find_all_files(["**/export_bundle*/predictions"])
PRED_DIR = pick_latest_path(pred_dirs)

sample_meta_candidates = find_all_files(["**/export_bundle*/sample_meta.csv"])
SAMPLE_META_FP = pick_latest_path(sample_meta_candidates)

final_run_mapping_fp = find_first_file(["**/final_stage_mapping/final_run_task_mapping.csv"])
final_stage_group_mapping_fp = find_first_file(["**/final_stage_mapping/final_stage_group_mapping.csv"])
final_native_eval_mapping_fp = find_first_file(["**/final_stage_mapping/final_native_eval_mapping.csv"])

strict_eval_mapping_fp = find_first_file(["**/llamafactory_eval_mapping_strict/strict_eval_run_mapping.csv"])

if PRED_DIR is None:
    raise FileNotFoundError("Cannot find export_bundle*/predictions under ROOT_DEPRESSED.")

print("=" * 100)
print("[Input] prediction dir:", PRED_DIR)
print("[Input] sample_meta:", SAMPLE_META_FP)
print("[Input] final_run_mapping:", final_run_mapping_fp)
print("[Input] final_stage_group_mapping:", final_stage_group_mapping_fp)
print("[Input] final_native_eval_mapping:", final_native_eval_mapping_fp)
print("[Input] strict_eval_mapping:", strict_eval_mapping_fp)
print("=" * 100)

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, "*.csv")))
if len(pred_files) == 0:
    raise FileNotFoundError(f"No prediction csvs found under {PRED_DIR}")

all_runs = []
run_file_rows = []

for fp in pred_files:
    try:
        df = pd.read_csv(fp)
    except Exception as e:
        print("[Skip read fail]", fp, e)
        continue

    task = extract_task_from_filename(fp)
    model_name = extract_model_from_filename(fp)

    safe_col(df, "model_name", model_name)
    safe_col(df, "setting", "unknown")
    safe_col(df, "source_jsonl", np.nan)
    safe_col(df, "source_file", np.nan)

    if "run_name" not in df.columns:
        df["run_name"] = df.apply(infer_run_name_from_row, axis=1)

    if "pred_file" not in df.columns:
        if df["source_jsonl"].notna().any():
            df["pred_file"] = df["source_jsonl"]
        elif df["source_file"].notna().any():
            df["pred_file"] = df["source_file"]
        else:
            df["pred_file"] = fp

    df["task"] = task
    df["pred_csv"] = fp
    df["model_size_b"] = df["model_name"].map(parse_model_size_b)

    all_runs.append(df)

    run_level_small = df[["model_name", "model_size_b", "setting", "run_name", "task", "pred_file", "pred_csv"]].copy()
    run_level_small = run_level_small.drop_duplicates()
    run_file_rows.append(run_level_small)

pred_df_all = pd.concat(all_runs, axis=0, ignore_index=True)
run_files_df = pd.concat(run_file_rows, axis=0, ignore_index=True).drop_duplicates()

if SAMPLE_META_FP is not None and os.path.exists(SAMPLE_META_FP):
    sample_meta = pd.read_csv(SAMPLE_META_FP)
else:
    sample_meta = pd.DataFrame(columns=["sample_id", "confidence_score", "is_disagreement_sample"])

safe_col(sample_meta, "sample_id", "")
sample_meta["sample_id"] = sample_meta["sample_id"].astype(str)

safe_col(sample_meta, "confidence_score", np.nan)
safe_col(sample_meta, "is_disagreement_sample", 0)

def load_csv_or_empty(fp):
    if fp is None or (not os.path.exists(fp)):
        return pd.DataFrame()
    try:
        return pd.read_csv(fp)
    except:
        return pd.DataFrame()

final_run_map = load_csv_or_empty(final_run_mapping_fp)
final_stage_group_map = load_csv_or_empty(final_stage_group_mapping_fp)
native_map = load_csv_or_empty(final_native_eval_mapping_fp)
strict_run_map = load_csv_or_empty(strict_eval_mapping_fp)

# normalize mapping columns
for dfx in [final_run_map, final_stage_group_map, native_map, strict_run_map]:
    if len(dfx) == 0:
        continue
    for c in ["model_name", "run_name", "final_task", "final_stage_name", "stage_name_inferred",
              "source_task", "target_task", "count_bucket", "eval_dir_name", "pred_file",
              "pred_file_final", "setting", "final_task_source", "stage_role_in_group",
              "final_stage_role"]:
        safe_col(dfx, c, np.nan)


run_map_merge = run_files_df.copy()

# Merge on model_name + run_name if possible
if len(final_run_map) > 0:
    tmp = final_run_map.copy()
    if "run_name" not in tmp.columns and "eval_dir_name" in tmp.columns:
        tmp["run_name"] = tmp["eval_dir_name"]
    cols_keep = [c for c in [
        "model_name", "run_name", "count_bucket", "final_task", "final_task_source",
        "final_stage_name", "final_stage_role", "pred_file_final", "setting"
    ] if c in tmp.columns]
    tmp = tmp[cols_keep].drop_duplicates()
    run_map_merge = run_map_merge.merge(tmp, on=["model_name", "run_name"], how="left")

if len(strict_run_map) > 0:
    tmp = strict_run_map.copy()
    if "run_name" not in tmp.columns and "eval_dir_name" in tmp.columns:
        tmp["run_name"] = tmp["eval_dir_name"]
    cols_keep = [c for c in [
        "model_name", "run_name", "count_bucket", "final_task", "final_stage_name", "stage_role_in_group"
    ] if c in tmp.columns]
    tmp = tmp[cols_keep].drop_duplicates()
    tmp = tmp.rename(columns={
        "count_bucket": "count_bucket_strict",
        "final_task": "final_task_strict",
        "final_stage_name": "final_stage_name_strict",
        "stage_role_in_group": "stage_role_in_group_strict",
    })
    run_map_merge = run_map_merge.merge(tmp, on=["model_name", "run_name"], how="left")

# fill final task
safe_col(run_map_merge, "final_task", np.nan)
safe_col(run_map_merge, "final_task_strict", np.nan)
run_map_merge["task_final"] = run_map_merge["final_task"]
run_map_merge.loc[run_map_merge["task_final"].isna(), "task_final"] = run_map_merge["final_task_strict"]
run_map_merge.loc[run_map_merge["task_final"].isna(), "task_final"] = run_map_merge["task"]

safe_col(run_map_merge, "count_bucket", np.nan)
safe_col(run_map_merge, "count_bucket_strict", np.nan)
run_map_merge["count_bucket_final"] = run_map_merge["count_bucket"]
run_map_merge.loc[run_map_merge["count_bucket_final"].isna(), "count_bucket_final"] = run_map_merge["count_bucket_strict"]

safe_col(run_map_merge, "final_stage_name", np.nan)
safe_col(run_map_merge, "final_stage_name_strict", np.nan)
run_map_merge["stage_type"] = run_map_merge["final_stage_name"]
run_map_merge.loc[run_map_merge["stage_type"].isna(), "stage_type"] = run_map_merge["final_stage_name_strict"]

# infer stage_type fallback
def infer_stage_type_fallback(row):
    st = row.get("stage_type", np.nan)
    if pd.notna(st) and str(st).strip():
        return str(st)
    rn = str(row.get("run_name", ""))
    pf = str(row.get("pred_file", ""))
    s = (rn + " || " + pf).lower()
    if "from2to255" in s:
        return "from2to255"
    if "from4to255" in s:
        return "from4to255"
    if "from2to8" in s:
        return "from2to8"
    if "native_eval" in s:
        return "native"
    return "direct"

run_map_merge["stage_type"] = run_map_merge.apply(infer_stage_type_fallback, axis=1)

# if count bucket missing, infer from task
def infer_count_bucket(row):
    cb = row.get("count_bucket_final", np.nan)
    if pd.notna(cb):
        return str(cb)
    task_final = str(row.get("task_final", ""))
    if task_final in ["binary", "fourclass"]:
        return "deptweet_task"
    if task_final in ["eightlabel", "class255"]:
        return "depressionemo_task"
    return "unknown"

run_map_merge["count_bucket_final"] = run_map_merge.apply(infer_count_bucket, axis=1)

# pred_file_final fallback
safe_col(run_map_merge, "pred_file_final", np.nan)
run_map_merge["pred_file_final"] = run_map_merge["pred_file_final"].fillna(run_map_merge["pred_file"])

save_table(run_map_merge, "run_inventory_enriched.csv")

binary_runs = run_map_merge[run_map_merge["task_final"] == "binary"].copy()
four_runs = run_map_merge[run_map_merge["task_final"] == "fourclass"].copy()
eight_runs = run_map_merge[run_map_merge["task_final"] == "eightlabel"].copy()
class255_runs = run_map_merge[run_map_merge["task_final"] == "class255"].copy()

print(f"[Info] binary runs found: {len(binary_runs)}")
print(f"[Info] fourclass runs found: {len(four_runs)}")
print(f"[Info] eightlabel runs found: {len(eight_runs)}")
print(f"[Info] class255 runs found: {len(class255_runs)}")


pred_cache = {}

def load_prediction_run(pred_ref):
    """
    pred_ref may be:
    - a jsonl/source path kept in pred_file
    - or standardized csv path
    We resolve using pred_df_all rows.
    """
    if pd.isna(pred_ref):
        return pd.DataFrame()

    pred_ref = str(pred_ref)

    if pred_ref in pred_cache:
        return pred_cache[pred_ref].copy()

    # direct pred_csv match
    d = pred_df_all[pred_df_all["pred_csv"] == pred_ref].copy()
    if len(d) > 0:
        pred_cache[pred_ref] = d.copy()
        return d

    # pred_file match
    d = pred_df_all[pred_df_all["pred_file"].astype(str) == pred_ref].copy()
    if len(d) > 0:
        pred_cache[pred_ref] = d.copy()
        return d

    # source_jsonl/source_file match
    d1 = pred_df_all[pred_df_all["source_jsonl"].astype(str) == pred_ref].copy()
    d2 = pred_df_all[pred_df_all["source_file"].astype(str) == pred_ref].copy()
    d = pd.concat([d1, d2], axis=0, ignore_index=True).drop_duplicates()
    if len(d) > 0:
        pred_cache[pred_ref] = d.copy()
        return d

    # maybe it's a basename
    base = os.path.basename(pred_ref)
    d = pred_df_all[pred_df_all["pred_csv"].map(os.path.basename) == base].copy()
    if len(d) > 0:
        pred_cache[pred_ref] = d.copy()
        return d

    pred_cache[pred_ref] = pd.DataFrame()
    return pd.DataFrame()

def compute_binary_run_metrics(df):
    if len(df) == 0:
        return {}
    d = df.copy()
    d = d.dropna(subset=["true_label", "pred_label"])
    if len(d) == 0:
        return {}
    y_true = d["true_label"].astype(int).values
    y_pred = d["pred_label"].astype(int).values
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "n": len(d),
    }

def compute_fourclass_run_metrics(df):
    if len(df) == 0:
        return {}
    d = df.copy()
    d = d.dropna(subset=["true_label", "pred_label"])
    if len(d) == 0:
        return {}
    y_true = d["true_label"].astype(int).values
    y_pred = d["pred_label"].astype(int).values
    cm = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "n": len(d),
        "cm": cm,
        "report": classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    }

def compute_eightlabel_run_metrics(df):
    if len(df) == 0:
        return {}
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    return {
        "exact_match": accuracy_score(Yt, Yp),
        "macro_f1": f1_score(Yt, Yp, average="macro", zero_division=0),
        "micro_f1": f1_score(Yt, Yp, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(Yt, Yp),
        "n": len(d),
        "Y_true": Yt,
        "Y_pred": Yp
    }

def compute_class255_run_metrics(df):
    if len(df) == 0:
        return {}
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    yt255 = d["true_255"].astype(int).values if "true_255" in d.columns else np.array([vec_to_int(v) for v in d["true_vec_list"]], dtype=int)
    yp255 = d["pred_255"].fillna(-1).astype(int).values if "pred_255" in d.columns else np.array([vec_to_int(v) for v in d["pred_vec_list"]], dtype=int)
    return {
        "accuracy": accuracy_score(yt255, yp255),
        "macro_f1": f1_score(yt255, yp255, average="macro", zero_division=0),
        "weighted_f1": f1_score(yt255, yp255, average="weighted", zero_division=0),
        "micro_f1_symptom": f1_score(Yt, Yp, average="micro", zero_division=0),
        "macro_f1_symptom": f1_score(Yt, Yp, average="macro", zero_division=0),
        "n": len(d),
        "Y_true": Yt,
        "Y_pred": Yp,
        "yt255": yt255,
        "yp255": yp255,
    }

# attach run-level metrics back
def attach_metrics(run_df):
    rows = []
    for _, r in run_df.iterrows():
        pred_file = r["pred_file_final"]
        d = load_prediction_run(pred_file)
        task = r["task_final"]
        out = r.to_dict()
        if task == "binary":
            met = compute_binary_run_metrics(d)
        elif task == "fourclass":
            met = compute_fourclass_run_metrics(d)
        elif task == "eightlabel":
            met = compute_eightlabel_run_metrics(d)
        elif task == "class255":
            met = compute_class255_run_metrics(d)
        else:
            met = {}
        for k, v in met.items():
            if isinstance(v, (np.ndarray, dict)):
                continue
            out[k] = v
        rows.append(out)
    return pd.DataFrame(rows)

binary_runs_m = attach_metrics(binary_runs)
four_runs_m = attach_metrics(four_runs)
eight_runs_m = attach_metrics(eight_runs)
class255_runs_m = attach_metrics(class255_runs)

save_table(binary_runs_m, "binary_runs_metrics.csv")
save_table(four_runs_m, "fourclass_runs_metrics.csv")
save_table(eight_runs_m, "eightlabel_runs_metrics.csv")
save_table(class255_runs_m, "class255_runs_metrics.csv")

def pick_best(df, metric_col):
    if len(df) == 0 or metric_col not in df.columns:
        return None
    d = df[df[metric_col].notna()].copy()
    if len(d) == 0:
        return None
    d = d.sort_values(metric_col, ascending=False).reset_index(drop=True)
    return d.iloc[0]

best_binary = pick_best(binary_runs_m, "f1")
best_four = pick_best(four_runs_m, "macro_f1")
best_eight = pick_best(eight_runs_m, "macro_f1")
best_class255 = pick_best(class255_runs_m, "macro_f1")

best_report_text = []
for name, row in [
    ("Best binary", best_binary),
    ("Best fourclass", best_four),
    ("Best eightlabel", best_eight),
    ("Best class255", best_class255),
]:
    best_report_text.append(f"{name}:\n{row}\n")

save_text("\n\n".join(best_report_text), "best_runs_summary.txt")

def get_best_run_df(best_row):
    if best_row is None:
        return pd.DataFrame()
    d = load_prediction_run(best_row["pred_file_final"]).copy()
    if "sample_id" in d.columns:
        d["sample_id"] = d["sample_id"].astype(str)
        d = d.merge(sample_meta, on="sample_id", how="left")
    else:
        safe_col(d, "confidence_score", np.nan)
        safe_col(d, "is_disagreement_sample", 0)
    return d

best_binary_df = get_best_run_df(best_binary)
best_four_df = get_best_run_df(best_four)
best_eight_df = get_best_run_df(best_eight)
best_class255_df = get_best_run_df(best_class255)

if len(best_four_df) > 0:
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)

    cm = confusion_matrix(d["true_label"], d["pred_label"], labels=[0,1,2,3])
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sum, where=row_sum != 0)

    rows = []
    for i in range(4):
        for j in range(4):
            rows.append({
                "true_label": i,
                "pred_label": j,
                "count": int(cm[i, j]),
                "fraction": float(cm_norm[i, j])
            })
    cm_df = pd.DataFrame(rows)
    save_df(cm_df, "Fig1A_fourclass_confusion_normalized.csv")

    fig, ax = plt.subplots(figsize=(8.6, 8.2))
    im = ax.imshow(cm_norm, cmap=CMAP_BLUE, vmin=0, vmax=np.max(cm_norm))
    for i in range(4):
        for j in range(4):
            ax.text(
                j, i,
                f"{cm[i,j]}\n{cm_norm[i,j]*100:.1f}%",
                ha="center", va="center",
                fontsize=16, fontweight="bold", color=PALETTE["black"]
            )
    ax.set_xticks([0,1,2,3])
    ax.set_yticks([0,1,2,3])
    ax.set_xlabel("Predicted severity")
    ax.set_ylabel("True severity")
    style_ax(ax)
    save_fig(fig, "Fig1A_fourclass_confusion_normalized")

if len(best_four_df) > 0:
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()

    d["err_type"] = np.where(
        d["abs_err"] == 0, "Correct",
        np.where(d["abs_err"] == 1, "Adjacent", "Far")
    )

    err_order = ["Correct", "Adjacent", "Far"]
    err_df = (
        d["err_type"].value_counts(normalize=True)
        .reindex(err_order).fillna(0)
        .reset_index()
    )
    err_df.columns = ["error_type", "fraction"]
    err_df["n"] = err_df["error_type"].map(d["err_type"].value_counts().to_dict())
    save_df(err_df, "Fig1B_fourclass_error_structure.csv")

    adj = d[d["abs_err"] == 1].copy()
    boundary_df = pd.DataFrame([
        {"transition": "1→2", "n": int(((adj["true_label"] == 1) & (adj["pred_label"] == 2)).sum())},
        {"transition": "2→1", "n": int(((adj["true_label"] == 2) & (adj["pred_label"] == 1)).sum())},
    ])
    boundary_df["fraction_within_adjacent"] = boundary_df["n"] / max(len(adj), 1)
    save_df(boundary_df, "Fig1B2_fourclass_adjacent_boundary_breakdown.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.0))

    ax = axes[0]
    colors = [PALETTE["green"], PALETTE["orange"], PALETTE["pink"]]
    ax.bar(err_df["error_type"], err_df["fraction"], color=colors, width=0.58)
    for i, r in err_df.iterrows():
        ax.text(i, r["fraction"] + 0.02, f"{r['fraction']*100:.1f}%", ha="center", fontweight="bold")
    ax.set_ylim(0, min(1.0, max(err_df["fraction"]) + 0.12))
    ax.set_ylabel("Fraction of test samples")
    style_ax(ax)

    ax = axes[1]
    ax.bar(boundary_df["transition"], boundary_df["n"], color=[PALETTE["orange"], PALETTE["blue"]], width=0.58)
    for i, r in boundary_df.iterrows():
        ax.text(i, r["n"] + max(boundary_df["n"]) * 0.04, f"{int(r['n'])}", ha="center", fontweight="bold")
    ax.set_ylabel("No. of boundary cases")
    style_ax(ax)

    save_fig(fig, "Fig1B_fourclass_error_structure_and_boundary")

if len(best_four_df) > 0 and ("confidence_score" in best_four_df.columns):
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()
    d["err_type"] = np.where(
        d["abs_err"] == 0, "Correct",
        np.where(d["abs_err"] == 1, "Adjacent", "Far")
    )
    d["confidence_score"] = pd.to_numeric(d["confidence_score"], errors="coerce")
    d["is_disagreement_sample"] = pd.to_numeric(d["is_disagreement_sample"], errors="coerce").fillna(0).astype(int)

    anchor_rows = []
    for et in ["Correct", "Adjacent", "Far"]:
        dd = d[d["err_type"] == et].copy()
        anchor_rows.append({
            "error_type": et,
            "n": len(dd),
            "mean_confidence": mean_no_nan(dd["confidence_score"]),
            "median_confidence": median_no_nan(dd["confidence_score"]),
            "disagreement_rate": mean_no_nan(dd["is_disagreement_sample"])
        })
    anchor_df = pd.DataFrame(anchor_rows)
    save_df(anchor_df, "Fig1C_fourclass_confidence_anchor.csv")

    # odds ratio vs Correct
    correct = d[d["err_type"] == "Correct"]
    enrich_rows = []
    for et in ["Adjacent", "Far"]:
        dd = d[d["err_type"] == et]
        a = int(dd["is_disagreement_sample"].sum())
        b = int(len(dd) - a)
        c = int(correct["is_disagreement_sample"].sum())
        d0 = int(len(correct) - c)
        enrich_rows.append({
            "compare_to_correct": et,
            "n_group": len(dd),
            "disagreement_rate_group": mean_no_nan(dd["is_disagreement_sample"]),
            "disagreement_rate_correct": mean_no_nan(correct["is_disagreement_sample"]),
            "odds_ratio": odds_ratio_2x2(a, b, c, d0)
        })
    enrich_df = pd.DataFrame(enrich_rows)
    save_df(enrich_df, "Fig1C2_disagreement_enrichment.csv")

    # combined odds rate table
    odds_rows = []
    for et in ["Correct", "Adjacent", "Far"]:
        dd = d[d["err_type"] == et]
        odds_rows.append({
            "error_type": et,
            "n": len(dd),
            "n_disagreement": int(dd["is_disagreement_sample"].sum()),
            "disagreement_rate": mean_no_nan(dd["is_disagreement_sample"])
        })
    odds_df = pd.DataFrame(odds_rows)
    save_df(odds_df, "Fig1C3_disagreement_odds_rate.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))

    ax = axes[0]
    order = ["Correct", "Adjacent", "Far"]
    cols = [PALETTE["green"], PALETTE["orange"], PALETTE["pink"]]
    xs = np.arange(3)
    for i, et in enumerate(order):
        dd = d[(d["err_type"] == et) & d["confidence_score"].notna()].copy()
        if len(dd) == 0:
            continue
        vals = dd["confidence_score"].values
        q1, med, q3 = np.percentile(vals, [25, 50, 75])
        vmin, vmax = np.min(vals), np.max(vals)
        ax.hlines(i, vmin, vmax, color=cols[i], lw=3.5, alpha=0.9)
        ax.add_patch(plt.Rectangle((q1, i - 0.075), q3 - q1, 0.15, color=cols[i], alpha=0.95))
        ax.scatter([med], [i], s=120, color=PALETTE["black"], zorder=5)
        ax.text(vmax + 0.015, i, f"median={med:.2f}   n={len(dd)}", va="center", fontweight="bold")
    ax.set_yticks(xs)
    ax.set_yticklabels(order)
    ax.set_xlabel("Annotation confidence")
    ax.set_xlim(
        max(0.0, d["confidence_score"].dropna().min() - 0.05),
        min(1.05, d["confidence_score"].dropna().max() + 0.03)
    )
    style_ax(ax)

    ax = axes[1]
    vals = odds_df["disagreement_rate"].values
    ax.bar(odds_df["error_type"], vals, color=cols, width=0.58)
    for i, r in odds_df.iterrows():
        ax.text(i, r["disagreement_rate"] + 0.01, f"{r['disagreement_rate']:.3f}", ha="center", fontweight="bold")
    ax.set_ylabel("Disagreement rate")
    style_ax(ax)

    save_fig(fig, "Fig1C_human_uncertainty_anchor")

if len(best_binary_df) > 0 and len(best_four_df) > 0:
    # binary head
    db = best_binary_df.dropna(subset=["true_label", "pred_label"]).copy()
    db["true_label"] = db["true_label"].astype(int)
    db["pred_label"] = db["pred_label"].astype(int)

    yb_true = db["true_label"].values
    yb_pred = db["pred_label"].values

    # fourclass collapsed to binary
    df4 = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    df4["true_label"] = df4["true_label"].astype(int)
    df4["pred_label"] = df4["pred_label"].astype(int)

    y4_true_bin = (df4["true_label"] > 0).astype(int).values
    y4_pred_bin = (df4["pred_label"] > 0).astype(int).values

    # severe threshold
    y4_true_severe = (df4["true_label"] >= 2).astype(int).values
    y4_pred_severe = (df4["pred_label"] >= 2).astype(int).values

    utility_rows = []
    for task_name, yt, yp in [
        ("binary", yb_true, yb_pred),
        ("fourclass_collapsed", y4_true_bin, y4_pred_bin),
        ("fourclass_high_risk_threshold", y4_true_severe, y4_pred_severe),
    ]:
        utility_rows.append({
            "task_view": task_name,
            "accuracy": accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "f1": f1_score(yt, yp, zero_division=0),
            "n": len(yt)
        })
    utility_df = pd.DataFrame(utility_rows)
    save_df(utility_df, "Fig1D_fourclass_threshold_utility.csv")

    severe_rows = []
    severe_rows.append({
        "view": "fourclass_high_risk_threshold",
        "tp": int(((y4_true_severe == 1) & (y4_pred_severe == 1)).sum()),
        "fn": int(((y4_true_severe == 1) & (y4_pred_severe == 0)).sum()),
        "fp": int(((y4_true_severe == 0) & (y4_pred_severe == 1)).sum()),
        "tn": int(((y4_true_severe == 0) & (y4_pred_severe == 0)).sum()),
    })
    save_df(pd.DataFrame(severe_rows), "Fig1D2_severe_routing.csv")

    fig, ax = plt.subplots(figsize=(10.5, 5.0))
    plot_df = utility_df.melt(id_vars="task_view", value_vars=["accuracy", "precision", "recall", "f1"],
                              var_name="metric", value_name="score")
    metrics_order = ["accuracy", "precision", "recall", "f1"]
    task_order = ["binary", "fourclass_collapsed", "fourclass_high_risk_threshold"]
    x = np.arange(len(task_order))
    width = 0.18
    color_map = {
        "accuracy": PALETTE["gray"],
        "precision": PALETTE["yellow"],
        "recall": PALETTE["blue"],
        "f1": PALETTE["orange"],
    }
    offsets = [-1.5*width, -0.5*width, 0.5*width, 1.5*width]
    for off, met in zip(offsets, metrics_order):
        dd = plot_df[plot_df["metric"] == met].set_index("task_view").reindex(task_order).reset_index()
        ax.bar(x + off, dd["score"], width=width, color=color_map[met], label=met)
    ax.set_xticks(x)
    ax.set_xticklabels(["Binary", "4-class→bin", "Severe\nthreshold"])
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.0)
    ax.legend(frameon=False, ncol=4, loc="upper left")
    style_ax(ax)
    save_fig(fig, "Fig1D_fourclass_risk_sensitive_utility")

def per_symptom_metrics_from_multilabel(df):
    if len(df) == 0:
        return pd.DataFrame()

    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return pd.DataFrame()

    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)

    rows = []
    for i, lab in enumerate(EMOTION_NAMES):
        yt = Yt[:, i]
        yp = Yp[:, i]
        tp = int(((yt == 1) & (yp == 1)).sum())
        fn = int(((yt == 1) & (yp == 0)).sum())
        fp = int(((yt == 0) & (yp == 1)).sum())
        tn = int(((yt == 0) & (yp == 0)).sum())
        support = int(yt.sum())

        omission = fn / max(tp + fn, 1)
        commission = fp / max(fp + tn, 1)
        prevalence_true = float(yt.mean())
        prevalence_pred = float(yp.mean())

        rows.append({
            "label": lab,
            "label_short": EMOTION_TO_SHORT[lab],
            "support": support,
            "f1": f1_score(yt, yp, zero_division=0),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0),
            "omission_rate": omission,
            "commission_rate": commission,
            "true_prevalence": prevalence_true,
            "pred_prevalence": prevalence_pred,
            "prevalence_bias": prevalence_pred - prevalence_true
        })
    return pd.DataFrame(rows)

def true_cooccurrence_centrality(df):
    if len(df) == 0:
        return pd.DataFrame()
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna()].copy()
    if len(d) == 0:
        return pd.DataFrame()
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    co = Yt.T @ Yt
    np.fill_diagonal(co, 0)
    deg = co.sum(axis=1).astype(float)
    deg_norm = deg / max(deg.max(), 1.0)
    out = pd.DataFrame({
        "label": EMOTION_NAMES,
        "label_short": [EMOTION_TO_SHORT[x] for x in EMOTION_NAMES],
        "centrality": deg_norm
    })
    co_df = pd.DataFrame(co, index=EMOTION_NAMES, columns=EMOTION_NAMES).reset_index().rename(columns={"index": "label"})
    return out, co_df

eight_sym_df = per_symptom_metrics_from_multilabel(best_eight_df)
class255_sym_df = per_symptom_metrics_from_multilabel(best_class255_df)

# centrality based on best eightlabel true structure if available; fallback class255
if len(best_eight_df) > 0:
    centrality_df, true_cooc_df = true_cooccurrence_centrality(best_eight_df)
elif len(best_class255_df) > 0:
    centrality_df, true_cooc_df = true_cooccurrence_centrality(best_class255_df)
else:
    centrality_df = pd.DataFrame(columns=["label", "label_short", "centrality"])
    true_cooc_df = pd.DataFrame()

if len(centrality_df) > 0:
    base_sym_df = centrality_df.copy()
    if len(class255_sym_df) > 0:
        base_sym_df = base_sym_df.merge(
            class255_sym_df[["label", "f1", "omission_rate", "commission_rate", "prevalence_bias", "support"]]
            .rename(columns={
                "f1": "class255_f1",
                "omission_rate": "class255_omission",
                "commission_rate": "class255_commission",
                "prevalence_bias": "class255_prev_bias",
                "support": "class255_support"
            }),
            on="label", how="left"
        )
    if len(eight_sym_df) > 0:
        base_sym_df = base_sym_df.merge(
            eight_sym_df[["label", "f1", "omission_rate", "commission_rate", "prevalence_bias", "support"]]
            .rename(columns={
                "f1": "eightlabel_f1",
                "omission_rate": "eightlabel_omission",
                "commission_rate": "eightlabel_commission",
                "prevalence_bias": "eightlabel_prev_bias",
                "support": "eightlabel_support"
            }),
            on="label", how="left"
        )
else:
    base_sym_df = pd.DataFrame()

if len(base_sym_df) > 0:
    save_df(base_sym_df, "Fig2_symptom_structure_metrics.csv")

if len(base_sym_df) > 0 and base_sym_df["class255_omission"].notna().sum() >= 3:
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

    panels = [
        ("class255_omission", "Omission rate", PALETTE["orange"], "Fig2A"),
        ("class255_commission", "Commission rate", PALETTE["pink"], "Fig2B"),
        ("class255_prev_bias", "Predicted prevalence − true prevalence", PALETTE["blue"], "Fig2C"),
    ]

    for ax, (ycol, ylab, color, _) in zip(axes, panels):
        dd = base_sym_df[["label_short", "centrality", ycol]].dropna().copy()
        ax.scatter(dd["centrality"], dd[ycol], s=220, color=color, edgecolor="none")
        r = corr_or_nan(dd["centrality"], dd[ycol], method="pearson")
        for _, r0 in dd.iterrows():
            ax.text(r0["centrality"] + 0.01, r0[ycol], r0["label_short"], fontsize=12, fontweight="bold")
        ax.set_xlabel("Symptom centrality")
        ax.set_ylabel(ylab)
        ax.text(0.98, 0.97, f"r={r:.2f}" if pd.notna(r) else "r=NA",
                ha="right", va="top", transform=ax.transAxes, fontweight="bold")
        if "prevalence" in ylab:
            ax.axhline(0, lw=1.6, color=PALETTE["lightgray"])
        style_ax(ax)

    save_fig(fig, "Fig2ABC_construct_closure")


if len(eight_sym_df) > 0:
    plot_df = eight_sym_df.sort_values("f1", ascending=False).reset_index(drop=True)
    save_df(plot_df, "Fig2D_per_symptom_f1.csv")

    fig, ax = plt.subplots(figsize=(9.8, 5.6))
    colors = [PALETTE["blue"]] * len(plot_df)
    low_lab = "brain dysfunction (forget)"
    top_lab = "suicide intent"
    colors = [PALETTE["orange"] if x == top_lab else (PALETTE["pink"] if x == low_lab else PALETTE["blue"]) for x in plot_df["label"]]
    ax.barh(plot_df["label_short"], plot_df["f1"], color=colors, edgecolor="none")
    for i, r in plot_df.iterrows():
        ax.text(r["f1"] + 0.01, i, f"{r['f1']:.3f}  (n={int(r['support'])})", va="center", fontweight="bold", fontsize=13)
    ax.set_xlabel("F1 score")
    ax.invert_yaxis()
    style_ax(ax)
    save_fig(fig, "Fig2D_per_symptom_f1")

if len(true_cooc_df) > 0:
    save_df(true_cooc_df, "Fig2E_eightlabel_true_cooccurrence.csv")

    mat = true_cooc_df.set_index("label")[EMOTION_NAMES].values.astype(float)
    mat_norm = mat / max(np.max(mat), 1.0)

    fig, ax = plt.subplots(figsize=(8.8, 8.0))
    ax.imshow(mat_norm, cmap=CMAP_PINK, vmin=0, vmax=np.max(mat_norm))
    for i in range(len(EMOTION_NAMES)):
        for j in range(len(EMOTION_NAMES)):
            ax.text(j, i, f"{mat_norm[i,j]:.2f}", ha="center", va="center", fontsize=12, fontweight="bold")
    ax.set_xticks(np.arange(len(EMOTION_NAMES)))
    ax.set_yticks(np.arange(len(EMOTION_NAMES)))
    ax.set_xticklabels([EMOTION_TO_SHORT[x] for x in EMOTION_NAMES], rotation=45, ha="right")
    ax.set_yticklabels([EMOTION_TO_SHORT[x] for x in EMOTION_NAMES])
    style_ax(ax)
    save_fig(fig, "Fig2E_eightlabel_true_cooccurrence")

if len(best_class255_df) > 0:
    d = best_class255_df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    d["true_255"] = d["true_255"].astype(int)
    d["pred_255"] = d["pred_255"].fillna(-1).astype(int)
    d["correct"] = (d["true_255"] == d["pred_255"]).astype(int)
    d["k_true"] = d["true_vec_list"].map(symptom_count_from_vec)
    d["k_pred"] = d["pred_vec_list"].map(symptom_count_from_vec)

    # A
    acc_by_k = d.groupby("k_true").agg(
        n=("correct", "size"),
        exact_accuracy=("correct", "mean")
    ).reset_index()
    save_df(acc_by_k, "Fig3A_class255_accuracy_by_symptom_count.csv")

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.bar(acc_by_k["k_true"], acc_by_k["exact_accuracy"], color=PALETTE["pink"], width=0.72)
    ax.plot(acc_by_k["k_true"], acc_by_k["exact_accuracy"], marker="o", ms=8, lw=2.8, color=PALETTE["orange"])
    for _, r in acc_by_k.iterrows():
        ax.text(r["k_true"], r["exact_accuracy"] + 0.015, f"{r['exact_accuracy']:.2f}", ha="center", fontweight="bold")
    ax.set_xlabel("No. of symptoms in true combination")
    ax.set_ylabel("Exact accuracy")
    ax.set_xticks(acc_by_k["k_true"].tolist())
    style_ax(ax)
    save_fig(fig, "Fig3A_class255_accuracy_by_symptom_count")

    # B
    comb = d.groupby("true_255").agg(
        support=("true_255", "size"),
        combination_accuracy=("correct", "mean")
    ).reset_index()
    save_df(comb, "Fig3B_class255_support_vs_accuracy.csv")

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    sizes = 80 + comb["support"].values * 8
    ax.scatter(comb["support"], comb["combination_accuracy"], s=sizes, color=PALETTE["green"], alpha=0.95, edgecolor="none")
    pearson_r = corr_or_nan(comb["support"], comb["combination_accuracy"], method="pearson")
    spearman_r = corr_or_nan(comb["support"], comb["combination_accuracy"], method="spearman")
    top_lab = comb.sort_values("support", ascending=False).head(4)
    for _, r0 in top_lab.iterrows():
        ax.text(r0["support"] + 0.8, r0["combination_accuracy"], f"{int(r0['true_255'])}", fontweight="bold", fontsize=12)
    ax.text(0.98, 0.97, f"Pearson r={pearson_r:.2f}\nSpearman ρ={spearman_r:.2f}",
            ha="right", va="top", transform=ax.transAxes, fontweight="bold")
    ax.set_xlabel("Combination support")
    ax.set_ylabel("Combination accuracy")
    style_ax(ax)
    save_fig(fig, "Fig3B_class255_support_vs_accuracy")

    # C
    count_cm = confusion_matrix(d["k_true"], d["k_pred"], labels=list(range(1, 9)))
    row_sum = count_cm.sum(axis=1, keepdims=True)
    count_cm_norm = np.divide(count_cm, row_sum, where=row_sum != 0)

    rows = []
    for i, kt in enumerate(range(1, 9)):
        for j, kp in enumerate(range(1, 9)):
            rows.append({
                "true_symptom_count": kt,
                "pred_symptom_count": kp,
                "count": int(count_cm[i, j]),
                "fraction": float(count_cm_norm[i, j])
            })
    save_df(pd.DataFrame(rows), "Fig3C_class255_symptom_count_confusion.csv")

    fig, ax = plt.subplots(figsize=(8.8, 7.2))
    ax.imshow(count_cm_norm, cmap=CMAP_ORANGE, vmin=0, vmax=np.max(count_cm_norm))
    for i in range(8):
        for j in range(8):
            ax.text(j, i, f"{count_cm_norm[i,j]:.2f}", ha="center", va="center", fontsize=12, fontweight="bold")
    ax.set_xticks(np.arange(8))
    ax.set_yticks(np.arange(8))
    ax.set_xticklabels(np.arange(1, 9))
    ax.set_yticklabels(np.arange(1, 9))
    ax.set_xlabel("Predicted symptom count")
    ax.set_ylabel("True symptom count")
    style_ax(ax)
    save_fig(fig, "Fig3C_class255_symptom_count_confusion")


def build_fourclass_cue_features(df):
    if len(df) == 0:
        return pd.DataFrame()
    d = df.copy()
    safe_col(d, "text", "")
    d["text"] = d["text"].astype(str)

    def has_any(txt, keywords):
        t = txt.lower()
        return int(any(k in t for k in keywords))

    cues = {
        "sadness_explicit": ["sad", "depressed", "cry", "empty", "down"],
        "hopelessness_explicit": ["hopeless", "no future", "nothing will change", "can't go on"],
        "worthlessness_explicit": ["worthless", "useless", "burden", "failure"],
        "cognitive_explicit": ["forget", "memory", "focus", "concentrate", "brain fog"],
        "suicide_explicit": ["suicide", "kill myself", "end my life", "die", "self-harm"],
        "high_first_person": None,
        "high_negation": None,
    }

    d["high_first_person"] = d["text"].str.lower().str.count(r"\b(i|me|my|mine|myself)\b") >= 4
    d["high_first_person"] = d["high_first_person"].astype(int)
    d["high_negation"] = d["text"].str.lower().str.count(r"\b(no|not|never|nothing|can't|cannot|don't|won't)\b") >= 2
    d["high_negation"] = d["high_negation"].astype(int)

    for cue, kws in cues.items():
        if kws is None:
            continue
        d[cue] = d["text"].map(lambda x: has_any(x, kws))

    return d

# fourclass cue dependence
if len(best_four_df) > 0:
    d = build_fourclass_cue_features(best_four_df.dropna(subset=["true_label", "pred_label"]).copy())
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()

    cue_names = [
        "cognitive_explicit",
        "sadness_explicit",
        "hopelessness_explicit",
        "worthlessness_explicit",
        "suicide_explicit",
        "high_first_person",
        "high_negation",
    ]

    cue_rows = []
    for cue in cue_names:
        pos = d[d[cue] == 1]
        neg = d[d[cue] == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        cue_rows.append({
            "cue": cue,
            "delta_mean_true_severity": pos["true_label"].mean() - neg["true_label"].mean(),
            "delta_fourclass_error_rate": (pos["abs_err"] > 0).mean() - (neg["abs_err"] > 0).mean(),
            "n_pos": len(pos),
            "n_neg": len(neg),
        })
    cue_four_df = pd.DataFrame(cue_rows)
    save_df(cue_four_df, "Fig4A_fourclass_cue_dependence.csv")

    fig, ax = plt.subplots(figsize=(9.6, 5.3))
    ax.axhline(0, color=PALETTE["lightgray"], lw=1.6)
    ax.axvline(0, color=PALETTE["lightgray"], lw=1.6)
    colors = []
    for cue in cue_four_df["cue"]:
        if cue == "cognitive_explicit":
            colors.append(PALETTE["orange"])
        elif cue in ["high_first_person", "high_negation", "suicide_explicit"]:
            colors.append(PALETTE["green"])
        else:
            colors.append(PALETTE["gray"])
    ax.scatter(cue_four_df["delta_mean_true_severity"], cue_four_df["delta_fourclass_error_rate"],
               s=260, color=colors, edgecolor="none")
    for _, r in cue_four_df.iterrows():
        ax.text(r["delta_mean_true_severity"] + 0.01, r["delta_fourclass_error_rate"], r["cue"], fontsize=12, fontweight="bold")
    ax.set_xlabel("Δ mean true severity")
    ax.set_ylabel("Δ fourclass error rate")
    style_ax(ax)
    save_fig(fig, "Fig4A_fourclass_cue_dependence")

# class255 cue dependence
if len(best_class255_df) > 0:
    d = build_fourclass_cue_features(best_class255_df.copy())
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    d["k_true"] = d["true_vec_list"].map(symptom_count_from_vec)
    d["true_suicide"] = d["true_vec_list"].map(lambda v: v[6])
    d["pred_suicide"] = d["pred_vec_list"].map(lambda v: v[6])
    d["suicide_fn"] = ((d["true_suicide"] == 1) & (d["pred_suicide"] == 0)).astype(int)

    cue_names = [
        "cognitive_explicit",
        "sadness_explicit",
        "hopelessness_explicit",
        "worthlessness_explicit",
        "suicide_explicit",
        "high_first_person",
        "high_negation",
    ]
    cue_rows = []
    for cue in cue_names:
        pos = d[d[cue] == 1]
        neg = d[d[cue] == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        cue_rows.append({
            "cue": cue,
            "delta_true_symptom_burden": pos["k_true"].mean() - neg["k_true"].mean(),
            "delta_suicide_fn_rate": pos["suicide_fn"].mean() - neg["suicide_fn"].mean(),
            "n_pos": len(pos),
            "n_neg": len(neg),
        })
    cue_255_df = pd.DataFrame(cue_rows)
    save_df(cue_255_df, "Fig4B_class255_cue_dependence.csv")

    fig, ax = plt.subplots(figsize=(9.6, 5.3))
    ax.axhline(0, color=PALETTE["lightgray"], lw=1.6)
    ax.axvline(0, color=PALETTE["lightgray"], lw=1.6)
    colors = []
    for cue in cue_255_df["cue"]:
        if cue == "cognitive_explicit":
            colors.append(PALETTE["orange"])
        elif cue in ["high_first_person", "high_negation", "suicide_explicit", "worthlessness_explicit"]:
            colors.append(PALETTE["green"])
        else:
            colors.append(PALETTE["gray"])
    ax.scatter(cue_255_df["delta_true_symptom_burden"], cue_255_df["delta_suicide_fn_rate"],
               s=260, color=colors, edgecolor="none")
    for _, r in cue_255_df.iterrows():
        ax.text(r["delta_true_symptom_burden"] + 0.02, r["delta_suicide_fn_rate"], r["cue"], fontsize=12, fontweight="bold")
    ax.set_xlabel("Δ true symptom burden")
    ax.set_ylabel("Δ suicide FN rate")
    style_ax(ax)
    save_fig(fig, "Fig4B_class255_cue_dependence")


def summarize_fourclass_stability(row):
    pred_file = row["pred_file_final"]
    d = load_prediction_run(pred_file).copy()
    if len(d) == 0:
        return None
    d = d.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return None
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()
    adjacent = (d["abs_err"] == 1)
    share_12 = 0.0
    if adjacent.sum() > 0:
        share_12 = (
            (((d["true_label"] == 1) & (d["pred_label"] == 2)) |
             ((d["true_label"] == 2) & (d["pred_label"] == 1))).sum()
            / max(adjacent.sum(), 1)
        )
    return {
        "adjacent_error_frac": adjacent.mean(),
        "share_12_within_adjacent": share_12,
        "macro_f1": f1_score(d["true_label"], d["pred_label"], average="macro", zero_division=0),
    }

def summarize_multilabel_structure(row):
    pred_file = row["pred_file_final"]
    d = load_prediction_run(pred_file).copy()
    if len(d) == 0:
        return None
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return None
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)

    brain_i = EMOTION_NAMES.index("brain dysfunction (forget)")
    sad_i = EMOTION_NAMES.index("sadness")

    yt_b = Yt[:, brain_i]
    yp_b = Yp[:, brain_i]
    yt_s = Yt[:, sad_i]
    yp_s = Yp[:, sad_i]

    brain_omission = (((yt_b == 1) & (yp_b == 0)).sum()) / max((yt_b == 1).sum(), 1)
    sadness_commission = (((yt_s == 0) & (yp_s == 1)).sum()) / max((yt_s == 0).sum(), 1)

    # mean core shift based on centrality weights if available
    if len(centrality_df) > 0:
        w = centrality_df.set_index("label").loc[EMOTION_NAMES]["centrality"].values.astype(float)
        core_true = (Yt * w).sum(axis=1)
        core_pred = (Yp * w).sum(axis=1)
        mean_core_shift = float(np.mean(core_pred - core_true))
    else:
        mean_core_shift = np.nan

    return {
        "brain_omission": brain_omission,
        "sadness_commission": sadness_commission,
        "mean_core_shift": mean_core_shift,
        "macro_f1": f1_score(Yt, Yp, average="macro", zero_division=0),
    }

# fourclass stability
four_stab_rows = []
for _, r in four_runs_m.iterrows():
    out = summarize_fourclass_stability(r)
    if out is None:
        continue
    out.update({
        "model_name": r["model_name"],
        "model_size_b": r["model_size_b"],
        "run_name": r["run_name"],
        "stage_type": r["stage_type"],
    })
    four_stab_rows.append(out)
four_stab_df = pd.DataFrame(four_stab_rows)
save_df(four_stab_df, "Fig4C_fourclass_run_stability.csv")

# class255 stability
c255_stab_rows = []
for _, r in class255_runs_m.iterrows():
    out = summarize_multilabel_structure(r)
    if out is None:
        continue
    out.update({
        "model_name": r["model_name"],
        "model_size_b": r["model_size_b"],
        "run_name": r["run_name"],
        "stage_type": r["stage_type"],
        "task": "class255"
    })
    c255_stab_rows.append(out)
c255_stab_df = pd.DataFrame(c255_stab_rows)
save_df(c255_stab_df, "Fig4D_class255_run_stability.csv")

# eightlabel stability
eight_stab_rows = []
for _, r in eight_runs_m.iterrows():
    out = summarize_multilabel_structure(r)
    if out is None:
        continue
    out.update({
        "model_name": r["model_name"],
        "model_size_b": r["model_size_b"],
        "run_name": r["run_name"],
        "stage_type": r["stage_type"],
        "task": "eightlabel"
    })
    eight_stab_rows.append(out)
eight_stab_df = pd.DataFrame(eight_stab_rows)
save_df(eight_stab_df, "Fig4E_eightlabel_run_stability.csv")

if len(four_stab_df) > 0:
    plot_df = pd.DataFrame([
        {
            "metric": "adjacent_error_frac",
            "mean": four_stab_df["adjacent_error_frac"].mean(),
            "min": four_stab_df["adjacent_error_frac"].min(),
            "max": four_stab_df["adjacent_error_frac"].max()
        },
        {
            "metric": "share_12_within_adjacent",
            "mean": four_stab_df["share_12_within_adjacent"].mean(),
            "min": four_stab_df["share_12_within_adjacent"].min(),
            "max": four_stab_df["share_12_within_adjacent"].max()
        },
    ])
    fig, ax = plt.subplots(figsize=(7.8, 4.8))
    y = np.arange(len(plot_df))
    ax.hlines(y, plot_df["min"], plot_df["max"], color=PALETTE["orange"], lw=3.0)
    ax.scatter(plot_df["mean"], y, s=260, color=PALETTE["orange"], zorder=5)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["metric"])
    ax.set_xlabel("Range / mean")
    style_ax(ax)
    save_fig(fig, "Fig4C_fourclass_run_stability")

if len(c255_stab_df) > 0 or len(eight_stab_df) > 0:
    rows = []
    if len(eight_stab_df) > 0:
        rows += [
            {
                "label": "eightlabel:sadness_commission",
                "mean": eight_stab_df["sadness_commission"].mean(),
                "min": eight_stab_df["sadness_commission"].min(),
                "max": eight_stab_df["sadness_commission"].max(),
                "task": "eightlabel"
            },
            {
                "label": "eightlabel:brain_omission",
                "mean": eight_stab_df["brain_omission"].mean(),
                "min": eight_stab_df["brain_omission"].min(),
                "max": eight_stab_df["brain_omission"].max(),
                "task": "eightlabel"
            },
            {
                "label": "eightlabel:mean_core_shift",
                "mean": eight_stab_df["mean_core_shift"].mean(),
                "min": eight_stab_df["mean_core_shift"].min(),
                "max": eight_stab_df["mean_core_shift"].max(),
                "task": "eightlabel"
            },
        ]
    if len(c255_stab_df) > 0:
        rows += [
            {
                "label": "class255:sadness_commission",
                "mean": c255_stab_df["sadness_commission"].mean(),
                "min": c255_stab_df["sadness_commission"].min(),
                "max": c255_stab_df["sadness_commission"].max(),
                "task": "class255"
            },
            {
                "label": "class255:brain_omission",
                "mean": c255_stab_df["brain_omission"].mean(),
                "min": c255_stab_df["brain_omission"].min(),
                "max": c255_stab_df["brain_omission"].max(),
                "task": "class255"
            },
            {
                "label": "class255:mean_core_shift",
                "mean": c255_stab_df["mean_core_shift"].mean(),
                "min": c255_stab_df["mean_core_shift"].min(),
                "max": c255_stab_df["mean_core_shift"].max(),
                "task": "class255"
            },
        ]
    stab_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(10.8, 5.8))
    y = np.arange(len(stab_df))
    color_list = [PALETTE["blue"] if t == "eightlabel" else PALETTE["pink"] for t in stab_df["task"]]
    for i, r in stab_df.iterrows():
        ax.hlines(i, r["min"], r["max"], color=color_list[i], lw=3.0)
        ax.scatter(r["mean"], i, s=260, color=color_list[i], zorder=5)
    ax.set_yticks(y)
    ax.set_yticklabels(stab_df["label"])
    ax.set_xlabel("Range / mean")
    style_ax(ax)
    save_fig(fig, "Fig4DE_multilabel_run_stability")


if len(best_four_df) > 0:
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["confidence_score"] = pd.to_numeric(d["confidence_score"], errors="coerce")
    d["is_disagreement_sample"] = pd.to_numeric(d["is_disagreement_sample"], errors="coerce").fillna(0).astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()

    d["boundary_type"] = "other"
    d.loc[(d["true_label"] == 1) & (d["pred_label"] == 2), "boundary_type"] = "1→2"
    d.loc[(d["true_label"] == 2) & (d["pred_label"] == 1), "boundary_type"] = "2→1"
    bd = d[d["boundary_type"].isin(["1→2", "2→1"])].copy()

    closure_rows = []
    for bt in ["1→2", "2→1"]:
        dd = bd[bd["boundary_type"] == bt].copy()
        closure_rows.append({
            "boundary_type": bt,
            "n": len(dd),
            "mean_confidence": mean_no_nan(dd["confidence_score"]),
            "median_confidence": median_no_nan(dd["confidence_score"]),
            "disagreement_rate": mean_no_nan(dd["is_disagreement_sample"])
        })
    boundary_closure_df = pd.DataFrame(closure_rows)
    save_df(boundary_closure_df, "Fig5A_boundary_uncertainty_closure.csv")

    # OR boundary vs non-boundary-adjacent
    adj = d[d["abs_err"] == 1].copy()
    adj["is_boundary12"] = adj["boundary_type"].isin(["1→2", "2→1"]).astype(int)
    a = int(adj[(adj["is_boundary12"] == 1)]["is_disagreement_sample"].sum())
    b = int((adj["is_boundary12"] == 1).sum() - a)
    c = int(adj[(adj["is_boundary12"] == 0)]["is_disagreement_sample"].sum())
    d0 = int((adj["is_boundary12"] == 0).sum() - c)
    boundary_or = odds_ratio_2x2(a, b, c, d0) if len(adj) > 0 else np.nan

    boundary_or_df = pd.DataFrame([{
        "contrast": "boundary_12_vs_other_adjacent",
        "odds_ratio_disagreement": boundary_or,
        "n_boundary": int((adj["is_boundary12"] == 1).sum()),
        "n_other_adjacent": int((adj["is_boundary12"] == 0).sum()),
    }])
    save_df(boundary_or_df, "Fig5A2_boundary_disagreement_odds_ratio.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8))

    ax = axes[0]
    if len(boundary_closure_df) > 0:
        ax.bar(boundary_closure_df["boundary_type"], boundary_closure_df["n"],
               color=[PALETTE["orange"], PALETTE["blue"]], width=0.58)
        for i, r in boundary_closure_df.iterrows():
            ax.text(i, r["n"] + max(boundary_closure_df["n"].max(), 1) * 0.05,
                    f"{int(r['n'])}", ha="center", fontweight="bold")
    ax.set_ylabel("No. of boundary cases")
    style_ax(ax)

    ax = axes[1]
    if len(boundary_closure_df) > 0:
        vals = boundary_closure_df["median_confidence"].values
        ax.bar(boundary_closure_df["boundary_type"], vals,
               color=[PALETTE["orange"], PALETTE["blue"]], width=0.58)
        for i, r in boundary_closure_df.iterrows():
            ax.text(i, r["median_confidence"] + 0.01,
                    f"median={r['median_confidence']:.2f}\ndis={r['disagreement_rate']:.3f}",
                    ha="center", fontweight="bold", fontsize=13)
    ax.set_ylabel("Median confidence")
    style_ax(ax)

    save_fig(fig, "Fig5A_boundary_uncertainty_closure")

if len(class255_sym_df) > 0 and len(eight_sym_df) > 0:
    cross_df = class255_sym_df[["label", "label_short", "omission_rate", "commission_rate", "prevalence_bias"]].merge(
        eight_sym_df[["label", "omission_rate", "commission_rate", "prevalence_bias"]],
        on="label",
        suffixes=("_class255", "_eightlabel")
    )
    cross_df["label_short"] = cross_df["label"].map(EMOTION_TO_SHORT)
    save_df(cross_df, "Fig5B_cross_task_symptom_bias_consistency.csv")

    fig, axes = plt.subplots(1, 3, figsize=(15.8, 4.8))

    panels = [
        ("omission_rate_eightlabel", "omission_rate_class255", "Omission", PALETTE["orange"]),
        ("commission_rate_eightlabel", "commission_rate_class255", "Commission", PALETTE["pink"]),
        ("prevalence_bias_eightlabel", "prevalence_bias_class255", "Prevalence bias", PALETTE["blue"]),
    ]

    for ax, (xcol, ycol, name, color) in zip(axes, panels):
        dd = cross_df[[xcol, ycol, "label_short"]].dropna().copy()
        ax.scatter(dd[xcol], dd[ycol], s=240, color=color, edgecolor="none")
        lo = min(dd[xcol].min(), dd[ycol].min())
        hi = max(dd[xcol].max(), dd[ycol].max())
        pad = (hi - lo) * 0.08 + 1e-6
        ax.plot([lo-pad, hi+pad], [lo-pad, hi+pad], color=PALETTE["lightgray"], lw=1.6)
        r = corr_or_nan(dd[xcol], dd[ycol], method="pearson")
        for _, r0 in dd.iterrows():
            ax.text(r0[xcol] + 0.005, r0[ycol], r0["label_short"], fontsize=12, fontweight="bold")
        ax.text(0.98, 0.97, f"r={r:.2f}" if pd.notna(r) else "r=NA",
                ha="right", va="top", transform=ax.transAxes, fontweight="bold")
        ax.set_xlabel(f"8-label {name.lower()}")
        ax.set_ylabel(f"255-class {name.lower()}")
        style_ax(ax)

    save_fig(fig, "Fig5B_cross_task_symptom_bias_consistency")

utility_rows = []

# binary overall
if len(best_binary_df) > 0:
    db = best_binary_df.dropna(subset=["true_label", "pred_label"]).copy()
    db["true_label"] = db["true_label"].astype(int)
    db["pred_label"] = db["pred_label"].astype(int)
    utility_rows.append({
        "view": "binary",
        "precision": precision_score(db["true_label"], db["pred_label"], zero_division=0),
        "recall": recall_score(db["true_label"], db["pred_label"], zero_division=0),
        "f1": f1_score(db["true_label"], db["pred_label"], zero_division=0),
    })

# fourclass collapsed
if len(best_four_df) > 0:
    d4 = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d4["true_label"] = d4["true_label"].astype(int)
    d4["pred_label"] = d4["pred_label"].astype(int)
    yt = (d4["true_label"] > 0).astype(int)
    yp = (d4["pred_label"] > 0).astype(int)
    utility_rows.append({
        "view": "fourclass_collapsed",
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
    })
    yt2 = (d4["true_label"] >= 2).astype(int)
    yp2 = (d4["pred_label"] >= 2).astype(int)
    utility_rows.append({
        "view": "fourclass_high_risk_threshold",
        "precision": precision_score(yt2, yp2, zero_division=0),
        "recall": recall_score(yt2, yp2, zero_division=0),
        "f1": f1_score(yt2, yp2, zero_division=0),
    })

# eightlabel suicide
if len(best_eight_df) > 0:
    d8 = best_eight_df.copy()
    d8["true_vec_list"] = d8["true_vec"].map(parse_vec8)
    d8["pred_vec_list"] = d8["pred_vec"].map(parse_vec8)
    d8 = d8[d8["true_vec_list"].notna() & d8["pred_vec_list"].notna()].copy()
    yt = d8["true_vec_list"].map(lambda v: v[6]).astype(int)
    yp = d8["pred_vec_list"].map(lambda v: v[6]).astype(int)
    utility_rows.append({
        "view": "eightlabel_suicide_intent",
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
    })

# class255 suicide
if len(best_class255_df) > 0:
    d255 = best_class255_df.copy()
    d255["true_vec_list"] = d255["true_vec"].map(parse_vec8)
    d255["pred_vec_list"] = d255["pred_vec"].map(parse_vec8)
    d255 = d255[d255["true_vec_list"].notna() & d255["pred_vec_list"].notna()].copy()
    yt = d255["true_vec_list"].map(lambda v: v[6]).astype(int)
    yp = d255["pred_vec_list"].map(lambda v: v[6]).astype(int)
    utility_rows.append({
        "view": "class255_suicide_intent",
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
    })

utility_syn_df = pd.DataFrame(utility_rows)
save_df(utility_syn_df, "Fig5C_high_risk_utility_synthesis.csv")

if len(utility_syn_df) > 0:
    fig, ax = plt.subplots(figsize=(10.8, 5.4))
    plot_df = utility_syn_df.melt(id_vars="view", value_vars=["precision", "recall", "f1"],
                                  var_name="metric", value_name="score")
    view_order = utility_syn_df["view"].tolist()
    x = np.arange(len(view_order))
    width = 0.22
    color_map = {
        "precision": PALETTE["orange"],
        "recall": PALETTE["blue"],
        "f1": PALETTE["green"],
    }
    offsets = [-width, 0, width]
    for off, met in zip(offsets, ["precision", "recall", "f1"]):
        dd = plot_df[plot_df["metric"] == met].set_index("view").reindex(view_order).reset_index()
        ax.bar(x + off, dd["score"], width=width, color=color_map[met], label=met)
    ax.set_xticks(x)
    ax.set_xticklabels(
        ["binary", "fourclass_collapsed", "fourclass_high_risk_threshold", "eightlabel_suicide_intent", "class255_suicide_intent"],
        rotation=20, ha="right"
    )
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.0)
    ax.legend(frameon=False, ncol=3, loc="upper left")
    style_ax(ax)
    save_fig(fig, "Fig5C_high_risk_utility_synthesis")

def build_stage_summary(task_df, metric_col):
    if len(task_df) == 0 or metric_col not in task_df.columns:
        return pd.DataFrame()
    d = task_df.copy()
    d = d[d[metric_col].notna()].copy()
    if len(d) == 0:
        return pd.DataFrame()
    out = d.groupby("stage_type")[metric_col].agg(["mean", "min", "max", "count"]).reset_index()
    return out

stage_four = build_stage_summary(four_runs_m, "macro_f1")
stage_eight = build_stage_summary(eight_runs_m, "macro_f1")
stage_255 = build_stage_summary(class255_runs_m, "macro_f1")

if len(stage_four) > 0 or len(stage_eight) > 0 or len(stage_255) > 0:
    if len(stage_four) > 0:
        stage_four["task"] = "fourclass"
        stage_four = stage_four.rename(columns={"macro_f1": "metric_value"})
    if len(stage_eight) > 0:
        stage_eight["task"] = "eightlabel"
    if len(stage_255) > 0:
        stage_255["task"] = "class255"
    stage_summary_df = pd.concat([stage_four, stage_eight, stage_255], ignore_index=True, sort=False)
    save_df(stage_summary_df, "Fig5D_stage_transfer_summary.csv")

    fig, ax = plt.subplots(figsize=(10.6, 5.2))
    x_map = {k: i for i, k in enumerate(sorted(stage_summary_df["stage_type"].dropna().unique().tolist()))}
    task_color = {"fourclass": PALETTE["blue"], "eightlabel": PALETTE["green"], "class255": PALETTE["orange"]}
    for task_name, dd in stage_summary_df.groupby("task"):
        xs = [x_map[s] for s in dd["stage_type"]]
        ax.scatter(xs, dd["mean"], s=220, color=task_color.get(task_name, PALETTE["gray"]), label=task_name)
        for _, r in dd.iterrows():
            xi = x_map[r["stage_type"]]
            ax.vlines(xi, r["min"], r["max"], color=task_color.get(task_name, PALETTE["gray"]), lw=2.4)
    ax.set_xticks(list(x_map.values()))
    ax.set_xticklabels(list(x_map.keys()), rotation=20, ha="right")
    ax.set_ylabel("Macro-F1")
    ax.legend(frameon=False, ncol=3, loc="upper left")
    style_ax(ax)
    save_fig(fig, "Fig5D_stage_transfer_summary")


summary_lines = []

if best_four is not None:
    summary_lines.append("FOURCLASS")
    summary_lines.append(str(best_four))
    summary_lines.append("")

if best_eight is not None:
    summary_lines.append("EIGHTLABEL")
    summary_lines.append(str(best_eight))
    summary_lines.append("")

if best_class255 is not None:
    summary_lines.append("CLASS255")
    summary_lines.append(str(best_class255))
    summary_lines.append("")

if len(base_sym_df) > 0:
    # key closure statements
    r1 = corr_or_nan(base_sym_df["centrality"], base_sym_df["class255_commission"], "pearson")
    r2 = corr_or_nan(base_sym_df["centrality"], base_sym_df["class255_omission"], "pearson")
    r3 = np.nan
    if "eightlabel_commission" in base_sym_df.columns and "class255_commission" in base_sym_df.columns:
        r3 = corr_or_nan(base_sym_df["eightlabel_commission"], base_sym_df["class255_commission"], "pearson")
    summary_lines.append("KEY CLOSURE STATISTICS")
    summary_lines.append(f"centrality vs class255 commission: r={r1:.3f}" if pd.notna(r1) else "centrality vs class255 commission: NA")
    summary_lines.append(f"centrality vs class255 omission: r={r2:.3f}" if pd.notna(r2) else "centrality vs class255 omission: NA")
    summary_lines.append(f"eightlabel vs class255 commission consistency: r={r3:.3f}" if pd.notna(r3) else "eightlabel vs class255 commission consistency: NA")
    summary_lines.append("")

save_text("\n".join(summary_lines), "analysis_summary.txt")


print("\n" + "=" * 100)
print("Done.")
print("Figures:", FIG_DIR)
print("Figure CSVs:", CSV_DIR)
print("Tables:", TAB_DIR)
print("Texts:", TXT_DIR)
print("=" * 100)

[Input] prediction dir: D:\Downloads\Depressed\Depressed\export_bundle_v4\predictions
[Input] sample_meta: D:\Downloads\Depressed\Depressed\export_bundle_v4\sample_meta.csv
[Input] final_run_mapping: D:\Downloads\Depressed\Depressed\final_stage_mapping\final_run_task_mapping.csv
[Input] final_stage_group_mapping: D:\Downloads\Depressed\Depressed\final_stage_mapping\final_stage_group_mapping.csv
[Input] final_native_eval_mapping: D:\Downloads\Depressed\Depressed\final_stage_mapping\final_native_eval_mapping.csv
[Input] strict_eval_mapping: None
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs\tables\run_inventory_enriched.csv
[Info] binary runs found: 43
[Info] fourclass runs found: 21
[Info] eightlabel runs found: 12
[Info] class255 runs found: 32
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs\tables\binary_runs_metrics.csv
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs\tables\fourclass_runs_metrics.csv
[Saved Table] D:\Downloads\Depr

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig1C_human_uncertainty_anchor.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig1C_human_uncertainty_anchor.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs\figure_csv\Fig1D_fourclass_threshold_utility.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs\figure_csv\Fig1D2_severe_routing.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig1D_fourclass_risk_sensitive_utility.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig1D_fourclass_risk_sensitive_utility.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs\figure_csv\Fig2_symptom_structure_metrics.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig2ABC_construct_closure.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig2ABC_construct_closure.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_output

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig3B_class255_support_vs_accuracy.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig3B_class255_support_vs_accuracy.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs\figure_csv\Fig3C_class255_symptom_count_confusion.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig3C_class255_symptom_count_confusion.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig3C_class255_symptom_count_confusion.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs\figure_csv\Fig4A_fourclass_cue_dependence.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig4A_fourclass_cue_dependence.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs\figures\Fig4A_fourclass_cue_dependence.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs\figure_csv\Fig4B_class255_cue_dependence.csv
[Saved] D:\Downloads\Depressed

In [ ]:
# -*- coding: utf-8 -*-
"""
Coverage:
1) Boundary mechanism (1↔2 why)
2) Severity -> symptom monotonicity (bridge-derived if available)
3) Core-periphery closure (true vs pred network)
4) Combination topology (graph-level)
5) Human uncertainty strengthening (calibration)
6) Counterfactual cue effect (quasi-counterfactual / stratified effect)
7) High-risk errors (suicide FN profile)
8) Cross-run stability (forest-style)
"""

import os
import re
import glob
import math
import warnings
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    hamming_loss,
)

warnings.filterwarnings("ignore")

ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"

EXPORT_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4")
PRED_DIR = os.path.join(EXPORT_DIR, "predictions")
SAMPLE_META_FP = os.path.join(EXPORT_DIR, "sample_meta.csv")
SUMMARY_FP = os.path.join(EXPORT_DIR, "result_summary.csv")

FINAL_MAP_DIR = os.path.join(ROOT_DEPRESSED, "final_stage_mapping")
FINAL_RUN_MAP_FP = os.path.join(FINAL_MAP_DIR, "final_run_task_mapping.csv")

OUT_DIR = os.path.join(ROOT_DEPRESSED, "final_pub_outputs_fullcover")
OUT_FIG = os.path.join(OUT_DIR, "figures")
OUT_CSV = os.path.join(OUT_DIR, "figure_csv")
OUT_TAB = os.path.join(OUT_DIR, "tables")
OUT_TXT = os.path.join(OUT_DIR, "texts")

for d in [OUT_DIR, OUT_FIG, OUT_CSV, OUT_TAB, OUT_TXT]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.linewidth": 2.0,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "axes.unicode_minus": False,
    "font.size": 15,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
})

DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#202020",
    "lightgray": "#CFCBC3",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "deepblue": "#6EA9C4",
    "deeporange": "#D88537",
    "deepgreen": "#5FA777",
}

CMAP_BLUE = LinearSegmentedColormap.from_list(
    "nature_blue", ["#F7F7F7", "#DCE8EE", "#B9D5E2", "#98CFE6"]
)
CMAP_PINK = LinearSegmentedColormap.from_list(
    "nature_pink", ["#F8F6F7", "#F0DDE7", "#E7C3D7", "#EEB7D3"]
)
CMAP_ORANGE = LinearSegmentedColormap.from_list(
    "nature_orange", ["#FAF8F4", "#F1E2CF", "#F0C48E", "#F39F4E"]
)
CMAP_GREEN = LinearSegmentedColormap.from_list(
    "nature_green", ["#F6F8F5", "#DDEDD9", "#BDE0B6", "#ADE7A8"]
)

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

EMOTION_SHORT = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

KEY_MONO_SYMPTOMS = [
    "sadness",
    "hopelessness",
    "suicide intent",
    "brain dysfunction (forget)"
]

LEXICON = {
    "sadness_explicit": ["sad", "depressed", "cry", "crying", "empty", "down", "miserable"],
    "hopelessness_explicit": ["hopeless", "no future", "nothing will change", "can't go on", "pointless"],
    "worthlessness_explicit": ["worthless", "useless", "burden", "failure", "not good enough", "hate myself"],
    "cognitive_explicit": ["forget", "memory", "focus", "concentrate", "brain fog", "can't think"],
    "suicide_explicit": ["suicide", "suicidal", "kill myself", "end my life", "die", "self-harm", "self harm"],
    "help_seeking": ["please help", "need help", "advice", "suggestions", "what do i do", "help me"],
    "absolutist": ["always", "never", "nothing", "nobody", "completely", "totally", "forever"],
}

def save_fig(fig, stem):
    png_fp = os.path.join(OUT_FIG, f"{stem}.png")
    eps_fp = os.path.join(OUT_FIG, f"{stem}.eps")
    fig.savefig(png_fp, dpi=DPI, facecolor="white", bbox_inches="tight")
    fig.savefig(eps_fp, dpi=DPI, facecolor="white", bbox_inches="tight", format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")

def save_csv(df, name):
    fp = os.path.join(OUT_CSV, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")

def save_table(df, name):
    fp = os.path.join(OUT_TAB, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved Table] {fp}")

def save_text(text, name):
    fp = os.path.join(OUT_TXT, name)
    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[Saved Text] {fp}")

def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.0)
    ax.spines["bottom"].set_linewidth(2.0)
    ax.tick_params(axis="both", width=2.0, length=7)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")

def safe_read_csv(fp):
    if fp is None or (not os.path.exists(fp)):
        return pd.DataFrame()
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.DataFrame()

def parse_model_size_b(model_name):
    if pd.isna(model_name):
        return np.nan
    s = str(model_name)
    m = re.search(r"(\d+(?:\.\d+)?)B", s, re.I)
    return float(m.group(1)) if m else np.nan

def parse_task_from_filename(fp):
    base = os.path.basename(fp).lower()
    if base.startswith("binary__"):
        return "binary"
    if base.startswith("fourclass__"):
        return "fourclass"
    if base.startswith("eightlabel__"):
        return "eightlabel"
    if base.startswith("class255__"):
        return "class255"
    return "unknown"

def parse_model_from_filename(fp):
    parts = os.path.basename(fp).split("__")
    return parts[1] if len(parts) >= 2 else "unknown_model"

def parse_vec8(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) == 8 and set(s) <= {"0", "1"}:
        return [int(c) for c in s]
    nums = re.findall(r"[01]", s)
    if len(nums) == 8:
        return [int(c) for c in nums]
    return None

def vec_to_int(v):
    if v is None:
        return np.nan
    return int("".join(str(int(z)) for z in v), 2)

def hamming(v1, v2):
    return sum(int(a != b) for a, b in zip(v1, v2))

def symptom_count(v):
    return np.nan if v is None else int(sum(v))

def mean_no_nan(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) == 0 else float(x.mean())

def median_no_nan(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) == 0 else float(x.median())

def corr_no_nan(x, y, method="pearson"):
    xx = pd.Series(x)
    yy = pd.Series(y)
    m = xx.notna() & yy.notna()
    if m.sum() < 3:
        return np.nan
    return float(xx[m].corr(yy[m], method=method))

def odds_ratio_2x2(a, b, c, d, eps=1e-9):
    return ((a + eps) * (d + eps)) / ((b + eps) * (c + eps))

def log_or_ci(a, b, c, d, eps=0.5):
    a, b, c, d = a + eps, b + eps, c + eps, d + eps
    lor = np.log((a * d) / (b * c))
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo = np.exp(lor - 1.96 * se)
    hi = np.exp(lor + 1.96 * se)
    return np.exp(lor), lo, hi

def bootstrap_ci(vals, n_boot=500, alpha=0.05, random_state=42):
    vals = np.asarray(pd.Series(vals).dropna(), dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    boots = []
    n = len(vals)
    for _ in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        boots.append(np.mean(vals[idx]))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1-alpha/2))

def has_any(txt, kws):
    t = str(txt).lower()
    return int(any(k in t for k in kws))

def safe_ratio(a, b):
    return np.nan if b == 0 else a / b


def clean_name(s):
    s = str(s)
    s = s.replace("_", " ")
    s = s.replace("brain dysfunction (forget)", "brain dysf.")
    s = s.replace("suicide intent", "suicide")
    s = s.replace("worthlessness", "worthless.")
    s = s.replace("hopelessness", "hopeless.")
    s = s.replace("loneliness", "lonely.")
    s = s.replace("class255", "255-way")
    s = s.replace("eightlabel", "8-label")
    return s

CUE_DISPLAY = {
    "sadness_explicit": "sadness explicit",
    "hopelessness_explicit": "hopelessness explicit",
    "worthlessness_explicit": "worthlessness explicit",
    "cognitive_explicit": "cognitive explicit",
    "suicide_explicit": "suicide explicit",
    "help_seeking": "help seeking",
    "absolutist": "absolutist",
}

VIEW_DISPLAY = {
    "class255_suicide_head": "255-way suicide head",
    "eightlabel_suicide_head": "8-label suicide head",
    "fourclass_high_risk_threshold": "4-class high-risk threshold",
    "binary_depressed": "binary depressed head",
}

METRIC_DISPLAY = {
    "adjacent_error_frac": "Adjacent error fraction",
    "far_error_frac": "Far error fraction",
    "mean_abs_err": "Mean absolute error",
    "share_12_within_adjacent": "Boundary share within adjacent",
    "boundary_confidence_gap": "Boundary confidence gap",
    "sadness_commission": "Sadness commission",
    "brain_omission": "Brain omission",
    "mean_core_shift": "Mean core shift",
    "burden_bias": "Burden bias",
}

FEATURE_DISPLAY = {
    "confidence_score": "confidence",
    "text_len": "text length",
    "first_person_density": "1st-person",
    "negation_density": "negation",
    "emotion_word_density": "emotion words",
}


def choose_bridge_run(primary_df, backup_df):
    if len(primary_df) > 0:
        return primary_df.copy(), "from4to255"
    if len(backup_df) > 0:
        return backup_df.copy(), "from4to8"
    return pd.DataFrame(), None


def prepare_bridge_vectors(bridge_df):
    if len(bridge_df) == 0 or "sample_id" not in bridge_df.columns or "pred_vec" not in bridge_df.columns:
        return pd.DataFrame()
    out = bridge_df.copy()
    out["sample_id"] = out["sample_id"].astype(str)
    out["pred_vec_list"] = out["pred_vec"].map(parse_vec8)
    out = out[out["pred_vec_list"].notna()].copy()
    return out


def combo_terms_from_vec(v):
    if v is None:
        return []
    return [EMOTION_SHORT[EMOTION_NAMES[i]] for i, z in enumerate(v) if int(z) == 1]


def combo_display_label(v, max_terms=3, with_count=True):
    terms = combo_terms_from_vec(v)
    if len(terms) == 0:
        core = "none"
    elif len(terms) <= max_terms:
        core = "+".join(terms)
    else:
        core = "+".join(terms[:max_terms]) + "+..."
    return f"{core} ({len(terms)})" if with_count else core

COMBO_ABBR = {
    "anger": "anger",
    "brain dysf.": "brain",
    "emptiness": "empty",
    "hopeless.": "hopeless",
    "lonely.": "lonely",
    "sadness": "sadness",
    "suicide": "suicide",
    "worthless.": "worthless",
}

def combo_display_compact(v, max_terms=2, with_count=True):
    terms = [COMBO_ABBR.get(t, t) for t in combo_terms_from_vec(v)]
    if len(terms) == 0:
        core = "none"
    elif len(terms) <= max_terms:
        core = "+".join(terms)
    else:
        core = "+".join(terms[:max_terms]) + f" +{len(terms)-max_terms}"
    return f"{core} [{len(terms)}]" if with_count else core


def valid_summary_metric(vals, min_n=3, min_span=1e-6):
    vals = np.asarray(pd.Series(vals).dropna(), dtype=float)
    if len(vals) < min_n:
        return False
    return bool(np.nanmax(vals) - np.nanmin(vals) >= min_span or len(vals) >= 5)

def robust_ylim(vals, floor_zero=True, upper_pad=0.18):
    vals = np.asarray(pd.Series(vals).dropna(), dtype=float)
    if len(vals) == 0:
        return (0.0, 1.0)
    lo = float(vals.min())
    hi = float(vals.max())
    if abs(hi - lo) < 1e-12:
        span = max(abs(hi), 1.0) * 0.2
        lo, hi = lo - span, hi + span
    else:
        span = hi - lo
        lo, hi = lo - 0.12 * span, hi + upper_pad * span
    if floor_zero:
        lo = min(0.0, lo) if lo < 0 else 0.0
    return lo, hi

def add_placeholder(ax, main_text, sub_text=None):
    ax.text(0.5, 0.58, main_text, ha="center", va="center", fontweight="bold", transform=ax.transAxes)
    if sub_text:
        ax.text(0.5, 0.45, sub_text, ha="center", va="center", transform=ax.transAxes, fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    style_ax(ax)

def bootstrap_mean_ci(arr, n_boot=500, alpha=0.05, random_state=42):
    arr = np.asarray(pd.Series(arr).dropna(), dtype=float)
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(arr, size=len(arr), replace=True)
        boots.append(float(np.mean(samp)))
    return float(np.mean(boots)), float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1-alpha/2))

def bootstrap_standardized_diff(a, b, n_boot=500, alpha=0.05, random_state=42):
    a = np.asarray(pd.Series(a).dropna(), dtype=float)
    b = np.asarray(pd.Series(b).dropna(), dtype=float)
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    def _smd(x, y):
        vx = np.var(x, ddof=1) if len(x) > 1 else 0.0
        vy = np.var(y, ddof=1) if len(y) > 1 else 0.0
        pooled = np.sqrt(max((((len(x)-1)*vx) + ((len(y)-1)*vy)) / max(len(x)+len(y)-2, 1), 1e-12))
        return float((np.mean(x) - np.mean(y)) / pooled)
    boots = []
    for _ in range(n_boot):
        xx = rng.choice(a, size=len(a), replace=True)
        yy = rng.choice(b, size=len(b), replace=True)
        boots.append(_smd(xx, yy))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1-alpha/2))

def point_interval_h(ax, df, x_col, lo_col, hi_col, y_labels, color, zero=None, size=180):
    y = np.arange(len(df))
    for i, rr in enumerate(df.itertuples()):
        ax.hlines(i, getattr(rr, lo_col), getattr(rr, hi_col), color=color, lw=2.5)
        ax.scatter(getattr(rr, x_col), i, s=size, color=color, zorder=5)
    if zero is not None:
        ax.axvline(zero, color=PALETTE["lightgray"], lw=1.5)
    ax.set_yticks(y)
    ax.set_yticklabels(y_labels)
    style_ax(ax)


def infer_run_name(df, fp):
    for c in ["run_name"]:
        if c in df.columns and df[c].notna().any():
            return str(df[c].dropna().iloc[0])
    for c in ["source_jsonl", "source_file"]:
        if c in df.columns and df[c].notna().any():
            p = str(df[c].dropna().iloc[0])
            return os.path.basename(os.path.dirname(p)) if c == "source_jsonl" else os.path.splitext(os.path.basename(p))[0]
    return os.path.splitext(os.path.basename(fp))[0]

try:
    import networkx as nx
    HAS_NX = True
except Exception:
    HAS_NX = False

if not os.path.isdir(PRED_DIR):
    raise FileNotFoundError(f"Missing prediction dir:\n{PRED_DIR}")

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, "*.csv")))
if len(pred_files) == 0:
    raise FileNotFoundError(f"No standardized prediction csv found in:\n{PRED_DIR}")

sample_meta = safe_read_csv(SAMPLE_META_FP)
summary_df = safe_read_csv(SUMMARY_FP)
run_map_df = safe_read_csv(FINAL_RUN_MAP_FP)

if len(sample_meta) > 0 and "sample_id" in sample_meta.columns:
    sample_meta["sample_id"] = sample_meta["sample_id"].astype(str)

print("=" * 100)
print("ROOT_DEPRESSED =", ROOT_DEPRESSED)
print("PRED_DIR       =", PRED_DIR)
print("SAMPLE_META_FP =", SAMPLE_META_FP, "| exists =", os.path.exists(SAMPLE_META_FP))
print("SUMMARY_FP     =", SUMMARY_FP, "| exists =", os.path.exists(SUMMARY_FP))
print("FINAL_RUN_MAP  =", FINAL_RUN_MAP_FP, "| exists =", os.path.exists(FINAL_RUN_MAP_FP))
print("OUT_DIR        =", OUT_DIR)
print("Prediction CSVs:", len(pred_files))
print("=" * 100)

all_dfs = []
run_rows = []

for fp in pred_files:
    df = pd.read_csv(fp)

    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)

    task = parse_task_from_filename(fp)
    model_name = parse_model_from_filename(fp)
    run_name = infer_run_name(df, fp)

    df["task"] = task
    df["model_name"] = model_name
    df["model_size_b"] = parse_model_size_b(model_name)
    df["run_name"] = run_name
    df["pred_csv"] = fp

    if len(sample_meta) > 0 and "sample_id" in df.columns:
        df = df.merge(sample_meta, on="sample_id", how="left")

    if "text" not in df.columns:
        df["text"] = ""
    df["text"] = df["text"].astype(str)
    word_count = np.maximum(df["text"].str.split().str.len().fillna(0), 1)

    df["text_len"] = df["text"].str.len()
    df["first_person_density"] = df["text"].str.lower().str.count(r"\b(i|me|my|mine|myself)\b") / word_count
    df["negation_density"] = df["text"].str.lower().str.count(r"\b(no|not|never|nothing|can't|cannot|don't|won't)\b") / word_count
    df["emotion_word_density"] = df["text"].str.lower().str.count(r"\b(sad|hopeless|depressed|cry|empty|worthless|lonely|suicidal|anxious)\b") / word_count

    for cue, kws in LEXICON.items():
        df[cue] = df["text"].map(lambda x: has_any(x, kws))

    all_dfs.append(df)

    run_rows.append({
        "task": task,
        "model_name": model_name,
        "model_size_b": parse_model_size_b(model_name),
        "run_name": run_name,
        "pred_csv": fp
    })

master_df = pd.concat(all_dfs, ignore_index=True, sort=False)
run_inventory = pd.DataFrame(run_rows).drop_duplicates()

if len(summary_df) > 0:
    s = summary_df.copy()
    if "task" in s.columns:
        s = s.rename(columns={"task": "task_summary"})
    if "setting" in s.columns:
        s = s.rename(columns={"setting": "setting_summary"})
    if {"model_name", "run_name"}.issubset(set(s.columns)):
        run_inventory = run_inventory.merge(s, on=["model_name", "run_name"], how="left")

if len(run_map_df) > 0:
    m = run_map_df.copy()
    if "run_name" not in m.columns and "eval_dir_name" in m.columns:
        m["run_name"] = m["eval_dir_name"]
    keep = [c for c in [
        "model_name", "run_name", "final_task", "final_stage_name", "final_stage_role",
        "count_bucket", "pred_file_final"
    ] if c in m.columns]
    if {"model_name", "run_name"}.issubset(set(keep)):
        run_inventory = run_inventory.merge(m[keep].drop_duplicates(), on=["model_name", "run_name"], how="left")

run_inventory["task_final"] = run_inventory["final_task"].fillna(run_inventory["task"]) if "final_task" in run_inventory.columns else run_inventory["task"]

def infer_stage_type(row):
    st = row["final_stage_name"] if "final_stage_name" in row and pd.notna(row["final_stage_name"]) else np.nan
    if pd.notna(st) and str(st).strip():
        return str(st)
    rn = str(row["run_name"]).lower()
    if "from4to255" in rn:
        return "from4to255"
    if "from4to8" in rn:
        return "from4to8"
    if "from2to255" in rn:
        return "from2to255"
    if "from2to8" in rn:
        return "from2to8"
    if "native_eval" in rn:
        return "native"
    return "direct"

run_inventory["stage_type"] = run_inventory.apply(infer_stage_type, axis=1)
if "count_bucket" in run_inventory.columns:
    run_inventory["count_bucket_final"] = run_inventory["count_bucket"].fillna(
        run_inventory["task_final"].map({
            "binary": "deptweet_task",
            "fourclass": "deptweet_task",
            "eightlabel": "depressionemo_task",
            "class255": "depressionemo_task"
        })
    )
else:
    run_inventory["count_bucket_final"] = run_inventory["task_final"].map({
        "binary": "deptweet_task",
        "fourclass": "deptweet_task",
        "eightlabel": "depressionemo_task",
        "class255": "depressionemo_task"
    })

save_table(run_inventory, "run_inventory_master.csv")


def get_run_df(task, model_name, run_name):
    return master_df[
        (master_df["task"] == task) &
        (master_df["model_name"] == model_name) &
        (master_df["run_name"] == run_name)
    ].copy()

def metrics_binary(df):
    d = df.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return {}
    yt = d["true_label"].astype(int).values
    yp = d["pred_label"].astype(int).values
    return {
        "accuracy": accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "n": len(d)
    }

def metrics_fourclass(df):
    d = df.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return {}
    yt = d["true_label"].astype(int).values
    yp = d["pred_label"].astype(int).values
    return {
        "accuracy": accuracy_score(yt, yp),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "n": len(d)
    }

def metrics_multilabel(df):
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    return {
        "exact_match": accuracy_score(Yt, Yp),
        "macro_f1": f1_score(Yt, Yp, average="macro", zero_division=0),
        "micro_f1": f1_score(Yt, Yp, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(Yt, Yp),
        "n": len(d)
    }

def metrics_class255(df):
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    yt255 = d["true_255"].astype(int).values if "true_255" in d.columns else np.array([vec_to_int(v) for v in d["true_vec_list"]])
    yp255 = d["pred_255"].fillna(-1).astype(int).values if "pred_255" in d.columns else np.array([vec_to_int(v) for v in d["pred_vec_list"]])
    return {
        "accuracy": accuracy_score(yt255, yp255),
        "macro_f1": f1_score(yt255, yp255, average="macro", zero_division=0),
        "weighted_f1": f1_score(yt255, yp255, average="weighted", zero_division=0),
        "macro_f1_symptom": f1_score(Yt, Yp, average="macro", zero_division=0),
        "micro_f1_symptom": f1_score(Yt, Yp, average="micro", zero_division=0),
        "n": len(d)
    }

metric_rows = []
for _, r in run_inventory.iterrows():
    d = get_run_df(r["task"], r["model_name"], r["run_name"])
    if r["task"] == "binary":
        met = metrics_binary(d)
    elif r["task"] == "fourclass":
        met = metrics_fourclass(d)
    elif r["task"] == "eightlabel":
        met = metrics_multilabel(d)
    elif r["task"] == "class255":
        met = metrics_class255(d)
    else:
        met = {}
    row = r.to_dict()
    row.update(met)
    metric_rows.append(row)

run_metrics_df = pd.DataFrame(metric_rows)
save_table(run_metrics_df, "run_metrics_fullcover.csv")

def pick_best(df, metric):
    d = df[df[metric].notna()].copy()
    if len(d) == 0:
        return None
    return d.sort_values(metric, ascending=False).iloc[0]

binary_runs = run_metrics_df[run_metrics_df["task"] == "binary"].copy()
four_runs = run_metrics_df[run_metrics_df["task"] == "fourclass"].copy()
eight_runs = run_metrics_df[run_metrics_df["task"] == "eightlabel"].copy()
class255_runs = run_metrics_df[run_metrics_df["task"] == "class255"].copy()

best_binary = pick_best(binary_runs, "f1")
best_four = pick_best(four_runs, "macro_f1")
best_eight = pick_best(eight_runs, "macro_f1")
best_class255 = pick_best(class255_runs, "macro_f1")

def pick_best_bridge(task_target, stage_keyword):
    d = run_metrics_df[
        (run_metrics_df["task"] == task_target) &
        (run_metrics_df["stage_type"].astype(str).str.contains(stage_keyword, case=False, na=False))
    ].copy()
    if len(d) == 0:
        return None
    metric = "macro_f1" if "macro_f1" in d.columns else ("f1" if "f1" in d.columns else None)
    if metric is None:
        return d.iloc[0]
    d = d[d[metric].notna()].copy()
    if len(d) == 0:
        return None
    return d.sort_values(metric, ascending=False).iloc[0]

bridge_4to8 = pick_best_bridge("eightlabel", "from4to8")
bridge_4to255 = pick_best_bridge("class255", "from4to255")

def best_df(best_row):
    if best_row is None:
        return pd.DataFrame()
    return get_run_df(best_row["task"], best_row["model_name"], best_row["run_name"])

best_binary_df = best_df(best_binary)
best_four_df = best_df(best_four)
best_eight_df = best_df(best_eight)
best_class255_df = best_df(best_class255)
bridge_4to8_df = best_df(bridge_4to8)
bridge_4to255_df = best_df(bridge_4to255)

summary_lines = []
summary_lines.append("BEST / BRIDGE RUNS\n")
summary_lines.append(f"best_binary:\n{best_binary}\n")
summary_lines.append(f"best_four:\n{best_four}\n")
summary_lines.append(f"best_eight:\n{best_eight}\n")
summary_lines.append(f"best_class255:\n{best_class255}\n")
summary_lines.append(f"bridge_4to8:\n{bridge_4to8}\n")
summary_lines.append(f"bridge_4to255:\n{bridge_4to255}\n")
save_text("\n".join(summary_lines), "best_and_bridge_runs.txt")

if len(best_four_df) > 0:
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["err"] = d["pred_label"] - d["true_label"]
    d["boundary_dir"] = np.where((d["true_label"] == 1) & (d["pred_label"] == 2), "1→2",
                          np.where((d["true_label"] == 2) & (d["pred_label"] == 1), "2→1", "other"))
    bd = d[d["boundary_dir"].isin(["1→2", "2→1"])].copy()

    feature_rows = []
    for name, dd in bd.groupby("boundary_dir"):
        feature_rows.append({
            "boundary_dir": name,
            "n": len(dd),
            "mean_text_len": mean_no_nan(dd["text_len"]),
            "mean_first_person_density": mean_no_nan(dd["first_person_density"]),
            "mean_negation_density": mean_no_nan(dd["negation_density"]),
            "mean_emotion_word_density": mean_no_nan(dd["emotion_word_density"]),
            "help_seeking_rate": mean_no_nan(dd["help_seeking"]),
            "absolutist_rate": mean_no_nan(dd["absolutist"]),
            "median_confidence": median_no_nan(pd.to_numeric(dd.get("confidence_score", np.nan), errors="coerce")),
            "disagreement_rate": mean_no_nan(pd.to_numeric(dd.get("is_disagreement_sample", 0), errors="coerce"))
        })
    boundary_text_df = pd.DataFrame(feature_rows)
    save_csv(boundary_text_df, "FigA1_boundary_text_style_contrast.csv")

    boundary_symptom_df = pd.DataFrame()
    bridge_df, bridge_source = choose_bridge_run(bridge_4to255_df, bridge_4to8_df)
    bridge_df = prepare_bridge_vectors(bridge_df)

    if len(bridge_df) > 0:
        merged = bd.merge(
            bridge_df[["sample_id", "pred_vec", "pred_vec_list"]],
            on="sample_id",
            how="inner",
            suffixes=("", "_bridge")
        )
        dir_counts = merged.groupby("boundary_dir").size().to_dict() if len(merged) > 0 else {}
        bridge_effective = (
            len(merged) >= 12 and
            dir_counts.get("1→2", 0) >= 5 and
            dir_counts.get("2→1", 0) >= 5
        )
        if bridge_effective:
            sym_rows = []
            for direction, dd in merged.groupby("boundary_dir"):
                if len(dd) == 0:
                    continue
                mat = np.array(dd["pred_vec_list"].tolist(), dtype=int)
                row = {"boundary_dir": direction, "n": len(dd), "bridge_source": bridge_source,
                       "predicted_symptom_burden": float(mat.sum(axis=1).mean())}
                for i, lab in enumerate(EMOTION_NAMES):
                    row[f"pred_{lab}"] = float(mat[:, i].mean())
                sym_rows.append(row)
            boundary_symptom_df = pd.DataFrame(sym_rows)
            save_csv(boundary_symptom_df, "FigA2_boundary_bridge_symptom_contrast.csv")

    case_rank = bd.copy()
    case_rank["confidence_score"] = pd.to_numeric(case_rank.get("confidence_score", np.nan), errors="coerce")
    case_rank["is_disagreement_sample"] = pd.to_numeric(case_rank.get("is_disagreement_sample", 0), errors="coerce").fillna(0)
    case_rank["priority"] = (
        case_rank["is_disagreement_sample"] * 10
        - case_rank["confidence_score"].fillna(case_rank["confidence_score"].median() if case_rank["confidence_score"].notna().any() else 0.5)
        + case_rank["emotion_word_density"]
    )
    top_cases = case_rank.sort_values("priority", ascending=False)[
        ["sample_id", "text", "true_label", "pred_label", "boundary_dir", "confidence_score", "is_disagreement_sample"]
    ].head(30)
    save_table(top_cases, "boundary_representative_cases.csv")

    if len(boundary_symptom_df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8))
        axes = np.asarray(axes)
    else:
        fig, ax_single = plt.subplots(1, 1, figsize=(7.2, 4.8))
        axes = np.asarray([ax_single])

    ax = axes[0]
    text_feats = [
        ("mean_first_person_density", "1st-person"),
        ("mean_negation_density", "Negation"),
        ("mean_emotion_word_density", "Emotion"),
        ("help_seeking_rate", "Help-seek"),
        ("absolutist_rate", "Absolutist")
    ]
    if len(boundary_text_df) > 0:
        plot = boundary_text_df.set_index("boundary_dir")
        x = np.arange(len(text_feats))
        w = 0.34
        vals1 = [plot.loc["1→2", c] if "1→2" in plot.index else np.nan for c, _ in text_feats]
        vals2 = [plot.loc["2→1", c] if "2→1" in plot.index else np.nan for c, _ in text_feats]
        ax.bar(x - w/2, vals1, width=w, color=PALETTE["orange"], label="1→2")
        ax.bar(x + w/2, vals2, width=w, color=PALETTE["blue"], label="2→1")
        ax.set_xticks(x)
        ax.set_xticklabels([lab for _, lab in text_feats], rotation=16, ha="right")
        ax.tick_params(axis="x", labelsize=12)
        ax.set_ylabel("Mean value")
        ax.legend(frameon=False, fontsize=12, loc="upper right")
        _, yhi = robust_ylim(vals1 + vals2, floor_zero=True)
        ax.set_ylim(0, yhi)
    else:
        add_placeholder(ax, "No boundary samples")
    style_ax(ax)

    if len(boundary_symptom_df) > 0:
        ax = axes[1]
        plot = boundary_symptom_df.set_index("boundary_dir")
        key_syms = ["sadness", "hopelessness", "worthlessness", "brain dysfunction (forget)"]
        x = np.arange(len(key_syms))
        w = 0.34
        vals1 = [plot.loc["1→2", f"pred_{s}"] if "1→2" in plot.index else np.nan for s in key_syms]
        vals2 = [plot.loc["2→1", f"pred_{s}"] if "2→1" in plot.index else np.nan for s in key_syms]
        ax.bar(x - w/2, vals1, width=w, color=PALETTE["orange"], label="1→2")
        ax.bar(x + w/2, vals2, width=w, color=PALETTE["blue"], label="2→1")
        ax.set_xticks(x)
        ax.set_xticklabels([EMOTION_SHORT[s] for s in key_syms], rotation=20, ha="right")
        ax.set_ylabel("Bridge-derived prevalence")
        ax.legend(frameon=False)
        _, yhi = robust_ylim(vals1 + vals2, floor_zero=True)
        ax.set_ylim(0, yhi)
        style_ax(ax)

    save_fig(fig, "FigA_boundary_mechanism")


monotonicity_df = pd.DataFrame()
fallback_mono_df = pd.DataFrame()

if len(best_four_df) > 0:
    bridge_df, bridge_source = choose_bridge_run(bridge_4to255_df, bridge_4to8_df)
    bridge_df = prepare_bridge_vectors(bridge_df)

    if len(bridge_df) > 0:
        sev = best_four_df[["sample_id", "true_label", "pred_label"]].copy()
        sev["sample_id"] = sev["sample_id"].astype(str)
        sev["true_label"] = pd.to_numeric(sev["true_label"], errors="coerce")
        sev["pred_label"] = pd.to_numeric(sev["pred_label"], errors="coerce")

        merged = sev.merge(bridge_df[["sample_id", "pred_vec_list"]], on="sample_id", how="inner")
        merged = merged[merged["true_label"].notna()].copy()
        level_counts = merged.groupby(merged["true_label"].astype(int)).size() if len(merged) > 0 else pd.Series(dtype=int)
        bridge_effective = (len(merged) >= 24) and ((level_counts >= 5).sum() >= 3)

        if bridge_effective:
            rows = []
            for s in sorted(merged["true_label"].dropna().astype(int).unique()):
                dd = merged[merged["true_label"].astype(int) == s]
                if len(dd) == 0:
                    continue
                mat = np.array(dd["pred_vec_list"].tolist(), dtype=int)
                row = {"severity": int(s), "n": len(dd), "bridge_source": bridge_source,
                       "mean_symptom_burden": float(mat.sum(axis=1).mean())}
                for i, lab in enumerate(EMOTION_NAMES):
                    row[f"pred_{lab}"] = float(mat[:, i].mean())
                rows.append(row)
            monotonicity_df = pd.DataFrame(rows)
            save_csv(monotonicity_df, "FigB_severity_to_symptom_monotonicity.csv")

    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = pd.to_numeric(d["true_label"], errors="coerce")
    d["pred_label"] = pd.to_numeric(d["pred_label"], errors="coerce")
    cue_cols = ["sadness_explicit", "hopelessness_explicit", "worthlessness_explicit", "cognitive_explicit", "suicide_explicit"]
    for c in cue_cols:
        if c not in d.columns:
            d[c] = 0
    d["lexicon_cue_burden"] = d[cue_cols].sum(axis=1)
    rows = []
    for s in sorted(d["true_label"].dropna().astype(int).unique()):
        dd = d[d["true_label"].astype(int) == s]
        if len(dd) == 0:
            continue
        m_pred, lo_pred, hi_pred = bootstrap_mean_ci(dd["pred_label"])
        row = {
            "severity": int(s),
            "n": len(dd),
            "mean_predicted_severity": m_pred,
            "pred_lo": lo_pred,
            "pred_hi": hi_pred,
            "mean_lexicon_cue_burden": float(dd["lexicon_cue_burden"].mean()),
        }
        for c in cue_cols:
            row[c] = float(dd[c].mean())
        rows.append(row)
    fallback_mono_df = pd.DataFrame(rows)
    if len(fallback_mono_df) > 0:
        save_csv(fallback_mono_df, "FigB_fallback_lexicon_monotonicity.csv")

    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8))

    ax = axes[0]
    if len(monotonicity_df) > 0:
        ax.plot(monotonicity_df["severity"], monotonicity_df["mean_symptom_burden"],
                marker="o", ms=8, lw=2.6, color=PALETTE["deeporange"])
        ax.set_xlabel("True severity")
        ax.set_ylabel("Bridge-derived symptom burden")
        ax.set_xticks(monotonicity_df["severity"].tolist())
        lo, hi = robust_ylim(monotonicity_df["mean_symptom_burden"].values, floor_zero=True)
        ax.set_ylim(lo, hi)
        style_ax(ax)
    elif len(fallback_mono_df) > 0:
        ax.plot(fallback_mono_df["severity"], fallback_mono_df["mean_predicted_severity"],
                marker="o", ms=8, lw=2.6, color=PALETTE["deeporange"])
        for rr in fallback_mono_df.itertuples():
            ax.vlines(rr.severity, rr.pred_lo, rr.pred_hi, color=PALETTE["deeporange"], lw=2.0)
        ax.set_xlabel("True severity")
        ax.set_ylabel("Mean predicted severity")
        ax.set_xticks(fallback_mono_df["severity"].tolist())
        lo, hi = robust_ylim(list(fallback_mono_df["pred_lo"].values) + list(fallback_mono_df["pred_hi"].values), floor_zero=False)
        ax.set_ylim(lo, hi)
        style_ax(ax)
    else:
        add_placeholder(ax, "No monotonicity data")

    ax = axes[1]
    if len(monotonicity_df) > 0:
        sym_cols = ["sadness", "hopelessness", "suicide intent", "brain dysfunction (forget)"]
        sym_colors = [PALETTE["blue"], PALETTE["orange"], PALETTE["red"], PALETTE["purple"]]
        for sym, col in zip(sym_cols, sym_colors):
            colname = f"pred_{sym}"
            if colname in monotonicity_df.columns:
                ax.plot(monotonicity_df["severity"], monotonicity_df[colname],
                        marker="o", ms=7, lw=2.2, color=col, label=EMOTION_SHORT[sym])
        ax.set_xlabel("True severity")
        ax.set_ylabel("Bridge-derived symptom prevalence")
        ax.set_xticks(monotonicity_df["severity"].tolist())
        ax.legend(frameon=False, ncol=2)
        style_ax(ax)
    elif len(fallback_mono_df) > 0:
        ax.plot(fallback_mono_df["severity"], fallback_mono_df["mean_lexicon_cue_burden"],
                marker="o", ms=8, lw=2.6, color=PALETTE["deepblue"])
        ax.set_xlabel("True severity")
        ax.set_ylabel("Lexicon-derived cue burden")
        ax.set_xticks(fallback_mono_df["severity"].tolist())
        lo, hi = robust_ylim(fallback_mono_df["mean_lexicon_cue_burden"].values, floor_zero=True)
        ax.set_ylim(lo, hi)
        style_ax(ax)
    else:
        add_placeholder(ax, "No bridge or fallback panel")

    save_fig(fig, "FigB_severity_to_symptom_monotonicity")

def cooccurrence_matrix(Y):
    co = Y.T @ Y
    np.fill_diagonal(co, 0)
    return co

def centrality_from_matrix(co):
    deg = co.sum(axis=1).astype(float)
    return deg / max(deg.max(), 1.0)

core_df = pd.DataFrame()
if len(best_eight_df) > 0:
    d = best_eight_df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()

    if len(d) > 0:
        Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
        Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)

        co_t = cooccurrence_matrix(Yt)
        co_p = cooccurrence_matrix(Yp)

        cent_t = centrality_from_matrix(co_t)
        cent_p = centrality_from_matrix(co_p)

        prev_t = Yt.mean(axis=0)
        prev_p = Yp.mean(axis=0)

        rows = []
        for i, lab in enumerate(EMOTION_NAMES):
            rows.append({
                "label": lab,
                "label_short": EMOTION_SHORT[lab],
                "true_centrality": cent_t[i],
                "pred_centrality": cent_p[i],
                "true_prevalence": prev_t[i],
                "pred_prevalence": prev_p[i],
                "prevalence_shift": prev_p[i] - prev_t[i]
            })
        core_df = pd.DataFrame(rows).sort_values("true_centrality", ascending=False).reset_index(drop=True)
        save_csv(core_df, "FigC_core_periphery_true_vs_pred_network.csv")

        co_true_df = pd.DataFrame(co_t, index=EMOTION_NAMES, columns=EMOTION_NAMES).reset_index().rename(columns={"index": "label"})
        co_pred_df = pd.DataFrame(co_p, index=EMOTION_NAMES, columns=EMOTION_NAMES).reset_index().rename(columns={"index": "label"})
        save_csv(co_true_df, "FigC1_true_network_matrix.csv")
        save_csv(co_pred_df, "FigC2_pred_network_matrix.csv")

        fig, axes = plt.subplots(1, 3, figsize=(17.2, 4.9))

        ax = axes[0]
        ax.scatter(core_df["true_centrality"], core_df["pred_centrality"], s=230, color=PALETTE["blue"], edgecolor="white", linewidth=0.8)
        lo = min(core_df["true_centrality"].min(), core_df["pred_centrality"].min())
        hi = max(core_df["true_centrality"].max(), core_df["pred_centrality"].max())
        pad = 0.06 * (hi - lo if hi > lo else 1.0)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=PALETTE["lightgray"], lw=1.6)
        r = corr_no_nan(core_df["true_centrality"], core_df["pred_centrality"])
        offsets = [(0.012, 0.010), (-0.060, 0.010), (0.010, -0.015), (-0.055, -0.015)]
        for idx, rr in enumerate(core_df.itertuples()):
            dx, dy = offsets[idx % len(offsets)]
            ax.text(rr.true_centrality + dx, rr.pred_centrality + dy, rr.label_short, fontsize=10, fontweight="bold")
        ax.text(0.98, 0.97, f"r={r:.2f}" if pd.notna(r) else "r=NA", ha="right", va="top", transform=ax.transAxes, fontweight="bold")
        ax.set_xlim(lo - 2*pad, hi + 2*pad)
        ax.set_ylim(lo - 2*pad, hi + 2*pad)
        ax.set_xlabel("True centrality")
        ax.set_ylabel("Predicted centrality")
        style_ax(ax)

        ax = axes[1]
        x = np.arange(len(core_df))
        ax.plot(x, core_df["true_prevalence"], marker="o", lw=2.4, color=PALETTE["deepgreen"], label="True")
        ax.plot(x, core_df["pred_prevalence"], marker="o", lw=2.4, color=PALETTE["orange"], label="Pred")
        ax.set_xticks(x)
        ax.set_xticklabels(core_df["label_short"], rotation=30, ha="right")
        ax.set_ylabel("Prevalence")
        ax.legend(frameon=False)
        style_ax(ax)

        ax = axes[2]
        cols = [PALETTE["orange"] if v > 0 else PALETTE["blue"] for v in core_df["prevalence_shift"]]
        x = np.arange(len(core_df))
        ax.bar(x, core_df["prevalence_shift"], color=cols, width=0.62)
        ax.axhline(0, color=PALETTE["lightgray"], lw=1.5)
        ax.set_xticks(x)
        ax.set_xticklabels(core_df["label_short"], rotation=30, ha="right")
        ax.set_ylabel("Pred − true prevalence")
        style_ax(ax)

        save_fig(fig, "FigC_core_periphery_true_vs_pred_network")

topology_nodes_df = pd.DataFrame()
topology_edges_df = pd.DataFrame()
topology_transition_df = pd.DataFrame()

if len(best_class255_df) > 0:
    d = best_class255_df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    d["true_255"] = d["true_255"].astype(int)
    d["pred_255"] = d["pred_255"].fillna(-1).astype(int)
    d["correct"] = (d["true_255"] == d["pred_255"]).astype(int)

    comb_stats = d.groupby("true_255").agg(
        support=("true_255", "size"),
        accuracy=("correct", "mean"),
        true_vec=("true_vec", "first")
    ).reset_index().sort_values("support", ascending=False)

    top_combs = comb_stats.head(18).copy()
    top_combs["true_vec_list"] = top_combs["true_vec"].map(parse_vec8)

    node_rows = []
    for _, rr in top_combs.iterrows():
        node_rows.append({
            "node": int(rr["true_255"]),
            "support": int(rr["support"]),
            "accuracy": float(rr["accuracy"]),
            "true_vec": rr["true_vec"],
            "symptom_count": symptom_count(rr["true_vec_list"]),
            "display_label": combo_display_compact(rr["true_vec_list"], max_terms=2, with_count=True),
            "short_transition_label": combo_display_compact(rr["true_vec_list"], max_terms=2, with_count=False),
            "full_label": combo_display_label(rr["true_vec_list"], max_terms=8, with_count=False)
        })
    topology_nodes_df = pd.DataFrame(node_rows)
    save_csv(topology_nodes_df, "FigD1_topology_nodes.csv")

    edge_rows = []
    for i in range(len(top_combs)):
        for j in range(i + 1, len(top_combs)):
            a = top_combs.iloc[i]
            b = top_combs.iloc[j]
            va, vb = a["true_vec_list"], b["true_vec_list"]
            dist = hamming(va, vb)
            if dist <= 2:
                edge_rows.append({
                    "node_a": int(a["true_255"]),
                    "node_b": int(b["true_255"]),
                    "hamming_distance": int(dist)
                })
    topology_edges_df = pd.DataFrame(edge_rows)
    save_csv(topology_edges_df, "FigD2_topology_edges.csv")

    label_map = topology_nodes_df.set_index("node")["short_transition_label"].to_dict() if len(topology_nodes_df) > 0 else {}
    trans = d.groupby(["true_255", "pred_255"]).size().reset_index(name="count")
    trans = trans[trans["true_255"] != trans["pred_255"]].sort_values("count", ascending=False).head(30)
    trans["transition_label"] = trans.apply(
        lambda r: f'{label_map.get(int(r["true_255"]), str(int(r["true_255"])))} → {label_map.get(int(r["pred_255"]), str(int(r["pred_255"])))}',
        axis=1
    )
    topology_transition_df = trans.copy()
    save_csv(topology_transition_df, "FigD3_top_confusion_transitions.csv")

    combo_lookup = topology_nodes_df[["node", "full_label", "display_label"]].copy() if len(topology_nodes_df) > 0 else pd.DataFrame()
    if len(combo_lookup) > 0:
        save_csv(combo_lookup, "FigD4_combination_label_lookup.csv")

    fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.0))

    ax = axes[0]
    plot_df = topology_nodes_df.sort_values(["support", "accuracy"], ascending=[False, False]).head(8).sort_values("accuracy", ascending=True)
    y = np.arange(len(plot_df))
    ax.hlines(y, 0, plot_df["accuracy"], color=PALETTE["lightgray"], lw=2.2)
    ax.scatter(plot_df["accuracy"], y,
               s=np.clip(plot_df["support"].values * 6.0, 90, 900),
               c=plot_df["symptom_count"].values, cmap=CMAP_ORANGE, edgecolor="white", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["display_label"])
    ax.tick_params(axis="y", labelsize=11)
    ax.set_xlabel("Combination accuracy")
    ax.set_xlim(0, 1.05)
    for i, rr in enumerate(plot_df.itertuples()):
        ax.text(min(rr.accuracy + 0.022, 1.03), i, f'n={rr.support}', va="center", fontsize=8.5, fontweight="bold")
    style_ax(ax)

    ax = axes[1]
    top_trans = topology_transition_df.head(6).copy().sort_values("count", ascending=True)
    y = np.arange(len(top_trans))
    ax.barh(y, top_trans["count"], color=PALETTE["pink"], height=0.62)
    ax.set_yticks(y)
    ax.set_yticklabels(top_trans["transition_label"])
    ax.tick_params(axis="y", labelsize=12)
    ax.set_xlabel("Transition count")
    style_ax(ax)

    save_fig(fig, "FigD_combination_topology")

calibration_df = pd.DataFrame()
or_df = pd.DataFrame()

if len(best_four_df) > 0 and "confidence_score" in best_four_df.columns:
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["confidence_score"] = pd.to_numeric(d["confidence_score"], errors="coerce")
    d["is_disagreement_sample"] = pd.to_numeric(d.get("is_disagreement_sample", 0), errors="coerce").fillna(0).astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()
    d["error_type"] = np.where(d["abs_err"] == 0, "Correct", np.where(d["abs_err"] == 1, "Adjacent", "Far"))
    d["is_error"] = (d["abs_err"] > 0).astype(int)

    d2 = d[d["confidence_score"].notna()].copy()
    if len(d2) > 0:
        d2["conf_bin"] = pd.qcut(d2["confidence_score"], q=min(5, d2["confidence_score"].nunique()), duplicates="drop")
        cal_rows = []
        for bin_name, dd in d2.groupby("conf_bin"):
            cal_rows.append({
                "conf_bin": str(bin_name),
                "n": len(dd),
                "mean_confidence": float(dd["confidence_score"].mean()),
                "error_rate": float(dd["is_error"].mean()),
                "adjacent_error_rate": float((dd["abs_err"] == 1).mean()),
                "far_error_rate": float((dd["abs_err"] >= 2).mean()),
                "correct_rate": float((dd["abs_err"] == 0).mean())
            })
        calibration_df = pd.DataFrame(cal_rows).sort_values("mean_confidence")
        save_csv(calibration_df, "FigE1_calibration_curve.csv")

    rows = []
    ref = d[d["error_type"] == "Correct"]
    a0 = int(ref["is_disagreement_sample"].sum())
    b0 = int(len(ref) - a0)
    for et in ["Adjacent", "Far"]:
        dd = d[d["error_type"] == et]
        a = int(dd["is_disagreement_sample"].sum())
        b = int(len(dd) - a)
        if len(dd) == 0 or len(ref) == 0:
            continue
        or_val, lo, hi = log_or_ci(a, b, a0, b0)
        rows.append({
            "contrast": f"{et} vs Correct",
            "odds_ratio": or_val,
            "ci_low": lo,
            "ci_high": hi,
            "log_odds_ratio": float(np.log(or_val)),
            "log_ci_low": float(np.log(lo)),
            "log_ci_high": float(np.log(hi)),
            "group_rate": float(dd["is_disagreement_sample"].mean()),
            "correct_rate": float(ref["is_disagreement_sample"].mean()),
            "n_group": len(dd)
        })
    or_df = pd.DataFrame(rows)
    save_csv(or_df, "FigE2_disagreement_odds_forest.csv")

    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8))

    ax = axes[0]
    if len(calibration_df) > 0:
        ax.plot(calibration_df["mean_confidence"], calibration_df["error_rate"],
                marker="o", ms=8, lw=2.6, color=PALETTE["orange"], label="Overall error")
        ax.plot(calibration_df["mean_confidence"], calibration_df["adjacent_error_rate"],
                marker="o", ms=7, lw=2.2, color=PALETTE["blue"], label="Adjacent error")
        ax.plot(calibration_df["mean_confidence"], calibration_df["far_error_rate"],
                marker="o", ms=7, lw=2.2, color=PALETTE["pink"], label="Far error")
        ax.set_xlabel("Mean annotation confidence")
        ax.set_ylabel("Error probability")
        ax.legend(frameon=False)
    else:
        add_placeholder(ax, "No calibration data")
    style_ax(ax)

    ax = axes[1]
    if len(or_df) > 0:
        plot = or_df.copy().sort_values("log_odds_ratio", ascending=True).reset_index(drop=True)
        point_interval_h(ax, plot, "log_odds_ratio", "log_ci_low", "log_ci_high", plot["contrast"], PALETTE["deepblue"], zero=0.0)
        ax.set_xlabel("Log odds ratio of annotator disagreement")
    else:
        add_placeholder(ax, "No disagreement enrichment data")
    style_ax(ax)

    save_fig(fig, "FigE_human_uncertainty_strengthening")


cue_effect_df = pd.DataFrame()

def stratified_effect(df, cue_col, y_col, strata_cols):
    dd = df.copy()
    dd = dd[df[cue_col].isin([0, 1])].copy()
    out = []
    grouped = dd.groupby(strata_cols) if len(strata_cols) > 0 else [(("all",), dd)]
    for key, g in grouped:
        pos = g[g[cue_col] == 1]
        neg = g[g[cue_col] == 0]
        if len(pos) < 5 or len(neg) < 5:
            continue
        out.append({
            "stratum": str(key),
            "n_pos": len(pos),
            "n_neg": len(neg),
            "effect": float(pos[y_col].mean() - neg[y_col].mean())
        })
    if len(out) == 0:
        return np.nan, np.nan, np.nan, len(out)
    effects = pd.DataFrame(out)["effect"].values
    m, lo, hi = bootstrap_ci(effects)
    return m, lo, hi, len(out)

if len(best_four_df) > 0:
    d = best_four_df.dropna(subset=["true_label", "pred_label"]).copy()
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["pred_severity"] = d["pred_label"].astype(float)
    d["text_len_bin"] = pd.qcut(d["text_len"], q=3, duplicates="drop")

    cue_rows = []
    for cue in ["sadness_explicit", "hopelessness_explicit", "worthlessness_explicit",
                "cognitive_explicit", "suicide_explicit", "help_seeking", "absolutist"]:
        m, lo, hi, nstrata = stratified_effect(
            d, cue, "pred_severity", ["true_label", "text_len_bin"]
        )
        cue_rows.append({
            "task": "fourclass",
            "cue": cue,
            "cue_display": CUE_DISPLAY[cue],
            "effect_target": "predicted_severity",
            "mean_effect": m,
            "ci_low": lo,
            "ci_high": hi,
            "n_strata": nstrata
        })
    cue_effect_df = pd.DataFrame(cue_rows)
    save_csv(cue_effect_df, "FigF1_quasi_counterfactual_cue_effect_fourclass.csv")

cue_effect_255_df = pd.DataFrame()
if len(best_class255_df) > 0:
    d = best_class255_df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    d["pred_burden"] = d["pred_vec_list"].map(symptom_count)
    d["suicide_fn"] = d["true_vec_list"].map(lambda v: v[6]).astype(int)
    d["suicide_pred"] = d["pred_vec_list"].map(lambda v: v[6]).astype(int)
    d["suicide_fn"] = ((d["suicide_fn"] == 1) & (d["suicide_pred"] == 0)).astype(int)
    d["text_len_bin"] = pd.qcut(d["text_len"], q=3, duplicates="drop")

    rows = []
    for cue in ["sadness_explicit", "hopelessness_explicit", "worthlessness_explicit",
                "cognitive_explicit", "suicide_explicit", "help_seeking", "absolutist"]:
        m1, lo1, hi1, n1 = stratified_effect(d, cue, "pred_burden", ["text_len_bin"])
        m2, lo2, hi2, n2 = stratified_effect(d, cue, "suicide_fn", ["text_len_bin"])
        rows.append({
            "cue": cue,
            "cue_display": CUE_DISPLAY[cue],
            "effect_target": "predicted_burden",
            "mean_effect": m1,
            "ci_low": lo1,
            "ci_high": hi1,
            "n_strata": n1
        })
        rows.append({
            "cue": cue,
            "cue_display": CUE_DISPLAY[cue],
            "effect_target": "suicide_fn_rate",
            "mean_effect": m2,
            "ci_low": lo2,
            "ci_high": hi2,
            "n_strata": n2
        })
    cue_effect_255_df = pd.DataFrame(rows)
    save_csv(cue_effect_255_df, "FigF2_quasi_counterfactual_cue_effect_class255.csv")

if len(cue_effect_df) > 0 or len(cue_effect_255_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14.6, 5.4))

    ax = axes[0]
    plot = cue_effect_df.dropna(subset=["mean_effect"]).copy().sort_values("mean_effect", ascending=True).reset_index(drop=True)
    if len(plot) > 0:
        point_interval_h(ax, plot, "mean_effect", "ci_low", "ci_high", plot["cue_display"], PALETTE["orange"], zero=0.0)
        ax.set_xlabel("Effect on predicted severity")
    else:
        add_placeholder(ax, "No severity cue effect")

    ax = axes[1]
    plot = cue_effect_255_df[cue_effect_255_df["effect_target"] == "suicide_fn_rate"].dropna(subset=["mean_effect"]).copy()
    plot = plot.sort_values("mean_effect", ascending=True).reset_index(drop=True)
    if len(plot) > 0:
        point_interval_h(ax, plot, "mean_effect", "ci_low", "ci_high", plot["cue_display"], PALETTE["blue"], zero=0.0)
        ax.set_xlabel("Effect on suicide FN rate")
    else:
        add_placeholder(ax, "No suicide FN cue effect")

    save_fig(fig, "FigF_quasi_counterfactual_cue_effect")

suicide_profile_df = pd.DataFrame()
routing_df = pd.DataFrame()
profile_effect_df = pd.DataFrame()

if len(best_class255_df) > 0:
    d255 = best_class255_df.copy()
    d255["true_vec_list"] = d255["true_vec"].map(parse_vec8)
    d255["pred_vec_list"] = d255["pred_vec"].map(parse_vec8)
    d255 = d255[d255["true_vec_list"].notna() & d255["pred_vec_list"].notna()].copy()
    d255["true_suicide"] = d255["true_vec_list"].map(lambda v: v[6]).astype(int)
    d255["pred_suicide"] = d255["pred_vec_list"].map(lambda v: v[6]).astype(int)

    pos = d255[d255["true_suicide"] == 1].copy()
    pos["profile"] = np.where(pos["pred_suicide"] == 1, "TP", "FN")

    rows = []
    for name, dd in pos.groupby("profile"):
        if len(dd) == 0:
            continue
        mat = np.array(dd["true_vec_list"].tolist(), dtype=int)
        row = {
            "profile": name,
            "n": len(dd),
            "mean_text_len": mean_no_nan(dd["text_len"]),
            "mean_confidence": mean_no_nan(pd.to_numeric(dd.get("confidence_score", np.nan), errors="coerce")),
            "mean_first_person_density": mean_no_nan(dd["first_person_density"]),
            "mean_negation_density": mean_no_nan(dd["negation_density"]),
            "mean_emotion_word_density": mean_no_nan(dd["emotion_word_density"]),
        }
        for i, lab in enumerate(EMOTION_NAMES):
            row[f"true_{lab}"] = float(mat[:, i].mean())
        rows.append(row)
    suicide_profile_df = pd.DataFrame(rows)
    save_csv(suicide_profile_df, "FigG1_suicide_fn_profile.csv")

    feature_rows = []
    fn_df = pos[pos["profile"] == "FN"].copy()
    tp_df = pos[pos["profile"] == "TP"].copy()
    for feat in ["confidence_score", "text_len", "first_person_density", "negation_density", "emotion_word_density"]:
        fn_vals = pd.to_numeric(fn_df.get(feat, np.nan), errors="coerce")
        tp_vals = pd.to_numeric(tp_df.get(feat, np.nan), errors="coerce")
        m, lo, hi = bootstrap_standardized_diff(fn_vals, tp_vals)
        feature_rows.append({
            "feature": feat,
            "feature_display": FEATURE_DISPLAY[feat],
            "mean_effect": m,
            "ci_low": lo,
            "ci_high": hi
        })
    profile_effect_df = pd.DataFrame(feature_rows)
    save_csv(profile_effect_df, "FigG3_profile_effect_sizes.csv")

    route_rows = []
    base_ids = set(pos["sample_id"].astype(str).tolist())

    route_rows.append({
        "view": "class255_suicide_head",
        "view_display": VIEW_DISPLAY["class255_suicide_head"],
        "captured_rate": float((pos["pred_suicide"] == 1).mean()) if len(pos) > 0 else np.nan,
        "n": len(pos)
    })

    if len(best_eight_df) > 0:
        d8 = best_eight_df.copy()
        d8["sample_id"] = d8["sample_id"].astype(str)
        d8["pred_vec_list"] = d8["pred_vec"].map(parse_vec8)
        d8 = d8[d8["pred_vec_list"].notna()].copy()
        d8 = d8[d8["sample_id"].isin(base_ids)].copy()
        if len(d8) > 0:
            route_rows.append({
                "view": "eightlabel_suicide_head",
                "view_display": VIEW_DISPLAY["eightlabel_suicide_head"],
                "captured_rate": float(d8["pred_vec_list"].map(lambda v: v[6]).mean()),
                "n": len(d8)
            })

    if len(best_four_df) > 0:
        d4 = best_four_df.copy()
        d4["sample_id"] = d4["sample_id"].astype(str)
        d4["pred_label"] = pd.to_numeric(d4["pred_label"], errors="coerce")
        d4 = d4[d4["sample_id"].isin(base_ids)].copy()
        if len(d4) > 0:
            route_rows.append({
                "view": "fourclass_high_risk_threshold",
                "view_display": VIEW_DISPLAY["fourclass_high_risk_threshold"],
                "captured_rate": float((d4["pred_label"] >= 2).mean()),
                "n": len(d4)
            })

    if len(best_binary_df) > 0:
        db = best_binary_df.copy()
        db["sample_id"] = db["sample_id"].astype(str)
        db["pred_label"] = pd.to_numeric(db["pred_label"], errors="coerce")
        db = db[db["sample_id"].isin(base_ids)].copy()
        if len(db) > 0:
            route_rows.append({
                "view": "binary_depressed",
                "view_display": VIEW_DISPLAY["binary_depressed"],
                "captured_rate": float((db["pred_label"] == 1).mean()),
                "n": len(db)
            })

    routing_df = pd.DataFrame(route_rows)
    save_csv(routing_df, "FigG2_high_risk_routing.csv")

    preferred = ["eightlabel_suicide_head", "class255_suicide_head"]
    route_plot = routing_df[routing_df["view"].isin(preferred)].dropna(subset=["captured_rate"]).copy()
    route_spread = (route_plot["captured_rate"].max() - route_plot["captured_rate"].min()) if len(route_plot) >= 2 else np.nan
    show_routing_panel = bool(len(route_plot) >= 2 and pd.notna(route_spread) and route_spread >= 0.03)

    if show_routing_panel:
        fig, axes = plt.subplots(1, 2, figsize=(14.4, 5.2))
        axes = np.asarray(axes)
    else:
        fig, ax_single = plt.subplots(1, 1, figsize=(7.6, 5.2))
        axes = np.asarray([ax_single])

    ax = axes[0]
    plot = profile_effect_df.dropna(subset=["mean_effect"]).copy().sort_values("mean_effect", ascending=True).reset_index(drop=True)
    if len(plot) > 0:
        point_interval_h(ax, plot, "mean_effect", "ci_low", "ci_high", plot["feature_display"], PALETTE["orange"], zero=0.0)
        ax.set_xlabel("Standardized FN − TP difference")
    else:
        add_placeholder(ax, "No FN/TP profile contrast")

    if show_routing_panel:
        ax = axes[1]
        plot = route_plot.sort_values("captured_rate", ascending=True).reset_index(drop=True)
        color_map = {
            "eightlabel_suicide_head": PALETTE["gray"],
            "class255_suicide_head": PALETTE["pink"],
        }
        y = np.arange(len(plot))
        ax.barh(y, plot["captured_rate"], color=[color_map.get(v, PALETTE["gray"]) for v in plot["view"]], height=0.62)
        ax.set_yticks(y)
        ax.set_yticklabels(plot["view_display"])
        ax.set_xlabel("Capture rate among true suicide samples")
        ax.set_xlim(0, min(1.0, max(0.82, plot["captured_rate"].max() * 1.12)))
        for i, rr in enumerate(plot.itertuples()):
            ax.text(min(rr.captured_rate + 0.02, 0.98), i, f'{rr.captured_rate:.2f}', va="center", fontsize=10, fontweight="bold")
        style_ax(ax)

    save_fig(fig, "FigG_high_risk_error_profile")

def compute_four_stability(run_row):
    d = get_run_df(run_row["task"], run_row["model_name"], run_row["run_name"]).copy()
    d = d.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return None
    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()
    adjacent = (d["abs_err"] == 1).mean()
    far = (d["abs_err"] >= 2).mean()
    mean_abs_err = float(d["abs_err"].mean())
    adj = d[d["abs_err"] == 1]
    share12 = (
        (((adj["true_label"] == 1) & (adj["pred_label"] == 2)) |
         ((adj["true_label"] == 2) & (adj["pred_label"] == 1))).mean()
        if len(adj) > 0 else np.nan
    )
    boundary = d[
        ((d["true_label"] == 1) & (d["pred_label"] == 2)) |
        ((d["true_label"] == 2) & (d["pred_label"] == 1))
    ].copy()
    non_boundary_adj = d[(d["abs_err"] == 1) & ~(
        ((d["true_label"] == 1) & (d["pred_label"] == 2)) |
        ((d["true_label"] == 2) & (d["pred_label"] == 1))
    )].copy()

    bc = median_no_nan(pd.to_numeric(boundary.get("confidence_score", np.nan), errors="coerce"))
    oc = median_no_nan(pd.to_numeric(non_boundary_adj.get("confidence_score", np.nan), errors="coerce"))
    return {
        "mean_abs_err": mean_abs_err,
        "adjacent_error_frac": adjacent,
        "far_error_frac": far,
        "share_12_within_adjacent": share12,
        "boundary_confidence_gap": bc - oc if pd.notna(bc) and pd.notna(oc) else np.nan
    }

def compute_multilabel_stability(run_row):
    d = get_run_df(run_row["task"], run_row["model_name"], run_row["run_name"]).copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return None
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)

    sad_i = EMOTION_NAMES.index("sadness")
    brain_i = EMOTION_NAMES.index("brain dysfunction (forget)")
    sadness_comm = ((Yt[:, sad_i] == 0) & (Yp[:, sad_i] == 1)).sum() / max((Yt[:, sad_i] == 0).sum(), 1)
    brain_omit = ((Yt[:, brain_i] == 1) & (Yp[:, brain_i] == 0)).sum() / max((Yt[:, brain_i] == 1).sum(), 1)

    co = cooccurrence_matrix(Yt)
    w = centrality_from_matrix(co)
    true_score = (Yt * w).sum(axis=1)
    pred_score = (Yp * w).sum(axis=1)
    core_shift = np.mean(pred_score - true_score)
    burden_bias = float(Yp.sum(axis=1).mean() - Yt.sum(axis=1).mean())

    return {
        "sadness_commission": sadness_comm,
        "brain_omission": brain_omit,
        "mean_core_shift": core_shift,
        "burden_bias": burden_bias
    }

four_stab_rows = []
for _, r in four_runs.iterrows():
    out = compute_four_stability(r)
    if out is None:
        continue
    out.update({
        "task": "fourclass",
        "model_name": r["model_name"],
        "model_size_b": r["model_size_b"],
        "run_name": r["run_name"],
        "stage_type": r["stage_type"]
    })
    four_stab_rows.append(out)
four_stab_df = pd.DataFrame(four_stab_rows)
save_csv(four_stab_df, "FigH1_fourclass_stability_metrics.csv")

multi_stab_rows = []
for source_df, task_name in [(eight_runs, "eightlabel"), (class255_runs, "class255")]:
    for _, r in source_df.iterrows():
        out = compute_multilabel_stability(r)
        if out is None:
            continue
        out.update({
            "task": task_name,
            "model_name": r["model_name"],
            "model_size_b": r["model_size_b"],
            "run_name": r["run_name"],
            "stage_type": r["stage_type"]
        })
        multi_stab_rows.append(out)
multi_stab_df = pd.DataFrame(multi_stab_rows)
save_csv(multi_stab_df, "FigH2_multilabel_stability_metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(15.2, 5.0))

ax = axes[0]
summary = []
if len(four_stab_df) > 0:
    left_metric_order = ["boundary_confidence_gap", "adjacent_error_frac", "mean_abs_err"]
    for metric in left_metric_order:
        vals = four_stab_df[metric].dropna().values
        if not valid_summary_metric(vals):
            continue
        m, lo, hi = bootstrap_ci(vals)
        summary.append({"metric": metric, "metric_display": METRIC_DISPLAY[metric], "mean": m, "lo": lo, "hi": hi})
four_summary_df = pd.DataFrame(summary)
save_csv(four_summary_df, "FigH3_fourclass_forest_summary.csv")
if len(four_summary_df) > 0:
    plot = four_summary_df.copy()
    plot["metric_display"] = plot["metric_display"].map({
        "Boundary confidence gap": "Boundary confidence gap",
        "Adjacent error fraction": "Adjacent error fraction",
        "Mean absolute error": "Mean absolute error",
    })
    plot["abs_mean"] = plot["mean"].abs()
    plot = plot.sort_values("abs_mean", ascending=True).reset_index(drop=True)
    point_interval_h(ax, plot, "mean", "lo", "hi", plot["metric_display"], PALETTE["orange"], zero=0.0, size=210)
    xmin = min(plot["lo"].min(), -0.02)
    xmax = max(plot["hi"].max(), 0.02)
    pad = max(0.05, 0.08 * (xmax - xmin))
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_xlabel("Mean ± bootstrap interval")
else:
    add_placeholder(ax, "No four-class stability")

ax = axes[1]
summary = []
if len(multi_stab_df) > 0:
    right_metric_order = ["burden_bias", "mean_core_shift", "brain_omission"]
    for task_name in ["eightlabel", "class255"]:
        dd = multi_stab_df[multi_stab_df["task"] == task_name]
        for metric in right_metric_order:
            vals = dd[metric].dropna().values
            if not valid_summary_metric(vals):
                continue
            m, lo, hi = bootstrap_ci(vals)
            summary.append({
                "task": task_name,
                "metric": metric,
                "metric_display": METRIC_DISPLAY[metric],
                "mean": m, "lo": lo, "hi": hi
            })
multi_summary_df = pd.DataFrame(summary)
save_csv(multi_summary_df, "FigH4_multilabel_forest_summary.csv")
if len(multi_summary_df) > 0:
    metric_order = ["Burden bias", "Mean core shift", "Brain omission"]
    metric_to_y = {m: i for i, m in enumerate(metric_order)}
    task_specs = [
        ("eightlabel", clean_name("eightlabel"), PALETTE["blue"], -0.12),
        ("class255", clean_name("class255"), PALETTE["pink"], +0.12),
    ]
    for task_name, task_label, col, yoff in task_specs:
        dd = multi_summary_df[multi_summary_df["task"] == task_name].copy()
        if len(dd) == 0:
            continue
        dd["plot_y"] = dd["metric_display"].map(metric_to_y).astype(float) + yoff
        for rr in dd.itertuples(index=True):
            ax.hlines(rr.plot_y, rr.lo, rr.hi, color=col, lw=2.8)
            ax.scatter(rr.mean, rr.plot_y, s=190, color=col, zorder=5, label=task_label if rr.Index == dd.index[0] else None)
    ax.axvline(0, color=PALETTE["lightgray"], lw=1.5)
    ax.set_yticks(np.arange(len(metric_order)))
    ax.set_yticklabels(metric_order)
    xmin = min(multi_summary_df["lo"].min(), -0.05)
    xmax = max(multi_summary_df["hi"].max(), 0.05)
    pad = max(0.06, 0.08 * (xmax - xmin))
    ax.set_xlim(xmin - pad, xmax + pad)
    ax.set_xlabel("Mean ± bootstrap interval")
    ax.legend(frameon=False, fontsize=12, loc="lower right")
    style_ax(ax)
else:
    add_placeholder(ax, "No multilabel stability")

save_fig(fig, "FigH_cross_run_stability_forest")

# =========================================================
# 15. FINAL SUMMARY TEXT
# =========================================================
final_lines = []
final_lines.append("FINAL FULL-COVER ANALYSIS COMPLETED\n")
final_lines.append("Coverage status:")
final_lines.append("1) Boundary mechanism (1↔2 why): DONE (main figure uses only strong-evidence panels)")
final_lines.append("2) Severity → symptom monotonicity: DONE (effective bridge gating + robust fallback)")
final_lines.append("3) Core–periphery closure (true vs pred network): DONE")
final_lines.append("4) Combination topology (graph-level): DONE")
final_lines.append("5) Human uncertainty strengthening (calibration): DONE")
final_lines.append("6) Quasi-counterfactual cue effect: DONE")
final_lines.append("7) High-risk errors (suicide FN profile): DONE")
final_lines.append("8) Cross-run stability (forest-style): DONE")
final_lines.append("")
final_lines.append(f"Best binary: {best_binary['model_name']} | {best_binary['run_name']}" if best_binary is not None else "Best binary: None")
final_lines.append(f"Best fourclass: {best_four['model_name']} | {best_four['run_name']}" if best_four is not None else "Best fourclass: None")
final_lines.append(f"Best eightlabel: {best_eight['model_name']} | {best_eight['run_name']}" if best_eight is not None else "Best eightlabel: None")
final_lines.append(f"Best class255: {best_class255['model_name']} | {best_class255['run_name']}" if best_class255 is not None else "Best class255: None")
final_lines.append(f"Bridge 4→8: {bridge_4to8['model_name']} | {bridge_4to8['run_name']}" if bridge_4to8 is not None else "Bridge 4→8: None")
final_lines.append(f"Bridge 4→255: {bridge_4to255['model_name']} | {bridge_4to255['run_name']}" if bridge_4to255 is not None else "Bridge 4→255: None")
save_text("\n".join(final_lines), "final_fullcover_summary.txt")

print("\n" + "=" * 100)
print("FINAL FULL-COVER ANALYSIS DONE")
print("Input root :", ROOT_DEPRESSED)
print("Read from  :", PRED_DIR)
print("Write to   :", OUT_DIR)
print("Figures    :", OUT_FIG)
print("Figure CSV :", OUT_CSV)
print("Tables     :", OUT_TAB)
print("Texts      :", OUT_TXT)
print("=" * 100)

ROOT_DEPRESSED = D:\Downloads\Depressed\Depressed
PRED_DIR       = D:\Downloads\Depressed\Depressed\export_bundle_v4\predictions
SAMPLE_META_FP = D:\Downloads\Depressed\Depressed\export_bundle_v4\sample_meta.csv | exists = True
SUMMARY_FP     = D:\Downloads\Depressed\Depressed\export_bundle_v4\result_summary.csv | exists = True
FINAL_RUN_MAP  = D:\Downloads\Depressed\Depressed\final_stage_mapping\final_run_task_mapping.csv | exists = True
OUT_DIR        = D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover
Prediction CSVs: 108
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\tables\run_inventory_master.csv
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\tables\run_metrics_fullcover.csv
[Saved Text] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\texts\best_and_bridge_runs.txt
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigA1_boundary_text_style_contrast.csv
[Saved Table] D

In [ ]:
# -*- coding: utf-8 -*-
"""

I. Transition-specific specificity
J. Boundary-local human ambiguity
K. Transition feature decomposition
------------
1) I is evaluated separately for 0→1 / 1→2 / 2→3 instead of a single omnibus test.
2) J is restricted to the local 1↔2 boundary and adaptively chooses the most
   informative human-ambiguity target among available supervision signals.
3) K decomposes stage mechanisms with feature-level standardized effects and CI.

"""

import os
import re
import glob
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"
EXPORT_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4")
PRED_DIR = os.path.join(EXPORT_DIR, "predictions")
SAMPLE_META_FP = os.path.join(EXPORT_DIR, "sample_meta.csv")
SUMMARY_FP = os.path.join(EXPORT_DIR, "result_summary.csv")
FINAL_MAP_DIR = os.path.join(ROOT_DEPRESSED, "final_stage_mapping")
FINAL_RUN_MAP_FP = os.path.join(FINAL_MAP_DIR, "final_run_task_mapping.csv")

OUT_DIR = os.path.join(ROOT_DEPRESSED, "final_pub_outputs_fullcover")
OUT_FIG = os.path.join(OUT_DIR, "figures")
OUT_CSV = os.path.join(OUT_DIR, "figure_csv")
OUT_TXT = os.path.join(OUT_DIR, "texts")
for d in [OUT_DIR, OUT_FIG, OUT_CSV, OUT_TXT]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.linewidth": 2.0,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "axes.unicode_minus": False,
    "font.size": 15,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
})
DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#202020",
    "lightgray": "#CFCBC3",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "deepblue": "#6EA9C4",
    "deeporange": "#D88537",
    "deepgreen": "#5FA777",
}

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness",
]

LEXICON = {
    "sadness_explicit": ["sad", "depressed", "cry", "crying", "empty", "down", "miserable"],
    "hopelessness_explicit": ["hopeless", "no future", "nothing will change", "can't go on", "pointless"],
    "worthlessness_explicit": ["worthless", "useless", "burden", "failure", "not good enough", "hate myself"],
    "cognitive_explicit": ["forget", "memory", "focus", "concentrate", "brain fog", "can't think"],
    "suicide_explicit": ["suicide", "suicidal", "kill myself", "end my life", "die", "self-harm", "self harm"],
    "help_seeking": ["please help", "need help", "advice", "suggestions", "what do i do", "help me"],
    "absolutist": ["always", "never", "nothing", "nobody", "completely", "totally", "forever"],
}

def save_fig(fig, stem):
    png_fp = os.path.join(OUT_FIG, f"{stem}.png")
    eps_fp = os.path.join(OUT_FIG, f"{stem}.eps")
    fig.savefig(png_fp, dpi=DPI, facecolor="white", bbox_inches="tight")
    fig.savefig(eps_fp, dpi=DPI, facecolor="white", bbox_inches="tight", format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")


def save_csv(df, name):
    fp = os.path.join(OUT_CSV, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")


def save_text(text, name):
    fp = os.path.join(OUT_TXT, name)
    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[Saved Text] {fp}")


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.0)
    ax.spines["bottom"].set_linewidth(2.0)
    ax.tick_params(axis="both", width=2.0, length=7)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")


def safe_read_csv(fp):
    if fp is None or (not os.path.exists(fp)):
        return pd.DataFrame()
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.DataFrame()


def parse_model_size_b(model_name):
    if pd.isna(model_name):
        return np.nan
    s = str(model_name)
    m = re.search(r"(\d+(?:\.\d+)?)B", s, re.I)
    return float(m.group(1)) if m else np.nan


def parse_task_from_filename(fp):
    base = os.path.basename(fp).lower()
    if base.startswith("binary__"):
        return "binary"
    if base.startswith("fourclass__"):
        return "fourclass"
    if base.startswith("eightlabel__"):
        return "eightlabel"
    if base.startswith("class255__"):
        return "class255"
    return "unknown"


def parse_model_from_filename(fp):
    parts = os.path.basename(fp).split("__")
    return parts[1] if len(parts) >= 2 else "unknown_model"


def infer_run_name(df, fp):
    for c in ["run_name"]:
        if c in df.columns and df[c].notna().any():
            return str(df[c].dropna().iloc[0])
    for c in ["source_jsonl", "source_file"]:
        if c in df.columns and df[c].notna().any():
            p = str(df[c].dropna().iloc[0])
            return os.path.basename(os.path.dirname(p)) if c == "source_jsonl" else os.path.splitext(os.path.basename(p))[0]
    return os.path.splitext(os.path.basename(fp))[0]


def parse_vec8(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) == 8 and set(s) <= {"0", "1"}:
        return [int(c) for c in s]
    nums = re.findall(r"[01]", s)
    if len(nums) == 8:
        return [int(c) for c in nums]
    return None


def symptom_count(v):
    return np.nan if v is None else int(sum(v))


def has_any(txt, kws):
    t = str(txt).lower()
    return int(any(k in t for k in kws))


def mean_no_nan(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) == 0 else float(x.mean())


def corr_no_nan(x, y):
    xx = pd.Series(x)
    yy = pd.Series(y)
    m = xx.notna() & yy.notna()
    if m.sum() < 3:
        return np.nan
    return float(xx[m].corr(yy[m]))


def zscore_series(x):
    s = pd.to_numeric(pd.Series(x), errors="coerce")
    mu = s.mean()
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd


def bootstrap_mean_ci(vals, n_boot=500, alpha=0.05, random_state=42):
    vals = np.asarray(pd.Series(vals).dropna(), dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    n = len(vals)
    boots = []
    for _ in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        boots.append(float(np.mean(vals[idx])))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


def effect_size_d(x_hi, x_lo):
    a = pd.to_numeric(pd.Series(x_hi), errors="coerce").dropna().values.astype(float)
    b = pd.to_numeric(pd.Series(x_lo), errors="coerce").dropna().values.astype(float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    va = a.var(ddof=1)
    vb = b.var(ddof=1)
    pooled = np.sqrt(((len(a) - 1) * va + (len(b) - 1) * vb) / max(len(a) + len(b) - 2, 1))
    if pooled == 0 or np.isnan(pooled):
        return np.nan
    return float((a.mean() - b.mean()) / pooled)


def bootstrap_effect_ci(x_hi, x_lo, n_boot=400, alpha=0.05, random_state=42):
    hi = pd.to_numeric(pd.Series(x_hi), errors="coerce").dropna().values.astype(float)
    lo = pd.to_numeric(pd.Series(x_lo), errors="coerce").dropna().values.astype(float)
    if len(hi) < 3 or len(lo) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    vals = []
    for _ in range(n_boot):
        hi_s = hi[rng.choice(np.arange(len(hi)), size=len(hi), replace=True)]
        lo_s = lo[rng.choice(np.arange(len(lo)), size=len(lo), replace=True)]
        vals.append(effect_size_d(hi_s, lo_s))
    vals = np.asarray([v for v in vals if np.isfinite(v)])
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    return float(vals.mean()), float(np.quantile(vals, alpha / 2)), float(np.quantile(vals, 1 - alpha / 2))


def rank_bins(series, n_bins=3):
    s = pd.to_numeric(pd.Series(series), errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype=object)
    m = s.notna()
    if m.sum() < max(12, n_bins * 5):
        return out
    ranks = s[m].rank(method="average", pct=True)
    cuts = np.linspace(0, 1, n_bins + 1)
    labels = [f"Q{i+1}" for i in range(n_bins)]
    # include lowest to ensure all data are assigned
    out.loc[m] = pd.cut(ranks, bins=cuts, labels=labels, include_lowest=True, duplicates="drop").astype(str)
    return out


def choose_n_splits(y, max_splits=5):
    y = pd.Series(y).dropna().astype(int)
    if len(y) == 0:
        return None
    counts = y.value_counts()
    if len(counts) < 2:
        return None
    min_count = int(counts.min())
    if min_count < 2:
        return None
    return max(2, min(max_splits, min_count))


def _ensure_feature_columns(df):
    df = df.copy()
    if "log_text_len" not in df.columns and "text_len" in df.columns:
        tl = pd.to_numeric(df["text_len"], errors="coerce")
        df["log_text_len"] = np.log1p(tl.clip(lower=0))
    if "text_len" not in df.columns and "log_text_len" in df.columns:
        lt = pd.to_numeric(df["log_text_len"], errors="coerce")
        df["text_len"] = np.expm1(lt)
    alias_pairs = [
        ("first_person_density", ["first_person_ratio", "pronoun_first_person_density"]),
        ("negation_density", ["negation_ratio"]),
        ("emotion_word_density", ["emotion_density", "affect_word_density"]),
    ]
    for target, aliases in alias_pairs:
        if target not in df.columns:
            for a in aliases:
                if a in df.columns:
                    df[target] = pd.to_numeric(df[a], errors="coerce")
                    break
    return df


def _resolve_feature_cols(df, feature_cols, min_non_na=1):
    df = _ensure_feature_columns(df)
    keep = []
    for c in feature_cols:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notna().sum() >= min_non_na:
            keep.append(c)
    return df, list(dict.fromkeys(keep))


def cv_logistic_binary(df, target_col, feature_cols, random_state=42):
    if target_col not in df.columns:
        return None
    dd, feature_cols = _resolve_feature_cols(df.copy(), feature_cols, min_non_na=3)
    if len(feature_cols) == 0:
        return None
    X = dd[feature_cols].apply(pd.to_numeric, errors="coerce")
    y = pd.to_numeric(dd[target_col], errors="coerce")
    req_non_na = max(1, int(np.ceil(0.6 * len(feature_cols))))
    mask = y.notna() & (X.notna().sum(axis=1) >= req_non_na)
    X = X.loc[mask].copy()
    y = y.loc[mask].astype(int)
    if len(X) < 30 or y.nunique() < 2:
        return None
    med = X.median(numeric_only=True)
    X = X.fillna(med).fillna(0.0)
    n_splits = choose_n_splits(y)
    if n_splits is None:
        return None
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    aucs, aps, oof_prob = [], [], np.zeros(len(y), dtype=float)
    for tr, te in skf.split(X, y):
        xtr, xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        if ytr.nunique() < 2 or yte.nunique() < 2:
            continue
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear", random_state=random_state)),
        ])
        clf.fit(xtr, ytr)
        prob = clf.predict_proba(xte)[:, 1]
        oof_prob[te] = prob
        aucs.append(roc_auc_score(yte, prob))
        aps.append(average_precision_score(yte, prob))
    if len(aucs) == 0:
        return None
    return {
        "auc": float(np.mean(aucs)),
        "ap": float(np.mean(aps)),
        "n": int(len(X)),
        "positive_rate": float(y.mean()),
        "features": feature_cols,
        "oof_prob": oof_prob,
        "y": y.values,
        "index": X.index,
    }


def normalize_name(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())


def get_col(df, candidates):
    name_map = {normalize_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_name(cand)
        if key in name_map:
            return name_map[key]
    return None


def add_text_features(df):
    if "text" not in df.columns:
        df["text"] = ""
    txt = df["text"].astype(str)
    wc = np.maximum(txt.str.split().str.len().fillna(0), 1)
    df["text_len"] = txt.str.len()
    df["first_person_density"] = txt.str.lower().str.count(r"\b(i|me|my|mine|myself)\b") / wc
    df["negation_density"] = txt.str.lower().str.count(r"\b(no|not|never|nothing|can't|cannot|don't|won't)\b") / wc
    df["emotion_word_density"] = txt.str.lower().str.count(r"\b(sad|hopeless|depressed|cry|empty|worthless|lonely|suicidal|anxious)\b") / wc
    for cue, kws in LEXICON.items():
        df[cue] = txt.map(lambda x: has_any(x, kws))
    return df


def pick_best_run(metric_df, task, metric="macro_f1"):
    d = metric_df[metric_df["task"] == task].copy()
    if metric not in d.columns:
        return None
    d = d[d[metric].notna()].copy()
    if len(d) == 0:
        return None
    return d.sort_values(metric, ascending=False).iloc[0]


def pick_best_bridge(metric_df, task_target, stage_keyword):
    d = metric_df[(metric_df["task"] == task_target) & (metric_df["stage_type"].astype(str).str.contains(stage_keyword, case=False, na=False))].copy()
    metric = "macro_f1" if "macro_f1" in d.columns else None
    if metric is None or len(d) == 0:
        return None
    d = d[d[metric].notna()].copy()
    if len(d) == 0:
        return None
    return d.sort_values(metric, ascending=False).iloc[0]


def get_run_df(master_df, task, model_name, run_name):
    return master_df[(master_df["task"] == task) & (master_df["model_name"] == model_name) & (master_df["run_name"] == run_name)].copy()

if not os.path.isdir(PRED_DIR):
    raise FileNotFoundError(f"Missing prediction dir:\n{PRED_DIR}")
pred_files = sorted(glob.glob(os.path.join(PRED_DIR, "*.csv")))
if len(pred_files) == 0:
    raise FileNotFoundError(f"No prediction csv found in:\n{PRED_DIR}")

sample_meta = safe_read_csv(SAMPLE_META_FP)
summary_df = safe_read_csv(SUMMARY_FP)
run_map_df = safe_read_csv(FINAL_RUN_MAP_FP)
if len(sample_meta) > 0 and "sample_id" in sample_meta.columns:
    sample_meta["sample_id"] = sample_meta["sample_id"].astype(str)

all_dfs, run_rows = [], []
for fp in pred_files:
    df = pd.read_csv(fp)
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    task = parse_task_from_filename(fp)
    model_name = parse_model_from_filename(fp)
    run_name = infer_run_name(df, fp)
    df["task"] = task
    df["model_name"] = model_name
    df["model_size_b"] = parse_model_size_b(model_name)
    df["run_name"] = run_name
    df["pred_csv"] = fp
    if len(sample_meta) > 0 and "sample_id" in df.columns:
        df = df.merge(sample_meta, on="sample_id", how="left")
    df = add_text_features(df)
    all_dfs.append(df)
    run_rows.append({
        "task": task,
        "model_name": model_name,
        "model_size_b": parse_model_size_b(model_name),
        "run_name": run_name,
        "pred_csv": fp,
    })

master_df = pd.concat(all_dfs, ignore_index=True, sort=False)
run_inventory = pd.DataFrame(run_rows).drop_duplicates()

if len(summary_df) > 0 and {"model_name", "run_name"}.issubset(summary_df.columns):
    # avoid clobbering core registry columns such as task/model_size_b/pred_csv during merge
    sum_keep = ["model_name", "run_name"]
    sum_keep += [c for c in summary_df.columns if c not in set(run_inventory.columns) | {"model_name", "run_name"}]
    run_inventory = run_inventory.merge(summary_df[sum_keep].drop_duplicates(), on=["model_name", "run_name"], how="left")

if len(run_map_df) > 0:
    m = run_map_df.copy()
    if "run_name" not in m.columns and "eval_dir_name" in m.columns:
        m["run_name"] = m["eval_dir_name"]
    keep = [c for c in ["model_name", "run_name", "final_task", "final_stage_name"] if c in m.columns]
    if {"model_name", "run_name"}.issubset(keep):
        run_inventory = run_inventory.merge(m[keep].drop_duplicates(), on=["model_name", "run_name"], how="left")

# recover canonical columns if any previous merge introduced suffixed names
for base in ["task", "model_size_b", "pred_csv"]:
    if base not in run_inventory.columns:
        for alt in [f"{base}_x", f"{base}_left", f"{base}_orig", f"{base}_y"]:
            if alt in run_inventory.columns:
                run_inventory[base] = run_inventory[alt]
                break


def infer_stage_type(row):
    st = row["final_stage_name"] if "final_stage_name" in row and pd.notna(row["final_stage_name"]) else np.nan
    if pd.notna(st) and str(st).strip():
        return str(st)
    rn = str(row["run_name"]).lower()
    if "from4to255" in rn:
        return "from4to255"
    if "from4to8" in rn:
        return "from4to8"
    return "direct"


run_inventory["stage_type"] = run_inventory.apply(infer_stage_type, axis=1)

def metrics_fourclass(df):
    d = df.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return {}
    yt = d["true_label"].astype(int).values
    yp = d["pred_label"].astype(int).values
    # macro_f1 may already be in summary, but keep a simple proxy if absent
    from sklearn.metrics import f1_score
    return {"macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)), "n": len(d)}


def metrics_multilabel(df):
    d = _ensure_feature_columns(df.copy())
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    from sklearn.metrics import f1_score
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    return {"macro_f1": float(f1_score(Yt, Yp, average="macro", zero_division=0)), "n": len(d)}

metric_rows = []
for _, r in run_inventory.iterrows():
    task_val = r.get("task", r.get("task_x", r.get("task_y", np.nan)))
    d = get_run_df(master_df, task_val, r["model_name"], r["run_name"])
    met = {}
    if task_val == "fourclass":
        met = metrics_fourclass(d)
    elif task_val in ["eightlabel", "class255"] and "pred_vec" in d.columns and "true_vec" in d.columns:
        met = metrics_multilabel(d)
    row = r.to_dict()
    row.update(met)
    metric_rows.append(row)
run_metrics_df = pd.DataFrame(metric_rows)

best_four = pick_best_run(run_metrics_df, "fourclass", metric="macro_f1")
bridge_4to8 = pick_best_bridge(run_metrics_df, "eightlabel", "from4to8")
bridge_4to255 = pick_best_bridge(run_metrics_df, "class255", "from4to255")

if best_four is None:
    raise RuntimeError("No valid fourclass run found for I/J/K deepening.")

best_four_df = get_run_df(master_df, best_four.get("task", best_four.get("task_x", best_four.get("task_y"))), best_four["model_name"], best_four["run_name"])
bridge_4to8_df = get_run_df(master_df, bridge_4to8.get("task", bridge_4to8.get("task_x", bridge_4to8.get("task_y"))), bridge_4to8["model_name"], bridge_4to8["run_name"]) if bridge_4to8 is not None else pd.DataFrame()
bridge_4to255_df = get_run_df(master_df, bridge_4to255.get("task", bridge_4to255.get("task_x", bridge_4to255.get("task_y"))), bridge_4to255["model_name"], bridge_4to255["run_name"]) if bridge_4to255 is not None else pd.DataFrame()

def choose_bridge_for_four():
    if len(bridge_4to255_df) > 0:
        return bridge_4to255_df.copy(), "from4to255"
    if len(bridge_4to8_df) > 0:
        return bridge_4to8_df.copy(), "from4to8"
    return pd.DataFrame(), None


def build_bridge_feature_table(bridge_df):
    if len(bridge_df) == 0 or "sample_id" not in bridge_df.columns or "pred_vec" not in bridge_df.columns:
        return pd.DataFrame()
    b = bridge_df.copy()
    b["sample_id"] = b["sample_id"].astype(str)
    b["pred_vec_list"] = b["pred_vec"].map(parse_vec8)
    b = b[b["pred_vec_list"].notna()].copy()
    if len(b) == 0:
        return pd.DataFrame()

    def _get(v, i):
        try:
            return int(v[i])
        except Exception:
            return 0

    out = pd.DataFrame({"sample_id": b["sample_id"].astype(str)})
    out["bridge_symptom_burden"] = b["pred_vec_list"].map(symptom_count).astype(float)
    out["bridge_hopelessness"] = b["pred_vec_list"].map(lambda v: _get(v, 3)).astype(float)
    out["bridge_sadness"] = b["pred_vec_list"].map(lambda v: _get(v, 5)).astype(float)
    out["bridge_suicide"] = b["pred_vec_list"].map(lambda v: _get(v, 6)).astype(float)
    out["bridge_worthlessness"] = b["pred_vec_list"].map(lambda v: _get(v, 7)).astype(float)
    out["bridge_distortion_core"] = out[["bridge_hopelessness", "bridge_worthlessness"]].sum(axis=1)
    out["bridge_risk_core"] = out[["bridge_hopelessness", "bridge_suicide", "bridge_worthlessness"]].sum(axis=1)
    return out.drop_duplicates(subset=["sample_id"])


def prepare_four_deep_df():
    d = best_four_df.copy()
    if len(d) == 0:
        return pd.DataFrame(), None
    if "sample_id" in d.columns:
        d["sample_id"] = d["sample_id"].astype(str)
    d["true_label"] = pd.to_numeric(d.get("true_label"), errors="coerce")
    d["pred_label"] = pd.to_numeric(d.get("pred_label"), errors="coerce")
    d["confidence_score"] = pd.to_numeric(d.get("confidence_score"), errors="coerce")
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()

    # human-supervision columns (robust matching)
    disagree_col = get_col(d, ["is_disagreement_sample", "disagreement", "human_disagreement", "annotator_disagreement"])
    human_conf_col = get_col(d, ["human_confidence", "annotator_confidence", "human_confidence_score", "confidence_human"])
    if disagree_col is not None:
        d["is_disagreement_sample"] = pd.to_numeric(d[disagree_col], errors="coerce").fillna(0).astype(int)
    else:
        d["is_disagreement_sample"] = 0
    if human_conf_col is not None:
        d["human_confidence"] = pd.to_numeric(d[human_conf_col], errors="coerce")
        thr = float(d["human_confidence"].quantile(0.30)) if d["human_confidence"].notna().sum() >= 20 else np.nan
        d["low_human_confidence"] = ((d["human_confidence"].notna()) & (d["human_confidence"] <= thr)).astype(int) if np.isfinite(thr) else 0
    else:
        d["human_confidence"] = np.nan
        d["low_human_confidence"] = 0

    # cue structure
    d["cue_burden"] = d[[c for c in LEXICON if c in d.columns]].sum(axis=1).astype(float)
    d["distortion_core"] = d[[c for c in ["hopelessness_explicit", "worthlessness_explicit", "absolutist"] if c in d.columns]].sum(axis=1).astype(float)
    d["risk_combo"] = d[[c for c in ["suicide_explicit", "hopelessness_explicit", "worthlessness_explicit"] if c in d.columns]].sum(axis=1).astype(float)
    d["help_hopeless_conflict"] = ((d.get("help_seeking", 0) == 1) & (d.get("hopelessness_explicit", 0) == 1)).astype(int)

    bridge_df, bridge_source = choose_bridge_for_four()
    bf = build_bridge_feature_table(bridge_df)
    if len(bf) > 0 and "sample_id" in d.columns:
        d = d.merge(bf, on="sample_id", how="left")
    else:
        bridge_source = None

    return d, bridge_source

def get_feature_groups(df):
    df = _ensure_feature_columns(df.copy())
    control_candidates = [
        "log_text_len", "text_len", "first_person_density", "negation_density",
        "emotion_word_density", "confidence_score"
    ]
    cue_candidates = [
        "sadness_explicit", "hopelessness_explicit", "worthlessness_explicit", "cognitive_explicit",
        "suicide_explicit", "help_seeking", "absolutist", "cue_burden", "distortion_core",
        "risk_combo", "help_hopeless_conflict"
    ]
    bridge_candidates = [
        "bridge_symptom_burden", "bridge_hopelessness", "bridge_sadness", "bridge_suicide",
        "bridge_worthlessness", "bridge_distortion_core", "bridge_risk_core"
    ]
    def _avail(cols):
        out=[]
        for c in cols:
            if c in df.columns and pd.to_numeric(df[c], errors="coerce").notna().sum() > 0:
                out.append(c)
        return list(dict.fromkeys(out))
    return _avail(control_candidates), _avail(cue_candidates), _avail(bridge_candidates)

deep_four_df, deep_bridge_source = prepare_four_deep_df()
deep_four_df = _ensure_feature_columns(deep_four_df)

CONTROL_COLS = [c for c in [
    "text_len", "first_person_density", "negation_density", "emotion_word_density", "confidence_score"
] if c in deep_four_df.columns and pd.to_numeric(deep_four_df[c], errors="coerce").notna().sum() > 0]
CUE_COLS = [c for c in [
    "sadness_explicit", "hopelessness_explicit", "worthlessness_explicit", "cognitive_explicit",
    "suicide_explicit", "help_seeking", "absolutist", "cue_burden", "distortion_core",
    "risk_combo", "help_hopeless_conflict"
] if c in deep_four_df.columns and pd.to_numeric(deep_four_df[c], errors="coerce").notna().sum() > 0]
BRIDGE_COLS = [c for c in [
    "bridge_symptom_burden", "bridge_hopelessness", "bridge_sadness", "bridge_suicide",
    "bridge_worthlessness", "bridge_distortion_core", "bridge_risk_core"
] if c in deep_four_df.columns and pd.to_numeric(deep_four_df[c], errors="coerce").notna().sum() > 0]
STRUCTURE_COLS = list(dict.fromkeys(CUE_COLS + BRIDGE_COLS))

I_nested_rows, I_gain_rows = [], []
transition_order = ["0→1", "1→2", "2→3"]
for lo, hi in [(0, 1), (1, 2), (2, 3)]:
    sub = deep_four_df[deep_four_df["true_label"].isin([lo, hi])].copy()
    if len(sub) < 40:
        continue
    sub["upper_state"] = (sub["true_label"] == hi).astype(int)
    label = f"{lo}→{hi}"
    perf = {}
    for name, cols in [
        ("Controls", CONTROL_COLS),
        ("Controls + cues", list(dict.fromkeys(CONTROL_COLS + CUE_COLS))),
        ("Controls + cues + bridge", list(dict.fromkeys(CONTROL_COLS + CUE_COLS + BRIDGE_COLS))),
    ]:
        use_cols = [c for c in cols if c in sub.columns]
        if len(use_cols) == 0:
            continue
        res = cv_logistic_binary(sub, "upper_state", use_cols)
        if res is None:
            continue
        perf[name] = res
        I_nested_rows.append({
            "transition": label,
            "spec": name,
            "n": res["n"],
            "cv_auc": res["auc"],
            "cv_ap": res["ap"],
            "positive_rate": res["positive_rate"],
            "feature_count": len(res["features"]),
            "bridge_source": deep_bridge_source,
        })
    base_auc = perf["Controls"]["auc"] if "Controls" in perf else np.nan
    cue_auc = perf["Controls + cues"]["auc"] if "Controls + cues" in perf else np.nan
    full_auc = perf["Controls + cues + bridge"]["auc"] if "Controls + cues + bridge" in perf else np.nan
    I_gain_rows.append({
        "transition": label,
        "controls_auc": base_auc,
        "cue_increment": cue_auc - base_auc if np.isfinite(base_auc) and np.isfinite(cue_auc) else np.nan,
        "bridge_increment": full_auc - cue_auc if np.isfinite(full_auc) and np.isfinite(cue_auc) else np.nan,
        "full_auc": full_auc if np.isfinite(full_auc) else cue_auc,
    })

I_nested_df = pd.DataFrame(I_nested_rows)
I_gain_df = pd.DataFrame(I_gain_rows)
save_csv(I_nested_df, "FigI1_transition_specific_nested_cv.csv")
save_csv(I_gain_df, "FigI2_transition_specific_gains.csv")

if len(I_nested_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.9))
    ax = axes[0]
    xpos = np.arange(len(transition_order))
    colors = {
        "Controls": PALETTE["gray"],
        "Controls + cues": PALETTE["orange"],
        "Controls + cues + bridge": PALETTE["blue"],
    }
    for spec in ["Controls", "Controls + cues", "Controls + cues + bridge"]:
        dd = I_nested_df[I_nested_df["spec"] == spec].set_index("transition")
        vals = [dd.loc[t, "cv_auc"] if t in dd.index else np.nan for t in transition_order]
        if np.isfinite(np.asarray(vals, dtype=float)).sum() == 0:
            continue
        ax.plot(xpos, vals, marker="o", ms=8, lw=2.8, color=colors[spec], label=spec)
    ax.set_xticks(xpos)
    ax.set_xticklabels(transition_order)
    ax.set_xlabel("Adjacent transition")
    ax.set_ylabel("Cross-validated AUC")
    # annotate structure gain for each transition
    base_map = K_auc_df[K_auc_df["spec"] == "Controls"].set_index("transition")["cv_auc"].to_dict()
    full_map = K_auc_df[K_auc_df["spec"] == "Controls + structure"].set_index("transition")["cv_auc"].to_dict()
    for i, t in enumerate(transition_order):
        if t in base_map and t in full_map:
            delta = full_map[t] - base_map[t]
            ax.text(i, max(base_map[t], full_map[t]) + 0.008, f"Δ{delta:.3f}", ha="center", va="bottom", fontsize=10)
    ax.legend(frameon=False, loc="upper left")
    style_ax(ax)

    ax = axes[1]
    plot = I_gain_df.set_index("transition") if len(I_gain_df) > 0 else pd.DataFrame()
    x = np.arange(len(transition_order))
    cue_vals = np.array([plot.loc[t, "cue_increment"] if t in plot.index else np.nan for t in transition_order], dtype=float)
    bridge_vals = np.array([plot.loc[t, "bridge_increment"] if t in plot.index else np.nan for t in transition_order], dtype=float)
    # Make near-zero bridge effects visible instead of visually disappearing.
    for i, v in enumerate(cue_vals):
        if np.isfinite(v):
            ax.vlines(i - 0.10, 0, v, color=PALETTE["orange"], lw=5.0, alpha=0.95)
            ax.scatter(i - 0.10, v, s=170, color=PALETTE["orange"], zorder=5,
                       label="Cue increment" if i == 0 else None)
            ax.text(i - 0.10, v + 0.004, f"{v:.3f}", ha="center", va="bottom", fontsize=10)
    show_bridge_labels = not (np.isfinite(bridge_vals).any() and np.nanmax(np.abs(bridge_vals)) < 1e-3)
    for i, v in enumerate(bridge_vals):
        if np.isfinite(v):
            ax.vlines(i + 0.10, 0, v, color=PALETTE["blue"], lw=5.0, alpha=0.95)
            face = PALETTE["blue"] if abs(v) >= 1e-3 else "white"
            edge = PALETTE["blue"]
            ax.scatter(i + 0.10, v, s=170, facecolor=face, edgecolor=edge, linewidth=2.2, zorder=6,
                       label="Bridge increment" if i == 0 else None)
            if show_bridge_labels:
                txt_y = v + (0.004 if v >= 0 else -0.006)
                ax.text(i + 0.10, txt_y, f"{v:.3f}", ha="center", va=("bottom" if v >= 0 else "top"), fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(transition_order)
    ax.set_xlabel("Adjacent transition")
    ax.set_ylabel("Increment over previous block (AUC)")
    ax.axhline(0, color=PALETTE["lightgray"], lw=1.3)
    finite_all = np.concatenate([cue_vals[np.isfinite(cue_vals)], bridge_vals[np.isfinite(bridge_vals)]]) if (np.isfinite(cue_vals).any() or np.isfinite(bridge_vals).any()) else np.array([0.0])
    ymin = min(-0.01, float(finite_all.min()) - 0.01)
    ymax = max(0.03, float(finite_all.max()) + 0.015)
    ax.set_ylim(ymin, ymax)
    if np.isfinite(bridge_vals).any() and np.nanmax(np.abs(bridge_vals)) < 1e-3:
        ax.text(0.98, 0.96, "Bridge increment ≈ 0 across transitions", ha="right", va="top", transform=ax.transAxes, fontsize=9)
    ax.legend(frameon=False, loc="upper right")
    style_ax(ax)
    save_fig(fig, "FigI_specificity_beyond_shallow_controls")


def _safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def _rankcorr(a, b):
    a = pd.Series(a).rank(method="average").to_numpy(dtype=float)
    b = pd.Series(b).rank(method="average").to_numpy(dtype=float)
    if len(a) < 3 or np.nanstd(a) < 1e-12 or np.nanstd(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _norm01(s):
    s = _safe_numeric(s)
    lo = np.nanmin(s.values.astype(float)) if s.notna().any() else np.nan
    hi = np.nanmax(s.values.astype(float)) if s.notna().any() else np.nan
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return pd.Series(np.nan, index=s.index)
    return (s - lo) / (hi - lo)


def _fixed_rank_bins(s, q=3, prefix="Q"):
    s = _safe_numeric(s)
    if s.notna().sum() < q:
        return pd.Series([np.nan] * len(s), index=s.index)
    ranks = s.rank(method="average", pct=True)
    cuts = np.linspace(0, 1, q + 1)
    labels = [f"{prefix}{i}" for i in range(1, q + 1)]
    out = pd.cut(ranks, bins=cuts, labels=labels, include_lowest=True, duplicates="drop")
    return out.astype(object)


def _bootstrap_mean_ci(x, n_boot=2000, seed=42):
    x = pd.Series(x).dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    if len(x) == 1:
        v = float(x[0])
        return v, v, v
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        samp = rng.choice(x, size=len(x), replace=True)
        boots.append(float(np.mean(samp)))
    boots = np.array(boots, dtype=float)
    return float(np.mean(x)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def _cv_ridge_rankcorr(df, target_col, feat_cols, n_splits=5, seed=42):
    df2, feat_cols = _resolve_feature_cols(df.copy(), feat_cols, min_non_na=3)
    cols = [target_col] + feat_cols
    dd = df2[cols].copy()
    if len(dd) < 50:
        return None
    dd[target_col] = _safe_numeric(dd[target_col])
    dd = dd[dd[target_col].notna()].copy()
    if len(dd) < 50 or dd[target_col].nunique() < 8:
        return None

    feat_cols = [c for c in feat_cols if c in dd.columns and pd.to_numeric(dd[c], errors="coerce").notna().sum() > 0]
    if len(feat_cols) == 0:
        return None

    X = dd[feat_cols].apply(_safe_numeric)
    for c in X.columns:
        med = float(X[c].median()) if X[c].notna().sum() > 0 else 0.0
        X[c] = X[c].fillna(med)
    y = dd[target_col].astype(float).to_numpy()

    n_splits = int(max(3, min(n_splits, len(dd) // 20)))
    if n_splits < 3:
        return None

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for tr, te in kf.split(X):
        Xtr = X.iloc[tr].to_numpy(dtype=float)
        Xte = X.iloc[te].to_numpy(dtype=float)
        ytr = y[tr]
        yte = y[te]

        sc = StandardScaler()
        Xtr = sc.fit_transform(Xtr)
        Xte = sc.transform(Xte)

        model = Ridge(alpha=1.0, random_state=seed)
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
        rc = _rankcorr(yte, pred)
        if np.isfinite(rc):
            scores.append(float(rc))

    if len(scores) == 0:
        return None

    return {
        "n": int(len(dd)),
        "rankcorr": float(np.mean(scores)),
        "rankcorr_std": float(np.std(scores)),
        "feature_count": int(len(feat_cols)),
    }


def _build_rethought_boundary_df(df):
    d = _ensure_feature_columns(df.copy())
    need = ["true_label", "pred_label", "confidence_score"]
    miss = [c for c in need if c not in d.columns]
    if len(miss) > 0:
        return pd.DataFrame(), "missing_required_columns"

    d["true_label"] = _safe_numeric(d["true_label"])
    d["pred_label"] = _safe_numeric(d["pred_label"])
    d["confidence_score"] = _safe_numeric(d["confidence_score"])

    d = d[d["true_label"].notna() & d["pred_label"].notna() & d["confidence_score"].notna()].copy()
    if len(d) == 0:
        return pd.DataFrame(), "empty_after_basic_filter"

    d["true_label"] = d["true_label"].astype(int)
    d["pred_label"] = d["pred_label"].astype(int)
    d["abs_err"] = (d["pred_label"] - d["true_label"]).abs()

    # Boundary-local zone: any sample touching the 1↔2 neighborhood, not global.
    boundary_mask = (
        d["true_label"].isin([1, 2]) |
        d["pred_label"].isin([1, 2])
    )
    bd = d[boundary_mask].copy()
    if len(bd) == 0:
        return pd.DataFrame(), "no_boundary_zone"

    # Primary continuous uncertainty component: inverted confidence percentile.
    conf_pct = bd["confidence_score"].rank(method="average", pct=True)
    bd["unc_lowconf"] = 1.0 - conf_pct

    # Optional disagreement component.
    disagreement_ok = False
    if "is_disagreement_sample" in bd.columns:
        bd["is_disagreement_sample"] = _safe_numeric(bd["is_disagreement_sample"]).fillna(0).astype(float)
        disagreement_ok = bd["is_disagreement_sample"].nunique() >= 2 and bd["is_disagreement_sample"].sum() > 0

    # Optional local boundary error component (captures boundary difficulty, not generic global error).
    bd["is_boundary_adjacent_error"] = (
        (bd["abs_err"] == 1) &
        (bd["true_label"].isin([1, 2]) | bd["pred_label"].isin([1, 2]))
    ).astype(float)

    # Continuous human uncertainty score.
    if disagreement_ok:
        bd["human_uncertainty_score"] = (
            0.65 * bd["unc_lowconf"].astype(float) +
            0.35 * bd["is_disagreement_sample"].astype(float)
        )
        source = "low_confidence+disagreement"
    else:
        bd["human_uncertainty_score"] = bd["unc_lowconf"].astype(float)
        source = "low_confidence_only"

    # Binary target only for optional secondary summaries.
    bd["high_uncertainty"] = (
        bd["human_uncertainty_score"] >= bd["human_uncertainty_score"].quantile(0.65)
    ).astype(int)

    # Build structure score from cue/bridge groups.
    CONTROL_COLS, CUE_COLS, BRIDGE_COLS = get_feature_groups(bd)
    struct_cols = [c for c in (CUE_COLS + BRIDGE_COLS) if c in bd.columns]
    if len(struct_cols) == 0:
        return pd.DataFrame(), "no_structure_features"

    zcols = []
    for c in struct_cols:
        s = _safe_numeric(bd[c])
        med = float(s.median()) if s.notna().sum() > 0 else 0.0
        s = s.fillna(med).astype(float)
        sd = float(s.std()) if np.isfinite(float(s.std())) else 0.0
        if sd < 1e-12:
            z = pd.Series(np.zeros(len(s)), index=s.index)
        else:
            z = (s - float(s.mean())) / sd
        zcols.append(np.abs(z.to_numpy(dtype=float)))
    bd["structure_score"] = np.nanmean(np.column_stack(zcols), axis=1)

    return bd, source


ambiguity_nested_df = pd.DataFrame()
ambiguity_bin_df = pd.DataFrame()

if len(deep_four_df) > 0:
    boundary_df, ambiguity_source = _build_rethought_boundary_df(deep_four_df)

    if len(boundary_df) > 0 and boundary_df["human_uncertainty_score"].notna().sum() >= 50 and max(3, int(boundary_df["human_uncertainty_score"].nunique())) >= 3:
        CONTROL_COLS, CUE_COLS, BRIDGE_COLS = get_feature_groups(boundary_df)
        STRUCTURE_COLS = [c for c in (CUE_COLS + BRIDGE_COLS + ["structure_score"]) if c in boundary_df.columns]

        spec_defs = [
            ("Controls", CONTROL_COLS),
            ("Controls + structure", CONTROL_COLS + STRUCTURE_COLS),
        ]

        rows = []
        for name, cols in spec_defs:
            res = _cv_ridge_rankcorr(boundary_df, "human_uncertainty_score", cols)
            if res is None:
                continue
            rows.append({
                "spec": name,
                "n": res["n"],
                "cv_rankcorr": res["rankcorr"],
                "cv_rankcorr_std": res["rankcorr_std"],
                "feature_count": res["feature_count"],
                "ambiguity_source": ambiguity_source,
                "boundary_n": len(boundary_df),
            })
        ambiguity_nested_df = pd.DataFrame(rows)
        save_csv(ambiguity_nested_df, "FigJ1_boundary_uncertainty_model_cv.csv")

        boundary_df["structure_bin"] = _fixed_rank_bins(boundary_df["structure_score"], q=3, prefix="Q")
        qrows = []
        for qlab, dd in boundary_df.groupby("structure_bin", dropna=True):
            m1, lo1, hi1 = _bootstrap_mean_ci(dd["human_uncertainty_score"])
            row = {
                "bin": str(qlab),
                "mean_structure_score": float(dd["structure_score"].mean()),
                "uncertainty_mean": m1,
                "uncertainty_ci_low": lo1,
                "uncertainty_ci_high": hi1,
                "low_conf_rate": float((dd["unc_lowconf"] >= dd["unc_lowconf"].quantile(0.65)).mean()),
                "n": int(len(dd)),
            }
            if "is_disagreement_sample" in dd.columns and dd["is_disagreement_sample"].nunique() >= 2:
                row["disagreement_rate"] = float(dd["is_disagreement_sample"].mean())
            else:
                row["disagreement_rate"] = np.nan
            qrows.append(row)

        ambiguity_bin_df = pd.DataFrame(qrows).sort_values("mean_structure_score").reset_index(drop=True)
        save_csv(ambiguity_bin_df, "FigJ2_boundary_uncertainty_bins.csv")

        fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.9))

        ax = axes[0]
        plot = ambiguity_nested_df.copy()
        if len(plot) > 0:
            x = np.arange(len(plot))
            cols = [PALETTE["gray"], PALETTE["orange"]][:len(plot)]
            ax.bar(x, plot["cv_rankcorr"], color=cols, width=0.64)
            for i, rr in plot.iterrows():
                ax.text(i, rr["cv_rankcorr"] + 0.01, f"{rr['cv_rankcorr']:.3f}",
                        ha="center", va="bottom", fontsize=11, fontweight="bold")
            ax.set_xticks(x)
            ax.set_xticklabels(plot["spec"], rotation=16, ha="right")
            ax.set_ylabel("Cross-validated rank correlation\nfor boundary-local uncertainty")
            style_ax(ax)
        else:
            ax.text(0.5, 0.5, "No boundary-local uncertainty model", ha="center", va="center", fontweight="bold")
            ax.set_axis_off()

        ax = axes[1]
        plot = ambiguity_bin_df.copy()
        if len(plot) > 0:
            x = np.arange(len(plot))

            # main continuous uncertainty line with CI
            y = plot["uncertainty_mean"].to_numpy(dtype=float)
            lo = plot["uncertainty_ci_low"].to_numpy(dtype=float)
            hi = plot["uncertainty_ci_high"].to_numpy(dtype=float)
            ax.plot(x, y, marker="o", ms=8, lw=2.8, color=PALETTE["orange"], label="Boundary-local uncertainty score")
            ax.vlines(x, lo, hi, color=PALETTE["orange"], lw=2.0)

            # optional low-confidence rate overlay
            if plot["low_conf_rate"].notna().sum() > 0:
                ax.plot(x, plot["low_conf_rate"], marker="o", ms=7, lw=2.2,
                        color=PALETTE["blue"], label="High low-confidence rate")

            # disagreement only if non-degenerate
            if "disagreement_rate" in plot.columns and plot["disagreement_rate"].notna().sum() >= 2:
                ax.plot(x, plot["disagreement_rate"], marker="o", ms=7, lw=2.0,
                        color=PALETTE["purple"], label="Disagreement rate")

            ax.set_xticks(x)
            ax.set_xticklabels(plot["bin"])
            ax.set_xlabel("Boundary structure-complexity tertile")
            ax.set_ylabel("Rate / score")
            ax.legend(frameon=False, loc="upper left")
            style_ax(ax)
        else:
            ax.text(0.5, 0.5, "No boundary-local uncertainty bins", ha="center", va="center", fontweight="bold")
            ax.set_axis_off()

        save_fig(fig, "FigJ_human_uncertainty_tracks_structure")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.9))
        axes[0].text(0.5, 0.5, "No valid boundary-local\nuncertainty score", ha="center", va="center", fontweight="bold")
        axes[0].set_axis_off()
        axes[1].text(0.5, 0.5, "No valid uncertainty bins", ha="center", va="center", fontweight="bold")
        axes[1].set_axis_off()
        save_fig(fig, "FigJ_human_uncertainty_tracks_structure")


K_auc_rows, K_feat_rows = [], []
trans_feature_defs = [
    ("Worthlessness cue", ["worthlessness_explicit"]),
    ("Distortion core", ["distortion_core", "bridge_distortion_core"]),
    ("Negation", ["negation_density"]),
    ("1st-person", ["first_person_density"]),
    ("Emotion words", ["emotion_word_density"]),
    ("Suicide cue", ["suicide_explicit", "bridge_suicide"]),
]

for lo, hi in [(0, 1), (1, 2), (2, 3)]:
    sub = deep_four_df[deep_four_df["true_label"].isin([lo, hi])].copy()
    if len(sub) < 40:
        continue
    sub["upper_state"] = (sub["true_label"] == hi).astype(int)
    label = f"{lo}→{hi}"
    local_controls, _, _ = get_feature_groups(sub)
    local_structure = [c for c in list(dict.fromkeys(local_controls + STRUCTURE_COLS)) if c in _ensure_feature_columns(sub).columns]
    res_base = cv_logistic_binary(sub, "upper_state", local_controls)
    res_full = cv_logistic_binary(sub, "upper_state", local_structure)
    if res_base is not None:
        K_auc_rows.append({"transition": label, "spec": "Controls", "cv_auc": res_base["auc"], "cv_ap": res_base["ap"], "n": res_base["n"]})
    if res_full is not None:
        K_auc_rows.append({"transition": label, "spec": "Controls + structure", "cv_auc": res_full["auc"], "cv_ap": res_full["ap"], "n": res_full["n"]})

    upper = sub[sub["upper_state"] == 1].copy()
    lower = sub[sub["upper_state"] == 0].copy()
    for feat_name, feat_cols in trans_feature_defs:
        use = [c for c in feat_cols if c in sub.columns and pd.to_numeric(sub[c], errors="coerce").notna().sum() > 0]
        if len(use) == 0:
            continue
        x_hi = upper[use].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        x_lo = lower[use].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        mean_d, lo_ci, hi_ci = bootstrap_effect_ci(x_hi, x_lo)
        if not np.isfinite(mean_d):
            continue
        K_feat_rows.append({
            "transition": label,
            "feature_display": feat_name,
            "effect_d": mean_d,
            "lo": lo_ci,
            "hi": hi_ci,
        })

K_auc_df = pd.DataFrame(K_auc_rows)
K_feat_df = pd.DataFrame(K_feat_rows)
# keep stage-representative features: top 2 per transition, then deduplicate for readable plotting
if len(K_feat_df) > 0:
    chosen = []
    for trans in transition_order:
        dd = K_feat_df[K_feat_df["transition"] == trans].copy()
        if len(dd) == 0:
            continue
        dd = dd.assign(abs_effect=np.abs(dd["effect_d"].astype(float))).sort_values("abs_effect", ascending=False)
        chosen.extend(dd["feature_display"].head(2).tolist())
    chosen = list(dict.fromkeys(chosen))
    if len(chosen) < 5:
        more = (K_feat_df.groupby("feature_display")["effect_d"].apply(lambda s: np.nanmax(np.abs(s.values.astype(float))))
                .sort_values(ascending=False).index.tolist())
        for m in more:
            if m not in chosen:
                chosen.append(m)
            if len(chosen) >= 5:
                break
    K_feat_df = K_feat_df[K_feat_df["feature_display"].isin(chosen)].copy()
    preferred_order = ["Worthlessness cue", "Distortion core", "Negation", "1st-person", "Emotion words", "Suicide cue"]
    feat_order = [f for f in preferred_order if f in K_feat_df["feature_display"].unique().tolist()]
    K_feat_df["feature_display"] = pd.Categorical(K_feat_df["feature_display"], categories=feat_order, ordered=True)

save_csv(K_auc_df, "FigK1_transition_decomposition_auc.csv")
save_csv(K_feat_df, "FigK2_transition_feature_effects.csv")

if len(K_auc_df) > 0 and len(K_feat_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.9))
    ax = axes[0]
    xpos = np.arange(len(transition_order))
    for spec_name, color in [("Controls", PALETTE["gray"]), ("Controls + structure", PALETTE["deeporange"] )]:
        dd = K_auc_df[K_auc_df["spec"] == spec_name].set_index("transition")
        vals = [dd.loc[t, "cv_auc"] if t in dd.index else np.nan for t in transition_order]
        if np.isfinite(np.asarray(vals, dtype=float)).sum() == 0:
            continue
        ax.plot(xpos, vals, marker="o", ms=8, lw=2.8, color=color, label=spec_name)
    ax.set_xticks(xpos)
    ax.set_xticklabels(transition_order)
    ax.set_xlabel("Adjacent transition")
    ax.set_ylabel("Cross-validated AUC")
    ax.legend(frameon=False, loc="upper left")
    style_ax(ax)

    ax = axes[1]
    feat_order = K_feat_df["feature_display"].cat.categories.tolist()
    ybase = np.arange(len(feat_order))[::-1]
    ymap = {feat: y for feat, y in zip(feat_order, ybase)}
    offsets = {"0→1": -0.22, "1→2": 0.0, "2→3": 0.22}
    trans_colors = {"0→1": PALETTE["gray"], "1→2": PALETTE["orange"], "2→3": PALETTE["blue"]}
    for trans in transition_order:
        dd = K_feat_df[K_feat_df["transition"] == trans].copy()
        if len(dd) == 0:
            continue
        for rr in dd.itertuples(index=False):
            y = ymap[str(rr.feature_display)] + offsets[trans]
            ax.hlines(y, rr.lo, rr.hi, color=trans_colors[trans], lw=2.4)
            ax.scatter(rr.effect_d, y, s=120, color=trans_colors[trans], zorder=5,
                       label=trans if rr.feature_display == feat_order[0] else None)
    ax.axvline(0, color=PALETTE["lightgray"], lw=1.4)
    ax.set_yticks(ybase)
    ax.set_yticklabels(feat_order)
    ax.set_xlabel("Upper-vs-lower standardized effect")
    ax.legend(frameon=False, loc="lower right")
    style_ax(ax)
    save_fig(fig, "FigK_ordered_transition_gates")

deep_lines = []
deep_lines.append("NHB-STYLE DEEPENING IJK OPTIMIZED COMPLETED\n")
deep_lines.append(f"Best fourclass run: {best_four['model_name']} | {best_four['run_name']}")
deep_lines.append(f"Bridge source for deepening: {deep_bridge_source}")
if len(I_gain_df) > 0:
    best_i = I_gain_df.sort_values("full_auc", ascending=False).iloc[0]
    deep_lines.append(f"Best transition in I: {best_i['transition']} | full_auc={best_i['full_auc']:.4f} | cue_increment={best_i['cue_increment']:.4f} | bridge_increment={best_i['bridge_increment']:.4f}")
if len(J_model_df) > 0:
    jj = J_model_df[J_model_df["target_source"] == J_source_used].sort_values("cv_auc", ascending=False).iloc[0]
    deep_lines.append(f"J selected target: {J_source_used} | best_spec={jj['spec']} | cv_auc={jj['cv_auc']:.4f}")
else:
    deep_lines.append("J selected target: none")
if len(K_auc_df) > 0:
    kk = []
    for trans, g in K_auc_df.groupby("transition"):
        base = g.loc[g["spec"] == "Controls", "cv_auc"]
        full = g.loc[g["spec"] == "Controls + structure", "cv_auc"]
        if len(base) > 0 and len(full) > 0:
            kk.append((trans, float(full.iloc[0] - base.iloc[0])))
    if len(kk) > 0:
        best_trans, best_gain = sorted(kk, key=lambda x: x[1], reverse=True)[0]
        deep_lines.append(f"Largest ordered-transition gate effect: {best_trans} | ΔAUC={best_gain:.4f}")

save_text("\n".join(deep_lines), "deepening_summary_ijk_optimized.txt")
print("IJK optimized deepening DONE: FigI / FigJ / FigK saved.")


[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigI1_transition_specific_nested_cv.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigI2_transition_specific_gains.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigJ1_boundary_uncertainty_model_cv.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigJ2_boundary_uncertainty_bins.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigJ_human_uncertainty_tracks_structure.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigJ_human_uncertainty_tracks_structure.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigK1_transition_decomposition_auc.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigK2_transition_feature_effects.csv
[Saved] D:\Downloads\Depre

In [ ]:
# -*- coding: utf-8 -*-
"""
L) Theory-specificity beyond shallow controls
M) Failure-regime map (when the structure fails)
N) Decision / triage utility under review-budget constraints
O) Stage-mechanism family abstraction (why the transitions differ)
P) Human-judgment alignment (optional, only if usable annotation signals exist)
"""

import os
import re
import glob
import math
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge

warnings.filterwarnings("ignore")

ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"
EXPORT_DIR = os.path.join(ROOT_DEPRESSED, "export_bundle_v4")
PRED_DIR = os.path.join(EXPORT_DIR, "predictions")
SAMPLE_META_FP = os.path.join(EXPORT_DIR, "sample_meta.csv")
SUMMARY_FP = os.path.join(EXPORT_DIR, "result_summary.csv")
FINAL_MAP_DIR = os.path.join(ROOT_DEPRESSED, "final_stage_mapping")
FINAL_RUN_MAP_FP = os.path.join(FINAL_MAP_DIR, "final_run_task_mapping.csv")

OUT_DIR = os.path.join(ROOT_DEPRESSED, "final_pub_outputs_fullcover")
OUT_FIG = os.path.join(OUT_DIR, "figures")
OUT_CSV = os.path.join(OUT_DIR, "figure_csv")
OUT_TAB = os.path.join(OUT_DIR, "tables")
OUT_TXT = os.path.join(OUT_DIR, "texts")
for d in [OUT_DIR, OUT_FIG, OUT_CSV, OUT_TAB, OUT_TXT]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.linewidth": 2.0,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "axes.unicode_minus": False,
    "font.size": 15,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
})
DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "yellow": "#FFDF97",
    "black": "#202020",
    "lightgray": "#CFCBC3",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "deepblue": "#6EA9C4",
    "deeporange": "#D88537",
    "deepgreen": "#5FA777",
}

CMAP_ORANGE = LinearSegmentedColormap.from_list(
    "nature_orange", ["#FAF8F4", "#F1E2CF", "#F0C48E", "#F39F4E"]
)
CMAP_BLUE = LinearSegmentedColormap.from_list(
    "nature_blue", ["#F7F7F7", "#DCE8EE", "#B9D5E2", "#98CFE6"]
)
CMAP_RED = LinearSegmentedColormap.from_list(
    "nature_red", ["#FCF6F6", "#F3DADA", "#EAB4B4", "#D95F5F"]
)

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness",
]
EMOTION_SHORT = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "empty",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless.",
}
LEXICON = {
    "sadness_explicit": ["sad", "depressed", "cry", "crying", "empty", "down", "miserable"],
    "hopelessness_explicit": ["hopeless", "no future", "nothing will change", "can't go on", "pointless"],
    "worthlessness_explicit": ["worthless", "useless", "burden", "failure", "not good enough", "hate myself"],
    "cognitive_explicit": ["forget", "memory", "focus", "concentrate", "brain fog", "can't think"],
    "suicide_explicit": ["suicide", "suicidal", "kill myself", "end my life", "die", "self-harm", "self harm"],
    "help_seeking": ["please help", "need help", "advice", "suggestions", "what do i do", "help me"],
    "absolutist": ["always", "never", "nothing", "nobody", "completely", "totally", "forever"],
}
CORE_IDX = [3, 6, 7]  # hopelessness, suicide intent, worthlessness
PERIPHERY_IDX = [0, 1, 2, 4, 5]

def save_fig(fig, stem):
    png_fp = os.path.join(OUT_FIG, f"{stem}.png")
    eps_fp = os.path.join(OUT_FIG, f"{stem}.eps")
    fig.savefig(png_fp, dpi=DPI, facecolor="white", bbox_inches="tight")
    fig.savefig(eps_fp, dpi=DPI, facecolor="white", bbox_inches="tight", format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")


def save_csv(df, name):
    fp = os.path.join(OUT_CSV, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")


def save_table(df, name):
    fp = os.path.join(OUT_TAB, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved Table] {fp}")


def save_text(text, name):
    fp = os.path.join(OUT_TXT, name)
    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[Saved Text] {fp}")


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.0)
    ax.spines["bottom"].set_linewidth(2.0)
    ax.tick_params(axis="both", width=2.0, length=7)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")


def add_clean_legend(ax, handles, loc="best", ncol=1):
    leg = ax.legend(handles=handles, frameon=False, loc=loc, ncol=ncol)
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontweight("bold")
    return leg


def safe_read_csv(fp):
    if fp is None or (not os.path.exists(fp)):
        return pd.DataFrame()
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.DataFrame()


def parse_model_size_b(model_name):
    if pd.isna(model_name):
        return np.nan
    s = str(model_name)
    m = re.search(r"(\d+(?:\.\d+)?)B", s, re.I)
    return float(m.group(1)) if m else np.nan


def parse_task_from_filename(fp):
    base = os.path.basename(fp).lower()
    if base.startswith("binary__"):
        return "binary"
    if base.startswith("fourclass__"):
        return "fourclass"
    if base.startswith("eightlabel__"):
        return "eightlabel"
    if base.startswith("class255__"):
        return "class255"
    return "unknown"


def parse_model_from_filename(fp):
    parts = os.path.basename(fp).split("__")
    return parts[1] if len(parts) >= 2 else "unknown_model"


def parse_vec8(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) == 8 and set(s) <= {"0", "1"}:
        return [int(c) for c in s]
    nums = re.findall(r"[01]", s)
    if len(nums) == 8:
        return [int(c) for c in nums]
    return None


def vec_to_int(v):
    if v is None:
        return np.nan
    return int("".join(str(int(z)) for z in v), 2)


def hamming(v1, v2):
    return sum(int(a != b) for a, b in zip(v1, v2))


def symptom_count(v):
    return np.nan if v is None else int(sum(v))


def mean_no_nan(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) == 0 else float(x.mean())


def median_no_nan(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) == 0 else float(x.median())


def corr_no_nan(x, y, method="pearson"):
    xx = pd.Series(x)
    yy = pd.Series(y)
    m = xx.notna() & yy.notna()
    if m.sum() < 3:
        return np.nan
    return float(xx[m].corr(yy[m], method=method))


def zscore(s):
    s = pd.to_numeric(pd.Series(s), errors="coerce")
    mu = s.mean()
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd


def safe_ratio(a, b):
    return np.nan if b == 0 else a / b


def has_any(txt, kws):
    t = str(txt).lower()
    return int(any(k in t for k in kws))


def infer_run_name(df, fp):
    for c in ["run_name"]:
        if c in df.columns and df[c].notna().any():
            return str(df[c].dropna().iloc[0])
    for c in ["source_jsonl", "source_file"]:
        if c in df.columns and df[c].notna().any():
            p = str(df[c].dropna().iloc[0])
            return os.path.basename(os.path.dirname(p)) if c == "source_jsonl" else os.path.splitext(os.path.basename(p))[0]
    return os.path.splitext(os.path.basename(fp))[0]


def existing_cols(df, cols, min_non_na=5):
    out = []
    if df is None or len(df) == 0:
        return out
    for c in cols:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            if s.notna().sum() >= min_non_na:
                out.append(c)
    return out


def ensure_basic_text_features(df):
    d = df.copy()
    if "text" not in d.columns:
        d["text"] = ""
    d["text"] = d["text"].astype(str)
    if "text_len" not in d.columns:
        d["text_len"] = d["text"].str.len()
    d["text_len"] = pd.to_numeric(d["text_len"], errors="coerce").fillna(d["text"].str.len())
    if "log_text_len" not in d.columns:
        d["log_text_len"] = np.log1p(np.maximum(d["text_len"], 0))

    word_count = np.maximum(d["text"].str.split().str.len().fillna(0), 1)
    if "first_person_density" not in d.columns:
        d["first_person_density"] = d["text"].str.lower().str.count(r"\b(i|me|my|mine|myself)\b") / word_count
    if "negation_density" not in d.columns:
        d["negation_density"] = d["text"].str.lower().str.count(r"\b(no|not|never|nothing|can't|cannot|don't|won't)\b") / word_count
    if "emotion_word_density" not in d.columns:
        d["emotion_word_density"] = d["text"].str.lower().str.count(r"\b(sad|hopeless|depressed|cry|empty|worthless|lonely|suicidal|anxious)\b") / word_count
    for cue, kws in LEXICON.items():
        if cue not in d.columns:
            d[cue] = d["text"].map(lambda x: has_any(x, kws))
    return d


def bootstrap_mean_ci(vals, n_boot=500, alpha=0.05, random_state=42):
    vals = np.asarray(pd.Series(vals).dropna(), dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    boots = []
    n = len(vals)
    for _ in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        boots.append(float(np.mean(vals[idx])))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1 - alpha/2))


def bootstrap_mean_diff(x_hi, x_lo, n_boot=500, alpha=0.05, random_state=42):
    hi = np.asarray(pd.Series(x_hi).dropna(), dtype=float)
    lo = np.asarray(pd.Series(x_lo).dropna(), dtype=float)
    if len(hi) < 3 or len(lo) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    boots = []
    for _ in range(n_boot):
        bh = hi[rng.choice(np.arange(len(hi)), size=len(hi), replace=True)]
        bl = lo[rng.choice(np.arange(len(lo)), size=len(lo), replace=True)]
        boots.append(float(np.mean(bh) - np.mean(bl)))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha/2)), float(np.quantile(boots, 1 - alpha/2))


def cv_logistic_binary(df, target_col, feature_cols, random_state=42):
    d = df.copy()
    feat_cols = existing_cols(d, feature_cols, min_non_na=max(5, int(0.1 * len(d))))
    if target_col not in d.columns or len(feat_cols) == 0:
        return None
    y = pd.to_numeric(d[target_col], errors="coerce")
    X = d[feat_cols].apply(pd.to_numeric, errors="coerce")
    mask = y.notna() & (X.notna().sum(axis=1) >= max(1, int(0.5 * len(feat_cols))))
    X = X[mask]
    y = y[mask].astype(int)
    if len(X) < 24 or y.nunique() < 2:
        return None
    class_counts = y.value_counts()
    min_class = int(class_counts.min())
    n_splits = min(5, max(3, min_class))
    if min_class < 3:
        return None

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=4000, class_weight="balanced", solver="liblinear", random_state=random_state)),
    ])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    aucs, aps = [], []
    oof = pd.DataFrame(index=X.index, columns=["y", "p"])
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        if ytr.nunique() < 2 or yte.nunique() < 2:
            continue
        pipe.fit(Xtr, ytr)
        p = pipe.predict_proba(Xte)[:, 1]
        try:
            aucs.append(roc_auc_score(yte, p))
            aps.append(average_precision_score(yte, p))
        except Exception:
            continue
        oof.loc[Xte.index, "y"] = yte.values
        oof.loc[Xte.index, "p"] = p
    if len(aucs) == 0:
        return None
    oof = oof.dropna().copy()
    return {
        "n": int(len(X)),
        "features": feat_cols,
        "auc_mean": float(np.mean(aucs)),
        "auc_lo": float(np.quantile(aucs, 0.025)) if len(aucs) > 1 else float(np.mean(aucs)),
        "auc_hi": float(np.quantile(aucs, 0.975)) if len(aucs) > 1 else float(np.mean(aucs)),
        "ap_mean": float(np.mean(aps)),
        "oof": oof,
    }


def cv_ridge_rankcorr(df, target_col, feature_cols, random_state=42):
    d = df.copy()
    feat_cols = existing_cols(d, feature_cols, min_non_na=max(5, int(0.1 * len(d))))
    if target_col not in d.columns or len(feat_cols) == 0:
        return None
    y = pd.to_numeric(d[target_col], errors="coerce")
    X = d[feat_cols].apply(pd.to_numeric, errors="coerce")
    mask = y.notna() & (X.notna().sum(axis=1) >= max(1, int(0.5 * len(feat_cols))))
    X = X[mask]
    y = y[mask].astype(float)
    if len(X) < 24 or y.nunique() < 3:
        return None
    n_splits = min(5, max(3, len(X) // 20))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0, random_state=random_state)),
    ])
    corrs = []
    oof = pd.DataFrame(index=X.index, columns=["y", "pred"])
    for tr, te in kf.split(X):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        cc = pd.Series(pred).corr(pd.Series(yte.values), method="spearman")
        if pd.notna(cc):
            corrs.append(float(cc))
        oof.loc[Xte.index, "y"] = yte.values
        oof.loc[Xte.index, "pred"] = pred
    if len(corrs) == 0:
        return None
    return {
        "n": int(len(X)),
        "features": feat_cols,
        "rho_mean": float(np.mean(corrs)),
        "rho_lo": float(np.quantile(corrs, 0.025)) if len(corrs) > 1 else float(np.mean(corrs)),
        "rho_hi": float(np.quantile(corrs, 0.975)) if len(corrs) > 1 else float(np.mean(corrs)),
        "oof": oof.dropna().copy(),
    }


def make_rank_bins(x, n_bins=3):
    s = pd.Series(x)
    valid = s.notna()
    if valid.sum() < n_bins * 6:
        return pd.Series(np.nan, index=s.index)
    ranks = s[valid].rank(method="average", pct=True)
    out = pd.Series(np.nan, index=s.index, dtype=object)
    edges = np.linspace(0, 1, n_bins + 1)
    labels = [f"Q{i+1}" for i in range(n_bins)]
    out.loc[valid] = pd.cut(ranks, bins=edges, labels=labels, include_lowest=True, duplicates="drop").astype(str)
    return out


def normalize_01(s):
    s = pd.to_numeric(pd.Series(s), errors="coerce")
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi <= lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

if not os.path.isdir(PRED_DIR):
    raise FileNotFoundError(f"Missing prediction dir:\n{PRED_DIR}")

pred_files = sorted(glob.glob(os.path.join(PRED_DIR, "*.csv")))
if len(pred_files) == 0:
    raise FileNotFoundError(f"No standardized prediction csv found in:\n{PRED_DIR}")

sample_meta = safe_read_csv(SAMPLE_META_FP)
summary_df = safe_read_csv(SUMMARY_FP)
run_map_df = safe_read_csv(FINAL_RUN_MAP_FP)
if len(sample_meta) > 0 and "sample_id" in sample_meta.columns:
    sample_meta["sample_id"] = sample_meta["sample_id"].astype(str)

print("=" * 100)
print("ROOT_DEPRESSED =", ROOT_DEPRESSED)
print("PRED_DIR       =", PRED_DIR)
print("Prediction CSVs:", len(pred_files))
print("=" * 100)

all_dfs = []
run_rows = []
for fp in pred_files:
    df = pd.read_csv(fp)
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    task = parse_task_from_filename(fp)
    model_name = parse_model_from_filename(fp)
    run_name = infer_run_name(df, fp)

    df["task"] = task
    df["model_name"] = model_name
    df["model_size_b"] = parse_model_size_b(model_name)
    df["run_name"] = run_name
    df["pred_csv"] = fp

    if len(sample_meta) > 0 and "sample_id" in df.columns:
        df = df.merge(sample_meta, on="sample_id", how="left")

    df = ensure_basic_text_features(df)
    all_dfs.append(df)
    run_rows.append({
        "task": task,
        "model_name": model_name,
        "model_size_b": parse_model_size_b(model_name),
        "run_name": run_name,
        "pred_csv": fp,
    })

master_df = pd.concat(all_dfs, ignore_index=True, sort=False)
run_inventory = pd.DataFrame(run_rows).drop_duplicates()

if len(summary_df) > 0 and {"model_name", "run_name"}.issubset(summary_df.columns):
    s = summary_df.copy()
    conflict = [c for c in ["task", "pred_csv", "model_size_b"] if c in s.columns]
    s = s.drop(columns=conflict, errors="ignore")
    run_inventory = run_inventory.merge(s, on=["model_name", "run_name"], how="left")

if len(run_map_df) > 0:
    m = run_map_df.copy()
    if "run_name" not in m.columns and "eval_dir_name" in m.columns:
        m["run_name"] = m["eval_dir_name"]
    keep = [c for c in [
        "model_name", "run_name", "final_task", "final_stage_name", "final_stage_role",
        "count_bucket", "pred_file_final"
    ] if c in m.columns]
    if {"model_name", "run_name"}.issubset(set(keep)):
        run_inventory = run_inventory.merge(m[keep].drop_duplicates(), on=["model_name", "run_name"], how="left")

if "task" not in run_inventory.columns:
    if "task_x" in run_inventory.columns:
        run_inventory["task"] = run_inventory["task_x"]
    elif "task_y" in run_inventory.columns:
        run_inventory["task"] = run_inventory["task_y"]

run_inventory["task_final"] = run_inventory["final_task"].fillna(run_inventory["task"]) if "final_task" in run_inventory.columns else run_inventory["task"]


def infer_stage_type(row):
    st = row["final_stage_name"] if "final_stage_name" in row and pd.notna(row["final_stage_name"]) else np.nan
    if pd.notna(st) and str(st).strip():
        return str(st)
    rn = str(row["run_name"]).lower()
    if "from4to255" in rn:
        return "from4to255"
    if "from4to8" in rn:
        return "from4to8"
    if "from2to255" in rn:
        return "from2to255"
    if "from2to8" in rn:
        return "from2to8"
    if "native_eval" in rn:
        return "native"
    return "direct"

run_inventory["stage_type"] = run_inventory.apply(infer_stage_type, axis=1)
run_inventory["count_bucket_final"] = run_inventory.get("count_bucket", np.nan)
run_inventory["count_bucket_final"] = run_inventory["count_bucket_final"].fillna(
    run_inventory["task_final"].map({
        "binary": "deptweet_task",
        "fourclass": "deptweet_task",
        "eightlabel": "depressionemo_task",
        "class255": "depressionemo_task",
    })
)
save_table(run_inventory, "run_inventory_nhb_gap_pack.csv")

def get_run_df(task, model_name, run_name):
    return master_df[
        (master_df["task"] == task) &
        (master_df["model_name"] == model_name) &
        (master_df["run_name"] == run_name)
    ].copy()


def metrics_binary(df):
    d = df.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return {}
    yt = d["true_label"].astype(int).values
    yp = d["pred_label"].astype(int).values
    return {"accuracy": accuracy_score(yt, yp), "f1": f1_score(yt, yp, zero_division=0), "n": len(d)}


def metrics_fourclass(df):
    d = df.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) == 0:
        return {}
    yt = d["true_label"].astype(int).values
    yp = d["pred_label"].astype(int).values
    return {"accuracy": accuracy_score(yt, yp), "macro_f1": f1_score(yt, yp, average="macro", zero_division=0), "n": len(d)}


def metrics_multilabel(df):
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    return {
        "exact_match": accuracy_score(Yt, Yp),
        "macro_f1": f1_score(Yt, Yp, average="macro", zero_division=0),
        "micro_f1": f1_score(Yt, Yp, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(Yt, Yp),
        "n": len(d),
    }


def metrics_class255(df):
    d = df.copy()
    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[d["true_vec_list"].notna() & d["pred_vec_list"].notna()].copy()
    if len(d) == 0:
        return {}
    Yt = np.array(d["true_vec_list"].tolist(), dtype=int)
    Yp = np.array(d["pred_vec_list"].tolist(), dtype=int)
    yt255 = d["true_255"].astype(int).values if "true_255" in d.columns else np.array([vec_to_int(v) for v in d["true_vec_list"]])
    yp255 = d["pred_255"].fillna(-1).astype(int).values if "pred_255" in d.columns else np.array([vec_to_int(v) for v in d["pred_vec_list"]])
    return {
        "accuracy": accuracy_score(yt255, yp255),
        "macro_f1": f1_score(yt255, yp255, average="macro", zero_division=0),
        "macro_f1_symptom": f1_score(Yt, Yp, average="macro", zero_division=0),
        "n": len(d),
    }

metric_rows = []
for _, r in run_inventory.iterrows():
    task_val = r.get("task", r.get("task_x", r.get("task_y", np.nan)))
    d = get_run_df(task_val, r["model_name"], r["run_name"])
    if task_val == "binary":
        met = metrics_binary(d)
    elif task_val == "fourclass":
        met = metrics_fourclass(d)
    elif task_val == "eightlabel":
        met = metrics_multilabel(d)
    elif task_val == "class255":
        met = metrics_class255(d)
    else:
        met = {}
    row = r.to_dict()
    row["task"] = task_val
    row.update(met)
    metric_rows.append(row)

run_metrics_df = pd.DataFrame(metric_rows)
save_table(run_metrics_df, "run_metrics_nhb_gap_pack.csv")


def pick_best(df, metric):
    d = df[df[metric].notna()].copy()
    if len(d) == 0:
        return None
    return d.sort_values(metric, ascending=False).iloc[0]


def pick_best_bridge(task_target, stage_keyword):
    d = run_metrics_df[
        (run_metrics_df["task"] == task_target) &
        (run_metrics_df["stage_type"].astype(str).str.contains(stage_keyword, case=False, na=False))
    ].copy()
    if len(d) == 0:
        return None
    metric = "macro_f1" if "macro_f1" in d.columns else "f1"
    d = d[d[metric].notna()].copy()
    if len(d) == 0:
        return None
    return d.sort_values(metric, ascending=False).iloc[0]

best_binary = pick_best(run_metrics_df[run_metrics_df["task"] == "binary"].copy(), "f1")
best_four = pick_best(run_metrics_df[run_metrics_df["task"] == "fourclass"].copy(), "macro_f1")
best_eight = pick_best(run_metrics_df[run_metrics_df["task"] == "eightlabel"].copy(), "macro_f1")
best_class255 = pick_best(run_metrics_df[run_metrics_df["task"] == "class255"].copy(), "macro_f1")
bridge_4to8 = pick_best_bridge("eightlabel", "from4to8")
bridge_4to255 = pick_best_bridge("class255", "from4to255")


def best_df(best_row):
    if best_row is None:
        return pd.DataFrame()
    return get_run_df(best_row["task"], best_row["model_name"], best_row["run_name"])

best_binary_df = best_df(best_binary)
best_four_df = best_df(best_four)
best_eight_df = best_df(best_eight)
best_class255_df = best_df(best_class255)
bridge_4to8_df = best_df(bridge_4to8)
bridge_4to255_df = best_df(bridge_4to255)

save_text(
    "\n".join([
        "BEST / BRIDGE RUNS",
        f"best_binary:\n{best_binary}",
        f"best_four:\n{best_four}",
        f"best_eight:\n{best_eight}",
        f"best_class255:\n{best_class255}",
        f"bridge_4to8:\n{bridge_4to8}",
        f"bridge_4to255:\n{bridge_4to255}",
    ]),
    "nhb_gap_pack_best_runs.txt"
)

if len(best_four_df) == 0:
    raise RuntimeError("A best fourclass run is required for the NHB gap-pack.")

four = ensure_basic_text_features(best_four_df.copy())
four["sample_id"] = four["sample_id"].astype(str)
four["true_label"] = pd.to_numeric(four["true_label"], errors="coerce")
four["pred_label"] = pd.to_numeric(four["pred_label"], errors="coerce")
for c in ["confidence_score", "is_disagreement_sample"]:
    if c in four.columns:
        four[c] = pd.to_numeric(four[c], errors="coerce")

analysis_df = four[[c for c in [
    "sample_id", "text", "true_label", "pred_label", "confidence_score", "is_disagreement_sample",
    "text_len", "log_text_len", "first_person_density", "negation_density", "emotion_word_density"
] if c in four.columns]].copy()

for cue in LEXICON.keys():
    if cue in four.columns:
        analysis_df[cue] = pd.to_numeric(four[cue], errors="coerce")

# Attach bridge symptom predictions if available; else fall back to best class255/eightlabel predictions.
symptom_source = None
symptom_df = pd.DataFrame()
if len(bridge_4to255_df) > 0:
    symptom_df = bridge_4to255_df.copy()
    symptom_source = "bridge_from4to255"
elif len(bridge_4to8_df) > 0:
    symptom_df = bridge_4to8_df.copy()
    symptom_source = "bridge_from4to8"
elif len(best_class255_df) > 0:
    symptom_df = best_class255_df.copy()
    symptom_source = "best_class255"
elif len(best_eight_df) > 0:
    symptom_df = best_eight_df.copy()
    symptom_source = "best_eightlabel"

if len(symptom_df) > 0 and "sample_id" in symptom_df.columns:
    symptom_df["sample_id"] = symptom_df["sample_id"].astype(str)
    symptom_df["symptom_pred_vec_list"] = symptom_df["pred_vec"].map(parse_vec8)
    symptom_df = symptom_df[symptom_df["symptom_pred_vec_list"].notna()].copy()
    analysis_df = analysis_df.merge(
        symptom_df[["sample_id", "symptom_pred_vec_list"]],
        on="sample_id", how="left"
    )
    analysis_df["symptom_source"] = symptom_source
else:
    analysis_df["symptom_pred_vec_list"] = None
    analysis_df["symptom_source"] = "none"

# Attach best class255 truth/pred for risk / failure modules.
if len(best_class255_df) > 0:
    c255 = ensure_basic_text_features(best_class255_df.copy())
    c255["sample_id"] = c255["sample_id"].astype(str)
    c255["true_vec_list"] = c255["true_vec"].map(parse_vec8)
    c255["pred_vec_list"] = c255["pred_vec"].map(parse_vec8)
    c255 = c255[c255["true_vec_list"].notna() & c255["pred_vec_list"].notna()].copy()
    keep = [c for c in ["sample_id", "true_vec_list", "pred_vec_list", "true_255", "pred_255"] if c in c255.columns]
    analysis_df = analysis_df.merge(c255[keep], on="sample_id", how="left")
else:
    analysis_df["true_vec_list"] = None
    analysis_df["pred_vec_list"] = None

analysis_df = ensure_basic_text_features(analysis_df)

# cue-level structure
analysis_df["cue_sum"] = 0
for cue in LEXICON.keys():
    if cue in analysis_df.columns:
        analysis_df[cue] = pd.to_numeric(analysis_df[cue], errors="coerce").fillna(0)
        analysis_df["cue_sum"] += analysis_df[cue]
analysis_df["cue_conflict"] = (
    analysis_df.get("help_seeking", 0).fillna(0) *
    (analysis_df.get("hopelessness_explicit", 0).fillna(0) + analysis_df.get("worthlessness_explicit", 0).fillna(0) + analysis_df.get("suicide_explicit", 0).fillna(0) > 0).astype(int)
)
analysis_df["distortion_cue_score"] = (
    analysis_df.get("hopelessness_explicit", 0).fillna(0) +
    analysis_df.get("worthlessness_explicit", 0).fillna(0) +
    analysis_df.get("absolutist", 0).fillna(0)
)
analysis_df["explicit_risk_score"] = (
    2.0 * analysis_df.get("suicide_explicit", 0).fillna(0) +
    1.0 * analysis_df.get("hopelessness_explicit", 0).fillna(0) +
    1.0 * analysis_df.get("worthlessness_explicit", 0).fillna(0)
)
analysis_df["affective_onset_score"] = (
    analysis_df.get("sadness_explicit", 0).fillna(0) +
    normalize_01(analysis_df["emotion_word_density"]) +
    normalize_01(analysis_df["first_person_density"])
)

# topology-level structure from symptom predictions
analysis_df["bridge_burden"] = np.nan
analysis_df["bridge_core_load"] = np.nan
analysis_df["bridge_periphery_load"] = np.nan
analysis_df["core_minus_periphery"] = np.nan
analysis_df["symptom_combo_id"] = np.nan
analysis_df["bridge_combo_rarity"] = np.nan
analysis_df["latent_risk_minus_explicit"] = np.nan

if analysis_df["symptom_pred_vec_list"].notna().any():
    combo_counter = Counter()
    for v in analysis_df["symptom_pred_vec_list"].dropna():
        combo_counter[tuple(v)] += 1
    total_combo = sum(combo_counter.values())

    def _burden(v):
        return np.nan if v is None else int(sum(v))

    def _core(v):
        return np.nan if v is None else float(sum(v[i] for i in CORE_IDX))

    def _peri(v):
        return np.nan if v is None else float(sum(v[i] for i in PERIPHERY_IDX))

    def _rarity(v):
        if v is None:
            return np.nan
        p = combo_counter.get(tuple(v), 0) / max(total_combo, 1)
        if p <= 0:
            return np.nan
        return float(-np.log(p))

    analysis_df["bridge_burden"] = analysis_df["symptom_pred_vec_list"].map(_burden)
    analysis_df["bridge_core_load"] = analysis_df["symptom_pred_vec_list"].map(_core)
    analysis_df["bridge_periphery_load"] = analysis_df["symptom_pred_vec_list"].map(_peri)
    analysis_df["core_minus_periphery"] = analysis_df["bridge_core_load"] - analysis_df["bridge_periphery_load"]
    analysis_df["symptom_combo_id"] = analysis_df["symptom_pred_vec_list"].map(lambda v: np.nan if v is None else vec_to_int(v))
    analysis_df["bridge_combo_rarity"] = analysis_df["symptom_pred_vec_list"].map(_rarity)
    analysis_df["latent_risk_minus_explicit"] = zscore(analysis_df["bridge_core_load"]) - zscore(analysis_df["explicit_risk_score"])

# class255 / risk targets if available
if "true_vec_list" in analysis_df.columns and analysis_df["true_vec_list"].notna().any():
    analysis_df["true_suicide"] = analysis_df["true_vec_list"].map(lambda v: np.nan if v is None else int(v[6]))
    analysis_df["pred_suicide"] = analysis_df["pred_vec_list"].map(lambda v: np.nan if v is None else int(v[6]))
    analysis_df["suicide_fn"] = ((analysis_df["true_suicide"] == 1) & (analysis_df["pred_suicide"] == 0)).astype(float)
    analysis_df["vec_hamming"] = [
        hamming(t, p) if (isinstance(t, list) and isinstance(p, list)) else np.nan
        for t, p in zip(analysis_df["true_vec_list"], analysis_df["pred_vec_list"])
    ]
    analysis_df["failure_regime"] = np.where(
        analysis_df["suicide_fn"] == 1, "suicide_FN",
        np.where(analysis_df["vec_hamming"] == 0, "correct",
                 np.where(analysis_df["vec_hamming"] == 1, "adjacent", "far"))
    )
else:
    analysis_df["true_suicide"] = np.nan
    analysis_df["pred_suicide"] = np.nan
    analysis_df["suicide_fn"] = np.nan
    analysis_df["vec_hamming"] = np.nan
    analysis_df["failure_regime"] = np.nan

# human uncertainty (optional but continuous)
conf = pd.to_numeric(analysis_df.get("confidence_score", np.nan), errors="coerce")
dis = pd.to_numeric(analysis_df.get("is_disagreement_sample", np.nan), errors="coerce")
analysis_df["human_uncertainty_score"] = np.nan
if conf.notna().sum() >= 30 and conf.nunique(dropna=True) >= 5:
    analysis_df["human_uncertainty_score"] = zscore(1 - normalize_01(conf))
if dis.notna().sum() >= 30 and dis.nunique(dropna=True) >= 2:
    add = zscore(dis)
    if analysis_df["human_uncertainty_score"].notna().sum() > 0:
        analysis_df["human_uncertainty_score"] = analysis_df["human_uncertainty_score"].fillna(0) + 0.6 * add.fillna(0)
    else:
        analysis_df["human_uncertainty_score"] = add

save_csv(analysis_df, "nhb_gap_pack_master_analysis_df.csv")

# Feature groups
CONTROL_COLS = ["log_text_len", "first_person_density", "negation_density", "emotion_word_density"]
CUE_COLS = list(LEXICON.keys()) + ["cue_conflict", "distortion_cue_score", "explicit_risk_score", "affective_onset_score"]
TOPO_COLS = ["bridge_burden", "bridge_core_load", "bridge_periphery_load", "core_minus_periphery", "bridge_combo_rarity", "latent_risk_minus_explicit"]

def existing_numeric_cols(df, cols, min_non_na=12):
    out = []
    for c in cols:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notna().sum() >= min_non_na:
            out.append(c)
    return out


def stable_rank_bins(series, n_bins=3, prefix="Q"):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(index=s.index, dtype="object")
    valid = s.notna()
    if valid.sum() < max(12, n_bins * 4):
        return out
    ranks = s[valid].rank(method="average", pct=True)
    edges = np.linspace(0, 1, n_bins + 1)
    labels = [f"{prefix}{i+1}" for i in range(n_bins)]
    bins = pd.cut(ranks, bins=edges, labels=labels, include_lowest=True)
    out.loc[valid] = bins.astype(str)
    return out


def _display_bin_labels(bins):
    mp = {"Q1": "low", "Q2": "mid", "Q3": "high"}
    return [mp.get(b, str(b)) for b in bins]


def _trim_heatmap_frame(df_wide):
    if df_wide is None or len(df_wide) == 0:
        return df_wide
    row_mask = df_wide.notna().any(axis=1)
    col_mask = df_wide.notna().any(axis=0)
    if row_mask.sum() == 0 or col_mask.sum() == 0:
        return df_wide
    return df_wide.loc[row_mask, col_mask]


rows = []
transitions = [(0, 1, "Affective onset"), (1, 2, "Distortion escalation"), (2, 3, "Suicidality crystallization")]
for lo, hi, stage_name in transitions:
    sub = analysis_df[analysis_df["true_label"].isin([lo, hi])].copy()
    sub["upper_state"] = (sub["true_label"] == hi).astype(int)
    base_cols = existing_numeric_cols(sub, CONTROL_COLS)
    cue_cols = existing_numeric_cols(sub, base_cols + CUE_COLS)
    topo_cols = existing_numeric_cols(sub, cue_cols + TOPO_COLS)
    for block_name, feat_cols in [
        ("controls", base_cols),
        ("controls_plus_cues", cue_cols),
        ("controls_plus_cues_plus_topology", topo_cols),
    ]:
        res = cv_logistic_binary(sub, "upper_state", feat_cols) if len(feat_cols) > 0 else None
        if res is None:
            continue
        rows.append({
            "transition": f"{lo}→{hi}",
            "stage_name": stage_name,
            "block": block_name,
            "n": res["n"],
            "auc_mean": res["auc_mean"],
            "auc_lo": res["auc_lo"],
            "auc_hi": res["auc_hi"],
            "ap_mean": res["ap_mean"],
            "n_features": len(res["features"]),
            "features": " | ".join(res["features"]),
        })

l_df = pd.DataFrame(rows)
save_csv(l_df, "FigL1_transition_theory_specificity.csv")

if len(l_df) > 0:
    inc_rows = []
    order = ["0→1", "1→2", "2→3"]
    stage_map = {"0→1": "Affective onset", "1→2": "Distortion escalation", "2→3": "Suicidality crystallization"}
    for tr in order:
        dd = l_df[l_df["transition"] == tr].set_index("block")
        c0 = dd.loc["controls", "auc_mean"] if "controls" in dd.index else np.nan
        c1 = dd.loc["controls_plus_cues", "auc_mean"] if "controls_plus_cues" in dd.index else np.nan
        c2 = dd.loc["controls_plus_cues_plus_topology", "auc_mean"] if "controls_plus_cues_plus_topology" in dd.index else np.nan
        inc_rows.append({
            "transition": tr,
            "cue_increment": c1 - c0 if pd.notna(c1) and pd.notna(c0) else np.nan,
            "topology_increment": c2 - c1 if pd.notna(c2) and pd.notna(c1) else np.nan,
            "total_increment": c2 - c0 if pd.notna(c2) and pd.notna(c0) else np.nan,
        })
    l_inc_df = pd.DataFrame(inc_rows)
    save_csv(l_inc_df, "FigL2_transition_theory_specificity_increments.csv")

    fig = plt.figure(figsize=(13.6, 4.9))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.18, 0.82], wspace=0.30)
    axes = [fig.add_subplot(gs[0, i]) for i in range(2)]

    ax = axes[0]
    x = np.arange(len(order))
    block_specs = [
        ("controls", PALETTE["gray"], "Controls"),
        ("controls_plus_cues", PALETTE["orange"], "Controls + cues"),
        ("controls_plus_cues_plus_topology", PALETTE["blue"], "Controls + cues + topology"),
    ]
    top_line = l_df[l_df["block"] == "controls_plus_cues_plus_topology"].copy().set_index("transition").reindex(order)
    for block, col, label in block_specs:
        dd = l_df[l_df["block"] == block].copy().set_index("transition").reindex(order)
        ax.plot(x, dd["auc_mean"], marker="o", markersize=8, lw=2.6, color=col, label=label)
        for xi, rr in enumerate(dd.itertuples()):
            if pd.notna(rr.auc_lo) and pd.notna(rr.auc_hi):
                ax.vlines(xi, rr.auc_lo, rr.auc_hi, color=col, lw=1.8)
    for xi, rr in enumerate(l_inc_df.itertuples()):
        y0 = top_line.iloc[xi]["auc_mean"]
        if pd.notna(rr.total_increment) and pd.notna(y0):
            ax.text(xi, y0 + 0.010, f"Δ{rr.total_increment:.3f}", ha="center", va="bottom", fontsize=10.5, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(order)
    ax.set_ylabel("Cross-validated AUROC")
    ax.set_xlim(-0.10, len(order)-0.90)
    style_ax(ax)
    add_clean_legend(ax, [
        Line2D([0], [0], color=PALETTE["gray"], marker="o", lw=2.6, label="Controls"),
        Line2D([0], [0], color=PALETTE["orange"], marker="o", lw=2.6, label="Controls + cues"),
        Line2D([0], [0], color=PALETTE["blue"], marker="o", lw=2.6, label="Controls + cues + topology"),
    ], loc="upper left")

    ax = axes[1]
    dd = l_inc_df.set_index("transition").reindex(order)
    y = np.arange(len(order))
    cue_inc = dd["cue_increment"].astype(float).fillna(0)
    topo_inc = dd["topology_increment"].astype(float).fillna(0)
    ax.hlines(y + 0.10, 0, cue_inc, color=PALETTE["orange"], lw=2.6)
    ax.scatter(cue_inc, y + 0.10, s=150, color=PALETTE["orange"], zorder=5)
    ax.hlines(y - 0.10, 0, topo_inc, color=PALETTE["blue"], lw=2.0, alpha=0.85)
    ax.scatter(topo_inc, y - 0.10, s=150, facecolor="white", edgecolor=PALETTE["blue"], linewidth=2.0, zorder=5)
    for yi, val in zip(y + 0.10, cue_inc):
        ax.text(val + 0.002, yi + 0.03, f"{val:.3f}", fontsize=10, fontweight="bold")
    ax.axvline(0, color=PALETTE["lightgray"], lw=1.5)
    ax.set_yticks(y)
    ax.set_yticklabels(order)
    ax.set_xlabel("Increment over previous block (AUROC)")
    xmax = max(0.02, float(np.nanmax(cue_inc)) + 0.018)
    ax.set_xlim(-0.005, xmax)
    style_ax(ax)
    topo_label = "Topology increment (≈0)" if np.nanmax(np.abs(topo_inc.values)) < 0.01 else "Topology increment"
    add_clean_legend(ax, [
        Line2D([0], [0], color=PALETTE["orange"], marker="o", lw=2.6, label="Cue increment"),
        Line2D([0], [0], color=PALETTE["blue"], marker="o", markerfacecolor="white", lw=2.0, label=topo_label),
    ], loc="upper right")
    save_fig(fig, "FigL_transition_theory_specificity")
else:
    save_text("FigL could not be computed because no transition-level model had enough usable features and class balance.", "FigL_transition_theory_specificity_notes.txt")

def _build_visibility_score(df):
    parts = []
    for c, w in [("explicit_risk_score", 1.2), ("sadness_explicit", 0.8), ("emotion_word_density", 0.5)]:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notna().sum() >= 10:
            parts.append(w * normalize_01(df[c]).fillna(0))
    if len(parts) == 0:
        return pd.Series(0.0, index=df.index)
    return sum(parts) / len(parts)


def _build_latent_score(df):
    parts = []
    for c in ["distortion_cue_score", "bridge_core_load", "bridge_combo_rarity", "cue_conflict", "latent_risk_minus_explicit"]:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notna().sum() >= 10:
            parts.append(zscore(df[c]).fillna(0))
    if len(parts) == 0:
        return zscore(df.get("explicit_risk_score", pd.Series(0, index=df.index))).fillna(0)
    return sum(parts) / len(parts)


m_rows = []
m_mode = "none"
# Prefer class255 if dense enough; otherwise fallback to four-class map.
if analysis_df["failure_regime"].notna().sum() >= 40:
    d = analysis_df.copy()
    d["visibility_score"] = _build_visibility_score(d)
    d["latent_topology_score"] = _build_latent_score(d)
    d["visibility_bin"] = stable_rank_bins(d["visibility_score"], n_bins=3)
    d["topology_bin"] = stable_rank_bins(d["latent_topology_score"], n_bins=3)
    m_mode = "class255"
    target_specs = [("adjacent", "Adjacent error", CMAP_ORANGE), ("far", "Far error", CMAP_ORANGE), ("suicide_FN", "Suicide-FN miss", CMAP_RED)]
else:
    d = analysis_df.dropna(subset=["true_label", "pred_label"]).copy()
    if len(d) >= 60:
        d["visibility_score"] = _build_visibility_score(d)
        d["latent_topology_score"] = _build_latent_score(d)
        d["visibility_bin"] = stable_rank_bins(d["visibility_score"], n_bins=3)
        d["topology_bin"] = stable_rank_bins(d["latent_topology_score"], n_bins=3)
        err = (d["pred_label"] - d["true_label"]).abs()
        d["reg_adjacent"] = (err == 1).astype(float)
        d["reg_far"] = (err >= 2).astype(float)
        d["reg_hidden_miss"] = ((d["true_label"] >= 2) & (d["pred_label"] <= 1) & (normalize_01(d["explicit_risk_score"]).fillna(0) < 0.5)).astype(float)
        m_mode = "fourclass"
        target_specs = [("reg_adjacent", "Adjacent error", CMAP_ORANGE), ("reg_far", "Far error", CMAP_ORANGE), ("reg_hidden_miss", "Hidden high-distress miss", CMAP_RED)]
    else:
        target_specs = []

if len(target_specs) > 0:
    for reg, _, _ in target_specs:
        for vb in ["Q1", "Q2", "Q3"]:
            for tb in ["Q1", "Q2", "Q3"]:
                dd = d[(d["visibility_bin"] == vb) & (d["topology_bin"] == tb)].copy()
                rate = np.nan if len(dd) == 0 else mean_no_nan(pd.to_numeric(dd[reg], errors="coerce"))
                m_rows.append({"regime": reg, "visibility_bin": vb, "topology_bin": tb, "n": len(dd), "rate": rate})
    m_df = pd.DataFrame(m_rows)
    save_csv(m_df, "FigM1_failure_regime_surface.csv")

    all_plot = m_df.pivot_table(index="topology_bin", columns="visibility_bin", values="rate", aggfunc="mean")
    all_plot = all_plot.reindex(index=["Q1", "Q2", "Q3"], columns=["Q1", "Q2", "Q3"])
    all_plot = _trim_heatmap_frame(all_plot)
    valid_rows = list(all_plot.index)
    valid_cols = list(all_plot.columns)
    if len(valid_rows) == 0:
        valid_rows = ["Q1", "Q2", "Q3"]
    if len(valid_cols) == 0:
        valid_cols = ["Q1", "Q2", "Q3"]

    orange_regs = [t[0] for t in target_specs if t[2] == CMAP_ORANGE]
    red_regs = [t[0] for t in target_specs if t[2] == CMAP_RED]
    orange_vmax = m_df[m_df["regime"].isin(orange_regs)]["rate"].max() if len(orange_regs) > 0 else np.nan
    red_vmax = m_df[m_df["regime"].isin(red_regs)]["rate"].max() if len(red_regs) > 0 else np.nan
    orange_vmax = max(0.10, float(orange_vmax)) if pd.notna(orange_vmax) else 0.10
    red_vmax = max(0.06, float(red_vmax)) if pd.notna(red_vmax) else 0.06

    fig = plt.figure(figsize=(14.9, 4.7))
    gs = fig.add_gridspec(1, 3, wspace=0.34)
    axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
    for ax, (reg, title, cmap) in zip(axes, target_specs):
        plot = m_df[m_df["regime"] == reg].pivot(index="topology_bin", columns="visibility_bin", values="rate")
        plot = plot.reindex(index=valid_rows, columns=valid_cols)
        arr = plot.values.astype(float)
        arr = np.ma.masked_invalid(arr)
        cmap_local = cmap.copy()
        cmap_local.set_bad("#F5F2EB")
        vmax = orange_vmax if cmap == CMAP_ORANGE else red_vmax
        im = ax.imshow(arr, cmap=cmap_local, aspect="equal", vmin=0, vmax=vmax)
        ax.set_xticks(np.arange(len(valid_cols)))
        ax.set_yticks(np.arange(len(valid_rows)))
        ax.set_xticklabels(_display_bin_labels(valid_cols))
        ax.set_yticklabels(_display_bin_labels(valid_rows))
        ax.set_xlabel("Explicit visibility")
        if ax is axes[0]:
            ax.set_ylabel("Latent structure")
        else:
            ax.set_ylabel("")
        ax.set_title(title, fontsize=12, fontweight="bold")
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                val = plot.iloc[i, j]
                if pd.notna(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10, fontweight="bold")
        ax.set_xticks(np.arange(-0.5, len(valid_cols), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(valid_rows), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=1.8)
        ax.tick_params(which="minor", bottom=False, left=False)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel("Rate", rotation=90, fontweight="bold")
        style_ax(ax)
    save_fig(fig, "FigM_failure_regime_map")
    save_text(f"FigM mode: {m_mode}", "FigM_failure_regime_map_notes.txt")
else:
    save_text("FigM skipped: neither class255 nor four-class failure-regime mapping had enough usable samples.", "FigM_failure_regime_map_notes.txt")

def recall_at_budget(df, truth_col, score_col, budgets=(0.01, 0.02, 0.05, 0.10, 0.20)):
    out = []
    d = df.copy()
    if truth_col not in d.columns or score_col not in d.columns:
        return pd.DataFrame(out)
    d[truth_col] = pd.to_numeric(d[truth_col], errors="coerce")
    d[score_col] = pd.to_numeric(d[score_col], errors="coerce")
    d = d[d[truth_col].notna() & d[score_col].notna()].copy()
    if len(d) == 0 or d[truth_col].sum() == 0:
        return pd.DataFrame(out)
    d = d.sort_values(score_col, ascending=False).reset_index(drop=True)
    total_pos = float((d[truth_col] == 1).sum())
    for b in budgets:
        k = max(1, int(math.ceil(len(d) * b)))
        top = d.iloc[:k]
        tp = float((top[truth_col] == 1).sum())
        out.append({"budget_frac": b, "k": k, "recall": safe_ratio(tp, total_pos), "precision": safe_ratio(tp, k)})
    return pd.DataFrame(out)


n_rows = []
d = analysis_df.copy()
d["severity_risk_score"] = normalize_01(d["pred_label"]).fillna(0)
d["cue_risk_score"] = (
    2.0 * d.get("suicide_explicit", 0).fillna(0) +
    1.2 * d.get("hopelessness_explicit", 0).fillna(0) +
    1.0 * d.get("worthlessness_explicit", 0).fillna(0) +
    0.8 * normalize_01(d.get("distortion_cue_score", 0)).fillna(0)
)
topo_parts = []
for c in ["bridge_core_load", "bridge_combo_rarity", "latent_risk_minus_explicit", "cue_conflict", "distortion_cue_score"]:
    if c in d.columns and pd.to_numeric(d[c], errors="coerce").notna().sum() >= 10:
        topo_parts.append(zscore(d[c]).fillna(0))
d["topology_risk_score"] = sum(topo_parts) / len(topo_parts) if len(topo_parts) > 0 else zscore(d["cue_risk_score"]).fillna(0)
d["hybrid_risk_score"] = 0.45 * zscore(d["severity_risk_score"]).fillna(0) + 0.20 * zscore(d["cue_risk_score"]).fillna(0) + 0.35 * zscore(d["topology_risk_score"]).fillna(0)

if d["true_suicide"].notna().sum() >= 20 and d["true_suicide"].sum() >= 5:
    d["target_primary"] = (d["true_suicide"] == 1).astype(float)
    d["target_secondary"] = ((d["true_suicide"] == 1) & (normalize_01(d["explicit_risk_score"]).fillna(0) < 0.5)).astype(float)
    target_map = {"target_primary": "Recall of true suicide-intent", "target_secondary": "Recall of hidden high-risk suicide"}
    n_mode = "class255"
else:
    d = d.dropna(subset=["true_label", "pred_label"]).copy()
    d["target_primary"] = (d["true_label"] >= 2).astype(float)
    d["target_secondary"] = ((d["true_label"] >= 2) & (normalize_01(d["explicit_risk_score"]).fillna(0) < 0.5)).astype(float)
    target_map = {"target_primary": "Recall of true high-distress", "target_secondary": "Recall of hidden high-distress"}
    n_mode = "fourclass"

for strategy, sc in [("severity", "severity_risk_score"), ("cue", "cue_risk_score"), ("topology", "topology_risk_score"), ("hybrid", "hybrid_risk_score")]:
    for target in ["target_primary", "target_secondary"]:
        dd = recall_at_budget(d, target, sc)
        if len(dd) > 0:
            dd["target"] = target
            dd["strategy"] = strategy
            n_rows.append(dd)

n_df = pd.concat(n_rows, ignore_index=True) if len(n_rows) > 0 else pd.DataFrame()
if len(n_df) > 0:
    save_csv(n_df, "FigN1_triage_utility_curves.csv")
    fig = plt.figure(figsize=(13.8, 5.0))
    gs = fig.add_gridspec(1, 2, wspace=0.24)
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
    color_map = {"severity": PALETTE["gray"], "cue": PALETTE["orange"], "topology": PALETTE["blue"], "hybrid": PALETTE["red"]}
    label_map = {"severity": "Severity score", "cue": "Cue-aware risk", "topology": "Topology-aware risk", "hybrid": "Hybrid routing"}
    for ax, target in zip(axes, ["target_primary", "target_secondary"]):
        plot = n_df[n_df["target"] == target].copy()
        for strategy in ["severity", "cue", "topology", "hybrid"]:
            dd = plot[plot["strategy"] == strategy].sort_values("budget_frac")
            if len(dd) == 0:
                continue
            ax.plot(dd["budget_frac"] * 100, dd["recall"], marker="o", ms=8, lw=2.6, color=color_map[strategy], label=label_map[strategy])
        ax.set_xlabel("Review budget (%)")
        ax.set_ylabel(target_map[target])
        ax.set_xlim(0.8, 20.5)
        ax.set_ylim(0, 1.0)
        ax.set_xticks([1, 2, 5, 10, 20])
        style_ax(ax)
    handles = [Line2D([0], [0], color=color_map[k], marker="o", lw=2.6, label=v) for k, v in label_map.items()]
    fig.legend(handles=handles, frameon=False, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.03))
    fig.subplots_adjust(top=0.82)
    save_fig(fig, "FigN_triage_utility")
    save_text(f"FigN mode: {n_mode}", "FigN_triage_utility_notes.txt")
else:
    save_text("FigN skipped: no target produced enough positive samples for budgeted-routing analysis.", "FigN_triage_utility_notes.txt")

o_df_rows = []
analysis_df["family_affective_onset"] = (
    zscore(analysis_df.get("sadness_explicit", 0)).fillna(0) +
    zscore(analysis_df.get("emotion_word_density", 0)).fillna(0) +
    zscore(analysis_df.get("first_person_density", 0)).fillna(0)
) / 3.0
analysis_df["family_distortion_escalation"] = (
    zscore(analysis_df.get("hopelessness_explicit", 0)).fillna(0) +
    zscore(analysis_df.get("worthlessness_explicit", 0)).fillna(0) +
    zscore(analysis_df.get("absolutist", 0)).fillna(0) +
    zscore(analysis_df.get("negation_density", 0)).fillna(0)
) / 4.0
suicidality_parts = [zscore(analysis_df.get("suicide_explicit", 0)).fillna(0), zscore(analysis_df.get("explicit_risk_score", 0)).fillna(0)]
if pd.to_numeric(analysis_df.get("bridge_core_load", np.nan), errors="coerce").notna().sum() >= 10:
    suicidality_parts.append(zscore(analysis_df["bridge_core_load"]).fillna(0))
analysis_df["family_suicidality_crystallization"] = sum(suicidality_parts) / len(suicidality_parts)

family_cols = [("family_affective_onset", "Affective onset"), ("family_distortion_escalation", "Distortion escalation"), ("family_suicidality_crystallization", "Suicidality crystallization")]
for lo, hi, stage_name in transitions:
    sub = analysis_df[analysis_df["true_label"].isin([lo, hi])].copy()
    lo_df = sub[sub["true_label"] == lo].copy(); hi_df = sub[sub["true_label"] == hi].copy()
    if len(lo_df) < 8 or len(hi_df) < 8:
        continue
    for col, disp in family_cols:
        mean_eff, ci_lo, ci_hi = bootstrap_mean_diff(hi_df[col], lo_df[col], n_boot=600)
        o_df_rows.append({"transition": f"{lo}→{hi}", "stage_name": stage_name, "family": disp, "effect": mean_eff, "ci_low": ci_lo, "ci_high": ci_hi})

o_df = pd.DataFrame(o_df_rows)
save_csv(o_df, "FigO1_stage_mechanism_family_effects.csv")

if len(o_df) > 0:
    fig = plt.figure(figsize=(13.8, 5.0))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.16, 0.84], wspace=0.30)
    axes = [fig.add_subplot(gs[0, i]) for i in range(2)]
    ax = axes[0]
    order = ["0→1", "1→2", "2→3"]
    fam_order = ["Affective onset", "Distortion escalation", "Suicidality crystallization"]
    color_map = {"Affective onset": PALETTE["green"], "Distortion escalation": PALETTE["orange"], "Suicidality crystallization": PALETTE["red"]}
    x = np.arange(len(order))
    for fam in fam_order:
        dd = o_df[o_df["family"] == fam].set_index("transition").reindex(order)
        ax.plot(x, dd["effect"], marker="o", lw=2.6, color=color_map[fam], label=fam)
        for xi, rr in enumerate(dd.itertuples()):
            if pd.notna(rr.ci_low) and pd.notna(rr.ci_high):
                ax.vlines(xi, rr.ci_low, rr.ci_high, color=color_map[fam], lw=1.8)
    ax.axhline(0, color=PALETTE["lightgray"], lw=1.5)
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set_ylabel("Upper-minus-lower effect")
    style_ax(ax)
    add_clean_legend(ax, [Line2D([0], [0], color=color_map[f], marker="o", lw=2.6, label=f) for f in fam_order], loc="upper left")

    ax = axes[1]
    summary = []
    for tr in order:
        dd = o_df[o_df["transition"] == tr].copy().sort_values("effect", ascending=False)
        if len(dd) == 0:
            continue
        top = dd.iloc[0]
        summary.append({"transition": tr, "dominant_family": top["family"], "effect": top["effect"]})
    summary_df2 = pd.DataFrame(summary).set_index("transition").reindex(order)
    if len(summary_df2) > 0:
        ax.bar(np.arange(len(summary_df2)), summary_df2["effect"], color=[color_map.get(f, PALETTE["gray"]) for f in summary_df2["dominant_family"]], width=0.62)
        ax.set_xticks(np.arange(len(summary_df2)))
        ax.set_xticklabels(summary_df2.index)
        ymax = float(np.nanmax(summary_df2["effect"])) if summary_df2["effect"].notna().sum() > 0 else 1.0
        ax.set_ylim(0, ymax + 0.35)
        for i, rr in enumerate(summary_df2.itertuples()):
            ax.text(i, rr.effect + 0.05, rr.dominant_family.replace(" ", "\n"), ha="center", va="bottom", fontsize=11, fontweight="bold")
        ax.set_ylabel("Dominant family effect")
    style_ax(ax)
    save_fig(fig, "FigO_stage_mechanism_families")
else:
    save_text("FigO skipped: stage-family effects were too sparse after transition splitting.", "FigO_stage_mechanism_families_notes.txt")

p_rows = []
if analysis_df["human_uncertainty_score"].notna().sum() >= 24 and analysis_df["human_uncertainty_score"].nunique(dropna=True) >= 3:
    rename_map = {
        "cue_conflict": "Cue conflict",
        "distortion_cue_score": "Distortion cue score",
        "bridge_combo_rarity": "Topology rarity",
        "bridge_core_load": "Core latent risk",
        "latent_risk_minus_explicit": "Latent-over-explicit risk",
    }
    for col in rename_map:
        if col in analysis_df.columns and pd.to_numeric(analysis_df[col], errors="coerce").notna().sum() >= 20:
            p_rows.append({"subset": "all", "feature": rename_map[col], "spearman_r": corr_no_nan(analysis_df[col], analysis_df["human_uncertainty_score"], method="spearman")})

    bd = analysis_df[analysis_df["true_label"].isin([1, 2])].copy()
    if len(bd) >= 24 and bd["human_uncertainty_score"].notna().sum() >= 18:
        parts = []
        for c in ["distortion_cue_score", "cue_conflict", "bridge_combo_rarity", "bridge_core_load"]:
            if c in bd.columns and pd.to_numeric(bd[c], errors="coerce").notna().sum() >= 10:
                parts.append(zscore(bd[c]).fillna(0))
        bd["structure_score"] = sum(parts) / len(parts) if len(parts) > 0 else zscore(bd["distortion_cue_score"]).fillna(0)
        bd["structure_bin"] = stable_rank_bins(bd["structure_score"], n_bins=3)
        for bb in ["Q1", "Q2", "Q3"]:
            dd = bd[bd["structure_bin"] == bb]
            if len(dd) == 0:
                continue
            p_rows.append({"subset": "boundary_1_2", "feature": bb, "spearman_r": mean_no_nan(dd["human_uncertainty_score"])})
    p_df = pd.DataFrame(p_rows)
    save_csv(p_df, "FigP1_human_judgment_alignment.csv")

    fig = plt.figure(figsize=(13.2, 4.6))
    gs = fig.add_gridspec(1, 2, width_ratios=[0.95, 1.05], wspace=0.22)
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])]
    ax = axes[0]
    plot = p_df[p_df["subset"] == "all"].dropna().sort_values("spearman_r")
    if len(plot) > 0:
        y = np.arange(len(plot))
        ax.hlines(y, 0, plot["spearman_r"], color=PALETTE["purple"], lw=2.6)
        ax.scatter(plot["spearman_r"], y, s=150, color=PALETTE["purple"], zorder=5)
        for yi, val in zip(y, plot["spearman_r"]):
            ax.text(val + 0.0007, yi + 0.08, f"{val:.3f}", fontsize=9.5, fontweight="bold")
        ax.axvline(0, color=PALETTE["lightgray"], lw=1.5)
        ax.set_yticks(y)
        ax.set_yticklabels(plot["feature"])
        xmin = min(-0.002, float(np.nanmin(plot["spearman_r"])) - 0.002)
        xmax = max(0.015, float(np.nanmax(plot["spearman_r"])) + 0.004)
        ax.set_xlim(xmin, xmax)
        ax.set_xlabel("Spearman with human uncertainty")
    else:
        ax.text(0.5, 0.5, "No stable full-sample alignment", ha="center", va="center", fontweight="bold")
    style_ax(ax)

    ax = axes[1]
    plot = p_df[p_df["subset"] == "boundary_1_2"].copy().set_index("feature")
    bin_order = [b for b in ["Q1", "Q2", "Q3"] if b in plot.index and pd.notna(plot.loc[b, "spearman_r"])]
    if len(bin_order) >= 2:
        vals = [plot.loc[b, "spearman_r"] for b in bin_order]
        ax.plot(np.arange(len(bin_order)), vals, marker="o", lw=2.6, color=PALETTE["purple"])
        ax.set_xticks(np.arange(len(bin_order)))
        ax.set_xticklabels(_display_bin_labels(bin_order))
        ax.set_xlabel("Boundary-local structure bin")
        ax.set_ylabel("Mean human uncertainty")
    else:
        ax.text(0.5, 0.5, "Boundary-local uncertainty unavailable", ha="center", va="center", fontweight="bold")
    style_ax(ax)
    save_fig(fig, "FigP_human_judgment_alignment")
else:
    save_text("FigP skipped: human-uncertainty signal was too sparse or too degenerate.", "FigP_human_judgment_alignment_notes.txt")

summary_lines = []
summary_lines.append(f"rows in aligned NHB analysis frame: {len(analysis_df)}")
summary_lines.append(f"symptom source: {analysis_df['symptom_source'].dropna().mode().iloc[0] if analysis_df['symptom_source'].notna().sum() > 0 else 'none'}")
summary_lines.append(f"human uncertainty usable: {analysis_df['human_uncertainty_score'].notna().sum() >= 24 and analysis_df['human_uncertainty_score'].nunique(dropna=True) >= 3}")
summary_lines.append(f"class255 risk usable: {analysis_df['true_suicide'].notna().sum() >= 20 if 'true_suicide' in analysis_df.columns else False}")
summary_lines.append(f"FigM mode: {m_mode}")
summary_lines.append(f"FigN mode: {n_mode}")
save_text("\n".join(summary_lines), "nhb_gap_pack_summary.txt")

print("\n[DONE] NHB gap-pack complete.")

ROOT_DEPRESSED = D:\Downloads\Depressed\Depressed
PRED_DIR       = D:\Downloads\Depressed\Depressed\export_bundle_v4\predictions
Prediction CSVs: 108
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\tables\run_inventory_nhb_gap_pack.csv
[Saved Table] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\tables\run_metrics_nhb_gap_pack.csv
[Saved Text] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\texts\nhb_gap_pack_best_runs.txt
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\nhb_gap_pack_master_analysis_df.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigL1_transition_theory_specificity.csv
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigL2_transition_theory_specificity_increments.csv


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity.eps
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigM1_failure_regime_surface.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigM_failure_regime_map.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigM_failure_regime_map.eps
[Saved Text] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\texts\FigM_failure_regime_map_notes.txt
[Saved CSV] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figure_csv\FigN1_triage_utility_curves.csv
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigN_triage_utility.png
[Saved] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigN_triage_utility.eps
[Saved Te

In [ ]:
# -*- coding: utf-8 -*-
"""
Q) Stage-conditional residual specificity
R) Failure interaction surface
S) Hidden-risk enrichment beyond severity baseline
T) Targeted human boundary alignment / re-annotation pool
U) Minimal-edit counterfactual ladder
"""

import os
import re
import math
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

warnings.filterwarnings("ignore")


ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"
OUT_DIR = os.path.join(ROOT_DEPRESSED, "final_pub_outputs_fullcover")
OUT_FIG = os.path.join(OUT_DIR, "figures")
OUT_CSV = os.path.join(OUT_DIR, "figure_csv")
OUT_TAB = os.path.join(OUT_DIR, "tables")
OUT_TXT = os.path.join(OUT_DIR, "texts")
for d in [OUT_DIR, OUT_FIG, OUT_CSV, OUT_TAB, OUT_TXT]:
    os.makedirs(d, exist_ok=True)

MASTER_ANALYSIS_FP = os.path.join(OUT_CSV, "nhb_gap_pack_master_analysis_df.csv")
TARGETED_ANN_FP = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "targeted_boundary_annotations.csv")
MINIMAL_EDIT_HUMAN_FP = os.path.join(ROOT_DEPRESSED, "export_bundle_v4", "minimal_edit_human_ratings.csv")


plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.linewidth": 2.0,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "axes.unicode_minus": False,
    "font.size": 15,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
})
DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "green": "#9FD39B",
    "orange": "#F39F4E",
    "pink": "#E7B1CF",
    "gray": "#D8D4CD",
    "lightgray": "#CFCBC3",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "black": "#202020",
    "deepblue": "#6EA9C4",
    "deeporange": "#D88537",
    "deepgreen": "#5FA777",
}
CMAP_ORANGE = LinearSegmentedColormap.from_list("nature_orange", ["#FAF8F4", "#F1E2CF", "#F0C48E", "#F39F4E"])
CMAP_RED = LinearSegmentedColormap.from_list("nature_red", ["#FCF6F6", "#F3DADA", "#EAB4B4", "#D95F5F"])
CMAP_PURPLE = LinearSegmentedColormap.from_list("nature_purple", ["#FAF8FD", "#E7DFF7", "#CCBDEC", "#B39DDB"])


LEXICON = {
    "sadness_explicit": ["sad", "depressed", "cry", "crying", "empty", "down", "miserable"],
    "hopelessness_explicit": ["hopeless", "no future", "nothing will change", "can't go on", "pointless"],
    "worthlessness_explicit": ["worthless", "useless", "burden", "failure", "not good enough", "hate myself"],
    "cognitive_explicit": ["forget", "memory", "focus", "concentrate", "brain fog", "can't think"],
    "suicide_explicit": ["suicide", "suicidal", "kill myself", "end my life", "die", "self-harm", "self harm"],
    "help_seeking": ["please help", "need help", "advice", "suggestions", "what do i do", "help me"],
    "absolutist": ["always", "never", "nothing", "nobody", "completely", "totally", "forever"],
}
BASE_EDIT_PHRASES = {
    "worthlessness_family": {
        1: " I feel like a burden and not good enough.",
        2: " I feel worthless, like a burden, and that nothing about me is good enough.",
    },
    "distortion_family": {
        1: " Nothing ever seems to get better and I keep thinking in circles.",
        2: " Nothing will ever get better, I can't stop the thoughts, and everything feels permanent.",
    },
    "suicidality_family": {
        1: " Sometimes I think about disappearing and not being here.",
        2: " I think about ending my life and I do not want to keep going.",
    },
}

def save_fig(fig, stem):
    png_fp = os.path.join(OUT_FIG, f"{stem}.png")
    eps_fp = os.path.join(OUT_FIG, f"{stem}.eps")
    fig.savefig(png_fp, dpi=DPI, facecolor="white", bbox_inches="tight")
    fig.savefig(eps_fp, dpi=DPI, facecolor="white", bbox_inches="tight", format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")


def save_csv(df, name):
    fp = os.path.join(OUT_CSV, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")


def save_text(text, name):
    fp = os.path.join(OUT_TXT, name)
    with open(fp, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[Saved Text] {fp}")


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.0)
    ax.spines["bottom"].set_linewidth(2.0)
    ax.tick_params(axis="both", width=2.0, length=7)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")


def add_clean_legend(ax, handles, loc="best", ncol=1):
    leg = ax.legend(handles=handles, frameon=False, loc=loc, ncol=ncol)
    if leg is not None:
        for txt in leg.get_texts():
            txt.set_fontweight("bold")
    return leg


def has_any(txt, kws):
    t = str(txt).lower()
    return int(any(k in t for k in kws))


def safe_read_csv(fp):
    if fp is None or (not os.path.exists(fp)):
        return pd.DataFrame()
    try:
        return pd.read_csv(fp)
    except Exception:
        return pd.DataFrame()


def normalize_01(x):
    s = pd.to_numeric(pd.Series(x), errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def zscore(x):
    s = pd.to_numeric(pd.Series(x), errors="coerce")
    mu, sd = s.mean(), s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mu) / sd


def corr_no_nan(x, y, method="spearman"):
    xx = pd.Series(x)
    yy = pd.Series(y)
    m = xx.notna() & yy.notna()
    if m.sum() < 4:
        return np.nan
    return float(xx[m].corr(yy[m], method=method))


def existing_cols(df, cols, min_non_na=8):
    out = []
    if df is None or len(df) == 0:
        return out
    for c in cols:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            if s.notna().sum() >= min_non_na:
                out.append(c)
    return out


def mean_no_nan(x):
    s = pd.Series(x).dropna()
    return np.nan if len(s) == 0 else float(s.mean())


def stable_rank_bins(series, n_bins=3, prefix="Q"):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(index=s.index, dtype="object")
    valid = s.notna()
    if valid.sum() < max(12, n_bins * 4):
        return out
    ranks = s[valid].rank(method="average", pct=True)
    edges = np.linspace(0, 1, n_bins + 1)
    labels = [f"{prefix}{i+1}" for i in range(n_bins)]
    bins = pd.cut(ranks, bins=edges, labels=labels, include_lowest=True)
    out.loc[valid] = bins.astype(str)
    return out


def bootstrap_mean_ci(vals, n_boot=600, alpha=0.05, random_state=42):
    vals = np.asarray(pd.Series(vals).dropna(), dtype=float)
    if len(vals) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    boots = []
    n = len(vals)
    for _ in range(n_boot):
        idx = rng.choice(np.arange(n), size=n, replace=True)
        boots.append(float(np.mean(vals[idx])))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


def bootstrap_mean_diff(x_hi, x_lo, n_boot=600, alpha=0.05, random_state=42):
    hi = np.asarray(pd.Series(x_hi).dropna(), dtype=float)
    lo = np.asarray(pd.Series(x_lo).dropna(), dtype=float)
    if len(hi) < 4 or len(lo) < 4:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(random_state)
    boots = []
    for _ in range(n_boot):
        bh = hi[rng.choice(np.arange(len(hi)), size=len(hi), replace=True)]
        bl = lo[rng.choice(np.arange(len(lo)), size=len(lo), replace=True)]
        boots.append(float(np.mean(bh) - np.mean(bl)))
    boots = np.asarray(boots)
    return float(np.mean(boots)), float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


def ensure_basic_text_features(df):
    d = df.copy()
    if "text" not in d.columns:
        d["text"] = ""
    d["text"] = d["text"].astype(str)
    if "text_len" not in d.columns:
        d["text_len"] = d["text"].str.len()
    d["text_len"] = pd.to_numeric(d["text_len"], errors="coerce").fillna(d["text"].str.len())
    if "log_text_len" not in d.columns:
        d["log_text_len"] = np.log1p(np.maximum(d["text_len"], 0))

    wc = np.maximum(d["text"].str.split().str.len().fillna(0), 1)
    if "first_person_density" not in d.columns:
        d["first_person_density"] = d["text"].str.lower().str.count(r"\b(i|me|my|mine|myself)\b") / wc
    if "negation_density" not in d.columns:
        d["negation_density"] = d["text"].str.lower().str.count(r"\b(no|not|never|nothing|can't|cannot|don't|won't)\b") / wc
    if "emotion_word_density" not in d.columns:
        d["emotion_word_density"] = d["text"].str.lower().str.count(r"\b(sad|hopeless|depressed|cry|empty|worthless|lonely|suicidal|anxious)\b") / wc
    for cue, kws in LEXICON.items():
        if cue not in d.columns:
            d[cue] = d["text"].map(lambda x: has_any(x, kws))
    return d


def fit_logistic_full(df, target_col, feature_cols, random_state=42):
    d = df.copy()
    feat_cols = existing_cols(d, feature_cols, min_non_na=max(8, int(0.1 * len(d))))
    if target_col not in d.columns or len(feat_cols) == 0:
        return None
    y = pd.to_numeric(d[target_col], errors="coerce")
    X = d[feat_cols].apply(pd.to_numeric, errors="coerce")
    mask = y.notna() & (X.notna().sum(axis=1) >= max(1, int(0.6 * len(feat_cols))))
    X = X[mask]
    y = y[mask].astype(int)
    if len(X) < 24 or y.nunique() < 2:
        return None
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=4000, class_weight="balanced", solver="liblinear", random_state=random_state)),
    ])
    pipe.fit(X, y)
    p = pipe.predict_proba(X)[:, 1]
    ll_model = log_loss(y, p, labels=[0, 1], normalize=False)
    base_p = np.repeat(float(y.mean()), len(y))
    ll_null = log_loss(y, base_p, labels=[0, 1], normalize=False)
    dev_model = 2.0 * ll_model
    dev_null = 2.0 * ll_null
    pseudo_r2 = np.nan if dev_null == 0 else 1.0 - (dev_model / dev_null)
    auc = roc_auc_score(y, p) if y.nunique() >= 2 else np.nan
    return {
        "pipe": pipe,
        "X": X,
        "y": y,
        "features": feat_cols,
        "p": p,
        "ll": ll_model,
        "deviance": dev_model,
        "null_deviance": dev_null,
        "pseudo_r2": pseudo_r2,
        "auc": auc,
    }


def cv_auc_binary(df, target_col, feature_cols, random_state=42):
    d = df.copy()
    feat_cols = existing_cols(d, feature_cols, min_non_na=max(8, int(0.1 * len(d))))
    if target_col not in d.columns or len(feat_cols) == 0:
        return np.nan, np.nan
    y = pd.to_numeric(d[target_col], errors="coerce")
    X = d[feat_cols].apply(pd.to_numeric, errors="coerce")
    mask = y.notna() & (X.notna().sum(axis=1) >= max(1, int(0.6 * len(feat_cols))))
    X = X[mask]
    y = y[mask].astype(int)
    if len(X) < 24 or y.nunique() < 2:
        return np.nan, np.nan
    min_class = int(y.value_counts().min())
    if min_class < 3:
        return np.nan, np.nan
    n_splits = min(5, max(3, min_class))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    aucs = []
    for tr, te in skf.split(X, y):
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=4000, class_weight="balanced", solver="liblinear", random_state=random_state)),
        ])
        pipe.fit(X.iloc[tr], y.iloc[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        aucs.append(roc_auc_score(y.iloc[te], p))
    return float(np.mean(aucs)), float(np.std(aucs))


def permutation_drop_auc(df, target_col, base_cols, perm_cols, random_state=42):
    d = df.copy()
    all_cols = list(dict.fromkeys(base_cols + perm_cols))
    feat_cols = existing_cols(d, all_cols, min_non_na=max(8, int(0.1 * len(d))))
    perm_cols = [c for c in perm_cols if c in feat_cols]
    if target_col not in d.columns or len(feat_cols) == 0 or len(perm_cols) == 0:
        return np.nan
    y = pd.to_numeric(d[target_col], errors="coerce")
    X = d[feat_cols].apply(pd.to_numeric, errors="coerce")
    mask = y.notna() & (X.notna().sum(axis=1) >= max(1, int(0.6 * len(feat_cols))))
    X = X[mask]
    y = y[mask].astype(int)
    if len(X) < 24 or y.nunique() < 2:
        return np.nan
    min_class = int(y.value_counts().min())
    if min_class < 3:
        return np.nan
    n_splits = min(5, max(3, min_class))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rng = np.random.RandomState(random_state)
    drops = []
    for tr, te in skf.split(X, y):
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=4000, class_weight="balanced", solver="liblinear", random_state=random_state)),
        ])
        Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
        ytr, yte = y.iloc[tr], y.iloc[te]
        pipe.fit(Xtr, ytr)
        p = pipe.predict_proba(Xte)[:, 1]
        auc_base = roc_auc_score(yte, p)
        Xperm = Xte.copy()
        for c in perm_cols:
            Xperm[c] = rng.permutation(Xperm[c].values)
        pp = pipe.predict_proba(Xperm)[:, 1]
        auc_perm = roc_auc_score(yte, pp)
        drops.append(float(auc_base - auc_perm))
    return float(np.mean(drops)) if len(drops) > 0 else np.nan


def semipartial_corr(df, y_col, target_col, control_cols):
    d = df.copy()
    if y_col not in d.columns or target_col not in d.columns:
        return np.nan
    use_controls = existing_cols(d, control_cols, min_non_na=max(8, int(0.1 * len(d))))
    y = pd.to_numeric(d[y_col], errors="coerce")
    t = pd.to_numeric(d[target_col], errors="coerce")
    if len(use_controls) == 0:
        return corr_no_nan(y, t, method="spearman")
    X = d[use_controls].apply(pd.to_numeric, errors="coerce")
    mask = y.notna() & t.notna() & (X.notna().sum(axis=1) >= max(1, int(0.6 * len(use_controls))))
    if mask.sum() < 20:
        return np.nan
    ridge = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0)),
    ])
    ridge.fit(X.loc[mask], t.loc[mask])
    resid = t.loc[mask] - ridge.predict(X.loc[mask])
    return corr_no_nan(y.loc[mask], resid, method="spearman")


def load_master_analysis_df():
    df = safe_read_csv(MASTER_ANALYSIS_FP)
    if len(df) == 0:
        raise RuntimeError(
            "Missing nhb_gap_pack_master_analysis_df.csv. Run the prior NHB gap-pack first."
        )
    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
    df = ensure_basic_text_features(df)
    for c in [
        "true_label", "pred_label", "confidence_score", "is_disagreement_sample", "cue_sum",
        "explicit_risk_score", "affective_onset_score", "distortion_cue_score", "bridge_burden",
        "bridge_core_load", "bridge_periphery_load", "core_minus_periphery", "bridge_combo_rarity",
        "latent_risk_minus_explicit"
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # Ensure cue columns exist
    for cue, kws in LEXICON.items():
        if cue not in df.columns:
            df[cue] = df["text"].map(lambda x: has_any(x, kws))
        df[cue] = pd.to_numeric(df[cue], errors="coerce").fillna(0)
    # Rebuild engineered features if needed
    if "cue_sum" not in df.columns:
        df["cue_sum"] = sum(df[c] for c in LEXICON if c in df.columns)
    if "cue_conflict" not in df.columns:
        df["cue_conflict"] = (
            df.get("help_seeking", 0).fillna(0) *
            ((df.get("hopelessness_explicit", 0).fillna(0) + df.get("worthlessness_explicit", 0).fillna(0) + df.get("suicide_explicit", 0).fillna(0)) > 0).astype(int)
        )
    if "distortion_cue_score" not in df.columns:
        df["distortion_cue_score"] = df.get("hopelessness_explicit", 0).fillna(0) + df.get("worthlessness_explicit", 0).fillna(0) + df.get("absolutist", 0).fillna(0)
    if "explicit_risk_score" not in df.columns:
        df["explicit_risk_score"] = 2.0 * df.get("suicide_explicit", 0).fillna(0) + df.get("hopelessness_explicit", 0).fillna(0) + df.get("worthlessness_explicit", 0).fillna(0)
    if "affective_onset_score" not in df.columns:
        df["affective_onset_score"] = df.get("sadness_explicit", 0).fillna(0) + normalize_01(df["emotion_word_density"]) + normalize_01(df["first_person_density"])
    # Theory family scores for Q/O/R/S/U
    df["family_affective_onset"] = (
        zscore(df.get("sadness_explicit", 0)).fillna(0) +
        zscore(df.get("emotion_word_density", 0)).fillna(0) +
        zscore(df.get("first_person_density", 0)).fillna(0)
    ) / 3.0
    df["family_distortion_escalation"] = (
        zscore(df.get("hopelessness_explicit", 0)).fillna(0) +
        zscore(df.get("absolutist", 0)).fillna(0) +
        zscore(df.get("negation_density", 0)).fillna(0)
    ) / 3.0
    df["family_worthlessness"] = (
        zscore(df.get("worthlessness_explicit", 0)).fillna(0) +
        zscore(df.get("cue_conflict", 0)).fillna(0) +
        zscore(df.get("first_person_density", 0)).fillna(0)
    ) / 3.0
    df["family_suicidality_crystallization"] = (
        zscore(df.get("suicide_explicit", 0)).fillna(0) +
        zscore(df.get("explicit_risk_score", 0)).fillna(0) +
        zscore(df.get("bridge_core_load", 0)).fillna(0)
    ) / 3.0

    # Structure / visibility composites
    df["explicit_visibility"] = (
        normalize_01(df.get("explicit_risk_score", 0)).fillna(0) * 0.7 +
        normalize_01(df.get("cue_sum", 0)).fillna(0) * 0.3
    )
    latent_parts = []
    for c in ["bridge_core_load", "bridge_combo_rarity", "bridge_burden", "family_distortion_escalation", "family_worthlessness"]:
        if c in df.columns:
            latent_parts.append(zscore(df[c]).fillna(0))
    if len(latent_parts) == 0:
        df["latent_structure"] = 0.0
    else:
        df["latent_structure"] = sum(latent_parts) / float(len(latent_parts))
    # Dominant family
    fam_cols = [
        ("Affective onset", "family_affective_onset"),
        ("Distortion escalation", "family_distortion_escalation"),
        ("Worthlessness consolidation", "family_worthlessness"),
        ("Suicidality crystallization", "family_suicidality_crystallization"),
    ]
    fam_frame = pd.DataFrame({k: pd.to_numeric(df[c], errors="coerce") for k, c in fam_cols})
    if len(fam_frame) > 0:
        df["dominant_family"] = fam_frame.idxmax(axis=1)
    else:
        df["dominant_family"] = np.nan

    # Outcome / routing features
    df["true_label"] = pd.to_numeric(df.get("true_label", np.nan), errors="coerce")
    df["pred_label"] = pd.to_numeric(df.get("pred_label", np.nan), errors="coerce")
    if "confidence_score" not in df.columns:
        df["confidence_score"] = np.nan
    df["severity_score"] = normalize_01(df.get("pred_label", 0)).fillna(0) + 0.15 * normalize_01(df.get("confidence_score", 0)).fillna(0)
    df["cue_aware_risk"] = (
        0.55 * normalize_01(df.get("pred_label", 0)).fillna(0) +
        0.30 * normalize_01(df.get("explicit_risk_score", 0)).fillna(0) +
        0.15 * normalize_01(df.get("family_distortion_escalation", 0)).fillna(0)
    )
    df["topology_aware_risk"] = (
        0.45 * normalize_01(df.get("pred_label", 0)).fillna(0) +
        0.30 * normalize_01(df.get("bridge_core_load", 0)).fillna(0) +
        0.25 * normalize_01(df.get("latent_structure", 0)).fillna(0)
    )
    df["hybrid_routing"] = (
        0.45 * df["severity_score"] + 0.30 * df["cue_aware_risk"] + 0.25 * df["topology_aware_risk"]
    )
    # Targets
    df["adjacent_error"] = ((df["pred_label"] - df["true_label"]).abs() == 1).astype(float)
    df["far_error"] = ((df["pred_label"] - df["true_label"]).abs() >= 2).astype(float)
    df["understate"] = (df["pred_label"] < df["true_label"]).astype(float)
    df["true_high_distress"] = (df["true_label"] >= 2).astype(float)
    vis_bin = stable_rank_bins(df["explicit_visibility"], n_bins=3)
    df["visibility_bin"] = vis_bin
    df["hidden_high_distress"] = ((df["true_high_distress"] == 1) & (vis_bin == "Q1")).astype(float)
    df["hidden_high_distress_miss"] = ((df["hidden_high_distress"] == 1) & (df["pred_label"] < 2)).astype(float)
    # Human uncertainty continuous
    if "human_uncertainty_score" not in df.columns or pd.to_numeric(df["human_uncertainty_score"], errors="coerce").notna().sum() < 10:
        conf = pd.to_numeric(df.get("confidence_score", np.nan), errors="coerce")
        dis = pd.to_numeric(df.get("is_disagreement_sample", np.nan), errors="coerce")
        hu = pd.Series(np.nan, index=df.index)
        if conf.notna().sum() >= 20:
            hu = zscore(1 - normalize_01(conf))
        if dis.notna().sum() >= 20 and dis.nunique(dropna=True) >= 2:
            hu = hu.fillna(0) + 0.5 * zscore(dis).fillna(0)
        df["human_uncertainty_score"] = hu
    return df


analysis_df = load_master_analysis_df()


Q_CONTROL_COLS = ["log_text_len", "first_person_density", "negation_density", "emotion_word_density", "suicide_explicit"]
Q_FAMILIES = {
    "Distortion family": ["family_distortion_escalation"],
    "Worthlessness family": ["family_worthlessness"],
    "Suicidality family": ["family_suicidality_crystallization"],
}

q_rows = []
for lo, hi in [(0, 1), (1, 2), (2, 3)]:
    sub = analysis_df[analysis_df["true_label"].isin([lo, hi])].copy()
    if len(sub) < 24:
        continue
    sub["upper_state"] = (sub["true_label"] == hi).astype(int)
    base_fit = fit_logistic_full(sub, "upper_state", Q_CONTROL_COLS)
    if base_fit is None:
        continue
    for fam_name, fam_cols in Q_FAMILIES.items():
        full_cols = list(dict.fromkeys(Q_CONTROL_COLS + fam_cols))
        full_fit = fit_logistic_full(sub, "upper_state", full_cols)
        if full_fit is None:
            continue
        perm_drop = permutation_drop_auc(sub, "upper_state", Q_CONTROL_COLS, fam_cols)
        semi = semipartial_corr(sub, "upper_state", fam_cols[0], Q_CONTROL_COLS)
        delta_pseudo = full_fit["pseudo_r2"] - base_fit["pseudo_r2"]
        dev_drop = base_fit["deviance"] - full_fit["deviance"]
        q_rows.append({
            "transition": f"{lo}→{hi}",
            "family": fam_name,
            "base_auc": base_fit["auc"],
            "full_auc": full_fit["auc"],
            "delta_auc": full_fit["auc"] - base_fit["auc"],
            "base_pseudo_r2": base_fit["pseudo_r2"],
            "full_pseudo_r2": full_fit["pseudo_r2"],
            "delta_pseudo_r2": delta_pseudo,
            "deviance_drop": dev_drop,
            "perm_auc_drop": perm_drop,
            "semipartial_corr": semi,
            "n": len(sub),
        })
q_df = pd.DataFrame(q_rows)
if len(q_df) > 0:
    save_csv(q_df, "FigQ1_stage_conditional_specificity.csv")
    # Heatmaps
    families = list(Q_FAMILIES.keys())
    transitions = ["0→1", "1→2", "2→3"]
    def _pivot(metric):
        p = q_df.pivot(index="family", columns="transition", values=metric).reindex(index=families, columns=transitions)
        return p
    q_pseudo = _pivot("delta_pseudo_r2")
    q_dev = _pivot("deviance_drop")
    q_semi = _pivot("semipartial_corr")
    fig, axes = plt.subplots(1, 3, figsize=(14.8, 4.8), constrained_layout=True)
    for ax, mat, title, cmap, fmt in [
        (axes[0], q_pseudo, "Δpseudo-R²", CMAP_ORANGE, ".03f"),
        (axes[1], q_dev, "Deviance drop", CMAP_ORANGE, ".1f"),
        (axes[2], q_semi, "Semipartial correlation", CMAP_PURPLE, ".03f"),
    ]:
        arr = mat.values.astype(float)
        im = ax.imshow(arr, cmap=cmap, aspect="auto")
        ax.set_xticks(range(len(transitions)))
        ax.set_xticklabels(transitions)
        ax.set_yticks(range(len(families)))
        ax.set_yticklabels(families)
        ax.set_title(title, fontsize=16, fontweight="bold")
        style_ax(ax)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                val = arr[i, j]
                if pd.notna(val):
                    ax.text(j, i, format(val, fmt), ha="center", va="center", fontsize=12, fontweight="bold")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        cbar.ax.tick_params(width=1.6, length=5)
    save_fig(fig, "FigQ_stage_conditional_specificity")
    best_lines = []
    for tr in transitions:
        dd = q_df[q_df["transition"] == tr].sort_values("delta_pseudo_r2", ascending=False)
        if len(dd) > 0:
            rr = dd.iloc[0]
            best_lines.append(f"{tr}: strongest residual family = {rr['family']} (Δpseudo-R²={rr['delta_pseudo_r2']:.3f}, perm-drop={rr['perm_auc_drop']:.3f})")
    save_text("\n".join(best_lines), "FigQ_stage_conditional_specificity_notes.txt")
else:
    save_text("No valid stage-conditional specificity result.", "FigQ_stage_conditional_specificity_notes.txt")


r_df = analysis_df.copy()
# Use underestimation as the generic miss outcome, which stays meaningful across all stages.
r_df["latent_bin"] = stable_rank_bins(r_df["latent_structure"], n_bins=3)
r_df["explicit_bin"] = stable_rank_bins(r_df["explicit_visibility"], n_bins=3)
family_order = ["Affective onset", "Distortion escalation", "Suicidality crystallization"]
# Map worthlessness into distortion family for stage-level presentation
r_df["dominant_stage_family"] = r_df["dominant_family"].replace({"Worthlessness consolidation": "Distortion escalation"})
r_rows = []
r_surface_frames = {}
for fam in family_order:
    sub = r_df[r_df["dominant_stage_family"] == fam].copy()
    if len(sub) < 30:
        continue
    model_df = sub[["understate", "latent_structure", "explicit_visibility"]].copy()
    model_df["latent_x_explicit"] = model_df["latent_structure"] * model_df["explicit_visibility"]
    fit = fit_logistic_full(model_df, "understate", ["latent_structure", "explicit_visibility", "latent_x_explicit"])
    if fit is None:
        continue
    # Approximate standardized OR from scaled logistic coefficients
    clf = fit["pipe"].named_steps["clf"]
    feats = fit["features"]
    coef_map = {f: float(c) for f, c in zip(feats, clf.coef_.ravel())}
    for term in ["latent_structure", "explicit_visibility", "latent_x_explicit"]:
        if term in coef_map:
            r_rows.append({
                "family": fam,
                "term": term,
                "log_odds": coef_map[term],
                "odds_ratio": float(np.exp(coef_map[term])),
            })
    # Predicted surface on observed tertile centers
    lb = stable_rank_bins(sub["latent_structure"], 3)
    eb = stable_rank_bins(sub["explicit_visibility"], 3)
    centers_lat = sub.groupby(lb)["latent_structure"].median().to_dict()
    centers_exp = sub.groupby(eb)["explicit_visibility"].median().to_dict()
    labels = [b for b in ["Q1", "Q2", "Q3"] if b in centers_lat and b in centers_exp]
    wide = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for lv in labels:
        for ev in labels:
            row = pd.DataFrame({
                "latent_structure": [centers_lat[lv]],
                "explicit_visibility": [centers_exp[ev]],
                "latent_x_explicit": [centers_lat[lv] * centers_exp[ev]],
            })
            pp = fit["pipe"].predict_proba(row)[:, 1][0]
            wide.loc[lv, ev] = pp
    r_surface_frames[fam] = wide
save_csv(pd.DataFrame(r_rows), "FigR1_failure_interaction_terms.csv")
if len(r_surface_frames) > 0:
    fig, axes = plt.subplots(1, len(r_surface_frames), figsize=(5.2 * len(r_surface_frames), 4.8), constrained_layout=True)
    if len(r_surface_frames) == 1:
        axes = [axes]
    for ax, fam in zip(axes, family_order):
        if fam not in r_surface_frames:
            ax.axis("off")
            continue
        wide = r_surface_frames[fam]
        arr = wide.values.astype(float)
        im = ax.imshow(arr, cmap=CMAP_RED, aspect="auto", vmin=max(0, np.nanmin(arr)), vmax=np.nanmax(arr) if np.nanmax(arr) > 0 else 1)
        show_bins = list(wide.columns)
        label_map = {"Q1": "low", "Q2": "mid", "Q3": "high"}
        ax.set_xticks(range(len(show_bins)))
        ax.set_xticklabels([label_map[b] for b in show_bins])
        ax.set_yticks(range(len(show_bins)))
        ax.set_yticklabels([label_map[b] for b in show_bins])
        ax.set_xlabel("Explicit visibility")
        ax.set_ylabel("Latent structure")
        ax.set_title(fam, fontsize=15, fontweight="bold")
        style_ax(ax)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                ax.text(j, i, f"{arr[i, j]:.02f}", ha="center", va="center", fontsize=12, fontweight="bold")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        cbar.set_label("Predicted understate probability", fontweight="bold")
    save_fig(fig, "FigR_failure_interaction_surface")
else:
    save_text("No valid family-specific interaction surface could be estimated.", "FigR_failure_interaction_surface_notes.txt")


def review_metrics(df, score_col, budgets=(1,2,5,10,20)):
    d = df.copy()
    if score_col not in d.columns:
        return pd.DataFrame()
    d = d[pd.to_numeric(d[score_col], errors="coerce").notna()].copy()
    d = d.sort_values(score_col, ascending=False).reset_index(drop=True)
    rows = []
    n = len(d)
    base_hidden_prev = mean_no_nan(d["hidden_high_distress"])
    for b in budgets:
        k = max(1, int(math.ceil(n * b / 100.0)))
        top = d.iloc[:k].copy()
        rest = d.iloc[k:].copy()
        tp = float(top["true_high_distress"].sum())
        fp = float(len(top) - tp)
        hidden_tp = float(top["hidden_high_distress"].sum())
        rows.append({
            "strategy": score_col,
            "budget": b,
            "reviewed_n": k,
            "recall_true_high_distress": safe_num(tp / max(float(d["true_high_distress"].sum()), 1.0)),
            "recall_hidden_high_distress": safe_num(hidden_tp / max(float(d["hidden_high_distress"].sum()), 1.0)),
            "hidden_precision": safe_num(hidden_tp / k),
            "hidden_enrichment_vs_global": safe_num((hidden_tp / k) / max(base_hidden_prev, 1e-9)),
            "benefit_harm_ratio": safe_num(tp / max(fp, 1.0)),
            "miss_hidden_share_after_review": safe_num(rest["hidden_high_distress"].sum() / max(rest["true_high_distress"].sum(), 1.0)),
            "miss_visible_share_after_review": safe_num((rest["true_high_distress"].sum() - rest["hidden_high_distress"].sum()) / max(rest["true_high_distress"].sum(), 1.0)),
        })
    return pd.DataFrame(rows)


def safe_num(x):
    try:
        return float(x)
    except Exception:
        return np.nan

s_frames = []
for strat in ["severity_score", "cue_aware_risk", "topology_aware_risk", "hybrid_routing"]:
    dd = review_metrics(analysis_df, strat)
    if len(dd) > 0:
        s_frames.append(dd)
s_df = pd.concat(s_frames, ignore_index=True) if len(s_frames) > 0 else pd.DataFrame()
if len(s_df) > 0:
    save_csv(s_df, "FigS1_hidden_risk_enrichment.csv")
    fig = plt.figure(figsize=(15.8, 4.8), constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.1, 1.0])
    ax1 = fig.add_subplot(gs[0,0])
    ax2 = fig.add_subplot(gs[0,1])
    ax3 = fig.add_subplot(gs[0,2])
    strat_map = {
        "severity_score": ("Severity baseline", PALETTE["gray"]),
        "cue_aware_risk": ("Cue-aware", PALETTE["orange"]),
        "topology_aware_risk": ("Topology-aware", PALETTE["blue"]),
        "hybrid_routing": ("Hybrid", PALETTE["red"]),
    }
    # left: hidden-risk enrichment over severity baseline
    sev = s_df[s_df["strategy"] == "severity_score"][["budget", "hidden_enrichment_vs_global"]].rename(columns={"hidden_enrichment_vs_global": "sev_enrich"})
    tmp = s_df.merge(sev, on="budget", how="left")
    for strat, (lab, col) in strat_map.items():
        dd = tmp[tmp["strategy"] == strat].copy().sort_values("budget")
        if len(dd) == 0:
            continue
        yy = dd["hidden_enrichment_vs_global"] / dd["sev_enrich"]
        ax1.plot(dd["budget"], yy, marker="o", lw=2.6, ms=7, color=col, label=lab)
    ax1.axhline(1.0, color=PALETTE["lightgray"], lw=1.5)
    ax1.set_xlabel("Review budget (%)")
    ax1.set_ylabel("Hidden-risk enrichment\n(relative to severity baseline)")
    style_ax(ax1)
    add_clean_legend(ax1, [Line2D([0],[0], color=v[1], marker='o', lw=2.6, label=v[0]) for v in strat_map.values()], loc="upper left")
    # middle: missed-case composition after routing at 10%
    dd10 = s_df[s_df["budget"] == 10].copy()
    x = np.arange(len(dd10))
    vis = dd10["miss_visible_share_after_review"].values
    hid = dd10["miss_hidden_share_after_review"].values
    ax2.bar(x, vis, color=PALETTE["gray"], width=0.66, label="Visible among missed")
    ax2.bar(x, hid, bottom=vis, color=PALETTE["pink"], width=0.66, label="Hidden among missed")
    ax2.set_xticks(x)
    ax2.set_xticklabels([strat_map[s][0] for s in dd10["strategy"]], rotation=15, ha="right")
    ax2.set_ylim(0, 1.0)
    ax2.set_ylabel("Composition of unrecovered\nhigh-distress cases (10% budget)")
    style_ax(ax2)
    add_clean_legend(ax2, [Patch(facecolor=PALETTE["gray"], label="Visible among missed"), Patch(facecolor=PALETTE["pink"], label="Hidden among missed")], loc="upper left")
    # right: benefit-harm ratio
    for strat, (lab, col) in strat_map.items():
        dd = s_df[s_df["strategy"] == strat].copy().sort_values("budget")
        if len(dd) == 0:
            continue
        ax3.plot(dd["budget"], dd["benefit_harm_ratio"], marker="o", lw=2.6, ms=7, color=col, label=lab)
    ax3.set_xlabel("Review budget (%)")
    ax3.set_ylabel("Benefit-harm ratio\n(true high-distress / reviewed non-target)")
    style_ax(ax3)
    save_fig(fig, "FigS_hidden_risk_enrichment")
    # concise notes
    best_hybrid = s_df[s_df["strategy"] == "hybrid_routing"].sort_values("budget")
    best_sev = s_df[s_df["strategy"] == "severity_score"].sort_values("budget")
    notes = []
    for b in [5,10,20]:
        hh = mean_no_nan(best_hybrid[best_hybrid["budget"] == b]["hidden_enrichment_vs_global"])
        ss = mean_no_nan(best_sev[best_sev["budget"] == b]["hidden_enrichment_vs_global"])
        notes.append(f"Budget {b}%: hidden-risk enrichment, hybrid={hh:.3f}, severity={ss:.3f}, relative={safe_num(hh/max(ss,1e-9)):.3f}")
    save_text("\n".join(notes), "FigS_hidden_risk_enrichment_notes.txt")
else:
    save_text("No valid review-utility result.", "FigS_hidden_risk_enrichment_notes.txt")


t_df = analysis_df.copy()
# Candidate strata
cand1 = t_df[(t_df["true_label"].isin([1,2]) | t_df["pred_label"].isin([1,2]))].copy()
if "confidence_score" in cand1.columns and pd.to_numeric(cand1["confidence_score"], errors="coerce").notna().sum() > 10:
    thr = pd.to_numeric(cand1["confidence_score"], errors="coerce").quantile(0.35)
    cand1 = cand1[pd.to_numeric(cand1["confidence_score"], errors="coerce") <= thr].copy()
cand1["candidate_stratum"] = "1↔2 boundary"

cand2 = t_df[(t_df["true_high_distress"] == 1)].copy()
vis_bin = stable_rank_bins(cand2["explicit_visibility"], 3)
lat_bin = stable_rank_bins(cand2["latent_structure"], 3)
cand2 = cand2[(vis_bin == "Q1") & (lat_bin == "Q3")].copy()
cand2["candidate_stratum"] = "Hidden-risk / low explicit"

cand = pd.concat([cand1, cand2], ignore_index=True, sort=False)
if len(cand) > 0:
    # prioritize diversity and manageable size
    cand = cand.drop_duplicates(subset=["sample_id"]).copy()
    cand = cand.sort_values(["candidate_stratum", "latent_structure", "explicit_visibility"], ascending=[True, False, True])
    cand_pool = cand[[c for c in [
        "sample_id", "candidate_stratum", "text", "true_label", "pred_label", "confidence_score",
        "explicit_visibility", "latent_structure", "dominant_stage_family", "distortion_cue_score",
        "family_distortion_escalation", "family_suicidality_crystallization"
    ] if c in cand.columns]].copy()
    # blank columns for future annotation
    for col in ["annotator_id", "human_label", "human_confidence", "notes"]:
        cand_pool[col] = ""
    save_csv(cand_pool, "FigT1_targeted_reannotation_candidates.csv")

    # If annotation file exists, compute actual alignment; else show pool design.
    ann = safe_read_csv(TARGETED_ANN_FP)
    if len(ann) > 0 and {"sample_id", "annotator_id", "human_label"}.issubset(ann.columns):
        ann["sample_id"] = ann["sample_id"].astype(str)
        merged = ann.merge(cand_pool[[c for c in ["sample_id", "candidate_stratum", "explicit_visibility", "latent_structure", "confidence_score"] if c in cand_pool.columns]], on="sample_id", how="left")
        # entropy / agreement per sample
        rows = []
        for sid, gg in merged.groupby("sample_id"):
            vc = gg["human_label"].value_counts(normalize=True)
            entropy = float(-(vc * np.log2(vc + 1e-12)).sum())
            pair_agree = safe_num(vc.max())
            mean_conf = mean_no_nan(pd.to_numeric(gg.get("human_confidence", np.nan), errors="coerce"))
            rows.append({
                "sample_id": sid,
                "annotator_entropy": entropy,
                "pairwise_agreement": pair_agree,
                "mean_human_confidence": mean_conf,
            })
        align = pd.DataFrame(rows).merge(cand_pool[[c for c in ["sample_id", "candidate_stratum", "explicit_visibility", "latent_structure", "confidence_score"] if c in cand_pool.columns]], on="sample_id", how="left")
        save_csv(align, "FigT2_targeted_human_alignment_metrics.csv")
        fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.6), constrained_layout=True)
        ax1, ax2 = axes
        for strat, col in [("1↔2 boundary", PALETTE["orange"]), ("Hidden-risk / low explicit", PALETTE["purple"] )]:
            dd = align[align["candidate_stratum"] == strat]
            if len(dd) == 0:
                continue
            ax1.scatter(dd["latent_structure"], dd["annotator_entropy"], s=70, color=col, alpha=0.8, label=strat)
        ax1.set_xlabel("Latent structure")
        ax1.set_ylabel("Annotator entropy")
        style_ax(ax1)
        add_clean_legend(ax1, [Patch(facecolor=PALETTE["orange"], label="1↔2 boundary"), Patch(facecolor=PALETTE["purple"], label="Hidden-risk / low explicit")], loc="upper left")
        bins = stable_rank_bins(align["latent_structure"], 3)
        dd = align.groupby(bins)["annotator_entropy"].mean().dropna()
        if len(dd) > 0:
            ax2.plot(range(len(dd)), dd.values, marker="o", lw=2.6, color=PALETTE["purple"])
            ax2.set_xticks(range(len(dd)))
            ax2.set_xticklabels(dd.index.tolist())
        ax2.set_xlabel("Latent-structure tertile")
        ax2.set_ylabel("Mean annotator entropy")
        style_ax(ax2)
        save_fig(fig, "FigT_targeted_human_boundary_alignment")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.6), constrained_layout=True)
        ax1, ax2 = axes
        cnt = cand_pool["candidate_stratum"].value_counts()
        ax1.bar(cnt.index.tolist(), cnt.values, color=[PALETTE["orange"], PALETTE["purple"]][:len(cnt)])
        ax1.set_ylabel("Candidate count")
        ax1.set_xticklabels(cnt.index.tolist(), rotation=12, ha="right")
        style_ax(ax1)
        for i, v in enumerate(cnt.values):
            ax1.text(i, v + 0.5, str(int(v)), ha="center", va="bottom", fontweight="bold")
        if {"explicit_visibility", "latent_structure"}.issubset(cand_pool.columns):
            for strat, col in [("1↔2 boundary", PALETTE["orange"]), ("Hidden-risk / low explicit", PALETTE["purple"] )]:
                dd = cand_pool[cand_pool["candidate_stratum"] == strat]
                if len(dd) == 0:
                    continue
                ax2.scatter(dd["explicit_visibility"], dd["latent_structure"], s=70, color=col, alpha=0.75, label=strat)
            ax2.set_xlabel("Explicit visibility")
            ax2.set_ylabel("Latent structure")
            style_ax(ax2)
            add_clean_legend(ax2, [Patch(facecolor=PALETTE["orange"], label="1↔2 boundary"), Patch(facecolor=PALETTE["purple"], label="Hidden-risk / low explicit")], loc="upper left")
        save_fig(fig, "FigT_targeted_human_boundary_alignment")
        save_text(
            "No targeted annotation file was found. The figure shows the re-annotation pool design, and FigT1_targeted_reannotation_candidates.csv can be used directly for new human labeling.",
            "FigT_targeted_human_boundary_alignment_notes.txt"
        )
else:
    save_text("No valid targeted re-annotation candidate pool could be built.", "FigT_targeted_human_boundary_alignment_notes.txt")

# Surrogate models from current data
u_df = analysis_df.copy()
SURR_FEATS = existing_cols(u_df, [
    "log_text_len", "first_person_density", "negation_density", "emotion_word_density",
    "sadness_explicit", "hopelessness_explicit", "worthlessness_explicit", "suicide_explicit",
    "absolutist", "help_seeking", "cue_conflict", "distortion_cue_score", "explicit_risk_score",
    "bridge_core_load", "bridge_burden", "bridge_combo_rarity", "latent_structure",
    "family_distortion_escalation", "family_worthlessness", "family_suicidality_crystallization"
])
severity_model = None
hidden_model = None
if len(SURR_FEATS) >= 4 and u_df["true_label"].notna().sum() >= 40:
    X = u_df[SURR_FEATS].apply(pd.to_numeric, errors="coerce")
    y = pd.to_numeric(u_df["true_label"], errors="coerce")
    mask = y.notna() & (X.notna().sum(axis=1) >= max(3, int(0.5 * len(SURR_FEATS))))
    if mask.sum() >= 40:
        severity_model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("reg", Ridge(alpha=1.0)),
        ])
        severity_model.fit(X.loc[mask], y.loc[mask])
if len(SURR_FEATS) >= 4 and u_df["hidden_high_distress"].notna().sum() >= 40 and u_df["hidden_high_distress"].nunique() >= 2:
    fit_h = fit_logistic_full(u_df, "hidden_high_distress", SURR_FEATS)
    if fit_h is not None:
        hidden_model = fit_h["pipe"]

base_pool = u_df[(u_df["explicit_visibility"] <= u_df["explicit_visibility"].quantile(0.35)) & (u_df["true_label"].isin([1, 2]))].copy()
if len(base_pool) < 20:
    base_pool = u_df[u_df["true_label"].isin([1, 2])].copy()
base_pool = base_pool.drop_duplicates(subset=["sample_id"]).copy()
base_pool = base_pool.head(60).copy()

u_rows = []
edit_text_rows = []
if severity_model is not None and len(base_pool) >= 12:
    for rr in base_pool.itertuples(index=False):
        base_text = str(getattr(rr, "text", ""))
        base_row = pd.DataFrame([rr._asdict()])
        base_row = ensure_basic_text_features(base_row)
        # recompute local engineered features on base row
        def engineer_one(d):
            d = ensure_basic_text_features(d.copy())
            d["cue_sum"] = sum(pd.to_numeric(d.get(c, 0), errors="coerce").fillna(0) for c in LEXICON)
            d["cue_conflict"] = (
                d.get("help_seeking", 0).fillna(0) *
                ((d.get("hopelessness_explicit", 0).fillna(0) + d.get("worthlessness_explicit", 0).fillna(0) + d.get("suicide_explicit", 0).fillna(0)) > 0).astype(int)
            )
            d["distortion_cue_score"] = d.get("hopelessness_explicit", 0).fillna(0) + d.get("worthlessness_explicit", 0).fillna(0) + d.get("absolutist", 0).fillna(0)
            d["explicit_risk_score"] = 2.0 * d.get("suicide_explicit", 0).fillna(0) + d.get("hopelessness_explicit", 0).fillna(0) + d.get("worthlessness_explicit", 0).fillna(0)
            d["family_distortion_escalation"] = (
                zscore(d.get("hopelessness_explicit", 0)).fillna(0) + zscore(d.get("absolutist", 0)).fillna(0) + zscore(d.get("negation_density", 0)).fillna(0)
            ) / 3.0
            d["family_worthlessness"] = (
                zscore(d.get("worthlessness_explicit", 0)).fillna(0) + zscore(d.get("cue_conflict", 0)).fillna(0) + zscore(d.get("first_person_density", 0)).fillna(0)
            ) / 3.0
            d["family_suicidality_crystallization"] = (
                zscore(d.get("suicide_explicit", 0)).fillna(0) + zscore(d.get("explicit_risk_score", 0)).fillna(0) + zscore(d.get("bridge_core_load", 0)).fillna(0)
            ) / 3.0
            d["latent_structure"] = (
                zscore(d.get("family_distortion_escalation", 0)).fillna(0) + zscore(d.get("family_worthlessness", 0)).fillna(0)
            ) / 2.0
            return d
        base_row = engineer_one(base_row)
        Xbase = base_row.reindex(columns=SURR_FEATS, fill_value=np.nan)
        base_sev = float(severity_model.predict(Xbase)[0])
        base_hidden = float(hidden_model.predict_proba(Xbase)[:,1][0]) if hidden_model is not None else np.nan
        for fam_key in ["worthlessness_family", "distortion_family", "suicidality_family"]:
            for level in [0, 1, 2]:
                txt = base_text if level == 0 else (base_text + BASE_EDIT_PHRASES[fam_key][level])
                row = base_row.copy()
                row["text"] = txt
                row = engineer_one(row)
                Xedit = row.reindex(columns=SURR_FEATS, fill_value=np.nan)
                pred_sev = float(severity_model.predict(Xedit)[0])
                pred_hidden = float(hidden_model.predict_proba(Xedit)[:,1][0]) if hidden_model is not None else np.nan
                u_rows.append({
                    "sample_id": str(getattr(rr, "sample_id", "")),
                    "family": fam_key,
                    "edit_level": level,
                    "predicted_severity": pred_sev,
                    "delta_severity": pred_sev - base_sev,
                    "predicted_hidden_risk": pred_hidden,
                    "delta_hidden_risk": pred_hidden - base_hidden if pd.notna(base_hidden) else np.nan,
                })
                edit_text_rows.append({
                    "sample_id": str(getattr(rr, "sample_id", "")),
                    "family": fam_key,
                    "edit_level": level,
                    "edited_text": txt,
                    "base_true_label": getattr(rr, "true_label", np.nan),
                    "base_pred_label": getattr(rr, "pred_label", np.nan),
                    "human_rating": "",
                    "human_confidence": "",
                    "notes": "",
                })

u_df_out = pd.DataFrame(u_rows)
if len(u_df_out) > 0:
    save_csv(u_df_out, "FigU1_minimal_edit_counterfactual_ladder.csv")
    save_csv(pd.DataFrame(edit_text_rows), "FigU2_minimal_edit_human_template.csv")
    fam_lab = {
        "worthlessness_family": "Hopelessness–worthlessness family",
        "distortion_family": "Absolutist / negation family",
        "suicidality_family": "Suicide-explicit family",
    }
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.8), constrained_layout=True)
    ax1, ax2 = axes
    color_map = {
        "worthlessness_family": PALETTE["orange"],
        "distortion_family": PALETTE["purple"],
        "suicidality_family": PALETTE["red"],
    }
    for fam in ["worthlessness_family", "distortion_family", "suicidality_family"]:
        dd = u_df_out[u_df_out["family"] == fam].groupby("edit_level")["delta_severity"].mean().reset_index()
        ax1.plot(dd["edit_level"], dd["delta_severity"], marker="o", lw=2.8, ms=8, color=color_map[fam], label=fam_lab[fam])
        if u_df_out["delta_hidden_risk"].notna().sum() > 0:
            dd2 = u_df_out[u_df_out["family"] == fam].groupby("edit_level")["delta_hidden_risk"].mean().reset_index()
            ax2.plot(dd2["edit_level"], dd2["delta_hidden_risk"], marker="o", lw=2.8, ms=8, color=color_map[fam], label=fam_lab[fam])
    ax1.set_xlabel("Edit level")
    ax1.set_ylabel("Model severity shift")
    ax1.set_xticks([0,1,2])
    style_ax(ax1)
    ax2.set_xlabel("Edit level")
    ax2.set_ylabel("Hidden-risk routing shift")
    ax2.set_xticks([0,1,2])
    style_ax(ax2)
    add_clean_legend(ax1, [Line2D([0],[0], color=color_map[f], marker='o', lw=2.8, label=fam_lab[f]) for f in color_map], loc="upper left")
    save_fig(fig, "FigU_minimal_edit_counterfactual_ladder")
    # Optional human rerating compare
    human_edit = safe_read_csv(MINIMAL_EDIT_HUMAN_FP)
    if len(human_edit) > 0 and {"sample_id", "family", "edit_level", "human_rating"}.issubset(human_edit.columns):
        merged = pd.DataFrame(edit_text_rows).merge(human_edit, on=["sample_id", "family", "edit_level"], how="left")
        save_csv(merged, "FigU3_minimal_edit_human_ratings_merged.csv")
else:
    save_text(
        "No valid minimal-edit ladder could be built. This can happen if the prerequisite master analysis table lacks enough moderate-severity, low-explicit posts or the surrogate severity model could not be fit.",
        "FigU_minimal_edit_counterfactual_ladder_notes.txt"
    )

summary_lines = [
    "NHB deepening pack Q-U complete.",
    "Q: stage-conditional residual specificity.",
    "R: model-based failure interaction surface.",
    "S: hidden-risk enrichment / decision utility beyond severity baseline.",
    "T: targeted human boundary alignment / re-annotation pool.",
    "U: minimal-edit counterfactual ladder (surrogate model based, with optional human-rerating template).",
]
save_text("\n".join(summary_lines), "nhb_deepening_Q_to_U_summary.txt")
print("\n[DONE] NHB deepening pack Q-U complete.")


In [ ]:

# -*- coding: utf-8 -*-
"""
- FigQ_stage_conditional_specificity
- FigR_failure_interaction_surface
- FigS_hidden_risk_enrichment
- FigT_targeted_human_boundary_alignment
- FigU_minimal_edit_counterfactual_ladder
"""

import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')

ROOT = Path(r"D:\Downloads\Depressed\Depressed")
if not ROOT.exists():
    ROOT = Path('.')
OUT = ROOT / 'final_pub_outputs_fullcover'
FIG_DIR = OUT / 'figures'
CSV_DIR = OUT / 'figure_csv'
TXT_DIR = OUT / 'texts'
for d in [FIG_DIR, CSV_DIR, TXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DPI = 800

PALETTE = {
    'gray': '#C9C5BF',
    'darkgray': '#7B756E',
    'orange': '#EE9B47',
    'blue': '#8DBED6',
    'green': '#9ED299',
    'red': '#D95C5C',
    'pink': '#D8A4C0',
    'purple': '#A993D2',
    'lightgray': '#BFB9B1',
    'beige': '#E9E4DC',
    'black': '#111111',
    'white': '#FBFBF8',
}

CMAP_ORANGE = LinearSegmentedColormap.from_list('cm_orange', ['#F4EFE8', '#F1C993', '#EE9B47'])
CMAP_RED = LinearSegmentedColormap.from_list('cm_red', ['#F4EEEE', '#E8B1B1', '#D95C5C'])
CMAP_PURPLE = LinearSegmentedColormap.from_list('cm_purple', ['#F0ECF7', '#C6B7E8', '#8F76C6'])

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Times'],
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.linewidth': 2.0,
    'xtick.major.width': 2.0,
    'ytick.major.width': 2.0,
    'xtick.major.size': 7,
    'ytick.major.size': 7,
    'legend.frameon': False,
})


def safe_read_csv(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return pd.DataFrame()
    return pd.DataFrame()


def save_fig(fig, stem: str):
    png = FIG_DIR / f'{stem}.png'
    eps = FIG_DIR / f'{stem}.eps'
    fig.savefig(png, dpi=DPI, bbox_inches='tight', facecolor=PALETTE['white'])
    fig.savefig(eps, dpi=DPI, bbox_inches='tight', facecolor=PALETTE['white'])
    plt.close(fig)


def save_text(txt: str, name: str):
    (TXT_DIR / name).write_text(txt, encoding='utf-8')


def style_ax(ax, left=True, bottom=True):
    ax.set_facecolor(PALETTE['white'])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(left)
    ax.spines['bottom'].set_visible(bottom)
    ax.tick_params(direction='out', pad=6)


def add_panel_label(ax, s):
    ax.text(-0.12, 1.04, s, transform=ax.transAxes, fontsize=18, fontweight='bold', va='bottom', ha='left')


def redraw_Q():
    q = safe_read_csv(CSV_DIR / 'FigQ1_stage_conditional_specificity.csv')
    if q.empty:
        save_text('Missing FigQ1_stage_conditional_specificity.csv; redraw skipped.', 'FigQ_redraw_notes.txt')
        return
    trans = ['0→1', '1→2', '2→3']
    fams = ['Distortion family', 'Worthlessness family', 'Suicidality family']
    metrics = [
        ('delta_pseudo_r2', 'Δpseudo-R²', CMAP_ORANGE, '.03f'),
        ('deviance_drop', 'Deviance drop', CMAP_ORANGE, '.1f'),
        ('semipartial_corr', 'Semipartial correlation', CMAP_PURPLE, '.03f'),
    ]
    fig = plt.figure(figsize=(15.6, 4.5), facecolor=PALETTE['white'])
    gs = fig.add_gridspec(1, 3, wspace=0.42)
    for i, (col, title, cmap, fmt) in enumerate(metrics):
        ax = fig.add_subplot(gs[0, i])
        mat = q.pivot(index='family', columns='transition', values=col).reindex(index=fams, columns=trans)
        arr = mat.values.astype(float)
        vmax = np.nanmax(np.abs(arr))
        if col == 'semipartial_corr':
            im = ax.imshow(arr, cmap=cmap, aspect='auto', vmin=-max(vmax, 1e-6), vmax=max(vmax, 1e-6))
        else:
            im = ax.imshow(arr, cmap=cmap, aspect='auto', vmin=0, vmax=np.nanmax(arr) if np.nanmax(arr)>0 else 1)
        ax.set_xticks(range(len(trans)))
        ax.set_xticklabels(trans)
        ax.set_yticks(range(len(fams)))
        ax.set_yticklabels(fams)
        ax.set_title(title, fontsize=15, pad=6)
        style_ax(ax)
        for r in range(arr.shape[0]):
            for c in range(arr.shape[1]):
                if pd.notna(arr[r, c]):
                    ax.text(c, r, format(arr[r, c], fmt), ha='center', va='center', fontsize=11.5, fontweight='bold')
        cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
        cbar.ax.tick_params(width=1.4, length=4)
        add_panel_label(ax, chr(65+i))
    save_fig(fig, 'FigQ_stage_conditional_specificity')


def redraw_R():
    surf_csv = CSV_DIR / 'FigR1_failure_interaction_terms.csv'
    save_text('FigR uses model-derived surfaces not fully stored in CSV. Keep current analytical figure; consider exporting surface matrices for a strict redraw.', 'FigR_redraw_notes.txt')


def redraw_S():
    s = safe_read_csv(CSV_DIR / 'FigS1_hidden_risk_enrichment.csv')
    if s.empty:
        save_text('Missing FigS1_hidden_risk_enrichment.csv; redraw skipped.', 'FigS_redraw_notes.txt')
        return
    strat_map = {
        'severity_score': ('Severity baseline', PALETTE['gray']),
        'cue_aware_risk': ('Cue-aware', PALETTE['orange']),
        'topology_aware_risk': ('Topology-aware', PALETTE['blue']),
        'hybrid_routing': ('Hybrid', PALETTE['red']),
    }
    order = [k for k in ['severity_score', 'cue_aware_risk', 'topology_aware_risk', 'hybrid_routing'] if k in s['strategy'].unique()]
    fig = plt.figure(figsize=(16.2, 4.8), facecolor=PALETTE['white'])
    gs = fig.add_gridspec(1, 3, width_ratios=[1.18, 1.0, 0.96], wspace=0.34)
    ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1]); ax3 = fig.add_subplot(gs[0,2])
    handles=[]
    for key in order:
        dd = s[s['strategy']==key].sort_values('budget')
        if dd.empty: continue
        lab, col = strat_map[key]
        if 'hidden_enrichment_vs_global' in dd.columns and 'strategy' in dd.columns:
            sev = s[s['strategy']=='severity_score'][['budget','hidden_enrichment_vs_global']].rename(columns={'hidden_enrichment_vs_global':'sev'})
            dd = dd.merge(sev, on='budget', how='left')
            yy = dd['hidden_enrichment_vs_global'] / dd['sev'].replace(0,np.nan)
        else:
            yy = np.nan * np.ones(len(dd))
        ax1.plot(dd['budget'], yy, marker='o', lw=2.5, ms=7, color=col)
        ax3.plot(dd['budget'], dd['benefit_harm_ratio'], marker='o', lw=2.5, ms=7, color=col)
        handles.append(Line2D([0],[0], color=col, marker='o', lw=2.5, label=lab))
    ax1.axhline(1.0, color=PALETTE['lightgray'], lw=1.3)
    ax1.set_xlabel('Review budget (%)')
    ax1.set_ylabel('Hidden-risk enrichment\n(relative to severity baseline)')
    ax1.set_xlim(0.5, 20.5)
    ax1.set_xticks([1,2,5,10,20])
    style_ax(ax1); add_panel_label(ax1, 'A')

    dd10 = s[s['budget']==10].copy()
    dd10 = dd10.set_index('strategy').reindex(order).reset_index()
    x = np.arange(len(dd10))
    vis = dd10['miss_visible_share_after_review'].fillna(0).values
    hid = dd10['miss_hidden_share_after_review'].fillna(0).values
    ax2.bar(x, vis, color=PALETTE['gray'], width=0.62)
    ax2.bar(x, hid, bottom=vis, color=PALETTE['pink'], width=0.62)
    ax2.set_xticks(x)
    ax2.set_xticklabels([strat_map[s0][0] for s0 in dd10['strategy']], rotation=10, ha='right')
    ax2.set_ylim(0,1)
    ax2.set_ylabel('Composition of unrecovered\nhigh-distress cases (10% budget)')
    style_ax(ax2); add_panel_label(ax2, 'B')
    ax2.legend(handles=[Patch(facecolor=PALETTE['gray'], label='Visible among missed'), Patch(facecolor=PALETTE['pink'], label='Hidden among missed')], loc='upper left')

    ax3.set_xlabel('Review budget (%)')
    ax3.set_ylabel('Benefit-harm ratio\n(true high-distress / reviewed non-target)')
    ax3.set_xlim(0.5, 20.5)
    ax3.set_xticks([1,2,5,10,20])
    style_ax(ax3); add_panel_label(ax3, 'C')
    fig.legend(handles=handles, loc='upper center', ncol=len(handles), bbox_to_anchor=(0.5, 1.03))
    save_fig(fig, 'FigS_hidden_risk_enrichment')


def redraw_T():
    t1 = safe_read_csv(CSV_DIR / 'FigT1_targeted_reannotation_candidates.csv')
    t2 = safe_read_csv(CSV_DIR / 'FigT2_targeted_human_alignment_metrics.csv')
    if t1.empty and t2.empty:
        save_text('Missing targeted re-annotation CSVs; redraw skipped.', 'FigT_redraw_notes.txt')
        return
    fig = plt.figure(figsize=(12.2, 4.4), facecolor=PALETTE['white'])
    gs = fig.add_gridspec(1,2, width_ratios=[0.84,1.16], wspace=0.22)
    ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1])
    if not t1.empty:
        cnt = t1['candidate_stratum'].value_counts()
        order = [x for x in ['1↔2 boundary','Hidden-risk / low explicit'] if x in cnt.index]
        cols = [PALETTE['orange'] if o=='1↔2 boundary' else PALETTE['purple'] for o in order]
        vals = [cnt[o] for o in order]
        xx = np.arange(len(order))
        bars = ax1.bar(xx, vals, color=cols, width=0.62)
        for b,v in zip(bars, vals):
            ax1.text(b.get_x()+b.get_width()/2, v + max(vals)*0.02 if max(vals)>0 else 0.1, str(int(v)), ha='center', va='bottom', fontsize=12.5)
        ax1.set_xticks(xx)
        ax1.set_xticklabels(order, rotation=10, ha='right')
        ax1.set_ylabel('Candidate count')
        # style_ax(ax1); add_panel_label(ax1,'A')
    else:
        ax1.axis('off')
    if not t2.empty:
        for strat, col in [('1↔2 boundary', PALETTE['orange']), ('Hidden-risk / low explicit', PALETTE['purple'])]:
            dd = t2[t2['candidate_stratum']==strat]
            if dd.empty: continue
            ax2.scatter(dd['explicit_visibility'], dd['latent_structure'], s=70, color=col, alpha=0.75, label=strat)
        ax2.set_xlabel('Explicit visibility')
        ax2.set_ylabel('Latent structure')
        # style_ax(ax2); add_panel_label(ax2,'B')
        ax2.legend(loc='upper left')
    else:
        # fallback pool-design scatter from candidate file
        if not t1.empty and {'explicit_visibility','latent_structure','candidate_stratum'}.issubset(t1.columns):
            for strat, col in [('1↔2 boundary', PALETTE['orange']), ('Hidden-risk / low explicit', PALETTE['purple'])]:
                dd = t1[t1['candidate_stratum']==strat]
                if dd.empty: continue
                ax2.scatter(dd['explicit_visibility'], dd['latent_structure'], s=70, color=col, alpha=0.75, label=strat)
            ax2.set_xlabel('Explicit visibility')
            ax2.set_ylabel('Latent structure')
            # style_ax(ax2); add_panel_label(ax2,'B')
            ax2.legend(loc='upper left')
        else:
            ax2.axis('off')
    save_fig(fig, 'FigT_targeted_human_boundary_alignment')


def redraw_U():
    u = safe_read_csv(CSV_DIR / 'FigU1_minimal_edit_counterfactual_ladder.csv')
    if u.empty:
        save_text('Missing FigU1_minimal_edit_counterfactual_ladder.csv; redraw skipped.', 'FigU_redraw_notes.txt')
        return
    fam_lab = {
        'worthlessness_family': 'Hopelessness–worthlessness family',
        'distortion_family': 'Absolutist / negation family',
        'suicidality_family': 'Suicide-explicit family',
    }
    colors = {
        'worthlessness_family': PALETTE['orange'],
        'distortion_family': PALETTE['purple'],
        'suicidality_family': PALETTE['red'],
    }
    fig = plt.figure(figsize=(11.8, 4.5), facecolor=PALETTE['white'])
    gs = fig.add_gridspec(1,2, wspace=0.24)
    ax1 = fig.add_subplot(gs[0,0]); ax2 = fig.add_subplot(gs[0,1])
    for fam in [x for x in ['worthlessness_family','distortion_family','suicidality_family'] if x in u['family'].unique()]:
        dd = u[u['family']==fam].groupby('edit_level')[['delta_severity','delta_hidden_risk']].mean().reset_index()
        ax1.plot(dd['edit_level'], dd['delta_severity'], marker='o', lw=2.6, ms=7, color=colors[fam], label=fam_lab[fam])
        if 'delta_hidden_risk' in dd.columns:
            ax2.plot(dd['edit_level'], dd['delta_hidden_risk'], marker='o', lw=2.6, ms=7, color=colors[fam], label=fam_lab[fam])
    for ax, ylabel, panel in [(ax1,'Model severity shift','A'), (ax2,'Hidden-risk routing shift','B')]:
        ax.axhline(0, color=PALETTE['lightgray'], lw=1.2)
        ax.set_xlabel('Edit level')
        ax.set_ylabel(ylabel)
        ax.set_xticks([0,1,2])
        style_ax(ax); add_panel_label(ax,panel)
    fig.legend(handles=[Line2D([0],[0], color=colors[f], marker='o', lw=2.6, label=fam_lab[f]) for f in colors if f in u['family'].unique()], loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.03))
    # if all effects are zero-like, add a compact note rather than leaving the reader guessing
    if np.nanmax(np.abs(pd.to_numeric(u.get('delta_severity', np.nan), errors='coerce').values)) < 1e-8 and np.nanmax(np.abs(pd.to_numeric(u.get('delta_hidden_risk', np.nan), errors='coerce').values)) < 1e-8:
        fig.text(0.5, 0.985, 'No measurable surrogate edit effect under the current ladder design', ha='center', va='top', fontsize=11.5)
    save_fig(fig, 'FigU_minimal_edit_counterfactual_ladder')


if __name__ == '__main__':
    redraw_Q()
    redraw_R()
    redraw_S()
    redraw_T()
    redraw_U()
    print('[DONE] Q–U nature-style redraw complete.')


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[DONE] Q–U nature-style redraw complete.


In [ ]:
# -*- coding: utf-8 -*-
"""
Standalone Figure 2E generator
"""

import os
import re
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")


ROOT_DEPRESSED = r"D:\Downloads\Depressed\Depressed"

TARGET_EIGHTLABEL_CSV = None

OUTPUT_DIR = os.path.join(ROOT_DEPRESSED, "final_pub_outputs")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
CSV_DIR = os.path.join(OUTPUT_DIR, "figure_csv")

for d in [OUTPUT_DIR, FIG_DIR, CSV_DIR]:
    os.makedirs(d, exist_ok=True)

DPI = 800


plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.unicode_minus": False,
    "font.size": 18,
    "axes.labelsize": 24,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 16,
    "axes.linewidth": 2.2,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 8,
    "ytick.major.size": 8,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

PALETTE = {
    "black": "#202020",
    "lightgray": "#CFCBC3",
}

CMAP_ORANGE = LinearSegmentedColormap.from_list(
    "nature_orange",
    ["#FBFAF7", "#F3E7D8", "#F0C990", "#F39F4E"]
)

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

EMOTION_SHORT = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.2)
    ax.spines["bottom"].set_linewidth(2.2)
    ax.tick_params(axis="both", width=2.0, length=8, pad=10)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")

def save_fig(fig, name):
    png_fp = os.path.join(FIG_DIR, f"{name}.png")
    eps_fp = os.path.join(FIG_DIR, f"{name}.eps")
    fig.savefig(png_fp, dpi=DPI, facecolor="white")
    fig.savefig(eps_fp, dpi=DPI, facecolor="white", format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")

def save_df(df, name):
    fp = os.path.join(CSV_DIR, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")

def parse_vec8(x):
    """
    Robust parsing for eight-label vectors.
    Supported:
    - b00010100
    - 00010100
    - [0,1,0,1,0,0,0,1]
    - '0 1 0 1 0 0 0 1'
    """
    if isinstance(x, list) and len(x) == 8:
        try:
            return [int(v) for v in x]
        except Exception:
            return None

    if pd.isna(x):
        return None

    s = str(x).strip()

    if s.startswith("b") and len(s) == 9 and set(s[1:]) <= {"0", "1"}:
        return [int(c) for c in s[1:]]

    if len(s) == 8 and set(s) <= {"0", "1"}:
        return [int(c) for c in s]

    nums = re.findall(r"[01]", s)
    if len(nums) == 8:
        return [int(c) for c in nums]

    return None

def macro_f1_from_eightlabel_csv(fp):
    try:
        df = pd.read_csv(fp)
    except Exception as e:
        print(f"[Skip read fail] {fp} | {e}")
        return None

    if ("true_vec" not in df.columns) or ("pred_vec" not in df.columns):
        print(f"[Skip missing true_vec/pred_vec] {fp}")
        return None

    df = df.copy()
    df["true_vec_list"] = df["true_vec"].map(parse_vec8)
    df["pred_vec_list"] = df["pred_vec"].map(parse_vec8)
    df = df[df["true_vec_list"].notna() & df["pred_vec_list"].notna()].copy()

    if len(df) == 0:
        print(f"[Skip no valid rows] {fp}")
        return None

    yt = np.array(df["true_vec_list"].tolist(), dtype=int)
    yp = np.array(df["pred_vec_list"].tolist(), dtype=int)

    try:
        score = f1_score(yt, yp, average="macro", zero_division=0)
    except Exception as e:
        print(f"[Skip metric fail] {fp} | {e}")
        return None

    return {
        "pred_csv": fp,
        "macro_f1": float(score),
        "n_valid": int(len(df))
    }

pred_dirs = sorted(glob.glob(os.path.join(ROOT_DEPRESSED, "**", "export_bundle*", "predictions"), recursive=True))
if len(pred_dirs) == 0:
    raise FileNotFoundError("Cannot find export_bundle*/predictions under ROOT_DEPRESSED.")

pred_dirs = [p for p in pred_dirs if os.path.isdir(p)]
pred_dirs = sorted(pred_dirs, key=lambda x: os.path.getmtime(x), reverse=True)
PRED_DIR = pred_dirs[0]

print("=" * 100)
print(f"[Using latest prediction dir] {PRED_DIR}")
print("=" * 100)

if TARGET_EIGHTLABEL_CSV is not None:
    if not os.path.exists(TARGET_EIGHTLABEL_CSV):
        raise FileNotFoundError(f"TARGET_EIGHTLABEL_CSV not found: {TARGET_EIGHTLABEL_CSV}")
    best_csv = TARGET_EIGHTLABEL_CSV
    print(f"[Manual target] {best_csv}")
else:
    eight_files = sorted(glob.glob(os.path.join(PRED_DIR, "eightlabel__*.csv")))
    if len(eight_files) == 0:
        raise FileNotFoundError(f"No eightlabel standardized csv found under {PRED_DIR}")

    metric_rows = []
    for fp in eight_files:
        res = macro_f1_from_eightlabel_csv(fp)
        if res is not None:
            metric_rows.append(res)

    if len(metric_rows) == 0:
        raise RuntimeError("No valid eightlabel CSV could be scored.")

    metrics_df = pd.DataFrame(metric_rows).sort_values(["macro_f1", "n_valid"], ascending=[False, False]).reset_index(drop=True)
    best_csv = metrics_df.iloc[0]["pred_csv"]

    print("[Auto-selected best eightlabel run]")
    print(metrics_df.head(10).to_string(index=False))
    print("-" * 100)
    print(f"[Best CSV] {best_csv}")
    print(f"[Best macro-F1] {metrics_df.iloc[0]['macro_f1']:.6f}")
    print(f"[Valid rows] {int(metrics_df.iloc[0]['n_valid'])}")

best_df = pd.read_csv(best_csv)
if ("true_vec" not in best_df.columns) or ("pred_vec" not in best_df.columns):
    raise KeyError("Best CSV must contain true_vec and pred_vec columns.")

best_df["true_vec_list"] = best_df["true_vec"].map(parse_vec8)
best_df["pred_vec_list"] = best_df["pred_vec"].map(parse_vec8)
best_df = best_df[best_df["true_vec_list"].notna() & best_df["pred_vec_list"].notna()].copy()

if len(best_df) == 0:
    raise RuntimeError("Best CSV has no valid parsed eight-label rows.")

Y_true = np.array(best_df["true_vec_list"].tolist(), dtype=int)

# Raw co-occurrence count and raw probability
co_count = Y_true.T @ Y_true
co_prob = co_count / float(len(Y_true))

# Final publication version: remove diagonal prevalence
co_prob_off = co_prob.astype(float).copy()
np.fill_diagonal(co_prob_off, np.nan)

co_count_off = co_count.astype(float).copy()
np.fill_diagonal(co_count_off, np.nan)

labels_short = [EMOTION_SHORT[x] for x in EMOTION_NAMES]

# Save CSV with raw off-diagonal probabilities
csv_rows = []
for i, row_lab in enumerate(EMOTION_NAMES):
    row = {"symptom": EMOTION_SHORT[row_lab]}
    for j, col_lab in enumerate(EMOTION_NAMES):
        key = EMOTION_SHORT[col_lab]
        row[key] = co_prob_off[i, j]
    csv_rows.append(row)

cooc_csv_df = pd.DataFrame(csv_rows)
save_df(cooc_csv_df, "Fig2E_eightlabel_true_cooccurrence.csv")

fig, ax = plt.subplots(figsize=(9.2, 8.2))

cmap = CMAP_ORANGE.copy()
cmap.set_bad(color="white")  # diagonal blank

vmax = np.nanmax(co_prob_off)
if (not np.isfinite(vmax)) or (vmax <= 0):
    vmax = 1.0

im = ax.imshow(
    co_prob_off,
    cmap=cmap,
    aspect="equal",
    vmin=0,
    vmax=vmax
)

# Full annotation for every off-diagonal cell
for i in range(co_prob_off.shape[0]):
    for j in range(co_prob_off.shape[1]):
        val = co_prob_off[i, j]
        if not np.isfinite(val):
            continue
        txt_color = "white" if val >= 0.68 * vmax else PALETTE["black"]
        ax.text(
            j, i, f"{val:.2f}",
            ha="center", va="center",
            fontsize=13,
            fontweight="bold",
            color=txt_color
        )

ax.set_xticks(np.arange(len(labels_short)))
ax.set_yticks(np.arange(len(labels_short)))
ax.set_xticklabels(labels_short, rotation=38, ha="right")
ax.set_yticklabels(labels_short)

ax.set_xlabel("Symptom", fontweight="bold")
ax.set_ylabel("Symptom", fontweight="bold")

style_ax(ax)

# Add thin gridlines between cells for publication readability
ax.set_xticks(np.arange(-0.5, len(labels_short), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(labels_short), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.set_ylabel("Pairwise co-occurrence probability", rotation=90, fontweight="bold")
cbar.outline.set_linewidth(1.2)
cbar.ax.tick_params(width=1.2, length=5)

fig.tight_layout()
save_fig(fig, "Fig2E_eightlabel_true_cooccurrence")

print("=" * 100)
print("[DONE] Final publication Fig2E generated successfully.")
print(f"[Source CSV] {best_csv}")
print(f"[Rows used] {len(best_df)}")
print("=" * 100)

[Using latest prediction dir] D:\Downloads\Depressed\Depressed\export_bundle_v4 0325\predictions
[Auto-selected best eightlabel run]
                                                                                                                                                      pred_csv  macro_f1  n_valid
                    D:\Downloads\Depressed\Depressed\export_bundle_v4 0325\predictions\eightlabel__MiniMax-M2.5__api_direct__MiniMax-M2.5_Depression_EIGHT.csv  0.635432      906
                    D:\Downloads\Depressed\Depressed\export_bundle_v4 0325\predictions\eightlabel__MiniMax-M2.7__api_direct__MiniMax-M2.7_Depression_EIGHT.csv  0.629148      906
                D:\Downloads\Depressed\Depressed\export_bundle_v4 0325\predictions\eightlabel__Step-3.5-Flash__api_direct__Step-3.5-Flash_Depression_EIGHT.csv  0.622120      906
                          D:\Downloads\Depressed\Depressed\export_bundle_v4 0325\predictions\eightlabel__kimi-k2.5__api_direct__kimi-k2.5_Depression_EIGHT.

In [ ]:
# -*- coding: utf-8 -*-
"""
Redraw Fig I / Fig L / Fig U 
"""

import os
import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")


ROOT_CANDIDATES = [
    Path(r"D:\Downloads\Depressed"),
    Path("."),
]

ROOT = None
for p in ROOT_CANDIDATES:
    if p.exists():
        ROOT = p
        break
if ROOT is None:
    ROOT = Path(".")

OUT = ROOT / "final_pub_outputs_fullcover"
FIG_DIR = OUT / "figures"
CSV_DIR = OUT / "figure_csv"
TXT_DIR = OUT / "texts"
ZIP_FP = ROOT / "final_pub_outputs_fullcover.zip"

for d in [FIG_DIR, CSV_DIR, TXT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DPI = 800

# =========================================================
# 1. STYLE
# =========================================================
PALETTE = {
    "gray": "#CBC7C0",
    "darkgray": "#7A746C",
    "orange": "#EE9B47",
    "blue": "#8FC6DE",
    "green": "#9CD497",
    "red": "#D95C5C",
    "purple": "#AD98D7",
    "lightgray": "#C9C3BC",
    "black": "#111111",
    "white": "#FBFBF8",
}

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 2.0,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "axes.unicode_minus": False,
    "font.size": 15,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 13,
    "savefig.facecolor": PALETTE["white"],
    "figure.facecolor": PALETTE["white"],
    "axes.facecolor": PALETTE["white"],
})


def read_csv_any(name_candidates):
    for name in name_candidates:
        fp = CSV_DIR / name
        if fp.exists():
            try:
                return pd.read_csv(fp)
            except Exception:
                pass
    if ZIP_FP.exists():
        try:
            with zipfile.ZipFile(ZIP_FP, "r") as z:
                for name in name_candidates:
                    arc = f"figure_csv/{name}"
                    if arc in z.namelist():
                        return pd.read_csv(io.BytesIO(z.read(arc)))
        except Exception:
            pass
    return pd.DataFrame()


def save_fig(fig, stem):
    png_fp = FIG_DIR / f"{stem}.png"
    eps_fp = FIG_DIR / f"{stem}.eps"
    fig.savefig(png_fp, dpi=DPI, bbox_inches="tight", facecolor=PALETTE["white"])
    fig.savefig(eps_fp, dpi=DPI, bbox_inches="tight", facecolor=PALETTE["white"], format="eps")
    plt.close(fig)
    print(f"[Saved] {png_fp}")
    print(f"[Saved] {eps_fp}")


def save_text(text, name):
    fp = TXT_DIR / name
    fp.write_text(text, encoding="utf-8")
    print(f"[Saved Text] {fp}")


def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(2.0)
    ax.spines["bottom"].set_linewidth(2.0)
    ax.tick_params(axis="both", width=2.0, length=7, pad=6)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")


def add_panel_label(ax, label):
    ax.text(-0.12, 1.04, label, transform=ax.transAxes,
            fontsize=21, fontweight="bold", ha="left", va="bottom")


def infer_transition_order(vals):
    pref = ["0→1", "1→2", "2→3"]
    vals = [str(v) for v in vals if pd.notna(v)]
    out = [x for x in pref if x in vals]
    out += [x for x in vals if x not in out]
    return out


def parse_nested_auc_df(df):
    """Convert either long-format or wide-format I/L nested model CSV into a standard wide table."""
    if df.empty:
        return pd.DataFrame()
    out = pd.DataFrame()

    # long format used by FigI1_transition_specific_nested_cv.csv
    if {"transition", "spec", "cv_auc"}.issubset(df.columns):
        piv = df.pivot_table(index="transition", columns="spec", values="cv_auc", aggfunc="mean")
        piv.columns = [str(c) for c in piv.columns]
        out = piv.reset_index()
        rename_map = {
            "Controls": "controls_auc",
            "Controls + cues": "controls_plus_cues_auc",
            "Controls + cues + bridge": "controls_plus_cues_plus_topology_auc",
            "Controls + cues + topology": "controls_plus_cues_plus_topology_auc",
            "Controls + structure": "controls_plus_structure_auc",
        }
        out = out.rename(columns=rename_map)
        return out

    # long format used by FigL1_transition_theory_specificity.csv
    if {"transition", "block", "auc_mean"}.issubset(df.columns):
        piv = df.pivot_table(index="transition", columns="block", values="auc_mean", aggfunc="mean")
        piv.columns = [str(c) for c in piv.columns]
        out = piv.reset_index()
        rename_map = {
            "controls": "controls_auc",
            "controls_plus_cues": "controls_plus_cues_auc",
            "controls_plus_cues_plus_topology": "controls_plus_cues_plus_topology_auc",
            "controls_plus_bridge": "controls_plus_cues_plus_topology_auc",
            "controls_plus_cues_plus_bridge": "controls_plus_cues_plus_topology_auc",
            "controls_plus_structure": "controls_plus_structure_auc",
        }
        out = out.rename(columns=rename_map)
        return out

    # already wide
    keep = [c for c in [
        "transition", "controls_auc", "controls_plus_cues_auc",
        "controls_plus_cues_plus_topology_auc", "controls_plus_structure_auc"
    ] if c in df.columns]
    return df[keep].copy() if keep else pd.DataFrame()


def draw_fig_I():
    i1 = read_csv_any(["FigI1_transition_specific_nested_cv.csv"])
    i2 = read_csv_any(["FigI2_transition_specific_gains.csv"])

    if i1.empty and i2.empty:
        save_text("Fig I layout-fix skipped because the required CSVs were not found.", "FigI_layoutfix_notes.txt")
        return

    nested = parse_nested_auc_df(i1)
    trans = infer_transition_order(nested["transition"].tolist() if not nested.empty else i2.get("transition", []).tolist())

    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), gridspec_kw={"wspace": 0.28})

    # ---------------- left ----------------
    ax = axes[0]
    if not nested.empty:
        dd = nested.set_index("transition").reindex(trans)
        x = np.arange(len(trans))
        series = [
            ("controls_auc", PALETTE["gray"], "Controls"),
            ("controls_plus_cues_auc", PALETTE["orange"], "Controls + cues"),
            ("controls_plus_cues_plus_topology_auc", PALETTE["blue"], "Controls + cues + topology"),
        ]
        handles = []
        ymax = 0
        ymin = 1
        for col, color, lab in series:
            if col not in dd.columns:
                continue
            y = dd[col].astype(float).values
            if np.isfinite(y).sum() == 0:
                continue
            ax.plot(x, y, color=color, lw=2.8, marker="o", ms=8, label=lab)
            ymax = max(ymax, np.nanmax(y))
            ymin = min(ymin, np.nanmin(y))
            handles.append(Line2D([0], [0], color=color, lw=2.8, marker="o", ms=8, label=lab))
        ax.set_xticks(x)
        ax.set_xticklabels(trans)
        ax.set_xlabel("Adjacent transition")
        ax.set_ylabel("Cross-validated AUROC")
        ax.set_ylim(max(0.5, ymin - 0.03), min(1.0, ymax + 0.05))
        ax.legend(handles=handles, loc="upper left", frameon=False)
        style_ax(ax)
        #add_panel_label(ax, "A")
    else:
        ax.axis("off")

    # ---------------- right ----------------
    ax = axes[1]
    if not i2.empty and "transition" in i2.columns:
        dd = i2.set_index("transition").reindex(trans)
        y = np.arange(len(trans))[::-1]
        cue = dd["cue_increment"].astype(float).values if "cue_increment" in dd.columns else np.full(len(dd), np.nan)
        topo = dd["bridge_increment"].astype(float).values if "bridge_increment" in dd.columns else np.full(len(dd), np.nan)

        for yi, val in zip(y, cue):
            if np.isfinite(val):
                ax.hlines(yi + 0.12, 0, val, color=PALETTE["orange"], lw=2.6)
                ax.scatter(val, yi + 0.12, s=170, color=PALETTE["orange"], zorder=5)
                ax.text(val + 0.0025, yi + 0.12, f"{val:.3f}", va="bottom", ha="left", fontsize=11)

        topo_is_near_zero = np.isfinite(topo).any() and np.nanmax(np.abs(topo)) < 1e-4
        for yi, val in zip(y, topo):
            if np.isfinite(val):
                # show a short stub even when effect is ~0 so it is visible
                xend = val if abs(val) > 1e-4 else 0.0003
                ax.hlines(yi - 0.12, 0, xend, color=PALETTE["blue"], lw=2.6)
                ax.scatter(val, yi - 0.12, s=170, facecolor=PALETTE["white"],
                           edgecolor=PALETTE["blue"], linewidth=2.6, zorder=6)

        handles = [
            Line2D([0], [0], color=PALETTE["orange"], lw=2.6, marker="o", ms=8, label="Cue increment"),
            Line2D([0], [0], color=PALETTE["blue"], lw=2.6, marker="o", ms=8,
                   markerfacecolor=PALETTE["white"], label="Topology increment"),
        ]
        ax.legend(handles=handles, loc="upper right", frameon=False)
        if topo_is_near_zero:
            ax.text(0.97, 0.06, "Topology increment ≈ 0 across transitions",
                    transform=ax.transAxes, ha="right", va="bottom", fontsize=10.5)
        ax.axvline(0, color=PALETTE["lightgray"], lw=1.4)
        ax.set_yticks(y)
        ax.set_yticklabels(trans)
        ax.set_xlabel("Increment over previous block (AUROC)")
        ax.set_xlim(-0.005, max(0.03, np.nanmax(cue) + 0.025 if np.isfinite(cue).any() else 0.03))
        style_ax(ax)
        #add_panel_label(ax, "B")
    else:
        ax.axis("off")

    save_fig(fig, "FigI_specificity_beyond_shallow_controls_layoutfix")



def draw_fig_L():
    l1 = read_csv_any(["FigL1_transition_theory_specificity.csv"])
    l2 = read_csv_any(["FigL2_transition_theory_specificity_increments.csv"])

    if l1.empty and l2.empty:
        save_text("Fig L layout-fix skipped because the required CSVs were not found.", "FigL_layoutfix_notes.txt")
        return

    nested = parse_nested_auc_df(l1)
    trans = infer_transition_order(nested["transition"].tolist() if not nested.empty else l2.get("transition", []).tolist())

    # prepare optional CI tables from raw long df
    ci_map = {}
    if {"transition", "block", "auc_mean", "auc_lo", "auc_hi"}.issubset(l1.columns):
        for _, r in l1.iterrows():
            blk = str(r["block"])
            blk = {
                "controls": "controls_auc",
                "controls_plus_cues": "controls_plus_cues_auc",
                "controls_plus_cues_plus_topology": "controls_plus_cues_plus_topology_auc",
                "controls_plus_cues_plus_bridge": "controls_plus_cues_plus_topology_auc",
            }.get(blk, blk)
            ci_map[(str(r["transition"]), blk)] = (float(r["auc_lo"]), float(r["auc_hi"]))

    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), gridspec_kw={"wspace": 0.28})

    # ---------------- left ----------------
    ax = axes[0]
    if not nested.empty:
        dd = nested.set_index("transition").reindex(trans)
        x = np.arange(len(trans))
        series = [
            ("controls_auc", PALETTE["gray"], "Controls"),
            ("controls_plus_cues_auc", PALETTE["orange"], "Controls + cues"),
            ("controls_plus_cues_plus_topology_auc", PALETTE["blue"], "Controls + cues + topology"),
        ]
        handles = []
        ymax = 0
        ymin = 1
        for col, color, lab in series:
            if col not in dd.columns:
                continue
            y = dd[col].astype(float).values
            if np.isfinite(y).sum() == 0:
                continue
            ax.plot(x, y, color=color, lw=2.8, marker="o", ms=8)
            # CI error bars if available
            lo_err, hi_err = [], []
            for t, yy in zip(trans, y):
                key = (t, col)
                if key in ci_map:
                    lo, hi = ci_map[key]
                    lo_err.append(max(0.0, yy - lo))
                    hi_err.append(max(0.0, hi - yy))
                else:
                    lo_err.append(np.nan)
                    hi_err.append(np.nan)
            if np.isfinite(np.array(lo_err, dtype=float)).sum() > 0:
                ax.errorbar(x, y, yerr=[lo_err, hi_err], fmt="none", ecolor=color,
                            elinewidth=1.8, capsize=0, alpha=0.95)
            ymax = max(ymax, np.nanmax(y))
            ymin = min(ymin, np.nanmin(y))
            handles.append(Line2D([0], [0], color=color, lw=2.8, marker="o", ms=8, label=lab))
        ax.set_xticks(x)
        ax.set_xticklabels(trans)
        ax.set_xlabel("Adjacent transition")
        ax.set_ylabel("Cross-validated AUROC")
        ax.set_ylim(max(0.55, ymin - 0.04), min(1.0, ymax + 0.05))
        ax.legend(handles=handles, loc="upper left", frameon=False)
        style_ax(ax)
        #add_panel_label(ax, "")
    else:
        ax.axis("off")

    # ---------------- right ----------------
    ax = axes[1]
    if not l2.empty and "transition" in l2.columns:
        dd = l2.set_index("transition").reindex(trans)
        y = np.arange(len(trans))[::-1]
        cue = dd["cue_increment"].astype(float).values if "cue_increment" in dd.columns else np.full(len(dd), np.nan)
        topo = dd["topology_increment"].astype(float).values if "topology_increment" in dd.columns else np.full(len(dd), np.nan)
        topo_is_near_zero = np.isfinite(topo).any() and np.nanmax(np.abs(topo)) < 1e-4

        for yi, val in zip(y, cue):
            if np.isfinite(val):
                ax.hlines(yi + 0.12, 0, val, color=PALETTE["orange"], lw=2.6)
                ax.scatter(val, yi + 0.12, s=170, color=PALETTE["orange"], zorder=5)
                ax.text(val + 0.0025, yi + 0.12, f"{val:.3f}", va="bottom", ha="left", fontsize=11)

        for yi, val in zip(y, topo):
            if np.isfinite(val):
                xend = val if abs(val) > 1e-4 else 0.0003
                ax.hlines(yi - 0.12, 0, xend, color=PALETTE["blue"], lw=2.6)
                ax.scatter(val, yi - 0.12, s=170, facecolor=PALETTE["white"],
                           edgecolor=PALETTE["blue"], linewidth=2.6, zorder=6)

        handles = [
            Line2D([0], [0], color=PALETTE["orange"], lw=2.6, marker="o", ms=8, label="Cue increment"),
            Line2D([0], [0], color=PALETTE["blue"], lw=2.6, marker="o", ms=8,
                   markerfacecolor=PALETTE["white"], label="Topology increment (≈0)" if topo_is_near_zero else "Topology increment"),
        ]
        ax.legend(handles=handles, loc="upper right", frameon=False)
        ax.axvline(0, color=PALETTE["lightgray"], lw=1.4)
        ax.set_yticks(y)
        ax.set_yticklabels(trans)
        ax.set_xlabel("Increment over previous block (AUROC)")
        ax.set_xlim(-0.005, max(0.03, np.nanmax(cue) + 0.025 if np.isfinite(cue).any() else 0.03))
        style_ax(ax)
        #add_panel_label(ax, "B")
    else:
        ax.axis("off")

    save_fig(fig, "FigL_transition_theory_specificity_layoutfix")



def draw_fig_U():
    u1 = read_csv_any(["FigU1_minimal_edit_counterfactual_ladder.csv"])
    u2 = read_csv_any(["FigU2_minimal_edit_human_template.csv"])

    if u1.empty:
        save_text("Fig U layout-fix skipped because FigU1_minimal_edit_counterfactual_ladder.csv was not found.", "FigU_layoutfix_notes.txt")
        return

    fam_order = ["worthlessness_family", "distortion_family", "suicidality_family"]
    fam_label = {
        "worthlessness_family": "Hopelessness–worthlessness family",
        "distortion_family": "Absolutist / negation family",
        "suicidality_family": "Suicide-explicit family",
    }
    fam_color = {
        "worthlessness_family": PALETTE["orange"],
        "distortion_family": PALETTE["purple"],
        "suicidality_family": PALETTE["red"],
    }

    # aggregate means per family x edit level
    agg = (
        u1.groupby(["family", "edit_level"], as_index=False)
          [["delta_severity", "delta_hidden_risk"]]
          .mean()
    )

    all_zero_sev = np.nanmax(np.abs(pd.to_numeric(agg["delta_severity"], errors="coerce"))) < 1e-8
    all_zero_risk = np.nanmax(np.abs(pd.to_numeric(agg["delta_hidden_risk"], errors="coerce"))) < 1e-8

    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.6), gridspec_kw={"wspace": 0.28})

    for ax, ycol, ylabel, panel in [
        (axes[0], "delta_severity", "Model severity shift", "A"),
        (axes[1], "delta_hidden_risk", "Hidden-risk routing shift", "B"),
    ]:
        ax.axhline(0, color=PALETTE["lightgray"], lw=1.4)
        handles = []
        plotted = False
        for fam in fam_order:
            dd = agg[agg["family"] == fam].sort_values("edit_level")
            if dd.empty:
                continue
            x = dd["edit_level"].astype(float).values
            y = dd[ycol].astype(float).values
            plotted = True

            # if the entire panel is zero-like, separate the families with tiny vertical offsets
            if (ycol == "delta_severity" and all_zero_sev) or (ycol == "delta_hidden_risk" and all_zero_risk):
                offset = {
                    "worthlessness_family": 0.0008,
                    "distortion_family": 0.0,
                    "suicidality_family": -0.0008,
                }[fam]
                yplot = y + offset
                ax.plot(x, yplot, color=fam_color[fam], lw=2.2, marker="o", ms=7)
                handles.append(Line2D([0], [0], color=fam_color[fam], lw=2.2, marker="o", ms=7, label=fam_label[fam]))
            else:
                ax.plot(x, y, color=fam_color[fam], lw=2.6, marker="o", ms=8)
                handles.append(Line2D([0], [0], color=fam_color[fam], lw=2.6, marker="o", ms=8, label=fam_label[fam]))

        ax.set_xticks([0, 1, 2])
        ax.set_xlabel("Edit level")
        ax.set_ylabel(ylabel)
        style_ax(ax)
        #add_panel_label(ax, panel)

        if plotted:
            ax.legend(handles=handles, loc="upper left", frameon=False)

    # compact note instead of giant overlap text
    # if all_zero_sev and all_zero_risk:
    #     fig.text(0.5, 0.985,
    #              "No measurable surrogate edit effect under the current ladder design",
    #              ha="center", va="top", fontsize=11.5, fontweight="bold")
    #     for ax in axes:
    #         ax.set_ylim(-0.05, 0.05)
    #         ax.set_yticks([-0.04, -0.02, 0.00, 0.02, 0.04])

    # # optional footer note from human template
    # if not u2.empty and "human_rating" in u2.columns and pd.to_numeric(u2["human_rating"], errors="coerce").notna().sum() == 0:
    #     fig.text(0.5, -0.02,
    #              "Human re-rating fields are currently empty in the template, so Fig U shows model-side surrogate edits only.",
    #              ha="center", va="top", fontsize=10.5)

    save_fig(fig, "FigU_minimal_edit_counterfactual_ladder_layoutfix")


# =========================================================
# 7. MAIN
# =========================================================
if __name__ == "__main__":
    draw_fig_I()
    draw_fig_L()
    draw_fig_U()
    print("[DONE] Fig I / Fig L / Fig U layout-fix complete.")


[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls_layoutfix.png
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls_layoutfix.eps


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity_layoutfix.png
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity_layoutfix.eps
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigU_minimal_edit_counterfactual_ladder_layoutfix.png
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigU_minimal_edit_counterfactual_ladder_layoutfix.eps
[DONE] Fig I / Fig L / Fig U layout-fix complete.


In [14]:
from pathlib import Path
from PIL import Image
import numpy as np

sources = {
    "FigI_specificity_beyond_shallow_controls_whitebg": Path(
        r"D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls_layoutfix.png"
    ),
    "FigL_transition_theory_specificity_whitebg": Path(
        r"D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity_layoutfix.png"
    ),
    "FigT_targeted_human_boundary_alignment_whitebg": Path(
        r"D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigT_targeted_human_boundary_alignment.png"
    ),
    "FigU_minimal_edit_counterfactual_ladder_whitebg": Path(
        r"D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigU_minimal_edit_counterfactual_ladder_layoutfix.png"
    ),
}

out_dir = Path(r"D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures")
out_dir.mkdir(parents=True, exist_ok=True)


def whiten_background(img: Image.Image) -> Image.Image:
    rgba = img.convert("RGBA")
    arr = np.array(rgba).copy()

    rgb = arr[:, :, :3]
    alpha = arr[:, :, 3]

    maxc = rgb.max(axis=2)
    minc = rgb.min(axis=2)

    # 只把近白、近灰、且不透明的背景像素改成纯白
    neutral = (maxc - minc) <= 6
    bright = minc >= 245
    opaque = alpha >= 250

    mask = neutral & bright & opaque
    arr[mask, 0] = 255
    arr[mask, 1] = 255
    arr[mask, 2] = 255
    arr[mask, 3] = 255

    return Image.fromarray(arr, mode="RGBA")


for stem, src in sources.items():
    if not src.exists():
        print(f"[Skip] File not found: {src}")
        continue

    img = Image.open(src)
    fixed = whiten_background(img).convert("RGB")

    png_path = out_dir / f"{stem}.png"
    eps_path = out_dir / f"{stem}.eps"

    fixed.save(png_path, format="PNG")
    fixed.save(eps_path, format="EPS")

    print(f"[Saved PNG] {png_path}")
    print(f"[Saved EPS] {eps_path}")

print("done")

[Saved PNG] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls_whitebg.png
[Saved EPS] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigI_specificity_beyond_shallow_controls_whitebg.eps
[Saved PNG] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity_whitebg.png
[Saved EPS] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigL_transition_theory_specificity_whitebg.eps
[Skip] File not found: D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigT_targeted_human_boundary_alignment.png
[Saved PNG] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigU_minimal_edit_counterfactual_ladder_whitebg.png
[Saved EPS] D:\Downloads\Depressed\Depressed\final_pub_outputs_fullcover\figures\FigU_minimal_edit_counterfactual_ladder_whitebg.eps
done


In [15]:
# -*- coding: utf-8 -*-
import os
import re
import glob
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.lines import Line2D
from sklearn.metrics import confusion_matrix

ROOT_DEPRESSED = r"D:\Downloads\Depressed"

EXPORT_ROOT = os.path.join(ROOT_DEPRESSED, "export_bundle_v4")
PRED_DIR = os.path.join(EXPORT_ROOT, "predictions")
SUMMARY_CSV = os.path.join(EXPORT_ROOT, "result_summary.csv")
META_CSV = os.path.join(EXPORT_ROOT, "sample_meta.csv")

OUT_ROOT = os.path.join(EXPORT_ROOT, "NHB_final_main_figures_final_v2")
OUT_FIGS = os.path.join(OUT_ROOT, "figures")
OUT_TABLES = os.path.join(OUT_ROOT, "tables")

os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)
os.makedirs(OUT_TABLES, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times New Roman PS MT", "DejaVu Serif"],
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.8,
    "xtick.major.width": 1.6,
    "ytick.major.width": 1.6,
    "xtick.major.size": 8,
    "ytick.major.size": 8,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "savefig.bbox": "tight",
    "axes.unicode_minus": False,
})

C = {
    "native": "#8FA6BF",
    "ft": "#F39F4E",
    "gain": "#9EDB9A",
    "blue_soft": "#98CFE6",
    "pink": "#EEB7D3",
    "yellow": "#F1D58C",
    "dark": "#222222",
    "lightgray": "#C8C5BE",
    "offwhite": "#F7F6F3",
    "softred": "#EAB4A1",
}

DIV_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "soft_div", [C["blue_soft"], C["offwhite"], C["softred"]]
)

QWEN_PATTERN = r"Qwen3\.5-(?:0\.8|2|4|9)B-Base"

EMOTION_ORDER = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

SHORT = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

def style_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for side in ["bottom", "left"]:
        ax.spines[side].set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=12, width=1.6, length=8, pad=8)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")

def pair_legend_handles():
    return [
        Line2D([0], [0], color=C["native"], marker="o", markersize=8, linewidth=2.5, label="Native"),
        Line2D([0], [0], color=C["ft"], marker="o", markersize=8, linewidth=2.5, label="Fine-tuned"),
    ]

def save_fig(fig, stem):
    png_path = os.path.join(OUT_FIGS, f"{stem}.png")
    eps_path = os.path.join(OUT_FIGS, f"{stem}.eps")
    fig.savefig(png_path, dpi=800, facecolor="white")
    fig.savefig(eps_path, dpi=800, facecolor="white", format="eps")
    print(f"[Saved] {png_path}")
    print(f"[Saved] {eps_path}")

def save_df(df, stem):
    out = os.path.join(OUT_TABLES, f"{stem}.csv")
    df.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"[Saved] {out}")

def model_size_to_float(model_name):
    m = re.search(r"Qwen3\.5-(0\.8|2|4|9)B-Base", str(model_name))
    if m:
        return float(m.group(1))
    return np.nan

def int_to_vec(x, n=8):
    try:
        x = int(x)
    except Exception:
        return None
    if x < 0 or x >= (2 ** n):
        return None
    return [int(b) for b in format(x, f"0{n}b")]

def vecstr_to_list(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.startswith("b"):
        s = s[1:]
    if len(s) != 8:
        return None
    if set(s) - {"0", "1"}:
        return None
    return [int(x) for x in s]

def hamming_dist(v1, v2):
    if v1 is None or v2 is None:
        return np.nan
    return int(np.sum(np.abs(np.array(v1) - np.array(v2))))

def attach_meta(df, meta_df):
    if len(df) == 0 or len(meta_df) == 0:
        return df.copy()
    if "sample_id" not in df.columns or "sample_id" not in meta_df.columns:
        return df.copy()
    out = df.copy()
    meta_use = meta_df.copy()
    out["sample_id"] = out["sample_id"].astype(str)
    meta_use["sample_id"] = meta_use["sample_id"].astype(str)
    meta_use = meta_use.drop_duplicates(subset=["sample_id"])
    cols_to_add = [c for c in meta_use.columns if c not in out.columns]
    meta_use = meta_use[["sample_id"] + cols_to_add]
    return out.merge(meta_use, on="sample_id", how="left")

def safe_text_y(y, ymin, ymax, margin=0.012):
    return min(max(y, ymin + margin), ymax - margin)

def smart_ylim(vals, low_pad=0.04, high_pad=0.06, lower_floor=None, upper_cap=None):
    vals = np.array(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return (0.0, 1.0)
    ymin = vals.min() - low_pad
    ymax = vals.max() + high_pad
    if lower_floor is not None:
        ymin = max(ymin, lower_floor)
    if upper_cap is not None:
        ymax = min(ymax, upper_cap)
    if ymax <= ymin:
        ymax = ymin + 0.1
    return ymin, ymax

summary = pd.read_csv(SUMMARY_CSV, encoding="utf-8-sig") if os.path.isfile(SUMMARY_CSV) else pd.DataFrame()
meta = pd.read_csv(META_CSV, encoding="utf-8-sig") if os.path.isfile(META_CSV) else pd.DataFrame()
pred_files = glob.glob(os.path.join(PRED_DIR, "*.csv"))

print("=" * 100)
print("ROOT_DEPRESSED =", ROOT_DEPRESSED)
print("EXPORT_ROOT    =", EXPORT_ROOT)
print("PRED_DIR       =", PRED_DIR)
print("SUMMARY_CSV    =", SUMMARY_CSV, "| exists =", os.path.isfile(SUMMARY_CSV))
print("META_CSV       =", META_CSV, "| exists =", os.path.isfile(META_CSV))
print("Prediction csv =", len(pred_files))
print("OUT_ROOT       =", OUT_ROOT)
print("=" * 100)

def load_run_by_row(task_prefix, row):
    if row is None or len(pred_files) == 0:
        return pd.DataFrame()
    model_name = str(row["model_name"])
    setting = str(row["setting"])
    run_name = str(row["run_name"]) if ("run_name" in row.index and pd.notna(row["run_name"])) else None
    exact, fallback = [], []
    for fp in pred_files:
        bn = os.path.basename(fp)
        if not bn.startswith(task_prefix + "__"):
            continue
        if f"__{model_name}__" not in bn:
            continue
        if run_name is not None and run_name in bn:
            exact.append(fp)
        elif f"__{setting}__" in bn:
            fallback.append(fp)
    cands = exact if len(exact) else fallback
    if len(cands) == 0:
        return pd.DataFrame()
    cands = sorted(cands, key=lambda x: len(os.path.basename(x)))
    df = pd.read_csv(cands[0], encoding="utf-8-sig")
    df = attach_meta(df, meta)
    print(f"[Loaded] {cands[0]}")
    return df

def pick_matched_qwen_native_ft_pair(task_name, metric_name):
    if len(summary) == 0:
        return pd.DataFrame(), pd.DataFrame(), None, None
    sub = summary[
        (summary["task"] == task_name) &
        (summary["model_name"].astype(str).str.contains(QWEN_PATTERN, regex=True, na=False))
    ].copy()
    if metric_name not in sub.columns:
        return pd.DataFrame(), pd.DataFrame(), None, None
    ft = sub[sub["setting"] == "ft_eval"].copy()
    ft[metric_name] = pd.to_numeric(ft[metric_name], errors="coerce")
    ft = ft[ft[metric_name].notna()].sort_values(metric_name, ascending=False)
    if len(ft) == 0:
        return pd.DataFrame(), pd.DataFrame(), None, None
    ft_row = ft.iloc[0]
    native = sub[
        (sub["setting"] == "native_base_eval") &
        (sub["model_name"].astype(str) == str(ft_row["model_name"]))
    ].copy()
    if len(native) == 0:
        size_ft = model_size_to_float(ft_row["model_name"])
        native = sub[
            (sub["setting"] == "native_base_eval") &
            (sub["model_name"].apply(model_size_to_float) == size_ft)
        ].copy()
    native_row = None if len(native) == 0 else native.iloc[0]
    native_df = load_run_by_row(task_name, native_row) if native_row is not None else pd.DataFrame()
    ft_df = load_run_by_row(task_name, ft_row)
    return native_df, ft_df, native_row, ft_row

def clean_fourclass_df(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if "true_label" not in out.columns or "pred_label" not in out.columns:
        return pd.DataFrame()
    out["true_label_num"] = pd.to_numeric(out["true_label"], errors="coerce")
    out["pred_label_num"] = pd.to_numeric(out["pred_label"], errors="coerce")
    out = out[out["true_label_num"].notna() & out["pred_label_num"].notna()].copy()
    out = out[out["true_label_num"].between(0, 3) & out["pred_label_num"].between(0, 3)].copy()
    out["true_label_num"] = out["true_label_num"].astype(int)
    out["pred_label_num"] = out["pred_label_num"].astype(int)
    if "sample_id" in out.columns:
        out["sample_id"] = out["sample_id"].astype(str)
    return out.reset_index(drop=True)

def clean_class255_df(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if "true_255" not in out.columns or "pred_255" not in out.columns:
        return pd.DataFrame()
    out["true_255_num"] = pd.to_numeric(out["true_255"], errors="coerce")
    out["pred_255_num"] = pd.to_numeric(out["pred_255"], errors="coerce")
    out = out[out["true_255_num"].notna() & out["pred_255_num"].notna()].copy()
    out = out[out["true_255_num"].between(1, 255) & out["pred_255_num"].between(1, 255)].copy()
    out["true_255_num"] = out["true_255_num"].astype(int)
    out["pred_255_num"] = out["pred_255_num"].astype(int)
    out["true_vec_list"] = out["true_vec"].apply(vecstr_to_list) if "true_vec" in out.columns else out["true_255_num"].apply(int_to_vec)
    out["pred_vec_list"] = out["pred_vec"].apply(vecstr_to_list) if "pred_vec" in out.columns else out["pred_255_num"].apply(int_to_vec)
    out = out[out["true_vec_list"].notna() & out["pred_vec_list"].notna()].copy()
    if "sample_id" in out.columns:
        out["sample_id"] = out["sample_id"].astype(str)
    return out.reset_index(drop=True)

def clean_eightlabel_df(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if "true_vec" not in out.columns or "pred_vec" not in out.columns:
        return pd.DataFrame()
    out["true_vec_list"] = out["true_vec"].apply(vecstr_to_list)
    out["pred_vec_list"] = out["pred_vec"].apply(vecstr_to_list)
    out = out[out["true_vec_list"].notna() & out["pred_vec_list"].notna()].copy()
    if "sample_id" in out.columns:
        out["sample_id"] = out["sample_id"].astype(str)
    return out.reset_index(drop=True)

four_native_raw, four_ft_raw, four_native_row, four_ft_row = pick_matched_qwen_native_ft_pair("fourclass", "macro_f1")
state_native_raw, state_ft_raw, state_native_row, state_ft_row = pick_matched_qwen_native_ft_pair("class255", "macro_f1")
eight_native_raw, eight_ft_raw, eight_native_row, eight_ft_row = pick_matched_qwen_native_ft_pair("eightlabel", "macro_f1")

four_native = clean_fourclass_df(four_native_raw)
four_ft = clean_fourclass_df(four_ft_raw)
state_native = clean_class255_df(state_native_raw)
state_ft = clean_class255_df(state_ft_raw)
eight_native = clean_eightlabel_df(eight_native_raw)
eight_ft = clean_eightlabel_df(eight_ft_raw)

if len(eight_native) and len(eight_ft):
    sym_native = eight_native.copy()
    sym_ft = eight_ft.copy()
    sym_source = "eightlabel"
else:
    sym_native = state_native.copy()
    sym_ft = state_ft.copy()
    sym_source = "class255_fallback"

print("=" * 100)
print("[Pair] fourclass native:", None if four_native_row is None else dict(four_native_row[["model_name", "setting", "run_name"]]))
print("[Pair] fourclass ft    :", None if four_ft_row is None else dict(four_ft_row[["model_name", "setting", "run_name"]]))
print("[Pair] class255 native :", None if state_native_row is None else dict(state_native_row[["model_name", "setting", "run_name"]]))
print("[Pair] class255 ft     :", None if state_ft_row is None else dict(state_ft_row[["model_name", "setting", "run_name"]]))
print("[Pair] eightlabel native:", None if eight_native_row is None else dict(eight_native_row[["model_name", "setting", "run_name"]]))
print("[Pair] eightlabel ft    :", None if eight_ft_row is None else dict(eight_ft_row[["model_name", "setting", "run_name"]]))
print("[Symptom source] =", sym_source)
print("=" * 100)

def boundary_stats(df):
    gap = np.abs(df["true_label_num"].values - df["pred_label_num"].values)
    return {"n": len(df), "mae": float(np.mean(gap)), "correct_rate": float(np.mean(gap == 0)),
            "adjacent_rate": float(np.mean(gap == 1)), "far_rate": float(np.mean(gap >= 2)),
            "acceptable_local_rate": float(np.mean(gap <= 1)), "long_jump_rate": float(np.mean(gap >= 2))}

def local_boundary_acceptance(df, a, b):
    sub = df[df["true_label_num"].isin([a, b])].copy()
    if len(sub) == 0:
        return np.nan
    gap = np.abs(sub["true_label_num"].values - sub["pred_label_num"].values)
    return float(np.mean(gap <= 1))

def cooccurrence_matrix(vecs):
    M = np.zeros((8, 8), dtype=float)
    n = len(vecs)
    if n == 0:
        return M
    for v in vecs:
        vv = np.array(v, dtype=int)
        M += np.outer(vv, vv)
    return M / n

def centrality_strength(mat):
    X = mat.copy()
    np.fill_diagonal(X, 0.0)
    s = X.sum(axis=1)
    mx = s.max()
    if mx <= 0:
        return np.zeros_like(s)
    return s / mx

def symptom_prevalence(vecs):
    A = np.array(vecs, dtype=int)
    return A.mean(axis=0)

def state_exact_acc(df, state_id):
    sub = df[df["true_255_num"] == state_id]
    if len(sub) == 0:
        return np.nan
    return float(np.mean(sub["pred_255_num"].values == state_id))

def state_neighborhood_stats(df, states):
    rows = []
    for s in states:
        sub = df[df["true_255_num"] == s].copy()
        if len(sub) == 0:
            continue
        d = np.array([hamming_dist(r["true_vec_list"], r["pred_vec_list"]) for _, r in sub.iterrows()], dtype=float)
        rows.append({"state255": s, "support": len(sub), "within1": float(np.mean(d <= 1)), "mean_hd": float(np.mean(d))})
    return pd.DataFrame(rows)

def best_row_per_size(task_name, metric_name, setting_name):
    sub = summary[(summary["task"] == task_name) &
                  (summary["setting"] == setting_name) &
                  (summary["model_name"].astype(str).str.contains(QWEN_PATTERN, regex=True, na=False))].copy()
    if len(sub) == 0 or metric_name not in sub.columns:
        return pd.DataFrame()
    sub["size_B"] = sub["model_name"].apply(model_size_to_float)
    sub[metric_name] = pd.to_numeric(sub[metric_name], errors="coerce")
    sub = sub[sub[metric_name].notna()].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    rows = []
    for size in sorted(sub["size_B"].dropna().unique()):
        ss = sub[sub["size_B"] == size].sort_values(metric_name, ascending=False)
        rows.append(ss.iloc[0])
    return pd.DataFrame(rows)

def make_fig1_boundary_refinement(native_df, ft_df):
    if len(native_df) == 0 or len(ft_df) == 0:
        print("[Skip] Fig1_boundary_refinement: missing data")
        return
    cm_n = confusion_matrix(native_df["true_label_num"], native_df["pred_label_num"], labels=[0, 1, 2, 3]).astype(float)
    cm_f = confusion_matrix(ft_df["true_label_num"], ft_df["pred_label_num"], labels=[0, 1, 2, 3]).astype(float)
    cm_n = cm_n / np.clip(cm_n.sum(axis=1, keepdims=True), 1e-12, None)
    cm_f = cm_f / np.clip(cm_f.sum(axis=1, keepdims=True), 1e-12, None)
    delta = cm_f - cm_n
    save_df(pd.DataFrame(delta, index=[0, 1, 2, 3], columns=[0, 1, 2, 3]).reset_index(),"Fig1A_delta_confusion_ft_minus_native")
    boundary_df = pd.DataFrame([{"boundary": f"{a}↔{b}", "native": local_boundary_acceptance(native_df, a, b),
                                 "fine_tuned": local_boundary_acceptance(ft_df, a, b),
                                 "gain": local_boundary_acceptance(ft_df, a, b) - local_boundary_acceptance(native_df, a, b)}
                                for a, b in [(0, 1), (1, 2), (2, 3)]])
    save_df(boundary_df, "Fig1B_boundary_local_closure")
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.9), gridspec_kw={"width_ratios": [1.05, 1.0]})
    ax = axes[0]
    vmax = max(abs(delta.min()), abs(delta.max()))
    ax.imshow(delta, cmap=DIV_CMAP, vmin=-vmax, vmax=vmax)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{delta[i, j]:+.2f}", ha="center", va="center", fontsize=8.2, fontweight="bold")
    ax.set_xticks(range(4), labels=["0", "1", "2", "3"])
    ax.set_yticks(range(4), labels=["0", "1", "2", "3"])
    ax.set_xlabel("Predicted severity")
    ax.set_ylabel("True severity")
    style_ax(ax)
    ax = axes[1]
    x = np.arange(len(boundary_df))
    ax.plot(x, boundary_df["native"], marker="o", linewidth=2.6, markersize=7.2, color=C["native"])
    ax.plot(x, boundary_df["fine_tuned"], marker="o", linewidth=2.6, markersize=7.2, color=C["ft"])
    ymin, ymax = smart_ylim(np.r_[boundary_df["native"].values, boundary_df["fine_tuned"].values], low_pad=0.035, high_pad=0.05, upper_cap=1.00)
    ax.set_ylim(ymin, ymax)
    for i, r in boundary_df.iterrows():
        ty = safe_text_y(r["fine_tuned"] + 0.012, ymin, ymax, margin=0.010)
        ax.text(i, ty, f"{r['gain']:+.2f}", ha="center", va="bottom", fontsize=8.1, fontweight="bold")
    ax.set_xticks(x, labels=boundary_df["boundary"].tolist())
    ax.set_ylabel("Locally acceptable fraction")
    style_ax(ax)
    ax.legend(handles=pair_legend_handles(), frameon=False, fontsize=8.5, loc="upper right")
    fig.subplots_adjust(left=0.08, right=0.98, bottom=0.18, top=0.96, wspace=0.26)
    save_fig(fig, "Fig1_boundary_refinement")
    plt.close(fig)

def Fig2_symptom_organization_refinement(eight_native, eight_ft):
    """
    Final publication-grade version:
    - move legend out of axes
    - unify legend at figure level
    - leave more top margin
    - keep panel spacing clean
    """
    if len(eight_native) == 0 or len(eight_ft) == 0:
        print("[Skip] Fig2_symptom_organization_refinement: missing data")
        return

    # ---------- compute ----------
    true_mat = cooccurrence_matrix(eight_ft["true_vec_list"].tolist())
    pred_n = cooccurrence_matrix(eight_native["pred_vec_list"].tolist())
    pred_f = cooccurrence_matrix(eight_ft["pred_vec_list"].tolist())

    ct = centrality_strength(true_mat)
    cn = centrality_strength(pred_n)
    cf = centrality_strength(pred_f)

    prev_true = np.mean(np.array(eight_ft["true_vec_list"].tolist()), axis=0)
    prev_n = np.mean(np.array(eight_native["pred_vec_list"].tolist()), axis=0)
    prev_f = np.mean(np.array(eight_ft["pred_vec_list"].tolist()), axis=0)

    delta_pairwise = np.abs(pred_f - true_mat) - np.abs(pred_n - true_mat)

    out_rows = []
    for i, lab in enumerate(EMOTION_ORDER):
        out_rows.append({
            "label": lab,
            "true_centrality": ct[i],
            "native_centrality": cn[i],
            "fine_tuned_centrality": cf[i],
            "true_prevalence": prev_true[i],
            "native_prevalence_residual": prev_n[i] - prev_true[i],
            "fine_tuned_prevalence_residual": prev_f[i] - prev_true[i],
        })
    save_df(pd.DataFrame(out_rows), "Fig2_symptom_organization_refinement_summary")
    save_df(
        pd.DataFrame(delta_pairwise, index=EMOTION_ORDER, columns=EMOTION_ORDER).reset_index(),
        "Fig2_delta_pairwise_structure_error"
    )

    # ---------- style ----------
    native_c = C.get("native", "#8EA3B8")
    ft_c = C.get("ft", "#D88B59")
    true_c = C.get("dark", "#222222")
    line_c = C.get("lightgray", "#C8C5BE")

    # ---------- plot ----------
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.6))
    ax1, ax2, ax3 = axes

    y = np.arange(len(EMOTION_ORDER))

    # (a) centrality alignment
    for i in range(len(EMOTION_ORDER)):
        lo = min(ct[i], cn[i], cf[i])
        hi = max(ct[i], cn[i], cf[i])
        ax1.plot([lo, hi], [i, i], color=line_c, linewidth=2.0, zorder=1)
        ax1.scatter(cn[i], i, s=180, color=native_c, zorder=3)
        ax1.scatter(cf[i], i, s=180, color=ft_c, zorder=4)
        ax1.plot([ct[i], ct[i]], [i - 0.18, i + 0.18], color=true_c, linewidth=2.2, zorder=5)

    ax1.set_yticks(y, labels=[SHORT[x] for x in EMOTION_ORDER])
    ax1.invert_yaxis()
    ax1.set_xlabel("Symptom centrality")
    ax1.set_xlim(0.20, 1.04)
    style_ax(ax1)

    # (b) prevalence residual
    for i in range(len(EMOTION_ORDER)):
        lo = min(prev_n[i] - prev_true[i], prev_f[i] - prev_true[i])
        hi = max(prev_n[i] - prev_true[i], prev_f[i] - prev_true[i])
        ax2.plot([lo, hi], [i, i], color=line_c, linewidth=2.0, zorder=1)
        ax2.scatter(prev_n[i] - prev_true[i], i, s=180, color=native_c, zorder=3)
        ax2.scatter(prev_f[i] - prev_true[i], i, s=180, color=ft_c, zorder=4)

    ax2.axvline(0, color=line_c, linewidth=1.6)
    ax2.set_yticks(y, labels=[SHORT[x] for x in EMOTION_ORDER])
    ax2.invert_yaxis()
    ax2.set_xlabel("Predicted prevalence − true prevalence")
    style_ax(ax2)

    # (c) pairwise structure delta heatmap
    vmax = float(np.max(np.abs(delta_pairwise)))
    if vmax < 1e-8:
        vmax = 0.10

    im = ax3.imshow(delta_pairwise, cmap="RdBu_r", vmin=-vmax, vmax=vmax)

    for i in range(8):
        for j in range(8):
            if i == j:
                continue
            ax3.text(
                j, i, f"{delta_pairwise[i, j]:+.2f}",
                ha="center", va="center",
                fontsize=6.6, fontweight="bold"
            )

    ax3.set_xticks(range(8), labels=[SHORT[x] for x in EMOTION_ORDER], rotation=32, ha="right")
    ax3.set_yticks(range(8), labels=[SHORT[x] for x in EMOTION_ORDER])
    ax3.set_xlabel("Δ pairwise structure error")
    style_ax(ax3)

    # ---------- figure-level legend ----------
    handles = [
        Line2D([0], [0], color=C["native"], marker="o", markersize=8, linewidth=2.5, label="Native"),
        Line2D([0], [0], color=C["ft"], marker="o", markersize=8, linewidth=2.5, label="Fine-tuned"),
        Line2D([0], [0], color=C["dark"], marker="|", markersize=12, linewidth=0, markeredgewidth=2.0, label="True"),
    ]

    fig.legend(
        handles=handles,
        loc="upper left",
        bbox_to_anchor=(0.86, 0.98),
        frameon=False,
        fontsize=9,
        borderaxespad=0.0,
        labelspacing=0.4,
        handletextpad=0.6
    )

    fig.subplots_adjust(left=0.06, right=0.84, bottom=0.24, top=0.94, wspace=0.35)
    save_fig(fig, "Fig2_symptom_organization_refinement")
    plt.close(fig)


def make_fig3_state_support_gain_and_residual(native_df, ft_df):
    if len(native_df) == 0 or len(ft_df) == 0:
        print("[Skip] Fig3_state_support_gain_and_residual: missing data")
        return
    true_counts = Counter(ft_df["true_255_num"].astype(int).tolist())
    pred_n = Counter(native_df["pred_255_num"].astype(int).tolist())
    pred_f = Counter(ft_df["pred_255_num"].astype(int).tolist())
    rows = []
    for s in sorted(true_counts.keys()):
        supp = true_counts[s]
        tf = supp / len(ft_df)
        nf = pred_n.get(s, 0) / len(native_df)
        ff = pred_f.get(s, 0) / len(ft_df)
        n_acc = state_exact_acc(native_df, s)
        f_acc = state_exact_acc(ft_df, s)
        rows.append({"state255": s, "support": supp, "native_acc": n_acc, "fine_tuned_acc": f_acc, "gain": f_acc - n_acc,
                     "true_freq": tf, "native_residual": nf - tf, "fine_tuned_residual": ff - tf})
    out = pd.DataFrame(rows)
    save_df(out, "Fig3_state_support_gain_and_residual_summary")
    top_resid = out.sort_values("support", ascending=False).head(12).copy()
    label_states = set(out.sort_values("support", ascending=False).head(5)["state255"].tolist())
    label_states |= set(out.sort_values("gain", ascending=False).head(3)["state255"].tolist())
    label_states |= set(out.sort_values("gain", ascending=True).head(2)["state255"].tolist())
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0), gridspec_kw={"width_ratios": [1.2, 1.15]})
    ax = axes[0]
    sizes = 40 + 23 * np.sqrt(out["support"].values)
    ax.scatter(out["support"], out["gain"], s=sizes, color=C["gain"], edgecolors="none", alpha=0.96)
    ax.axhline(0, color=C["lightgray"], linewidth=1.3)
    ymin, ymax = smart_ylim(out["gain"].values, low_pad=0.04, high_pad=0.06)
    ax.set_ylim(ymin, ymax)
    for _, r in out.iterrows():
        if r["state255"] in label_states:
            dx = 0.7 if r["support"] < out["support"].median() else 0.4
            ty = safe_text_y(r["gain"] + 0.015, ymin, ymax, margin=0.015)
            ax.text(r["support"] + dx, ty, str(int(r["state255"])), fontsize=7.8, fontweight="bold")
    ax.set_xlabel("True state support")
    ax.set_ylabel("Fine-tuned gain over native")
    style_ax(ax)
    ax = axes[1]
    y = np.arange(len(top_resid))
    for i, (_, r) in enumerate(top_resid.iterrows()):
        ax.hlines(i, xmin=min(r["native_residual"], r["fine_tuned_residual"]), xmax=max(r["native_residual"], r["fine_tuned_residual"]),
                  color=C["lightgray"], linewidth=2.0, zorder=1)
        ax.scatter(r["native_residual"], i, s=115, color=C["native"], zorder=4)
        ax.scatter(r["fine_tuned_residual"], i, s=115, color=C["ft"], zorder=5)
    ax.axvline(0, color=C["lightgray"], linewidth=1.4)
    ax.set_yticks(y, labels=[str(int(s)) for s in top_resid["state255"]])
    ax.invert_yaxis()
    ax.set_xlabel("Predicted prevalence − true prevalence")
    style_ax(ax)
    ax.legend(handles=pair_legend_handles(), frameon=False, fontsize=8.2, loc="lower right")
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.18, top=0.96, wspace=0.24)
    save_fig(fig, "Fig3_state_support_gain_and_residual")
    plt.close(fig)

def Fig4_state_neighborhood_refinement(state_native, state_ft, top_states=None):
    """
    Final publication-grade version:
    - move legend out of subplot
    - use one figure-level legend
    - keep both panels visually balanced
    """
    if len(state_native) == 0 or len(state_ft) == 0:
        print("[Skip] Fig4_state_neighborhood_refinement: missing data")
        return

    if top_states is None:
        cnt = Counter(state_ft["true_255_num"].astype(int).tolist())
        top_states = [s for s, _ in cnt.most_common(7)]

    def calc_stats(df, states):
        rows = []
        for s in states:
            sub = df[df["true_255_num"] == s].copy()
            if len(sub) == 0:
                continue

            # true vec from first matched row
            true_vec = None
            if "true_vec" in sub.columns:
                true_vec = vecstr_to_list(sub.iloc[0]["true_vec"])

            dists = []
            for _, r in sub.iterrows():
                pred_vec = None
                if "pred_vec" in sub.columns:
                    pred_vec = vecstr_to_list(r["pred_vec"])
                d = hamming_dist(true_vec, pred_vec)
                if not pd.isna(d):
                    dists.append(d)

            dists = np.array(dists, dtype=float) if len(dists) else np.array([np.nan])

            rows.append({
                "state255": s,
                "support": len(sub),
                "within1": float(np.nanmean(dists <= 1)),
                "mean_hd": float(np.nanmean(dists)),
            })
        return pd.DataFrame(rows)

    dn = calc_stats(state_native, top_states)
    df = calc_stats(state_ft, top_states)

    merged = pd.merge(
        dn, df, on=["state255", "support"], how="outer",
        suffixes=("_native", "_ft")
    ).sort_values("support", ascending=False).reset_index(drop=True)

    save_df(merged, "Fig4_state_neighborhood_refinement")

    native_c = C.get("native", "#8EA3B8")
    ft_c = C.get("ft", "#D88B59")
    line_c = C.get("lightgray", "#C8C5BE")

    fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.7))
    ax1, ax2 = axes
    y = np.arange(len(merged))

    # (a) within-1 recovery
    for i, r in merged.iterrows():
        x1 = r["within1_native"]
        x2 = r["within1_ft"]
        lo, hi = min(x1, x2), max(x1, x2)
        ax1.plot([lo, hi], [i, i], color=line_c, linewidth=2.0, zorder=1)
        ax1.scatter(x1, i, s=240, color=native_c, zorder=3)
        ax1.scatter(x2, i, s=240, color=ft_c, zorder=4)

    ax1.set_yticks(y, labels=[str(x) for x in merged["state255"]])
    ax1.invert_yaxis()
    ax1.set_xlabel("Within-1 neighborhood recovery")
    ax1.set_xlim(
        min(np.nanmin(merged["within1_native"]), np.nanmin(merged["within1_ft"])) - 0.05,
        max(np.nanmax(merged["within1_native"]), np.nanmax(merged["within1_ft"])) + 0.05
    )
    style_ax(ax1)

    # (b) mean Hamming distance
    for i, r in merged.iterrows():
        x1 = r["mean_hd_native"]
        x2 = r["mean_hd_ft"]
        lo, hi = min(x1, x2), max(x1, x2)
        ax2.plot([lo, hi], [i, i], color=line_c, linewidth=2.0, zorder=1)
        ax2.scatter(x1, i, s=240, color=native_c, zorder=3)
        ax2.scatter(x2, i, s=240, color=ft_c, zorder=4)

    ax2.set_yticks(y, labels=[str(x) for x in merged["state255"]])
    ax2.invert_yaxis()
    ax2.set_xlabel("Mean Hamming distance")
    style_ax(ax2)

    # ---------- figure-level legend ----------
    handles = [
        Line2D([0], [0], marker='o', color='none', markerfacecolor=native_c,
               markeredgecolor=native_c, markersize=10, label='Native'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor=ft_c,
               markeredgecolor=ft_c, markersize=10, label='Fine-tuned'),
    ]
    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        frameon=False,
        fontsize=11,
        handletextpad=0.8,
        columnspacing=1.6,
        borderaxespad=0.0
    )

    fig.subplots_adjust(top=0.84, wspace=0.28)
    save_fig(fig, "Fig4_state_neighborhood_refinement")
    plt.close(fig)
    
def make_fig5_capacity_release():
    rowsA = []
    natA = best_row_per_size("fourclass", "macro_f1", "native_base_eval")
    ftA = best_row_per_size("fourclass", "macro_f1", "ft_eval")
    for setting_name, ss in [("native", natA), ("fine_tuned", ftA)]:
        for _, row in ss.iterrows():
            run = clean_fourclass_df(load_run_by_row("fourclass", row))
            if len(run) == 0:
                continue
            st = boundary_stats(run)
            rowsA.append({"size_B": model_size_to_float(row["model_name"]), "setting": setting_name, "mae": st["mae"]})
    dfA = pd.DataFrame(rowsA)
    save_df(dfA, "Fig5A_fourclass_capacity_release")
    rowsB = []
    natB = best_row_per_size("class255", "macro_f1", "native_base_eval")
    ftB = best_row_per_size("class255", "macro_f1", "ft_eval")
    for setting_name, ss in [("native", natB), ("fine_tuned", ftB)]:
        for _, row in ss.iterrows():
            rowsB.append({"size_B": model_size_to_float(row["model_name"]), "setting": setting_name, "macro_f1": float(row["macro_f1"])})
    dfB = pd.DataFrame(rowsB)
    save_df(dfB, "Fig5B_class255_capacity_release")
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.8))
    ax = axes[0]
    native_sizes = set(dfA.loc[dfA["setting"] == "native", "size_B"].tolist())
    ft_sizes = set(dfA.loc[dfA["setting"] == "fine_tuned", "size_B"].tolist())
    matched_sizes = sorted(native_sizes.intersection(ft_sizes))
    dfA_plot = dfA[dfA["size_B"].isin(matched_sizes)].copy()
    for setting_name, color in [("native", C["native"]), ("fine_tuned", C["ft"])]:
        ss = dfA_plot[dfA_plot["setting"] == setting_name].sort_values("size_B")
        if len(ss):
            ax.plot(ss["size_B"], ss["mae"], marker="o", linewidth=2.6, markersize=7.5, color=color)
    ax.set_xlabel("Model size (B)")
    ax.set_ylabel("Mean absolute severity error")
    if matched_sizes:
        ax.set_xticks(matched_sizes, labels=[f"{x:g}" for x in matched_sizes])
    style_ax(ax)
    ax.legend(handles=pair_legend_handles(), frameon=False, fontsize=8.5, loc="upper right")
    ax = axes[1]
    for setting_name, color in [("native", C["native"]), ("fine_tuned", C["ft"])]:
        ss = dfB[dfB["setting"] == setting_name].sort_values("size_B")
        if len(ss):
            ax.plot(ss["size_B"], ss["macro_f1"], marker="o", linewidth=2.6, markersize=7.5, color=color)
    nat_dict = {r["size_B"]: r["macro_f1"] for _, r in dfB[dfB["setting"] == "native"].iterrows()}
    ft_dict = {r["size_B"]: r["macro_f1"] for _, r in dfB[dfB["setting"] == "fine_tuned"].iterrows()}
    ymin, ymax = smart_ylim(dfB["macro_f1"].values, low_pad=0.01, high_pad=0.012)
    for size in sorted(set(nat_dict.keys()).intersection(set(ft_dict.keys()))):
        gain = ft_dict[size] - nat_dict[size]
        ty = safe_text_y(ft_dict[size] + 0.004, ymin, ymax, margin=0.005)
        ax.text(size, ty, f"{gain:+.2f}", ha="center", va="bottom", fontsize=8.2, fontweight="bold")
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Model size (B)")
    ax.set_ylabel("Macro-F1")
    ax.set_xticks([0.8, 2.0, 4.0, 9.0], labels=["0.8", "2", "4", "9"])
    style_ax(ax)
    fig.subplots_adjust(left=0.08, right=0.98, bottom=0.18, top=0.96, wspace=0.28)
    save_fig(fig, "Fig5_capacity_release")
    plt.close(fig)

print("=" * 100)
print("Generating final Nature Human Behaviour style main-text figures ...")
print("=" * 100)
for fn, args in [
    (make_fig1_boundary_refinement, (four_native, four_ft)),
    (make_fig2_symptom_organization, (sym_native, sym_ft, sym_source)),
    (make_fig3_state_support_gain_and_residual, (state_native, state_ft)),
    (make_fig4_state_neighborhood_refinement, (state_native, state_ft)),
    (make_fig5_capacity_release, tuple()),
]:
    try:
        fn(*args)
    except Exception as e:
        print(f"[{fn.__name__} failed]", repr(e))

print("=" * 100)
print("Done.")
print("Output root:", OUT_ROOT)
print("Figure dir :", OUT_FIGS)
print("Table dir  :", OUT_TABLES)
print("=" * 100)


ROOT_DEPRESSED = D:\Downloads\Depressed
EXPORT_ROOT    = D:\Downloads\Depressed\export_bundle_v4
PRED_DIR       = D:\Downloads\Depressed\export_bundle_v4\predictions
SUMMARY_CSV    = D:\Downloads\Depressed\export_bundle_v4\result_summary.csv | exists = True
META_CSV       = D:\Downloads\Depressed\export_bundle_v4\sample_meta.csv | exists = True
Prediction csv = 126
OUT_ROOT       = D:\Downloads\Depressed\export_bundle_v4\NHB_final_main_figures_final_v2
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\fourclass__Qwen3.5-4B-Base__native_base_eval__Native_eval_26-03-18-02-45-11.csv
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\fourclass__Qwen3.5-4B-Base__ft_eval__eval_2026-03-16-02-46-08_1.csv
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\class255__Qwen3.5-9B-Base__native_base_eval__Native_eval_26-03-18-11-36-32.csv
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\class255__Qwen3.5-9B-Base__ft_eval__eval_2026-03-12-09-33-16_2.csv
[Pair

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


[Saved] D:\Downloads\Depressed\export_bundle_v4\NHB_final_main_figures_final_v2\figures\Fig3_state_support_gain_and_residual.png
[Saved] D:\Downloads\Depressed\export_bundle_v4\NHB_final_main_figures_final_v2\figures\Fig3_state_support_gain_and_residual.eps
[Saved] D:\Downloads\Depressed\export_bundle_v4\NHB_final_main_figures_final_v2\tables\Fig4_state_neighborhood_refinement.csv
[Saved] D:\Downloads\Depressed\export_bundle_v4\NHB_final_main_figures_final_v2\figures\Fig4_state_neighborhood_refinement.png
[Saved] D:\Downloads\Depressed\export_bundle_v4\NHB_final_main_figures_final_v2\figures\Fig4_state_neighborhood_refinement.eps
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\fourclass__Qwen3.5-0.8B-Base__native_base_eval__Native_eval_26-03-18-08-43-30.csv
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\fourclass__Qwen3.5-2B-Base__native_base_eval__Native_eval_26-03-18-08-43-30.csv
[Loaded] D:\Downloads\Depressed\export_bundle_v4\predictions\fourclass__Qwen3.

In [ ]:
# -*- coding: utf-8 -*-
"""
Standalone generator for FigD
"""

import os
import re
import glob
import argparse
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")


DEFAULT_ROOT = r"D:\Downloads\Depressed"
DPI = 800

EMOTION_NAMES = [
    "anger",
    "brain dysfunction (forget)",
    "emptiness",
    "hopelessness",
    "loneliness",
    "sadness",
    "suicide intent",
    "worthlessness"
]

EMOTION_SHORT = {
    "anger": "anger",
    "brain dysfunction (forget)": "brain dysf.",
    "emptiness": "emptiness",
    "hopelessness": "hopeless.",
    "loneliness": "lonely.",
    "sadness": "sadness",
    "suicide intent": "suicide",
    "worthlessness": "worthless."
}

COMBO_ABBR = {
    "anger": "anger",
    "brain dysf.": "brain",
    "emptiness": "empty",
    "hopeless.": "hopeless",
    "lonely.": "lonely",
    "sadness": "sadness",
    "suicide": "suicide",
    "worthless.": "worthless",
}

PALETTE = {
    "orange": "#F39F4E",
    "pink": "#EEB7D3",
    "lightgray": "#CFCBC3",
    "axis": "#000000",
    "text": "#000000",
    "white": "#FFFFFF",
}

CMAP_ORANGE = LinearSegmentedColormap.from_list(
    "nature_orange",
    ["#FAF8F4", "#F1E2CF", "#F0C48E", "#F39F4E"]
)

def set_global_style():
    mpl.rcParams.update({
        "font.family": "Times New Roman",
        "font.serif": ["Times New Roman"],
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.linewidth": 3.0,
        "axes.unicode_minus": False,
        "xtick.major.width": 3.0,
        "ytick.major.width": 3.0,
        "xtick.major.size": 10,
        "ytick.major.size": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.facecolor": "white",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "legend.frameon": False,
    })


def make_all_artists_opaque(fig):
    """
    防止 EPS 导出时透明度问题。
    强制所有 artist 的 alpha=1，背景纯白不透明。
    """
    fig.patch.set_facecolor("white")
    fig.patch.set_alpha(1.0)

    for ax in fig.axes:
        ax.set_facecolor("white")
        ax.patch.set_alpha(1.0)
        for artist in ax.get_children():
            try:
                artist.set_alpha(1.0)
            except Exception:
                pass


def force_bold_times(ax):
    """
    强制当前坐标轴上所有文字为 Times New Roman + bold。
    """
    for txt in [ax.xaxis.label, ax.yaxis.label, ax.title]:
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")

    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontname("Times New Roman")
        tick.set_fontweight("bold")

    for txt in ax.texts:
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")


def style_ax(ax, x_tick_size=24, y_tick_size=22, label_size=34):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(3.2)
    ax.spines["bottom"].set_linewidth(3.2)
    ax.spines["left"].set_color(PALETTE["axis"])
    ax.spines["bottom"].set_color(PALETTE["axis"])

    ax.tick_params(
        axis="x",
        width=3.2,
        length=10,
        labelsize=x_tick_size,
        colors=PALETTE["text"],
        pad=8
    )
    ax.tick_params(
        axis="y",
        width=3.2,
        length=10,
        labelsize=y_tick_size,
        colors=PALETTE["text"],
        pad=8
    )

    ax.xaxis.label.set_size(label_size)
    ax.yaxis.label.set_size(label_size)
    ax.grid(False)

    force_bold_times(ax)


def rotate_ylabels(
    ax,
    rotation=10,
    fontsize=25,
    ha="right",
    va="center"
):
    """
    轻微旋转 y 轴标签，适合长组合标签。
    """
    for lab in ax.get_yticklabels():
        lab.set_rotation(rotation)
        lab.set_ha(ha)
        lab.set_va(va)
        lab.set_rotation_mode("anchor")
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)


def force_xtick_style(ax, fontsize=26):
    for lab in ax.get_xticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)


def force_xlabel_style(ax, fontsize=38):
    ax.xaxis.label.set_fontname("Times New Roman")
    ax.xaxis.label.set_fontweight("bold")
    ax.xaxis.label.set_fontsize(fontsize)


def save_fig(fig, out_dir, stem):
    os.makedirs(out_dir, exist_ok=True)

    make_all_artists_opaque(fig)

    png_fp = os.path.join(out_dir, f"{stem}.png")
    pdf_fp = os.path.join(out_dir, f"{stem}.pdf")
    eps_fp = os.path.join(out_dir, f"{stem}.eps")

    for fp, fmt in [
        (png_fp, None),
        (pdf_fp, None),
        (eps_fp, "eps"),
    ]:
        save_kwargs = dict(
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.25,
            facecolor="white",
            edgecolor="white",
            transparent=False
        )
        if fmt is not None:
            save_kwargs["format"] = fmt
        fig.savefig(fp, **save_kwargs)

    plt.close(fig)

    print(f"[Saved] {png_fp}")
    print(f"[Saved] {pdf_fp}")
    print(f"[Saved] {eps_fp}")


def save_csv(df, out_dir, name):
    os.makedirs(out_dir, exist_ok=True)
    fp = os.path.join(out_dir, name)
    df.to_csv(fp, index=False, encoding="utf-8-sig")
    print(f"[Saved CSV] {fp}")

def parse_model_from_filename(fp):
    parts = os.path.basename(fp).split("__")
    return parts[1] if len(parts) >= 2 else "unknown_model"


def infer_run_name(df, fp):
    if "run_name" in df.columns and df["run_name"].notna().any():
        return str(df["run_name"].dropna().iloc[0])

    for c in ["source_jsonl", "source_file"]:
        if c in df.columns and df[c].notna().any():
            p = str(df[c].dropna().iloc[0])
            if c == "source_jsonl":
                return os.path.basename(os.path.dirname(p))
            return os.path.splitext(os.path.basename(p))[0]

    return os.path.splitext(os.path.basename(fp))[0]


def parse_vec8(x):
    if pd.isna(x):
        return None

    s = str(x).strip()
    if s.startswith("b"):
        s = s[1:]

    if len(s) == 8 and set(s) <= {"0", "1"}:
        return [int(c) for c in s]

    nums = re.findall(r"[01]", s)
    if len(nums) == 8:
        return [int(c) for c in nums]

    return None


def vec_to_int(v):
    if v is None:
        return np.nan
    return int("".join(str(int(z)) for z in v), 2)


def int_to_vec8(x):
    """
    将 255-way 类别编号解码为 8-bit vector。
    """
    try:
        i = int(x)
    except Exception:
        return None

    if i < 0 or i > 255:
        return None

    return [int(c) for c in format(i, "08b")]


def symptom_count(v):
    return np.nan if v is None else int(sum(v))


def combo_terms_from_vec(v):
    if v is None:
        return []
    return [
        EMOTION_SHORT[EMOTION_NAMES[i]]
        for i, z in enumerate(v)
        if int(z) == 1
    ]


def combo_display_compact(v, max_terms=2, with_count=True):
    terms = [COMBO_ABBR.get(t, t) for t in combo_terms_from_vec(v)]

    if len(terms) == 0:
        core = "none"
    elif len(terms) <= max_terms:
        core = "+".join(terms)
    else:
        core = "+".join(terms[:max_terms]) + f" +{len(terms) - max_terms}"

    return f"{core} [{len(terms)}]" if with_count else core


def combo_display_full(v):
    terms = [COMBO_ABBR.get(t, t) for t in combo_terms_from_vec(v)]
    return "+".join(terms) if len(terms) > 0 else "none"


def transition_label(true_id, pred_id, id_to_vec):
    """
    修复原代码中 predicted node 不在 top lookup 时仅显示数字的问题。
    """
    tv = id_to_vec.get(int(true_id), int_to_vec8(true_id))
    pv = id_to_vec.get(int(pred_id), int_to_vec8(pred_id))

    left = (
        combo_display_compact(tv, max_terms=2, with_count=False)
        if tv is not None
        else f"id={int(true_id)}"
    )
    right = (
        combo_display_compact(pv, max_terms=2, with_count=False)
        if pv is not None
        else f"id={int(pred_id)}"
    )

    return f"{left} → {right}"

def load_best_class255_run(pred_dir):
    pred_files = sorted(glob.glob(os.path.join(pred_dir, "class255__*.csv")))

    if len(pred_files) == 0:
        raise FileNotFoundError(
            f"No class255 prediction csv found in:\n{pred_dir}"
        )

    run_metrics = []
    run_dfs = []

    for fp in pred_files:
        df = pd.read_csv(fp)
        df["model_name"] = parse_model_from_filename(fp)
        df["run_name"] = infer_run_name(df, fp)
        df["pred_csv"] = fp

        if "true_vec" not in df.columns or "pred_vec" not in df.columns:
            continue

        df["true_vec_list"] = df["true_vec"].map(parse_vec8)
        df["pred_vec_list"] = df["pred_vec"].map(parse_vec8)
        df = df[
            df["true_vec_list"].notna()
            & df["pred_vec_list"].notna()
        ].copy()

        if len(df) == 0:
            continue

        if "true_255" not in df.columns:
            df["true_255"] = df["true_vec_list"].map(vec_to_int)
        if "pred_255" not in df.columns:
            df["pred_255"] = df["pred_vec_list"].map(vec_to_int)

        df["true_255"] = pd.to_numeric(df["true_255"], errors="coerce")
        df["pred_255"] = pd.to_numeric(df["pred_255"], errors="coerce")
        df = df[df["true_255"].notna() & df["pred_255"].notna()].copy()

        if len(df) == 0:
            continue

        df["true_255"] = df["true_255"].astype(int)
        df["pred_255"] = df["pred_255"].astype(int)
        df["correct"] = (df["true_255"] == df["pred_255"]).astype(int)

        yt = df["true_255"].values
        yp = df["pred_255"].values

        run_metrics.append({
            "model_name": df["model_name"].iloc[0],
            "run_name": df["run_name"].iloc[0],
            "pred_csv": fp,
            "n": len(df),
            "accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
            "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        })

        run_dfs.append(df)

    if len(run_metrics) == 0:
        raise RuntimeError("No valid class255 run found.")

    run_metrics_df = (
        pd.DataFrame(run_metrics)
        .sort_values("macro_f1", ascending=False)
        .reset_index(drop=True)
    )

    best_csv = run_metrics_df.iloc[0]["pred_csv"]

    best_df = None
    for df in run_dfs:
        if df["pred_csv"].iloc[0] == best_csv:
            best_df = df.copy()
            break

    return best_df, run_metrics_df

def compute_figd_tables(best_df):
    d = best_df.copy()

    d["true_vec_list"] = d["true_vec"].map(parse_vec8)
    d["pred_vec_list"] = d["pred_vec"].map(parse_vec8)
    d = d[
        d["true_vec_list"].notna()
        & d["pred_vec_list"].notna()
    ].copy()

    d["true_255"] = pd.to_numeric(d["true_255"], errors="coerce").astype(int)
    d["pred_255"] = pd.to_numeric(d["pred_255"], errors="coerce").astype(int)
    d["correct"] = (d["true_255"] == d["pred_255"]).astype(int)

    id_to_vec = {}

    for _, rr in d[["true_255", "true_vec_list"]].drop_duplicates("true_255").iterrows():
        id_to_vec[int(rr["true_255"])] = rr["true_vec_list"]

    for _, rr in d[["pred_255", "pred_vec_list"]].drop_duplicates("pred_255").iterrows():
        id_to_vec[int(rr["pred_255"])] = rr["pred_vec_list"]

    comb_stats = (
        d.groupby("true_255")
        .agg(
            support=("true_255", "size"),
            accuracy=("correct", "mean"),
            true_vec=("true_vec", "first")
        )
        .reset_index()
        .sort_values("support", ascending=False)
    )

    top_combs = comb_stats.head(18).copy()
    top_combs["true_vec_list"] = top_combs["true_vec"].map(parse_vec8)

    node_rows = []
    for _, rr in top_combs.iterrows():
        v = rr["true_vec_list"]
        node_rows.append({
            "node": int(rr["true_255"]),
            "support": int(rr["support"]),
            "accuracy": float(rr["accuracy"]),
            "true_vec": rr["true_vec"],
            "symptom_count": symptom_count(v),
            "display_label": combo_display_compact(v, max_terms=2, with_count=True),
            "short_transition_label": combo_display_compact(v, max_terms=2, with_count=False),
            "full_label": combo_display_full(v),
        })

    topology_nodes_df = pd.DataFrame(node_rows)

    edge_rows = []
    for i in range(len(top_combs)):
        for j in range(i + 1, len(top_combs)):
            a = top_combs.iloc[i]
            b = top_combs.iloc[j]

            va = a["true_vec_list"]
            vb = b["true_vec_list"]

            dist = sum(int(x != y) for x, y in zip(va, vb))

            if dist <= 2:
                edge_rows.append({
                    "node_a": int(a["true_255"]),
                    "node_b": int(b["true_255"]),
                    "hamming_distance": int(dist)
                })

    topology_edges_df = pd.DataFrame(edge_rows)

    trans = (
        d.groupby(["true_255", "pred_255"])
        .size()
        .reset_index(name="count")
    )

    trans = (
        trans[trans["true_255"] != trans["pred_255"]]
        .sort_values("count", ascending=False)
        .head(30)
    )

    trans["transition_label"] = trans.apply(
        lambda r: transition_label(
            int(r["true_255"]),
            int(r["pred_255"]),
            id_to_vec
        ),
        axis=1
    )

    topology_transition_df = trans.copy()

    combo_lookup_rows = []
    for node_id, vec in sorted(id_to_vec.items(), key=lambda x: x[0]):
        combo_lookup_rows.append({
            "node": int(node_id),
            "display_label": combo_display_compact(vec, max_terms=2, with_count=True),
            "short_label": combo_display_compact(vec, max_terms=2, with_count=False),
            "full_label": combo_display_full(vec),
            "symptom_count": symptom_count(vec)
        })

    combo_lookup_df = pd.DataFrame(combo_lookup_rows)

    return (
        topology_nodes_df,
        topology_edges_df,
        topology_transition_df,
        combo_lookup_df
    )


def plot_panel_a(topology_nodes_df, out_fig):
    plot_df = (
        topology_nodes_df
        .sort_values(["support", "accuracy"], ascending=[False, False])
        .head(8)
        .sort_values("accuracy", ascending=True)
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(14.8, 9.1))

    y = np.arange(len(plot_df))

    ax.hlines(
        y=y,
        xmin=0,
        xmax=plot_df["accuracy"].values,
        color=PALETTE["lightgray"],
        lw=3.2,
        zorder=1
    )

    sizes = np.clip(280 + plot_df["support"].values * 18.0, 420, 1700)

    norm = Normalize(
        vmin=max(1, int(plot_df["symptom_count"].min())),
        vmax=max(2, int(plot_df["symptom_count"].max()))
    )

    ax.scatter(
        plot_df["accuracy"].values,
        y,
        s=sizes,
        c=plot_df["symptom_count"].values,
        cmap=CMAP_ORANGE,
        norm=norm,
        edgecolor="white",
        linewidth=1.4,
        zorder=3
    )

    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["display_label"].tolist())

    ax.set_xlabel("Combination accuracy")
    ax.set_xlim(0, 1.08)

    xt = np.arange(0.0, 1.01, 0.2)
    ax.set_xticks(xt)
    ax.set_xticklabels([f"{v:.1f}" for v in xt])

    for i, rr in enumerate(plot_df.itertuples()):
        x_text = min(float(rr.accuracy) + 0.035, 1.035)
        ax.text(
            x_text,
            i,
            f"n={int(rr.support)}",
            va="center",
            ha="left",
            fontsize=22,
            fontname="Times New Roman",
            fontweight="bold",
            color=PALETTE["text"]
        )

    style_ax(ax, x_tick_size=27, y_tick_size=25, label_size=39)

    rotate_ylabels(
        ax,
        rotation=10,
        fontsize=25,
        ha="right",
        va="center"
    )

    force_xtick_style(ax, fontsize=27)
    force_xlabel_style(ax, fontsize=39)

    fig.subplots_adjust(
        left=0.36,
        right=0.985,
        bottom=0.19,
        top=0.97
    )

    save_fig(
        fig,
        out_fig,
        "FigD_combination_topology_a_combination_accuracy"
    )


def plot_panel_b(topology_transition_df, out_fig):
    top_trans = (
        topology_transition_df
        .head(6)
        .copy()
        .sort_values("count", ascending=True)
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(15.8, 9.0))

    y = np.arange(len(top_trans))

    ax.barh(
        y=y,
        width=top_trans["count"].values,
        color=PALETTE["pink"],
        height=0.64,
        edgecolor="none"
    )

    ax.set_yticks(y)
    ax.set_yticklabels(top_trans["transition_label"].tolist())

    ax.set_xlabel("Transition count")

    xmax = max(5, float(top_trans["count"].max()) * 1.10)
    step = 5 if xmax <= 50 else 10

    ax.set_xlim(0, xmax)
    ax.set_xticks(np.arange(0, xmax + 0.01, step))

    style_ax(ax, x_tick_size=27, y_tick_size=25, label_size=39)

    rotate_ylabels(
        ax,
        rotation=12,
        fontsize=25,
        ha="right",
        va="center"
    )

    force_xtick_style(ax, fontsize=27)
    force_xlabel_style(ax, fontsize=39)

    fig.subplots_adjust(
        left=0.58,
        right=0.985,
        bottom=0.19,
        top=0.97
    )

    save_fig(
        fig,
        out_fig,
        "FigD_combination_topology_b_top_transitions"
    )


def plot_combined(topology_nodes_df, topology_transition_df, out_fig):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(27.0, 8.8),
        gridspec_kw={"width_ratios": [1.05, 1.15]}
    )

    # -------------------------
    # Left panel
    # -------------------------
    ax = axes[0]

    plot_df = (
        topology_nodes_df
        .sort_values(["support", "accuracy"], ascending=[False, False])
        .head(8)
        .sort_values("accuracy", ascending=True)
        .reset_index(drop=True)
    )

    y = np.arange(len(plot_df))

    ax.hlines(
        y,
        0,
        plot_df["accuracy"],
        color=PALETTE["lightgray"],
        lw=3.0,
        zorder=1
    )

    sizes = np.clip(230 + plot_df["support"].values * 16.0, 380, 1500)

    ax.scatter(
        plot_df["accuracy"].values,
        y,
        s=sizes,
        c=plot_df["symptom_count"].values,
        cmap=CMAP_ORANGE,
        edgecolor="white",
        linewidth=1.2,
        zorder=3
    )

    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["display_label"].tolist())

    ax.set_xlabel("Combination accuracy")
    ax.set_xlim(0, 1.08)

    xt = np.arange(0.0, 1.01, 0.2)
    ax.set_xticks(xt)
    ax.set_xticklabels([f"{v:.1f}" for v in xt])

    for i, rr in enumerate(plot_df.itertuples()):
        ax.text(
            min(float(rr.accuracy) + 0.035, 1.035),
            i,
            f"n={int(rr.support)}",
            va="center",
            ha="left",
            fontsize=18,
            fontname="Times New Roman",
            fontweight="bold",
            color=PALETTE["text"]
        )

    style_ax(ax, x_tick_size=23, y_tick_size=21, label_size=32)

    rotate_ylabels(
        ax,
        rotation=10,
        fontsize=21,
        ha="right",
        va="center"
    )

    force_xtick_style(ax, fontsize=23)
    force_xlabel_style(ax, fontsize=32)

    # -------------------------
    # Right panel
    # -------------------------
    ax = axes[1]

    top_trans = (
        topology_transition_df
        .head(6)
        .copy()
        .sort_values("count", ascending=True)
        .reset_index(drop=True)
    )

    y = np.arange(len(top_trans))

    ax.barh(
        y,
        top_trans["count"].values,
        color=PALETTE["pink"],
        height=0.62,
        edgecolor="none"
    )

    ax.set_yticks(y)
    ax.set_yticklabels(top_trans["transition_label"].tolist())

    ax.set_xlabel("Transition count")

    xmax = max(5, float(top_trans["count"].max()) * 1.10)
    step = 5 if xmax <= 50 else 10

    ax.set_xlim(0, xmax)
    ax.set_xticks(np.arange(0, xmax + 0.01, step))

    style_ax(ax, x_tick_size=23, y_tick_size=21, label_size=32)

    rotate_ylabels(
        ax,
        rotation=12,
        fontsize=21,
        ha="right",
        va="center"
    )

    force_xtick_style(ax, fontsize=23)
    force_xlabel_style(ax, fontsize=32)

    fig.subplots_adjust(
        left=0.205,
        right=0.985,
        bottom=0.18,
        top=0.965,
        wspace=0.96
    )

    save_fig(
        fig,
        out_fig,
        "FigD_combination_topology"
    )

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--root",
        default=DEFAULT_ROOT,
        help=r"Project root, e.g. D:\Downloads\Depressed"
    )

    # 兼容 Jupyter / ipykernel 自动注入的参数
    args, unknown = parser.parse_known_args()

    if len(unknown) > 0:
        print("[Info] Ignored unknown runtime arguments:")
        for u in unknown:
            print("  ", u)

    set_global_style()

    root = args.root

    pred_dir = os.path.join(
        root,
        "export_bundle_v4",
        "predictions"
    )

    out_dir = os.path.join(
        root,
        "final_pub_outputs_fullcover"
    )

    out_fig = os.path.join(out_dir, "figures")
    out_csv = os.path.join(out_dir, "figure_csv")

    os.makedirs(out_fig, exist_ok=True)
    os.makedirs(out_csv, exist_ok=True)

    print("=" * 100)
    print("FigD combination topology standalone generator")
    print("ROOT     =", root)
    print("PRED_DIR =", pred_dir)
    print("OUT_FIG  =", out_fig)
    print("OUT_CSV  =", out_csv)
    print("=" * 100)

    best_df, run_metrics_df = load_best_class255_run(pred_dir)

    save_csv(
        run_metrics_df,
        out_csv,
        "FigD0_class255_run_metrics.csv"
    )

    print("\n[Best class255 run]")
    print(run_metrics_df.iloc[0].to_string())

    (
        topology_nodes_df,
        topology_edges_df,
        topology_transition_df,
        combo_lookup_df
    ) = compute_figd_tables(best_df)

    save_csv(
        topology_nodes_df,
        out_csv,
        "FigD1_topology_nodes.csv"
    )

    save_csv(
        topology_edges_df,
        out_csv,
        "FigD2_topology_edges.csv"
    )

    save_csv(
        topology_transition_df,
        out_csv,
        "FigD3_top_confusion_transitions.csv"
    )

    save_csv(
        combo_lookup_df,
        out_csv,
        "FigD4_combination_label_lookup.csv"
    )

    print("\n[Top nodes for panel A]")
    print(
        topology_nodes_df
        .sort_values(["support", "accuracy"], ascending=[False, False])
        .head(8)
        .to_string(index=False)
    )

    print("\n[Top transitions for panel B]")
    print(
        topology_transition_df
        .head(6)
        .to_string(index=False)
    )

    plot_panel_a(topology_nodes_df, out_fig)
    plot_panel_b(topology_transition_df, out_fig)
    plot_combined(topology_nodes_df, topology_transition_df, out_fig)

    print("\nDone.")
    print(f"Figures saved to: {out_fig}")
    print(f"CSVs saved to: {out_csv}")


if __name__ == "__main__":
    main()

[Info] Ignored unknown runtime arguments:
   --f=c:\Users\mhzho\AppData\Roaming\jupyter\runtime\kernel-v3b258096dcbe39942c95cfa7edc2274e05f9bfd66.json
FigD combination topology standalone generator
ROOT     = D:\Downloads\Depressed
PRED_DIR = D:\Downloads\Depressed\export_bundle_v4\predictions
OUT_FIG  = D:\Downloads\Depressed\final_pub_outputs_fullcover\figures
OUT_CSV  = D:\Downloads\Depressed\final_pub_outputs_fullcover\figure_csv
[Saved CSV] D:\Downloads\Depressed\final_pub_outputs_fullcover\figure_csv\FigD0_class255_run_metrics.csv

[Best class255 run]
model_name                                       Qwen3.5-9B-Base
run_name                              eval_2026-03-12-09-33-16_2
pred_csv       D:\Downloads\Depressed\export_bundle_v4\predic...
n                                                            906
accuracy                                                0.204194
macro_f1                                                0.088365
weighted_f1                                   

In [ ]:
# -*- coding: utf-8 -*-
"""
Redraw separated FigR, FigM and FigH panels
"""

import os
import argparse
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

warnings.filterwarnings("ignore")

DEFAULT_ROOT = r"D:\Downloads\Depressed"
DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "deepblue": "#6EA9C4",
    "green": "#ADE7A8",
    "deepgreen": "#5FA777",
    "orange": "#F39F4E",
    "deeporange": "#D88537",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "lightgray": "#CFCBC3",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "black": "#111111",
    "white": "#FFFFFF",
    "offwhite": "#FBFBF8",
}

CMAP_ORANGE = LinearSegmentedColormap.from_list(
    "nature_orange",
    ["#FAF8F4", "#F1E2CF", "#F0C48E", "#F39F4E"]
)

CMAP_RED = LinearSegmentedColormap.from_list(
    "nature_red",
    ["#FCF6F6", "#F3DADA", "#EAB4B4", "#D95F5F"]
)

CMAP_BLUE = LinearSegmentedColormap.from_list(
    "nature_blue",
    ["#F7F7F7", "#DCE8EE", "#B9D5E2", "#98CFE6"]
)

CMAP_PURPLE = LinearSegmentedColormap.from_list(
    "nature_purple",
    ["#F6F3FA", "#D8CDEE", "#B39DDB", "#8F76C6"]
)

# General typography
TICK_FS = 30
YTICK_FS = 30
LABEL_FS = 41
TITLE_FS = 34
LEGEND_FS = 25
CBAR_FS = 25

# Heatmap-specific typography for R1-R3 and M1-M3
HEAT_CELL_FS = 37
HEAT_LABEL_FS = 40
HEAT_TITLE_FS = 33
HEAT_CELL_FS = 31
HEAT_CBAR_FS = 24

# Forest-specific typography for H1-H2
FOREST_XTICK_FS = 24
FOREST_YTICK_FS = 30
FOREST_LABEL_FS = 39
FOREST_LEGEND_FS = 25


def set_global_style():
    mpl.rcParams.update({
        "font.family": "Times New Roman",
        "font.serif": ["Times New Roman"],
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.linewidth": 3.4,
        "axes.unicode_minus": False,
        "xtick.major.width": 3.4,
        "ytick.major.width": 3.4,
        "xtick.major.size": 12,
        "ytick.major.size": 12,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.facecolor": "white",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "legend.frameon": False,
    })


def make_all_artists_opaque(fig):
    fig.patch.set_facecolor("white")
    fig.patch.set_alpha(1.0)

    for ax in fig.axes:
        ax.set_facecolor("white")
        ax.patch.set_alpha(1.0)
        for artist in ax.get_children():
            try:
                artist.set_alpha(1.0)
            except Exception:
                pass


def force_text_style(
    ax,
    x_tick_size=TICK_FS,
    y_tick_size=YTICK_FS,
    label_size=LABEL_FS,
    title_size=TITLE_FS
):
    for txt in [ax.xaxis.label, ax.yaxis.label, ax.title]:
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")

    ax.xaxis.label.set_fontsize(label_size)
    ax.yaxis.label.set_fontsize(label_size)
    ax.title.set_fontsize(title_size)

    for lab in ax.get_xticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(x_tick_size)

    for lab in ax.get_yticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(y_tick_size)

    for txt in ax.texts:
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")


def style_ax(
    ax,
    x_tick_size=TICK_FS,
    y_tick_size=YTICK_FS,
    label_size=LABEL_FS,
    title_size=TITLE_FS
):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(3.4)
    ax.spines["bottom"].set_linewidth(3.4)
    ax.tick_params(axis="both", width=3.4, length=12, pad=9)
    ax.grid(False)

    force_text_style(
        ax,
        x_tick_size=x_tick_size,
        y_tick_size=y_tick_size,
        label_size=label_size,
        title_size=title_size,
    )


def rotate_ylabels(ax, rotation=12, fontsize=YTICK_FS):
    for lab in ax.get_yticklabels():
        lab.set_rotation(rotation)
        lab.set_ha("right")
        lab.set_va("center")
        lab.set_rotation_mode("anchor")
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)


def style_legend(leg, fontsize=LEGEND_FS):
    if leg is None:
        return

    frame = leg.get_frame()
    if frame is not None:
        frame.set_alpha(0.0)
        frame.set_facecolor("white")
        frame.set_edgecolor("white")

    for txt in leg.get_texts():
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")
        txt.set_fontsize(fontsize)


def choose_best_legend_location(ax):
    try:
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()

        x_mid = xlim[0] + 0.65 * (xlim[1] - xlim[0])
        y_mid = ylim[0] + 0.65 * (ylim[1] - ylim[0])

        pts = []
        for coll in ax.collections:
            try:
                arr = coll.get_offsets()
                if arr is not None and len(arr) > 0:
                    pts.extend(arr)
            except Exception:
                pass

        if len(pts) == 0:
            return "best"

        pts = np.asarray(pts, dtype=float)
        dense_upper_right = np.sum((pts[:, 0] >= x_mid) & (pts[:, 1] >= y_mid))

        return "lower right" if dense_upper_right >= 2 else "upper right"

    except Exception:
        return "best"


def compact_numeric_xaxis(ax, max_ticks=5, fontsize=FOREST_XTICK_FS, fmt="%.2f"):
    """
    Sparse numeric ticks for forest plots.
    This prevents H2 x-axis labels from overlapping.
    """
    ax.xaxis.set_major_locator(MaxNLocator(nbins=max_ticks))
    ax.xaxis.set_major_formatter(FormatStrFormatter(fmt))

    ax.tick_params(axis="x", labelsize=fontsize, pad=10)

    for lab in ax.get_xticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)
        lab.set_rotation(0)
        lab.set_ha("center")


def save_fig(fig, fig_dir: Path, stem: str):
    fig_dir.mkdir(parents=True, exist_ok=True)

    make_all_artists_opaque(fig)

    png = fig_dir / f"{stem}.png"
    pdf = fig_dir / f"{stem}.pdf"
    eps = fig_dir / f"{stem}.eps"

    for fp, fmt in [(png, None), (pdf, None), (eps, "eps")]:
        kwargs = dict(
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.28,
            facecolor="white",
            edgecolor="white",
            transparent=False,
        )
        if fmt is not None:
            kwargs["format"] = fmt
        fig.savefig(fp, **kwargs)

    plt.close(fig)

    print(f"[Saved] {png}")
    print(f"[Saved] {pdf}")
    print(f"[Saved] {eps}")


def safe_read_csv(path: Path) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()

    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return pd.DataFrame()

    return pd.DataFrame()


def write_note(txt_dir: Path, name: str, text: str):
    txt_dir.mkdir(parents=True, exist_ok=True)

    fp = txt_dir / name
    fp.write_text(text, encoding="utf-8")

    print(f"[Note] {fp}")


def save_csv_local(df: pd.DataFrame, csv_dir: Path, name: str):
    csv_dir.mkdir(parents=True, exist_ok=True)

    fp = csv_dir / name
    df.to_csv(fp, index=False, encoding="utf-8-sig")

    print(f"[Saved CSV] {fp}")


def display_bin_labels(bins):
    mp = {
        "Q1": "low",
        "Q2": "mid",
        "Q3": "high",
    }
    return [mp.get(str(b), str(b)) for b in bins]


def trim_heatmap_frame(df_wide):
    if df_wide is None or len(df_wide) == 0:
        return df_wide

    row_mask = df_wide.notna().any(axis=1)
    col_mask = df_wide.notna().any(axis=0)

    if row_mask.sum() == 0 or col_mask.sum() == 0:
        return df_wide

    return df_wide.loc[row_mask, col_mask]


def style_colorbar(cbar, label="Rate", fontsize=CBAR_FS):
    cbar.ax.set_ylabel(
        label,
        rotation=90,
        fontweight="bold",
        fontsize=fontsize
    )

    cbar.ax.tick_params(
        labelsize=fontsize - 3,
        width=2.4,
        length=7
    )

    for lab in cbar.ax.get_yticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")


def get_heatmap_cmap(title):
    t = str(title).lower()

    if "hidden" in t or "suicide" in t or "miss" in t or "fn" in t:
        return CMAP_RED

    if "topology" in t or "latent" in t:
        return CMAP_BLUE

    if "interaction" in t:
        return CMAP_PURPLE

    return CMAP_ORANGE

LEXICON = {
    "sadness_explicit": [
        "sad", "depressed", "cry", "crying", "empty", "down", "miserable"
    ],
    "hopelessness_explicit": [
        "hopeless", "no future", "nothing will change", "can't go on", "pointless"
    ],
    "worthlessness_explicit": [
        "worthless", "useless", "burden", "failure", "not good enough", "hate myself"
    ],
    "cognitive_explicit": [
        "forget", "memory", "focus", "concentrate", "brain fog", "can't think"
    ],
    "suicide_explicit": [
        "suicide", "suicidal", "kill myself", "end my life", "die",
        "self-harm", "self harm"
    ],
    "help_seeking": [
        "please help", "need help", "advice", "suggestions", "what do i do", "help me"
    ],
    "absolutist": [
        "always", "never", "nothing", "nobody", "completely", "totally", "forever"
    ],
}


def has_any(txt, kws):
    t = str(txt).lower()
    return int(any(k in t for k in kws))


def col_or_zero(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.zeros(len(df)), index=df.index)


def normalize_01(x):
    s = pd.to_numeric(pd.Series(x), errors="coerce")

    if s.notna().sum() == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)

    lo, hi = s.min(), s.max()

    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s - lo) / (hi - lo)


def zscore(x):
    s = pd.to_numeric(pd.Series(x), errors="coerce")

    mu = s.mean()
    sd = s.std(ddof=0)

    if pd.isna(sd) or sd == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s - mu) / sd


def normalize_col_01(df, col):
    return normalize_01(col_or_zero(df, col))


def zscore_col(df, col):
    return zscore(col_or_zero(df, col))


def existing_cols(df, cols, min_non_na=8):
    out = []

    if df is None or len(df) == 0:
        return out

    for c in cols:
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            if s.notna().sum() >= min_non_na:
                out.append(c)

    return out


def stable_rank_bins(series, n_bins=3, prefix="Q"):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(index=s.index, dtype="object")

    valid = s.notna()

    if valid.sum() < max(12, n_bins * 4):
        return out

    ranks = s[valid].rank(method="average", pct=True)
    edges = np.linspace(0, 1, n_bins + 1)
    labels = [f"{prefix}{i + 1}" for i in range(n_bins)]

    bins = pd.cut(
        ranks,
        bins=edges,
        labels=labels,
        include_lowest=True
    )

    out.loc[valid] = bins.astype(str)

    return out


def ensure_basic_text_features(df):
    d = df.copy()

    if "text" not in d.columns:
        d["text"] = ""

    d["text"] = d["text"].astype(str)

    if "text_len" not in d.columns:
        d["text_len"] = d["text"].str.len()

    d["text_len"] = (
        pd.to_numeric(d["text_len"], errors="coerce")
        .fillna(d["text"].str.len())
    )

    if "log_text_len" not in d.columns:
        d["log_text_len"] = np.log1p(np.maximum(d["text_len"], 0))

    wc = np.maximum(d["text"].str.split().str.len().fillna(0), 1)

    if "first_person_density" not in d.columns:
        d["first_person_density"] = (
            d["text"]
            .str.lower()
            .str.count(r"\b(i|me|my|mine|myself)\b") / wc
        )

    if "negation_density" not in d.columns:
        d["negation_density"] = (
            d["text"]
            .str.lower()
            .str.count(r"\b(no|not|never|nothing|can't|cannot|don't|won't)\b") / wc
        )

    if "emotion_word_density" not in d.columns:
        d["emotion_word_density"] = (
            d["text"]
            .str.lower()
            .str.count(
                r"\b(sad|hopeless|depressed|cry|empty|worthless|lonely|suicidal|anxious)\b"
            ) / wc
        )

    for cue, kws in LEXICON.items():
        if cue not in d.columns:
            d[cue] = d["text"].map(lambda x: has_any(x, kws))
        d[cue] = pd.to_numeric(d[cue], errors="coerce").fillna(0)

    return d


def fit_logistic_full(df, target_col, feature_cols, random_state=42):
    d = df.copy()

    feat_cols = existing_cols(
        d,
        feature_cols,
        min_non_na=max(8, int(0.1 * len(d)))
    )

    if target_col not in d.columns or len(feat_cols) == 0:
        return None

    y = pd.to_numeric(d[target_col], errors="coerce")
    X = d[feat_cols].apply(pd.to_numeric, errors="coerce")

    mask = (
        y.notna() &
        (X.notna().sum(axis=1) >= max(1, int(0.6 * len(feat_cols))))
    )

    X = X[mask]
    y = y[mask].astype(int)

    if len(X) < 24 or y.nunique() < 2:
        return None

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=4000,
            class_weight="balanced",
            solver="liblinear",
            random_state=random_state
        )),
    ])

    pipe.fit(X, y)

    p = pipe.predict_proba(X)[:, 1]

    ll_model = log_loss(y, p, labels=[0, 1], normalize=False)
    base_p = np.repeat(float(y.mean()), len(y))
    ll_null = log_loss(y, base_p, labels=[0, 1], normalize=False)

    dev_model = 2.0 * ll_model
    dev_null = 2.0 * ll_null

    pseudo_r2 = np.nan if dev_null == 0 else 1.0 - (dev_model / dev_null)
    auc = roc_auc_score(y, p) if y.nunique() >= 2 else np.nan

    return {
        "pipe": pipe,
        "X": X,
        "y": y,
        "features": feat_cols,
        "p": p,
        "ll": ll_model,
        "deviance": dev_model,
        "null_deviance": dev_null,
        "pseudo_r2": pseudo_r2,
        "auc": auc,
    }


def load_r_master_analysis_df(csv_dir: Path, txt_dir: Path):
    master_fp = csv_dir / "nhb_gap_pack_master_analysis_df.csv"

    if not master_fp.exists():
        write_note(
            txt_dir,
            "FigR_redraw_notes.txt",
            "FigR skipped: missing nhb_gap_pack_master_analysis_df.csv. "
            "Run the NHB gap-pack first, because FigR must be recomputed from the master analysis table."
        )
        return pd.DataFrame()

    df = safe_read_csv(master_fp)

    if len(df) == 0:
        write_note(
            txt_dir,
            "FigR_redraw_notes.txt",
            "FigR skipped: nhb_gap_pack_master_analysis_df.csv exists but could not be read."
        )
        return pd.DataFrame()

    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)

    df = ensure_basic_text_features(df)

    for c in [
        "true_label",
        "pred_label",
        "confidence_score",
        "is_disagreement_sample",
        "cue_sum",
        "explicit_risk_score",
        "affective_onset_score",
        "distortion_cue_score",
        "bridge_burden",
        "bridge_core_load",
        "bridge_periphery_load",
        "core_minus_periphery",
        "bridge_combo_rarity",
        "latent_risk_minus_explicit",
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if "cue_sum" not in df.columns:
        df["cue_sum"] = sum(col_or_zero(df, c) for c in LEXICON)

    if "cue_conflict" not in df.columns:
        df["cue_conflict"] = (
            col_or_zero(df, "help_seeking") *
            (
                (
                    col_or_zero(df, "hopelessness_explicit") +
                    col_or_zero(df, "worthlessness_explicit") +
                    col_or_zero(df, "suicide_explicit")
                ) > 0
            ).astype(int)
        )

    if "distortion_cue_score" not in df.columns:
        df["distortion_cue_score"] = (
            col_or_zero(df, "hopelessness_explicit") +
            col_or_zero(df, "worthlessness_explicit") +
            col_or_zero(df, "absolutist")
        )

    if "explicit_risk_score" not in df.columns:
        df["explicit_risk_score"] = (
            2.0 * col_or_zero(df, "suicide_explicit") +
            col_or_zero(df, "hopelessness_explicit") +
            col_or_zero(df, "worthlessness_explicit")
        )

    df["family_affective_onset"] = (
        zscore_col(df, "sadness_explicit").fillna(0) +
        zscore_col(df, "emotion_word_density").fillna(0) +
        zscore_col(df, "first_person_density").fillna(0)
    ) / 3.0

    df["family_distortion_escalation"] = (
        zscore_col(df, "hopelessness_explicit").fillna(0) +
        zscore_col(df, "absolutist").fillna(0) +
        zscore_col(df, "negation_density").fillna(0)
    ) / 3.0

    df["family_worthlessness"] = (
        zscore_col(df, "worthlessness_explicit").fillna(0) +
        zscore_col(df, "cue_conflict").fillna(0) +
        zscore_col(df, "first_person_density").fillna(0)
    ) / 3.0

    df["family_suicidality_crystallization"] = (
        zscore_col(df, "suicide_explicit").fillna(0) +
        zscore_col(df, "explicit_risk_score").fillna(0) +
        zscore_col(df, "bridge_core_load").fillna(0)
    ) / 3.0

    df["explicit_visibility"] = (
        normalize_col_01(df, "explicit_risk_score").fillna(0) * 0.7 +
        normalize_col_01(df, "cue_sum").fillna(0) * 0.3
    )

    latent_parts = []
    for c in [
        "bridge_core_load",
        "bridge_combo_rarity",
        "bridge_burden",
        "family_distortion_escalation",
        "family_worthlessness",
    ]:
        latent_parts.append(zscore_col(df, c).fillna(0))

    if len(latent_parts) == 0:
        df["latent_structure"] = 0.0
    else:
        df["latent_structure"] = sum(latent_parts) / float(len(latent_parts))

    fam_frame = pd.DataFrame({
        "Affective onset": pd.to_numeric(df["family_affective_onset"], errors="coerce"),
        "Distortion escalation": pd.to_numeric(df["family_distortion_escalation"], errors="coerce"),
        "Worthlessness consolidation": pd.to_numeric(df["family_worthlessness"], errors="coerce"),
        "Suicidality crystallization": pd.to_numeric(df["family_suicidality_crystallization"], errors="coerce"),
    })

    df["dominant_family"] = fam_frame.idxmax(axis=1)

    df["true_label"] = pd.to_numeric(df.get("true_label", np.nan), errors="coerce")
    df["pred_label"] = pd.to_numeric(df.get("pred_label", np.nan), errors="coerce")

    valid = df["true_label"].notna() & df["pred_label"].notna()
    df["understate"] = np.nan
    df.loc[valid, "understate"] = (
        df.loc[valid, "pred_label"] < df.loc[valid, "true_label"]
    ).astype(float)

    return df


def draw_single_r_surface_panel(wide, fam, stem, fig_dir: Path):
    arr = wide.values.astype(float)
    arr_masked = np.ma.masked_invalid(arr)

    cmap = CMAP_RED.copy()
    cmap.set_bad("#F5F2EB")

    if np.isfinite(np.nanmin(arr)):
        vmin = max(0.0, float(np.nanmin(arr)))
    else:
        vmin = 0.0

    if np.isfinite(np.nanmax(arr)):
        vmax = float(np.nanmax(arr))
    else:
        vmax = 1.0

    vmax = max(vmax, 0.05)

    fig, ax = plt.subplots(figsize=(9.4, 8.6))

    im = ax.imshow(
        arr_masked,
        cmap=cmap,
        aspect="auto",
        vmin=vmin,
        vmax=vmax
    )

    show_bins_x = list(wide.columns)
    show_bins_y = list(wide.index)

    ax.set_xticks(np.arange(len(show_bins_x)))
    ax.set_yticks(np.arange(len(show_bins_y)))
    ax.set_xticklabels(display_bin_labels(show_bins_x))
    ax.set_yticklabels(display_bin_labels(show_bins_y))

    ax.set_xlabel("Explicit visibility")
    ax.set_ylabel("Latent structure")
    ax.set_title(fam, pad=12)

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = wide.iloc[i, j]
            if pd.notna(val):
                ax.text(
                    j,
                    i,
                    f"{val:.2f}",
                    ha="center",
                    va="center",
                    fontsize=HEAT_CELL_FS,
                    fontweight="bold",
                    fontname="Times New Roman",
                    color=PALETTE["black"],
                )

    ax.set_xticks(np.arange(-0.5, len(show_bins_x), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(show_bins_y), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2.6)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.052, pad=0.042)

    style_colorbar(
        cbar,
        label="Predicted understate probability",
        fontsize=HEAT_CBAR_FS
    )

    style_ax(
        ax,
        x_tick_size=HEAT_TICK_FS,
        y_tick_size=HEAT_TICK_FS,
        label_size=HEAT_LABEL_FS,
        title_size=HEAT_TITLE_FS
    )
    rotate_ylabels(ax, rotation=12, fontsize=HEAT_TICK_FS)

    fig.subplots_adjust(
        left=0.30,
        right=0.88,
        bottom=0.22,
        top=0.88
    )

    save_fig(fig, fig_dir, stem)


def redraw_R(csv_dir: Path, fig_dir: Path, txt_dir: Path):
    analysis_df = load_r_master_analysis_df(csv_dir, txt_dir)

    if len(analysis_df) == 0:
        return

    r_df = analysis_df.copy()

    r_df["latent_bin"] = stable_rank_bins(r_df["latent_structure"], n_bins=3)
    r_df["explicit_bin"] = stable_rank_bins(r_df["explicit_visibility"], n_bins=3)

    family_order = [
        "Affective onset",
        "Distortion escalation",
        "Suicidality crystallization",
    ]

    r_df["dominant_stage_family"] = r_df["dominant_family"].replace({
        "Worthlessness consolidation": "Distortion escalation"
    })

    r_rows = []
    surface_long_rows = []
    r_surface_frames = {}

    for fam in family_order:
        sub = r_df[r_df["dominant_stage_family"] == fam].copy()

        if len(sub) < 30:
            continue

        model_df = sub[[
            "understate",
            "latent_structure",
            "explicit_visibility"
        ]].copy()

        model_df["latent_x_explicit"] = (
            model_df["latent_structure"] * model_df["explicit_visibility"]
        )

        fit = fit_logistic_full(
            model_df,
            "understate",
            [
                "latent_structure",
                "explicit_visibility",
                "latent_x_explicit"
            ]
        )

        if fit is None:
            continue

        clf = fit["pipe"].named_steps["clf"]
        feats = fit["features"]

        coef_map = {
            f: float(c)
            for f, c in zip(feats, clf.coef_.ravel())
        }

        for term in [
            "latent_structure",
            "explicit_visibility",
            "latent_x_explicit"
        ]:
            if term in coef_map:
                r_rows.append({
                    "family": fam,
                    "term": term,
                    "log_odds": coef_map[term],
                    "odds_ratio": float(np.exp(coef_map[term])),
                    "auc": fit["auc"],
                    "pseudo_r2": fit["pseudo_r2"],
                    "n": len(sub),
                })

        lb = stable_rank_bins(sub["latent_structure"], 3)
        eb = stable_rank_bins(sub["explicit_visibility"], 3)

        centers_lat = (
            sub.groupby(lb)["latent_structure"]
            .median()
            .to_dict()
        )

        centers_exp = (
            sub.groupby(eb)["explicit_visibility"]
            .median()
            .to_dict()
        )

        labels = [
            b for b in ["Q1", "Q2", "Q3"]
            if b in centers_lat and b in centers_exp
        ]

        if len(labels) == 0:
            continue

        wide = pd.DataFrame(
            index=labels,
            columns=labels,
            dtype=float
        )

        for lv in labels:
            for ev in labels:
                row = pd.DataFrame({
                    "latent_structure": [centers_lat[lv]],
                    "explicit_visibility": [centers_exp[ev]],
                    "latent_x_explicit": [
                        centers_lat[lv] * centers_exp[ev]
                    ],
                })

                pp = fit["pipe"].predict_proba(row)[:, 1][0]
                wide.loc[lv, ev] = pp

                surface_long_rows.append({
                    "family": fam,
                    "latent_bin": lv,
                    "explicit_bin": ev,
                    "predicted_understate_probability": float(pp),
                    "latent_center": float(centers_lat[lv]),
                    "explicit_center": float(centers_exp[ev]),
                    "n_family": len(sub),
                })

        r_surface_frames[fam] = wide

    if len(r_rows) > 0:
        save_csv_local(
            pd.DataFrame(r_rows),
            csv_dir,
            "FigR1_failure_interaction_terms.csv"
        )

    if len(surface_long_rows) > 0:
        save_csv_local(
            pd.DataFrame(surface_long_rows),
            csv_dir,
            "FigR2_failure_interaction_surface_long.csv"
        )

    if len(r_surface_frames) == 0:
        write_note(
            txt_dir,
            "FigR_failure_interaction_surface_notes.txt",
            "No valid family-specific interaction surface could be estimated."
        )
        return

    panel_idx = 1

    for fam in family_order:
        if fam not in r_surface_frames:
            continue

        draw_single_r_surface_panel(
            wide=r_surface_frames[fam],
            fam=fam,
            stem=f"FigR_failure_interaction_surface_R{panel_idx}",
            fig_dir=fig_dir
        )

        panel_idx += 1

    if panel_idx <= 3:
        write_note(
            txt_dir,
            "FigR_redraw_notes.txt",
            f"FigR source-based redraw finished, but only {panel_idx - 1} valid family surface panel(s) were estimated."
        )
    else:
        write_note(
            txt_dir,
            "FigR_redraw_notes.txt",
            "FigR source-based redraw finished. R1, R2 and R3 were generated from the master analysis table."
        )

def redraw_M(csv_dir: Path, fig_dir: Path, txt_dir: Path):
    m = safe_read_csv(csv_dir / "FigM1_failure_regime_surface.csv")

    if m.empty:
        write_note(
            txt_dir,
            "FigM_redraw_notes.txt",
            "FigM skipped: missing FigM1_failure_regime_surface.csv."
        )
        return

    required = {
        "regime",
        "visibility_bin",
        "topology_bin",
        "rate"
    }

    if not required.issubset(set(m.columns)):
        write_note(
            txt_dir,
            "FigM_redraw_notes.txt",
            f"FigM skipped: missing columns {required - set(m.columns)}."
        )
        return

    title_map = {
        "adjacent": "Adjacent error",
        "far": "Far error",
        "suicide_FN": "Suicide-FN miss",
        "reg_adjacent": "Adjacent error",
        "reg_far": "Far error",
        "reg_hidden_miss": "Hidden high-distress miss",
    }

    desired = [
        "adjacent",
        "far",
        "suicide_FN",
        "reg_adjacent",
        "reg_far",
        "reg_hidden_miss",
    ]

    available = list(m["regime"].dropna().astype(str).unique())

    ordered = [r for r in desired if r in available]
    ordered += [r for r in available if r not in ordered]
    ordered = ordered[:3]

    all_plot = m.pivot_table(
        index="topology_bin",
        columns="visibility_bin",
        values="rate",
        aggfunc="mean"
    )

    all_plot = all_plot.reindex(
        index=["Q1", "Q2", "Q3"],
        columns=["Q1", "Q2", "Q3"]
    )

    all_plot = trim_heatmap_frame(all_plot)

    valid_rows = (
        list(all_plot.index)
        if len(all_plot.index) > 0
        else ["Q1", "Q2", "Q3"]
    )

    valid_cols = (
        list(all_plot.columns)
        if len(all_plot.columns) > 0
        else ["Q1", "Q2", "Q3"]
    )

    for idx, reg in enumerate(ordered):
        title = title_map.get(reg, reg.replace("_", " "))

        dd = m[m["regime"].astype(str) == reg].copy()

        plot = dd.pivot(
            index="topology_bin",
            columns="visibility_bin",
            values="rate"
        )

        plot = plot.reindex(
            index=valid_rows,
            columns=valid_cols
        )

        arr = plot.values.astype(float)
        arr_masked = np.ma.masked_invalid(arr)

        cmap = get_heatmap_cmap(title).copy()
        cmap.set_bad("#F5F2EB")

        if np.isfinite(np.nanmax(arr)):
            vmax = max(float(np.nanmax(arr)), 0.06)
        else:
            vmax = 1.0

        fig, ax = plt.subplots(figsize=(9.4, 8.6))

        im = ax.imshow(
            arr_masked,
            cmap=cmap,
            aspect="equal",
            vmin=0,
            vmax=vmax
        )

        ax.set_xticks(np.arange(len(valid_cols)))
        ax.set_yticks(np.arange(len(valid_rows)))
        ax.set_xticklabels(display_bin_labels(valid_cols))
        ax.set_yticklabels(display_bin_labels(valid_rows))

        ax.set_xlabel("Explicit visibility")
        ax.set_ylabel("Latent structure")
        ax.set_title(title, pad=12)

        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                val = plot.iloc[i, j]
                if pd.notna(val):
                    ax.text(
                        j,
                        i,
                        f"{val:.2f}",
                        ha="center",
                        va="center",
                        fontsize=HEAT_CELL_FS,
                        fontweight="bold",
                        fontname="Times New Roman",
                        color=PALETTE["black"],
                    )

        ax.set_xticks(np.arange(-0.5, len(valid_cols), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(valid_rows), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=2.6)
        ax.tick_params(which="minor", bottom=False, left=False)

        cbar = fig.colorbar(
            im,
            ax=ax,
            fraction=0.052,
            pad=0.042
        )

        style_colorbar(cbar, label="Rate", fontsize=HEAT_CBAR_FS)

        style_ax(
            ax,
            x_tick_size=HEAT_TICK_FS,
            y_tick_size=HEAT_TICK_FS,
            label_size=HEAT_LABEL_FS,
            title_size=HEAT_TITLE_FS
        )
        rotate_ylabels(ax, rotation=12, fontsize=HEAT_TICK_FS)

        fig.subplots_adjust(
            left=0.30,
            right=0.89,
            bottom=0.22,
            top=0.88
        )

        save_fig(
            fig,
            fig_dir,
            f"FigM_failure_regime_map_M{idx + 1}"
        )

    if len(ordered) < 3:
        write_note(
            txt_dir,
            "FigM_redraw_notes.txt",
            f"FigM contained only {len(ordered)} usable regime panel(s). Saved M1–M{len(ordered)}."
        )

def redraw_H1(csv_dir: Path, fig_dir: Path, txt_dir: Path):
    four = safe_read_csv(
        csv_dir / "FigH3_fourclass_forest_summary.csv"
    )

    if four.empty:
        write_note(
            txt_dir,
            "FigH_redraw_notes.txt",
            "H1 skipped: missing FigH3_fourclass_forest_summary.csv."
        )
        return

    required = {
        "metric_display",
        "mean",
        "lo",
        "hi"
    }

    if not required.issubset(set(four.columns)):
        write_note(
            txt_dir,
            "FigH_redraw_notes.txt",
            f"H1 skipped: missing columns {required - set(four.columns)}."
        )
        return

    plot = four.copy()

    for c in ["mean", "lo", "hi"]:
        plot[c] = pd.to_numeric(plot[c], errors="coerce")

    plot = plot.dropna(subset=["mean", "lo", "hi"]).copy()
    plot["abs_mean"] = plot["mean"].abs()

    plot = (
        plot.sort_values("abs_mean", ascending=True)
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(11.6, 7.8))

    y = np.arange(len(plot))

    for i, rr in plot.iterrows():
        ax.hlines(
            i,
            rr["lo"],
            rr["hi"],
            color=PALETTE["orange"],
            lw=3.8
        )

        ax.scatter(
            rr["mean"],
            i,
            s=360,
            color=PALETTE["deeporange"],
            zorder=5
        )

    ax.axvline(
        0,
        color=PALETTE["lightgray"],
        lw=2.5
    )

    ax.set_yticks(y)
    ax.set_yticklabels(
        plot["metric_display"].astype(str).tolist()
    )

    xmin = min(plot["lo"].min(), -0.02)
    xmax = max(plot["hi"].max(), 0.02)
    pad = max(0.05, 0.08 * (xmax - xmin))

    ax.set_xlim(xmin - pad, xmax + pad)

    ax.set_xlabel("Mean ± bootstrap interval")

    style_ax(
        ax,
        x_tick_size=FOREST_XTICK_FS,
        y_tick_size=FOREST_YTICK_FS,
        label_size=FOREST_LABEL_FS,
        title_size=TITLE_FS
    )
    rotate_ylabels(ax, rotation=12, fontsize=FOREST_YTICK_FS)
    compact_numeric_xaxis(ax, max_ticks=5, fontsize=FOREST_XTICK_FS, fmt="%.2f")

    fig.subplots_adjust(
        left=0.45,
        right=0.97,
        bottom=0.25,
        top=0.94
    )

    save_fig(
        fig,
        fig_dir,
        "FigH_cross_run_stability_forest_H1"
    )


def redraw_H2(csv_dir: Path, fig_dir: Path, txt_dir: Path):
    multi = safe_read_csv(
        csv_dir / "FigH4_multilabel_forest_summary.csv"
    )

    if multi.empty:
        write_note(
            txt_dir,
            "FigH_redraw_notes.txt",
            "H2 skipped: missing FigH4_multilabel_forest_summary.csv."
        )
        return

    required = {
        "task",
        "metric_display",
        "mean",
        "lo",
        "hi"
    }

    if not required.issubset(set(multi.columns)):
        write_note(
            txt_dir,
            "FigH_redraw_notes.txt",
            f"H2 skipped: missing columns {required - set(multi.columns)}."
        )
        return

    plot = multi.copy()

    for c in ["mean", "lo", "hi"]:
        plot[c] = pd.to_numeric(plot[c], errors="coerce")

    plot = plot.dropna(subset=["mean", "lo", "hi"]).copy()

    metric_order = [
        "Burden bias",
        "Mean core shift",
        "Brain omission"
    ]

    metric_to_y = {
        m: i
        for i, m in enumerate(metric_order)
    }

    fig, ax = plt.subplots(figsize=(13.8, 7.8))

    task_specs = [
        ("eightlabel", "8-label", PALETTE["blue"], -0.15),
        ("class255", "255-way", PALETTE["pink"], +0.15),
    ]

    handles = []

    for task_name, task_label, col, yoff in task_specs:
        dd = plot[plot["task"] == task_name].copy()

        if dd.empty:
            continue

        dd["plot_y"] = (
            dd["metric_display"]
            .map(metric_to_y)
            .astype(float) + yoff
        )

        for _, rr in dd.iterrows():
            ax.hlines(
                rr["plot_y"],
                rr["lo"],
                rr["hi"],
                color=col,
                lw=3.8
            )

            ax.scatter(
                rr["mean"],
                rr["plot_y"],
                s=360,
                color=col,
                zorder=5
            )

        handles.append(
            Line2D(
                [0],
                [0],
                color=col,
                marker="o",
                lw=3.4,
                markersize=12,
                label=task_label,
            )
        )

    ax.axvline(
        0,
        color=PALETTE["lightgray"],
        lw=2.5
    )

    ax.set_yticks(np.arange(len(metric_order)))
    ax.set_yticklabels(metric_order)

    xmin = min(plot["lo"].min(), -0.05)
    xmax = max(plot["hi"].max(), 0.05)
    pad = max(0.06, 0.08 * (xmax - xmin))

    ax.set_xlim(xmin - pad, xmax + pad)

    ax.set_xlabel("Mean ± bootstrap interval")

    style_ax(
        ax,
        x_tick_size=FOREST_XTICK_FS,
        y_tick_size=FOREST_YTICK_FS,
        label_size=FOREST_LABEL_FS,
        title_size=TITLE_FS
    )
    rotate_ylabels(ax, rotation=12, fontsize=FOREST_YTICK_FS)
    compact_numeric_xaxis(ax, max_ticks=5, fontsize=FOREST_XTICK_FS, fmt="%.2f")

    loc = choose_best_legend_location(ax)
    leg = ax.legend(
        handles=handles,
        frameon=False,
        loc=loc
    )

    style_legend(leg, fontsize=FOREST_LEGEND_FS)

    fig.subplots_adjust(
        left=0.37,
        right=0.97,
        bottom=0.25,
        top=0.94
    )

    save_fig(
        fig,
        fig_dir,
        "FigH_cross_run_stability_forest_H2"
    )


def redraw_H(csv_dir: Path, fig_dir: Path, txt_dir: Path):
    redraw_H1(csv_dir, fig_dir, txt_dir)
    redraw_H2(csv_dir, fig_dir, txt_dir)

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--root",
        default=DEFAULT_ROOT,
        help=r"Project root, e.g. D:\Downloads\Depressed"
    )

    args, unknown = parser.parse_known_args()

    if len(unknown) > 0:
        print("[Info] Ignored unknown runtime arguments:")
        for u in unknown:
            print("  ", u)

    set_global_style()

    root = Path(args.root)

    out = root / "final_pub_outputs_fullcover"
    fig_dir = out / "figures"
    csv_dir = out / "figure_csv"
    txt_dir = out / "texts"

    if not csv_dir.exists():
        alt_out = root / "Depressed" / "final_pub_outputs_fullcover"
        if (alt_out / "figure_csv").exists():
            out = alt_out
            fig_dir = out / "figures"
            csv_dir = out / "figure_csv"
            txt_dir = out / "texts"

    fig_dir.mkdir(parents=True, exist_ok=True)
    csv_dir.mkdir(parents=True, exist_ok=True)
    txt_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 100)
    print("Redraw separated FigR, FigM and FigH panels")
    print("ROOT    =", root)
    print("OUT     =", out)
    print("CSV_DIR =", csv_dir)
    print("FIG_DIR =", fig_dir)
    print("=" * 100)

    redraw_R(csv_dir, fig_dir, txt_dir)
    redraw_M(csv_dir, fig_dir, txt_dir)
    redraw_H(csv_dir, fig_dir, txt_dir)

    print("\nDone.")
    print(f"Figures saved to: {fig_dir}")
    print(f"CSVs saved to:    {csv_dir}")
    print(f"Notes saved to:   {txt_dir}")


if __name__ == "__main__":
    main()

[Info] Ignored unknown runtime arguments:
   --f=c:\Users\mhzho\AppData\Roaming\jupyter\runtime\kernel-v3f84a6b90eb6cfb7d2aecef638096c6c2c0dc4f08.json
Redraw separated FigR, FigM and FigH panels
ROOT    = D:\Downloads\Depressed
OUT     = D:\Downloads\Depressed\final_pub_outputs_fullcover
CSV_DIR = D:\Downloads\Depressed\final_pub_outputs_fullcover\figure_csv
FIG_DIR = D:\Downloads\Depressed\final_pub_outputs_fullcover\figures
[Saved CSV] D:\Downloads\Depressed\final_pub_outputs_fullcover\figure_csv\FigR1_failure_interaction_terms.csv
[Saved CSV] D:\Downloads\Depressed\final_pub_outputs_fullcover\figure_csv\FigR2_failure_interaction_surface_long.csv
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigR_failure_interaction_surface_R1.png
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigR_failure_interaction_surface_R1.pdf
[Saved] D:\Downloads\Depressed\final_pub_outputs_fullcover\figures\FigR_failure_interaction_surface_R1.eps
[Saved] D:\Downloads\D

In [ ]:
# -*- coding: utf-8 -*-
"""
Redraw Fig3/Fig4 panels 
"""

import argparse
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, FormatStrFormatter

warnings.filterwarnings("ignore")

DEFAULT_ROOT = r"D:\Downloads\Depressed"
DPI = 800

PALETTE = {
    "blue": "#98CFE6",
    "deepblue": "#6EA9C4",
    "green": "#ADE7A8",
    "orange": "#F39F4E",
    "deeporange": "#D88537",
    "pink": "#EEB7D3",
    "gray": "#DBDAD3",
    "linegray": "#CFCBC5",
    "red": "#D95F5F",
    "purple": "#B39DDB",
    "black": "#111111",
    "white": "#FFFFFF",
}

CMAP_BLUE = LinearSegmentedColormap.from_list(
    "stageperf",
    [PALETTE["white"], "#D9EEF8", PALETTE["blue"]]
)

# General figure typography
TICK_FS = 28
YTICK_FS = 28
LABEL_FS = 38
TITLE_FS = 32
LEGEND_FS = 24

# Matrix-specific typography
MATRIX_XTICK_FS = 27
MATRIX_YTICK_FS = 30
MATRIX_CELL_FS = 31
MATRIX_LABEL_FS = 36
MATRIX_CBAR_FS = 24

# Long label figures
LONG_YTICK_FS = 28
LONG_Y_ROTATION = 12


def set_pub_style():
    mpl.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "Times New Roman",
        "font.serif": ["Times New Roman"],
        "font.weight": "bold",
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "axes.linewidth": 3.2,
        "xtick.major.width": 3.2,
        "ytick.major.width": 3.2,
        "xtick.major.size": 10,
        "ytick.major.size": 10,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.unicode_minus": False,
    })


def make_all_artists_opaque(fig):
    fig.patch.set_facecolor("white")
    fig.patch.set_alpha(1.0)

    for ax in fig.axes:
        ax.set_facecolor("white")
        ax.patch.set_alpha(1.0)
        for artist in ax.get_children():
            try:
                artist.set_alpha(1.0)
            except Exception:
                pass


def force_axis_text(ax, x_tick_size=TICK_FS, y_tick_size=YTICK_FS,
                    label_size=LABEL_FS, title_size=TITLE_FS):
    for txt in [ax.xaxis.label, ax.yaxis.label, ax.title]:
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")

    ax.xaxis.label.set_fontsize(label_size)
    ax.yaxis.label.set_fontsize(label_size)
    ax.title.set_fontsize(title_size)

    for lab in ax.get_xticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(x_tick_size)

    for lab in ax.get_yticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(y_tick_size)

    for txt in ax.texts:
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")


def style_ax(ax, x_tick_size=TICK_FS, y_tick_size=YTICK_FS,
             label_size=LABEL_FS, title_size=TITLE_FS):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(3.2)
    ax.spines["bottom"].set_linewidth(3.2)

    ax.tick_params(
        axis="both",
        which="major",
        length=10,
        width=3.2,
        colors=PALETTE["black"],
        pad=8
    )

    ax.grid(False)

    force_axis_text(
        ax,
        x_tick_size=x_tick_size,
        y_tick_size=y_tick_size,
        label_size=label_size,
        title_size=title_size
    )


def rotate_ylabels(ax, rotation=LONG_Y_ROTATION, fontsize=LONG_YTICK_FS):
    for lab in ax.get_yticklabels():
        lab.set_rotation(rotation)
        lab.set_ha("right")
        lab.set_va("center")
        lab.set_rotation_mode("anchor")
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)


def rotate_xlabels(ax, rotation=25, fontsize=TICK_FS):
    for lab in ax.get_xticklabels():
        lab.set_rotation(rotation)
        lab.set_ha("right")
        lab.set_va("top")
        lab.set_rotation_mode("anchor")
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)


def style_legend(leg, fontsize=LEGEND_FS):
    if leg is None:
        return

    frame = leg.get_frame()
    if frame is not None:
        frame.set_alpha(0.0)
        frame.set_facecolor("white")
        frame.set_edgecolor("white")

    for txt in leg.get_texts():
        txt.set_fontname("Times New Roman")
        txt.set_fontweight("bold")
        txt.set_fontsize(fontsize)


def add_legend_best(ax, handles=None, fontsize=LEGEND_FS, ncol=1):
    if handles is None:
        leg = ax.legend(frameon=False, loc="best", ncol=ncol)
    else:
        handles = [h for h in handles if h is not None]
        if len(handles) == 0:
            return
        leg = ax.legend(handles=handles, frameon=False, loc="best", ncol=ncol)

    style_legend(leg, fontsize=fontsize)


def compact_numeric_xaxis(ax, max_ticks=5, fontsize=TICK_FS, fmt="%.2f"):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=max_ticks))
    ax.xaxis.set_major_formatter(FormatStrFormatter(fmt))
    ax.tick_params(axis="x", labelsize=fontsize, pad=10)

    for lab in ax.get_xticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(fontsize)
        lab.set_rotation(0)
        lab.set_ha("center")


def save_fig(fig, fig_dir: Path, stem: str):
    fig_dir.mkdir(parents=True, exist_ok=True)
    make_all_artists_opaque(fig)

    for ext, fmt in [("png", None), ("pdf", None), ("eps", "eps")]:
        fp = fig_dir / f"{stem}.{ext}"
        kwargs = dict(
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.22,
            facecolor="white",
            edgecolor="white",
            transparent=False,
        )
        if fmt is not None:
            kwargs["format"] = fmt
        fig.savefig(fp, **kwargs)
        print("[Saved]", fp)

    plt.close(fig)


def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print("[Missing CSV]", path)
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("[Read failed]", path, e)
        return pd.DataFrame()


def pretty_stage(s):
    mp = {
        "native_class255": "native\nclass255",
        "direct_class255": "direct\nclass255",
        "from2to255": "2→255",
        "from4to255": "4→255",
    }
    return mp.get(str(s), str(s).replace("_", "\n"))


def pretty_cue(s):
    mp = {
        "suicide_explicit": "suicide\nexplicit",
        "hopelessness_explicit": "hopelessness\nexplicit",
        "worthlessness_explicit": "worthlessness\nexplicit",
        "sadness_explicit": "sadness\nexplicit",
        "cognitive_explicit": "cognitive\nexplicit",
        "high_first_person": "high first-\nperson",
        "high_negation": "high\nnegation",
    }
    return mp.get(str(s), str(s).replace("_", "\n"))


def resolve_output_dirs(root: str):
    root = Path(root)

    candidates = [
        root / "final_all_in_one_outputs_v2",
        root / "Depressed" / "final_all_in_one_outputs_v2",
    ]

    for out_dir in candidates:
        if (out_dir / "figure_csv").exists():
            fig_dir = out_dir / "figures"
            csv_dir = out_dir / "figure_csv"
            fig_dir.mkdir(parents=True, exist_ok=True)
            return out_dir, fig_dir, csv_dir

    # fallback to first candidate
    out_dir = candidates[0]
    fig_dir = out_dir / "figures"
    csv_dir = out_dir / "figure_csv"
    fig_dir.mkdir(parents=True, exist_ok=True)
    csv_dir.mkdir(parents=True, exist_ok=True)
    return out_dir, fig_dir, csv_dir


def redraw_fig3b(csv_dir: Path, fig_dir: Path):
    df = safe_read_csv(csv_dir / "Fig3B_stage_performance_matrix.csv")
    if df.empty:
        return

    if "metric" not in df.columns:
        first = df.columns[0]
        df = df.rename(columns={first: "metric"})

    mat_df = df.set_index("metric")
    stage_cols = list(mat_df.columns)
    mat = mat_df.values.astype(float)

    valid = mat[np.isfinite(mat)]
    vmin = float(np.nanmin(valid)) if len(valid) > 0 else 0.0
    vmax = float(np.nanmax(valid)) if len(valid) > 0 else 1.0

    fig, ax = plt.subplots(figsize=(8.8, 5.8))
    im = ax.imshow(
        mat,
        cmap=CMAP_BLUE,
        aspect="auto",
        vmin=vmin,
        vmax=vmax
    )

    ax.set_xticks(np.arange(len(stage_cols)))
    ax.set_xticklabels([pretty_stage(s) for s in stage_cols])
    ax.set_yticks(np.arange(len(mat_df.index)))
    ax.set_yticklabels(mat_df.index.tolist())

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if np.isfinite(mat[i, j]):
                ax.text(
                    j,
                    i,
                    f"{mat[i, j]:.3f}",
                    ha="center",
                    va="center",
                    fontsize=MATRIX_CELL_FS,
                    fontweight="bold",
                    fontname="Times New Roman",
                    color=PALETTE["black"],
                )

    ax.set_xticks(np.arange(-0.5, len(stage_cols), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(mat_df.index), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=2.4)
    ax.tick_params(which="minor", bottom=False, left=False)

    style_ax(
        ax,
        x_tick_size=MATRIX_XTICK_FS,
        y_tick_size=MATRIX_YTICK_FS,
        label_size=MATRIX_LABEL_FS,
        title_size=TITLE_FS
    )

    rotate_xlabels(ax, rotation=20, fontsize=MATRIX_XTICK_FS)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.035)
    cbar.ax.tick_params(labelsize=24, width=2.2, length=6)
    for lab in cbar.ax.get_yticklabels():
        lab.set_fontname("Times New Roman")
        lab.set_fontweight("bold")
        lab.set_fontsize(24)

    fig.subplots_adjust(left=0.19, right=0.94, bottom=0.25, top=0.96)

    save_fig(fig, fig_dir, "Fig3B_stage_performance_matrix")


# =========================================================
# 3. Fig3D
# =========================================================
def redraw_fig3d(csv_dir: Path, fig_dir: Path):
    df = safe_read_csv(csv_dir / "Fig3D_stage_scaling.csv")
    if df.empty:
        return

    required = {"stage_type", "model_size_b", "macro_f1"}
    if not required.issubset(df.columns):
        print("[Skip Fig3D] Missing columns:", required - set(df.columns))
        return

    df["model_size_b"] = pd.to_numeric(df["model_size_b"], errors="coerce")
    df["macro_f1"] = pd.to_numeric(df["macro_f1"], errors="coerce")
    df = df.dropna(subset=["model_size_b", "macro_f1"]).copy()

    fig, ax = plt.subplots(figsize=(8.6, 6.2))

    specs = [
        ("direct_class255", "direct class255", PALETTE["gray"]),
        ("from2to255", "2→255", PALETTE["orange"]),
        ("from4to255", "4→255", PALETTE["pink"]),
        ("native_class255", "native class255", PALETTE["blue"]),
    ]

    for st, label, color in specs:
        dd = df[df["stage_type"] == st].copy()
        if len(dd) == 0:
            continue
        agg = (
            dd.groupby("model_size_b")
            .agg(mean_macro_f1=("macro_f1", "mean"))
            .reset_index()
            .sort_values("model_size_b")
        )
        if len(agg) == 0:
            continue
        ax.plot(
            agg["model_size_b"],
            agg["mean_macro_f1"],
            marker="o",
            markersize=11,
            linewidth=3.4,
            color=color,
            label=label
        )

    ax.set_xlabel("Model size (B)")
    ax.set_ylabel("Macro-F1")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))

    style_ax(ax)
    compact_numeric_xaxis(ax, max_ticks=5, fontsize=TICK_FS, fmt="%.1f")
    add_legend_best(ax, fontsize=LEGEND_FS)

    fig.subplots_adjust(left=0.18, right=0.96, bottom=0.20, top=0.95)
    save_fig(fig, fig_dir, "Fig3D_stage_scaling")

def redraw_cue_dependence(
    csv_dir: Path,
    fig_dir: Path,
    csv_name: str,
    out_name: str,
    value_col: str,
    xlabel: str,
    color: str,
):
    df = safe_read_csv(csv_dir / f"{csv_name}.csv")
    if df.empty:
        return

    required = {"cue", "group", value_col}
    if not required.issubset(df.columns):
        print(f"[Skip {out_name}] Missing columns:", required - set(df.columns))
        return

    selected_cues = [
        "suicide_explicit",
        "hopelessness_explicit",
        "worthlessness_explicit",
        "high_first_person",
        "high_negation",
    ]

    tmp = df[df["cue"].isin(selected_cues)].copy()
    if len(tmp) == 0:
        tmp = df.copy()

    pivot = tmp.pivot_table(
        index="cue",
        columns="group",
        values=value_col,
        aggfunc="mean"
    ).reset_index()

    if "with_cue" not in pivot.columns or "without_cue" not in pivot.columns:
        print(f"[Skip {out_name}] with_cue/without_cue columns unavailable after pivot.")
        return

    pivot["delta"] = pivot["with_cue"] - pivot["without_cue"]
    pivot = pivot.sort_values("delta", ascending=True).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(9.8, 6.8))

    y = np.arange(len(pivot))
    ax.barh(
        y,
        pivot["delta"],
        color=color,
        edgecolor="none",
        height=0.66
    )

    ax.axvline(0, color=PALETTE["linegray"], linewidth=2.2)

    ax.set_yticks(y)
    ax.set_yticklabels([pretty_cue(x) for x in pivot["cue"].tolist()])
    ax.set_xlabel(xlabel)

    style_ax(
        ax,
        x_tick_size=TICK_FS,
        y_tick_size=LONG_YTICK_FS,
        label_size=LABEL_FS,
        title_size=TITLE_FS
    )
    rotate_ylabels(ax, rotation=LONG_Y_ROTATION, fontsize=LONG_YTICK_FS)
    compact_numeric_xaxis(ax, max_ticks=5, fontsize=TICK_FS, fmt="%.2f")

    fig.subplots_adjust(left=0.36, right=0.96, bottom=0.22, top=0.95)
    save_fig(fig, fig_dir, out_name)


def redraw_fig4a(csv_dir: Path, fig_dir: Path):
    redraw_cue_dependence(
        csv_dir=csv_dir,
        fig_dir=fig_dir,
        csv_name="Fig4A_fourclass_cue_dependence",
        out_name="Fig4A_fourclass_cue_dependence",
        value_col="error_rate",
        xlabel="Δ fourclass error rate\n(with cue − without)",
        color=PALETTE["orange"],
    )


def redraw_fig4b(csv_dir: Path, fig_dir: Path):
    redraw_cue_dependence(
        csv_dir=csv_dir,
        fig_dir=fig_dir,
        csv_name="Fig4B_class255_cue_dependence",
        out_name="Fig4B_class255_cue_dependence",
        value_col="suicide_fn_rate",
        xlabel="Δ suicide false-negative rate\n(with cue − without)",
        color=PALETTE["red"],
    )

def redraw_fig4c(csv_dir: Path, fig_dir: Path):
    df = safe_read_csv(csv_dir / "Fig4C_fourclass_run_stability.csv")
    if df.empty:
        return

    required = {"model_size_b", "adjacent_error_frac"}
    if not required.issubset(df.columns):
        print("[Skip Fig4C] Missing columns:", required - set(df.columns))
        return

    df["model_size_b"] = pd.to_numeric(df["model_size_b"], errors="coerce")
    df["adjacent_error_frac"] = pd.to_numeric(df["adjacent_error_frac"], errors="coerce")
    df = df.dropna(subset=["model_size_b", "adjacent_error_frac"]).copy()

    agg = (
        df.groupby("model_size_b")
        .agg(
            mean_adj=("adjacent_error_frac", "mean"),
            min_adj=("adjacent_error_frac", "min"),
            max_adj=("adjacent_error_frac", "max"),
        )
        .reset_index()
        .sort_values("model_size_b")
    )

    if len(agg) == 0:
        return

    fig, ax = plt.subplots(figsize=(8.8, 6.2))

    for _, r in agg.iterrows():
        ax.plot(
            [r["min_adj"], r["max_adj"]],
            [r["model_size_b"], r["model_size_b"]],
            color=PALETTE["orange"],
            linewidth=4.0,
            solid_capstyle="round"
        )
        ax.scatter(
            r["mean_adj"],
            r["model_size_b"],
            s=230,
            color=PALETTE["orange"],
            edgecolor="white",
            linewidth=1.1,
            zorder=4
        )

    ax.set_xlabel("Adjacent-error fraction")
    ax.set_ylabel("Model size (B)")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    style_ax(ax)
    compact_numeric_xaxis(ax, max_ticks=5, fontsize=TICK_FS, fmt="%.2f")

    handles = [
        Line2D([0], [0], color=PALETTE["orange"], lw=4.0, label="Range across runs"),
        Line2D([0], [0], color=PALETTE["orange"], marker="o", linestyle="None",
               markersize=12, label="Mean"),
    ]
    add_legend_best(ax, handles=handles, fontsize=LEGEND_FS)

    fig.subplots_adjust(left=0.18, right=0.96, bottom=0.20, top=0.95)
    save_fig(fig, fig_dir, "Fig4C_fourclass_run_stability")

def redraw_fig4d(csv_dir: Path, fig_dir: Path):
    df = safe_read_csv(csv_dir / "Fig4D_class255_run_stability.csv")
    if df.empty:
        return

    required = {"stage_type", "model_size_b", "mean_core_shift"}
    if not required.issubset(df.columns):
        print("[Skip Fig4D] Missing columns:", required - set(df.columns))
        return

    df["model_size_b"] = pd.to_numeric(df["model_size_b"], errors="coerce")
    df["mean_core_shift"] = pd.to_numeric(df["mean_core_shift"], errors="coerce")
    df = df.dropna(subset=["model_size_b", "mean_core_shift"]).copy()

    fig, ax = plt.subplots(figsize=(8.8, 6.2))

    specs = [
        ("direct_class255", "direct class255", PALETTE["gray"]),
        ("from2to255", "2→255", PALETTE["orange"]),
        ("from4to255", "4→255", PALETTE["pink"]),
        ("native_class255", "native class255", PALETTE["blue"]),
    ]

    for st, label, color in specs:
        dd = df[df["stage_type"] == st].copy()
        if len(dd) == 0:
            continue
        ax.scatter(
            dd["mean_core_shift"],
            dd["model_size_b"],
            s=230,
            color=color,
            edgecolor="white",
            linewidth=1.1,
            label=label,
            zorder=4
        )

    ax.axvline(0, color=PALETTE["linegray"], linewidth=2.2)

    ax.set_xlabel("Mean core-shift")
    ax.set_ylabel("Model size (B)")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

    style_ax(ax)
    compact_numeric_xaxis(ax, max_ticks=5, fontsize=TICK_FS, fmt="%.2f")
    add_legend_best(ax, fontsize=LEGEND_FS)

    fig.subplots_adjust(left=0.18, right=0.96, bottom=0.20, top=0.95)
    save_fig(fig, fig_dir, "Fig4D_class255_run_stability")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--root",
        default=DEFAULT_ROOT,
        help=r"Project root, e.g. D:\Downloads\Depressed"
    )

    args, unknown = parser.parse_known_args()
    if len(unknown) > 0:
        print("[Info] Ignored unknown runtime arguments:")
        for u in unknown:
            print("  ", u)

    set_pub_style()

    out_dir, fig_dir, csv_dir = resolve_output_dirs(args.root)

    print("=" * 100)
    print("Redraw selected Fig3/Fig4 panels")
    print("OUT_DIR =", out_dir)
    print("CSV_DIR =", csv_dir)
    print("FIG_DIR =", fig_dir)
    print("=" * 100)

    redraw_fig3b(csv_dir, fig_dir)
    redraw_fig3d(csv_dir, fig_dir)
    redraw_fig4a(csv_dir, fig_dir)
    redraw_fig4b(csv_dir, fig_dir)
    redraw_fig4c(csv_dir, fig_dir)
    redraw_fig4d(csv_dir, fig_dir)

    print("\nDone.")
    print("Figures saved to:", fig_dir)


if __name__ == "__main__":
    main()

[Info] Ignored unknown runtime arguments:
   --f=c:\Users\mhzho\AppData\Roaming\jupyter\runtime\kernel-v3f84a6b90eb6cfb7d2aecef638096c6c2c0dc4f08.json
Redraw selected Fig3/Fig4 panels
OUT_DIR = D:\Downloads\Depressed\final_all_in_one_outputs_v2
CSV_DIR = D:\Downloads\Depressed\final_all_in_one_outputs_v2\figure_csv
FIG_DIR = D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures
[Saved] D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures\Fig3B_stage_performance_matrix.png
[Saved] D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures\Fig3B_stage_performance_matrix.pdf
[Saved] D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures\Fig3B_stage_performance_matrix.eps
[Saved] D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures\Fig3D_stage_scaling.png
[Saved] D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures\Fig3D_stage_scaling.pdf
[Saved] D:\Downloads\Depressed\final_all_in_one_outputs_v2\figures\Fig3D_stage_scaling.eps
[Saved] D:\Downloads\Depressed\fina